In [1]:
# ======================================================================================
# Cell 0 — Setup and locked model contract
# ======================================================================================
#
# New notebook:
#   07_unified_fds_core_signal_threshold_research_v1.ipynb
#
# Purpose:
#   Start clean threshold/signal research using the locked unified denominator:
#
#       unified_fds_no_min_return
#
# Locked model contract:
#   - One exact model spec across all tenors.
#   - Tenors: 9, 12, 15, 18, 21, 24, 27, 30, 33.
#   - No blending.
#   - No threshold grid inside the denominator.
#   - No separate Front/Middle/Back forecast model.
#   - No candidate_min_return_* features.
#
# First notebook stage:
#   Cells 0–3 only build and validate the clean signal base panel.
#
# Not in scope yet:
#   - No Core threshold sweep.
#   - No Secondary.
#   - No production lock.
#   - No sizing changes.
#
# Important evaluation convention:
#   Because tenors have different DTEs, threshold research should evaluate trade
#   quality using average P&L per day:
#
#       normalized_pnl_per_day = normalized_pnl_dollars / tenor
#
#   Total P&L is still retained for portfolio-level diagnostics, but per-day P&L
#   will be the primary cross-tenor quality metric.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 320)

# --------------------------------------------------------------------------------------
# Project paths
# --------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

SOURCE_BRANCH = "vrp_front_middle_corsi_forecast_repair_v1"
NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

SOURCE_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / SOURCE_BRANCH

OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
OUT_CHART_DIR = OUT_AUDIT_DIR / "charts"

OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CHART_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# --------------------------------------------------------------------------------------
# Locked model / tenor contract
# --------------------------------------------------------------------------------------

LOCKED_MODEL_SPEC = "unified_fds_no_min_return"
EXISTING_REFERENCE_SPEC = "existing_har_downside_v1"

ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]
FRONT_TENORS = [9, 12, 15]
MIDDLE_TENORS = [18, 21, 24]
BACK_TENORS = [27, 30, 33]

TENOR_BUCKET_MAP = {
    9: "Front",
    12: "Front",
    15: "Front",
    18: "Middle",
    21: "Middle",
    24: "Middle",
    27: "Back",
    30: "Back",
    33: "Back",
}

BUCKET_ORDER_MAP = {
    "Front": 1,
    "Middle": 2,
    "Back": 3,
}

# Locked Back Core anchor. Front/Middle research comes later.
LOCKED_BACK_CORE_THRESHOLDS = {
    "model_vrp_log": 0.70,
    "model_vrp_z_3m": 0.70,
    "model_vrp_z_1y": 0.70,
    "RSI14_max": 70.0,
    "forecast_vol_pct_min": 8.5,
}

# Legacy Core reference only. Not locked for Front/Middle.
LEGACY_CORE_REFERENCE_THRESHOLDS = {
    "Front": {
        "model_vrp_log": 0.60,
        "model_vrp_z_3m": 0.55,
        "model_vrp_z_1y": 0.65,
        "RSI14_max": 70.0,
        "forecast_vol_pct_min": 8.5,
    },
    "Middle": {
        "model_vrp_log": 0.65,
        "model_vrp_z_3m": 0.75,
        "model_vrp_z_1y": 0.65,
        "RSI14_max": 68.0,
        "forecast_vol_pct_min": 8.5,
    },
    "Back": LOCKED_BACK_CORE_THRESHOLDS,
}

TARGET_VAR_COL = "target_realized_variance"
TARGET_LOG_COL = "target_log_variance"

ZSCORE_WINDOWS = {
    "model_vrp_z_3m": {
        "window": 63,
        "min_periods": 63,
    },
    "model_vrp_z_1y": {
        "window": 252,
        "min_periods": 252,
    },
}

EPS = 1e-12

print("=" * 100)
print("Cell 0 — Setup and locked model contract")
print("=" * 100)
print(f"Source branch:      {SOURCE_BRANCH}")
print(f"New branch:         {NEW_BRANCH}")
print(f"Locked model spec:  {LOCKED_MODEL_SPEC}")
print(f"All tenors:         {ALL_TENORS}")
print(f"Output processed:   {OUT_PROCESSED_DIR}")
print(f"Output audit:       {OUT_AUDIT_DIR}")
print()
print("Primary cross-tenor evaluation metric:")
print("  normalized_pnl_per_day = normalized_pnl_dollars / tenor")
print()
print("CELL 0 COMPLETE")


Cell 0 — Setup and locked model contract
Source branch:      vrp_front_middle_corsi_forecast_repair_v1
New branch:         vrp_unified_fds_core_signal_threshold_research_v1
Locked model spec:  unified_fds_no_min_return
All tenors:         [9, 12, 15, 18, 21, 24, 27, 30, 33]
Output processed:   C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1
Output audit:       C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1

Primary cross-tenor evaluation metric:
  normalized_pnl_per_day = normalized_pnl_dollars / tenor

CELL 0 COMPLETE


In [2]:
# ======================================================================================
# Cell 1 — Load locked unified denominator panel
# ======================================================================================
#
# Input:
#   Latest Cell 7A unified denominator output:
#
#       data\processed\vrp_front_middle_corsi_forecast_repair_v1\
#       07A_unified_fds_no_min_return_oos_forecast_panel_*.parquet
#
# Scope:
#   - Load the locked model forecast panel.
#   - Keep the locked model rows for signal construction.
#   - Keep existing_har_downside_v1 only as optional reference, not for signal use.
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


source_panel_path = latest_file(
    SOURCE_PROCESSED_DIR,
    "07A_unified_fds_no_min_return_oos_forecast_panel_*.parquet",
    required=True,
)

raw_panel = pd.read_parquet(source_panel_path)

duplicate_cols = raw_panel.columns[raw_panel.columns.duplicated()].tolist()
if duplicate_cols:
    print("Dropping duplicate column labels:")
    print(sorted(set(duplicate_cols)))

raw_panel = raw_panel.loc[:, ~raw_panel.columns.duplicated(keep="last")].copy()

# Normalize core date / tenor fields.
raw_panel["trade_date"] = pd.to_datetime(raw_panel["trade_date"], errors="coerce").dt.normalize()
raw_panel["tenor"] = pd.to_numeric(raw_panel["tenor"], errors="coerce").astype("Int64")

if "last_forward_rv_date" in raw_panel.columns:
    raw_panel["last_forward_rv_date"] = pd.to_datetime(
        raw_panel["last_forward_rv_date"],
        errors="coerce",
    ).dt.normalize()
else:
    raise KeyError("Expected last_forward_rv_date in locked model panel.")

required_cols = [
    "trade_date",
    "tenor",
    "model_spec",
    "last_forward_rv_date",
    "implied_variance",
    "forecast_variance_candidate",
    "candidate_forecast_vol_pct",
    TARGET_VAR_COL,
    TARGET_LOG_COL,
    "normalized_pnl_dollars",
    "win",
    "RSI14",
]

missing_cols = [c for c in required_cols if c not in raw_panel.columns]
if missing_cols:
    raise KeyError(f"Missing required columns in locked model panel: {missing_cols}")

for c in [
    "implied_variance",
    "forecast_variance_candidate",
    "candidate_forecast_vol_pct",
    TARGET_VAR_COL,
    TARGET_LOG_COL,
    "normalized_pnl_dollars",
    "win",
    "RSI14",
]:
    raw_panel[c] = pd.to_numeric(raw_panel[c], errors="coerce")

raw_panel = raw_panel[
    raw_panel["trade_date"].notna()
    & raw_panel["tenor"].isin(ALL_TENORS)
    & raw_panel["model_spec"].isin([LOCKED_MODEL_SPEC, EXISTING_REFERENCE_SPEC])
].copy()

locked_rows = raw_panel[raw_panel["model_spec"] == LOCKED_MODEL_SPEC].copy()
reference_rows = raw_panel[raw_panel["model_spec"] == EXISTING_REFERENCE_SPEC].copy()

if locked_rows.empty:
    raise RuntimeError(f"No rows found for locked model spec: {LOCKED_MODEL_SPEC}")

date_min = locked_rows["trade_date"].min()
date_max = locked_rows["trade_date"].max()

print("=" * 100)
print("Cell 1 — Loaded locked unified denominator panel")
print("=" * 100)
print(f"Source panel:            {source_panel_path}")
print(f"Raw rows loaded:          {len(raw_panel):,}")
print(f"Locked model rows:        {len(locked_rows):,}")
print(f"Reference model rows:     {len(reference_rows):,}")
print(f"Locked date range:        {date_min.date()} to {date_max.date()}")
print(f"Tenors found:             {sorted(locked_rows['tenor'].dropna().astype(int).unique().tolist())}")
print(f"Models found:             {sorted(raw_panel['model_spec'].dropna().unique().tolist())}")
print()
print("Rows by model:")
display(raw_panel["model_spec"].value_counts(dropna=False).to_frame("rows"))
print()
print("Rows by tenor for locked model:")
display(
    locked_rows
    .groupby("tenor", as_index=False)
    .agg(
        rows=("trade_date", "size"),
        date_min=("trade_date", "min"),
        date_max=("trade_date", "max"),
    )
)

print("\nCELL 1 COMPLETE")

Cell 1 — Loaded locked unified denominator panel
Source panel:            C:\Users\patri\vrp_project\data\processed\vrp_front_middle_corsi_forecast_repair_v1\07A_unified_fds_no_min_return_oos_forecast_panel_20200102_20260701_20260704_203242.parquet
Raw rows loaded:          31,140
Locked model rows:        14,688
Reference model rows:     16,452
Locked date range:        2020-01-02 to 2026-07-01
Tenors found:             [9, 12, 15, 18, 21, 24, 27, 30, 33]
Models found:             ['existing_har_downside_v1', 'unified_fds_no_min_return']

Rows by model:


,rows
model_spec,
existing_har_downside_v1,16452
unified_fds_no_min_return,14688



Rows by tenor for locked model:


,tenor,rows,date_min,date_max
0,9,1632,2020-01-02,2026-07-01
1,12,1632,2020-01-02,2026-07-01
2,15,1632,2020-01-02,2026-07-01
3,18,1632,2020-01-02,2026-07-01
4,21,1632,2020-01-02,2026-07-01
5,24,1632,2020-01-02,2026-07-01
6,27,1632,2020-01-02,2026-07-01
7,30,1632,2020-01-02,2026-07-01
8,33,1632,2020-01-02,2026-07-01



CELL 1 COMPLETE


In [3]:
# ======================================================================================
# Cell 2 — Build clean unified signal base panel
# ======================================================================================
#
# Objective:
#   Build one clean signal row per trade_date × tenor using only:
#
#       unified_fds_no_min_return
#
# Key fields created:
#   - implied_vol_pct
#   - forecast_vol_pct
#   - forward_realized_vol_pct_raw
#   - forward_realized_complete
#   - forward_realized_vol_pct_completed
#   - model_vrp_log
#   - model_vrp_z_3m, prior-only by tenor
#   - model_vrp_z_1y, prior-only by tenor
#   - normalized_pnl_per_day = normalized_pnl_dollars / tenor
#
# Important:
#   - Forward realized target may be incomplete near the end of the sample.
#   - We keep raw target fields for historical research.
#   - We flag completed forward rows for visual / realized-vol diagnostics.
#   - Threshold research can still evaluate only historical completed outcomes.
# ======================================================================================

def annualized_vol_pct_from_variance(x: pd.Series) -> pd.Series:
    return np.sqrt(pd.to_numeric(x, errors="coerce").clip(lower=0)) * 100.0


def compute_prior_z_by_tenor(
    df: pd.DataFrame,
    value_col: str,
    window: int,
    min_periods: int,
) -> pd.Series:
    """
    Prior-only rolling z-score by tenor.

    For each row:
        z_t = (x_t - mean(x_{t-window} ... x_{t-1})) / std(x_{t-window} ... x_{t-1})

    This intentionally excludes the current row from the rolling mean/std.
    """
    out = pd.Series(np.nan, index=df.index, dtype="float64")

    for tenor, g in df.sort_values(["tenor", "trade_date"]).groupby("tenor"):
        shifted = g[value_col].shift(1)
        roll_mean = shifted.rolling(window=window, min_periods=min_periods).mean()
        roll_std = shifted.rolling(window=window, min_periods=min_periods).std(ddof=1)
        z = (g[value_col] - roll_mean) / roll_std.replace(0.0, np.nan)
        out.loc[g.index] = z

    return out


# Start with locked model only.
signal = locked_rows.copy()

signal = signal.sort_values(["trade_date", "tenor"]).reset_index(drop=True)

# Defensive uniqueness check before building one-row-per-date-tenor panel.
dup_keys = signal.duplicated(["trade_date", "tenor"], keep=False)
if dup_keys.any():
    duplicate_sample = signal.loc[dup_keys, ["trade_date", "tenor", "model_spec"]].head(20)
    display(duplicate_sample)
    raise RuntimeError("Locked model panel has duplicate trade_date × tenor rows.")

signal["tenor"] = signal["tenor"].astype(int)
signal["tenor_bucket"] = signal["tenor"].map(TENOR_BUCKET_MAP)
signal["tenor_bucket_order"] = signal["tenor_bucket"].map(BUCKET_ORDER_MAP)

# Core vol / variance fields.
signal["implied_variance"] = pd.to_numeric(signal["implied_variance"], errors="coerce")
signal["forecast_variance"] = pd.to_numeric(signal["forecast_variance_candidate"], errors="coerce")
signal["target_realized_variance"] = pd.to_numeric(signal[TARGET_VAR_COL], errors="coerce")

signal["implied_vol_pct"] = annualized_vol_pct_from_variance(signal["implied_variance"])
signal["forecast_vol_pct"] = annualized_vol_pct_from_variance(signal["forecast_variance"])
signal["forward_realized_vol_pct_raw"] = annualized_vol_pct_from_variance(signal["target_realized_variance"])

# Max available trade date from the locked panel.
max_available_trade_date = signal["trade_date"].max()

signal["forward_realized_complete"] = (
    signal["last_forward_rv_date"].notna()
    & (signal["last_forward_rv_date"] <= max_available_trade_date)
)

signal["target_realized_variance_completed"] = np.where(
    signal["forward_realized_complete"],
    signal["target_realized_variance"],
    np.nan,
)

signal["forward_realized_vol_pct_completed"] = np.where(
    signal["forward_realized_complete"],
    signal["forward_realized_vol_pct_raw"],
    np.nan,
)

# VRP fields.
signal["model_vrp_log"] = np.log(signal["implied_variance"] / signal["forecast_variance"].clip(lower=EPS))
signal["model_vrp_ratio"] = signal["implied_variance"] / signal["forecast_variance"].clip(lower=EPS)

signal["model_vrp_z_3m"] = compute_prior_z_by_tenor(
    signal,
    value_col="model_vrp_log",
    window=ZSCORE_WINDOWS["model_vrp_z_3m"]["window"],
    min_periods=ZSCORE_WINDOWS["model_vrp_z_3m"]["min_periods"],
)

signal["model_vrp_z_1y"] = compute_prior_z_by_tenor(
    signal,
    value_col="model_vrp_log",
    window=ZSCORE_WINDOWS["model_vrp_z_1y"]["window"],
    min_periods=ZSCORE_WINDOWS["model_vrp_z_1y"]["min_periods"],
)

# P&L normalization.
signal["normalized_pnl_dollars"] = pd.to_numeric(signal["normalized_pnl_dollars"], errors="coerce")
signal["normalized_pnl_per_day"] = signal["normalized_pnl_dollars"] / signal["tenor"].replace(0, np.nan)

# Retain useful trade outcome fields.
signal["win"] = pd.to_numeric(signal["win"], errors="coerce")
signal["RSI14"] = pd.to_numeric(signal["RSI14"], errors="coerce")

# Current diagnostic thresholds, used only as reference labels here.
signal["legacy_reference_model_vrp_log_threshold"] = signal["tenor_bucket"].map(
    {k: v["model_vrp_log"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
)
signal["legacy_reference_model_vrp_z_3m_threshold"] = signal["tenor_bucket"].map(
    {k: v["model_vrp_z_3m"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
)
signal["legacy_reference_model_vrp_z_1y_threshold"] = signal["tenor_bucket"].map(
    {k: v["model_vrp_z_1y"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
)
signal["legacy_reference_RSI14_max"] = signal["tenor_bucket"].map(
    {k: v["RSI14_max"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
)
signal["legacy_reference_forecast_vol_pct_min"] = signal["tenor_bucket"].map(
    {k: v["forecast_vol_pct_min"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
)

signal["legacy_reference_core_pass"] = (
    (signal["model_vrp_log"] > signal["legacy_reference_model_vrp_log_threshold"])
    & (signal["model_vrp_z_3m"] > signal["legacy_reference_model_vrp_z_3m_threshold"])
    & (signal["model_vrp_z_1y"] > signal["legacy_reference_model_vrp_z_1y_threshold"])
    & (signal["RSI14"] < signal["legacy_reference_RSI14_max"])
    & (signal["forecast_vol_pct"] > signal["legacy_reference_forecast_vol_pct_min"])
)

# Keep a clean, explicit column order where possible.
preferred_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "last_forward_rv_date",
    "forward_realized_complete",
    "implied_variance",
    "forecast_variance",
    "target_realized_variance",
    "target_realized_variance_completed",
    "implied_vol_pct",
    "forecast_vol_pct",
    "forward_realized_vol_pct_raw",
    "forward_realized_vol_pct_completed",
    "model_vrp_log",
    "model_vrp_ratio",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
    "legacy_reference_model_vrp_log_threshold",
    "legacy_reference_model_vrp_z_3m_threshold",
    "legacy_reference_model_vrp_z_1y_threshold",
    "legacy_reference_RSI14_max",
    "legacy_reference_forecast_vol_pct_min",
    "legacy_reference_core_pass",
]

other_cols = [c for c in signal.columns if c not in preferred_cols]
signal = signal[preferred_cols + other_cols].copy()

# Save base panel.
base_panel_path = OUT_PROCESSED_DIR / (
    f"01_unified_fds_signal_base_panel_{signal['trade_date'].min():%Y%m%d}_"
    f"{signal['trade_date'].max():%Y%m%d}_{TIMESTAMP}.parquet"
)

signal.to_parquet(base_panel_path, index=False)

# Save summary audit.
summary_by_tenor = (
    signal.groupby(["tenor", "tenor_bucket"], as_index=False)
    .agg(
        rows=("trade_date", "size"),
        date_min=("trade_date", "min"),
        date_max=("trade_date", "max"),
        completed_forward_rows=("forward_realized_complete", "sum"),
        avg_implied_vol_pct=("implied_vol_pct", "mean"),
        avg_forecast_vol_pct=("forecast_vol_pct", "mean"),
        avg_model_vrp_log=("model_vrp_log", "mean"),
        z_3m_available_rows=("model_vrp_z_3m", lambda x: int(x.notna().sum())),
        z_1y_available_rows=("model_vrp_z_1y", lambda x: int(x.notna().sum())),
        avg_pnl_dollars=("normalized_pnl_dollars", "mean"),
        avg_pnl_per_day=("normalized_pnl_per_day", "mean"),
        median_pnl_per_day=("normalized_pnl_per_day", "median"),
    )
)

summary_by_bucket = (
    signal.groupby(["tenor_bucket", "tenor_bucket_order"], as_index=False)
    .agg(
        rows=("trade_date", "size"),
        completed_forward_rows=("forward_realized_complete", "sum"),
        avg_implied_vol_pct=("implied_vol_pct", "mean"),
        avg_forecast_vol_pct=("forecast_vol_pct", "mean"),
        avg_model_vrp_log=("model_vrp_log", "mean"),
        z_3m_available_rows=("model_vrp_z_3m", lambda x: int(x.notna().sum())),
        z_1y_available_rows=("model_vrp_z_1y", lambda x: int(x.notna().sum())),
        avg_pnl_dollars=("normalized_pnl_dollars", "mean"),
        avg_pnl_per_day=("normalized_pnl_per_day", "mean"),
        median_pnl_per_day=("normalized_pnl_per_day", "median"),
    )
    .sort_values("tenor_bucket_order")
)

summary_by_tenor_path = OUT_AUDIT_DIR / f"01_signal_base_summary_by_tenor_{TIMESTAMP}.csv"
summary_by_bucket_path = OUT_AUDIT_DIR / f"01_signal_base_summary_by_bucket_{TIMESTAMP}.csv"

summary_by_tenor.to_csv(summary_by_tenor_path, index=False)
summary_by_bucket.to_csv(summary_by_bucket_path, index=False)

print("=" * 100)
print("Cell 2 — Built clean unified signal base panel")
print("=" * 100)
print(f"Rows:                         {len(signal):,}")
print(f"Date range:                   {signal['trade_date'].min().date()} to {signal['trade_date'].max().date()}")
print(f"Max available trade date:      {max_available_trade_date.date()}")
print(f"Completed forward rows:        {int(signal['forward_realized_complete'].sum()):,} / {len(signal):,}")
print(f"Suppressed incomplete rows:    {int((~signal['forward_realized_complete']).sum()):,}")
print(f"Saved base panel:              {base_panel_path}")
print()
print("Summary by bucket:")
display(summary_by_bucket)
print()
print("Summary by tenor:")
display(summary_by_tenor)

print("\nCELL 2 COMPLETE")


Cell 2 — Built clean unified signal base panel
Rows:                         14,688
Date range:                   2020-01-02 to 2026-07-01
Max available trade date:      2026-07-01
Completed forward rows:        14,559 / 14,688
Suppressed incomplete rows:    129
Saved base panel:              C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\01_unified_fds_signal_base_panel_20200102_20260701_20260704_211636.parquet

Summary by bucket:


,tenor_bucket,tenor_bucket_order,rows,completed_forward_rows,avg_implied_vol_pct,avg_forecast_vol_pct,avg_model_vrp_log,z_3m_available_rows,z_1y_available_rows,avg_pnl_dollars,avg_pnl_per_day,median_pnl_per_day
1,Front,1,4896,4871,20.073584,14.867199,0.586669,4707,4140,1646.357888,133.312864,656.582042
2,Middle,2,4896,4853,20.489803,14.963731,0.592387,4707,4140,2988.726930,142.243849,512.556385
0,Back,3,4896,4835,20.862671,15.024010,0.606769,4707,4140,4731.834259,157.596048,454.394391



Summary by tenor:


,tenor,tenor_bucket,rows,date_min,date_max,completed_forward_rows,avg_implied_vol_pct,avg_forecast_vol_pct,avg_model_vrp_log,z_3m_available_rows,z_1y_available_rows,avg_pnl_dollars,avg_pnl_per_day,median_pnl_per_day
0,9,Front,1632,2020-01-02,2026-07-01,1625,19.913988,14.892609,0.574855,1569,1380,962.582306,106.953590,728.498588
1,12,Front,1632,2020-01-02,2026-07-01,1624,20.084706,14.657151,0.619382,1569,1380,1669.246539,139.103878,639.436381
2,15,Front,1632,2020-01-02,2026-07-01,1622,20.222058,15.051838,0.565772,1569,1380,2309.333426,153.955562,615.759612
3,18,Middle,1632,2020-01-02,2026-07-01,1620,20.363649,14.867733,0.599534,1569,1380,2495.032221,138.612901,541.025351
4,21,Middle,1632,2020-01-02,2026-07-01,1618,20.491143,15.045946,0.580522,1569,1380,3098.741526,147.559120,525.171067
5,24,Middle,1632,2020-01-02,2026-07-01,1615,20.614617,14.977513,0.597104,1569,1380,3373.566002,140.565250,475.901402
6,27,Back,1632,2020-01-02,2026-07-01,1614,20.742690,14.968975,0.606214,1569,1380,4227.971187,156.591525,475.015857
7,30,Back,1632,2020-01-02,2026-07-01,1611,20.861959,15.085066,0.597346,1569,1380,4660.809980,155.360333,453.444428
8,33,Back,1632,2020-01-02,2026-07-01,1610,20.983364,15.017990,0.616746,1569,1380,5307.797506,160.842349,436.928415



CELL 2 COMPLETE


In [4]:
# ======================================================================================
# Cell 3 — Validate locked signal base panel
# ======================================================================================
#
# Objective:
#   Validate that the clean signal base panel is suitable for Core threshold research.
#
# Checks:
#   - one locked model only
#   - one row per trade_date × tenor
#   - all exact tenors present
#   - bucket mapping correct
#   - prior-only z-scores exist after warmup
#   - no impossible variance values
#   - normalized_pnl_per_day created correctly
#   - incomplete forward realized targets are flagged
#   - Back locked threshold reference present
#
# No threshold sweep in this cell.
# ======================================================================================

validation_rows = []

def add_validation(check: str, status: str, detail: str):
    validation_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })


# --------------------------------------------------------------------------------------
# Basic structure
# --------------------------------------------------------------------------------------

models_found = sorted(signal["model_spec"].dropna().unique().tolist())
tenors_found = sorted(signal["tenor"].dropna().astype(int).unique().tolist())
missing_tenors = sorted(set(ALL_TENORS) - set(tenors_found))
extra_tenors = sorted(set(tenors_found) - set(ALL_TENORS))

duplicate_key_count = int(signal.duplicated(["trade_date", "tenor"]).sum())

add_validation(
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    "one_row_per_trade_date_tenor",
    "PASS" if duplicate_key_count == 0 else "FAIL",
    f"duplicate_key_count={duplicate_key_count}",
)

add_validation(
    "all_exact_tenors_present",
    "PASS" if not missing_tenors and not extra_tenors else "FAIL",
    f"missing={missing_tenors}; extra={extra_tenors}",
)

bucket_bad = signal[
    signal["tenor_bucket"] != signal["tenor"].map(TENOR_BUCKET_MAP)
]

add_validation(
    "bucket_mapping",
    "PASS" if bucket_bad.empty else "FAIL",
    f"bad_rows={len(bucket_bad):,}",
)

# --------------------------------------------------------------------------------------
# Variance / vol sanity
# --------------------------------------------------------------------------------------

bad_variance_rows = signal[
    (~np.isfinite(signal["implied_variance"]))
    | (~np.isfinite(signal["forecast_variance"]))
    | (signal["implied_variance"] <= 0)
    | (signal["forecast_variance"] <= 0)
]

add_validation(
    "positive_implied_and_forecast_variance",
    "PASS" if bad_variance_rows.empty else "FAIL",
    f"bad_rows={len(bad_variance_rows):,}",
)

bad_vrp_rows = signal[
    ~np.isfinite(signal["model_vrp_log"])
]

add_validation(
    "model_vrp_log_finite",
    "PASS" if bad_vrp_rows.empty else "FAIL",
    f"bad_rows={len(bad_vrp_rows):,}",
)

# --------------------------------------------------------------------------------------
# Z-score availability and prior-only warmup
# --------------------------------------------------------------------------------------

z_summary = (
    signal.groupby("tenor", as_index=False)
    .agg(
        rows=("trade_date", "size"),
        z_3m_available_rows=("model_vrp_z_3m", lambda x: int(x.notna().sum())),
        z_1y_available_rows=("model_vrp_z_1y", lambda x: int(x.notna().sum())),
        first_date=("trade_date", "min"),
        first_z_3m_date=("trade_date", lambda x: signal.loc[x.index[signal.loc[x.index, "model_vrp_z_3m"].notna()], "trade_date"].min()),
        first_z_1y_date=("trade_date", lambda x: signal.loc[x.index[signal.loc[x.index, "model_vrp_z_1y"].notna()], "trade_date"].min()),
    )
)

z_3m_all_have_values = bool((z_summary["z_3m_available_rows"] > 0).all())
z_1y_all_have_values = bool((z_summary["z_1y_available_rows"] > 0).all())

add_validation(
    "z_3m_available_after_warmup",
    "PASS" if z_3m_all_have_values else "FAIL",
    z_summary[["tenor", "z_3m_available_rows", "first_z_3m_date"]].to_dict("records").__repr__(),
)

add_validation(
    "z_1y_available_after_warmup",
    "PASS" if z_1y_all_have_values else "FAIL",
    z_summary[["tenor", "z_1y_available_rows", "first_z_1y_date"]].to_dict("records").__repr__(),
)

# Spot-check prior-only behavior:
# For each tenor, first non-null 3m z should occur no earlier than row index 63 within that tenor.
prior_only_rows = []

for tenor, g in signal.sort_values(["tenor", "trade_date"]).groupby("tenor"):
    g = g.reset_index(drop=True).copy()
    first_z3_idx = g["model_vrp_z_3m"].first_valid_index()
    first_z1_idx = g["model_vrp_z_1y"].first_valid_index()

    prior_only_rows.append({
        "tenor": int(tenor),
        "first_z3_idx": first_z3_idx,
        "first_z1_idx": first_z1_idx,
        "z3_prior_only_ok": first_z3_idx is not None and first_z3_idx >= ZSCORE_WINDOWS["model_vrp_z_3m"]["min_periods"],
        "z1_prior_only_ok": first_z1_idx is not None and first_z1_idx >= ZSCORE_WINDOWS["model_vrp_z_1y"]["min_periods"],
    })

prior_only_check = pd.DataFrame(prior_only_rows)

add_validation(
    "zscore_prior_only_warmup_check",
    "PASS" if prior_only_check["z3_prior_only_ok"].all() and prior_only_check["z1_prior_only_ok"].all() else "FAIL",
    prior_only_check.to_dict("records").__repr__(),
)

# --------------------------------------------------------------------------------------
# P&L per day validation
# --------------------------------------------------------------------------------------

pnl_check = signal.copy()
pnl_check["expected_pnl_per_day"] = pnl_check["normalized_pnl_dollars"] / pnl_check["tenor"]
pnl_check["pnl_per_day_diff"] = pnl_check["normalized_pnl_per_day"] - pnl_check["expected_pnl_per_day"]

bad_pnl_per_day = pnl_check[
    pnl_check["normalized_pnl_dollars"].notna()
    & pnl_check["normalized_pnl_per_day"].notna()
    & (pnl_check["pnl_per_day_diff"].abs() > 1e-8)
]

add_validation(
    "normalized_pnl_per_day_created",
    "PASS" if bad_pnl_per_day.empty else "FAIL",
    f"bad_rows={len(bad_pnl_per_day):,}; formula=normalized_pnl_dollars / tenor",
)

pnl_per_day_available = int(signal["normalized_pnl_per_day"].notna().sum())

add_validation(
    "normalized_pnl_per_day_available",
    "PASS" if pnl_per_day_available > 0 else "FAIL",
    f"available_rows={pnl_per_day_available:,}",
)

# --------------------------------------------------------------------------------------
# Forward realized completion guard
# --------------------------------------------------------------------------------------

max_available_trade_date = signal["trade_date"].max()

completion_check_bad = signal[
    signal["last_forward_rv_date"].notna()
    & (
        signal["forward_realized_complete"]
        != (signal["last_forward_rv_date"] <= max_available_trade_date)
    )
]

add_validation(
    "forward_realized_completion_guard",
    "PASS" if completion_check_bad.empty else "FAIL",
    f"bad_rows={len(completion_check_bad):,}; max_available_trade_date={max_available_trade_date.date()}",
)

latest_rows = signal[signal["trade_date"] == max_available_trade_date].copy()
latest_completed_count = int(latest_rows["forward_realized_complete"].sum())

add_validation(
    "latest_forward_completion_count",
    "PASS",
    f"latest_date={max_available_trade_date.date()}; completed_forward_tenors={latest_completed_count}/9",
)

# --------------------------------------------------------------------------------------
# Back locked threshold reference
# --------------------------------------------------------------------------------------

back_ref = LEGACY_CORE_REFERENCE_THRESHOLDS["Back"]

back_reference_ok = (
    back_ref["model_vrp_log"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"]
    and back_ref["model_vrp_z_3m"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"]
    and back_ref["model_vrp_z_1y"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_1y"]
    and back_ref["RSI14_max"] == LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"]
    and back_ref["forecast_vol_pct_min"] == LOCKED_BACK_CORE_THRESHOLDS["forecast_vol_pct_min"]
)

add_validation(
    "back_locked_threshold_reference",
    "PASS" if back_reference_ok else "FAIL",
    f"Back reference={back_ref}",
)

# --------------------------------------------------------------------------------------
# Save validation / manifest
# --------------------------------------------------------------------------------------

validation = pd.DataFrame(validation_rows)

validation_path = OUT_AUDIT_DIR / f"01_signal_base_validation_{TIMESTAMP}.csv"
z_summary_path = OUT_AUDIT_DIR / f"01_signal_base_zscore_summary_{TIMESTAMP}.csv"
prior_only_check_path = OUT_AUDIT_DIR / f"01_signal_base_prior_only_zscore_check_{TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"01_signal_base_manifest_{TIMESTAMP}.json"

validation.to_csv(validation_path, index=False)
z_summary.to_csv(z_summary_path, index=False)
prior_only_check.to_csv(prior_only_check_path, index=False)

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1.ipynb",
    "cell_stage": "Cells 0-3 — clean locked signal base panel",
    "timestamp": TIMESTAMP,
    "source_panel": str(source_panel_path),
    "locked_model_spec": LOCKED_MODEL_SPEC,
    "all_tenors": ALL_TENORS,
    "new_branch": NEW_BRANCH,
    "base_panel_path": str(base_panel_path),
    "summary_by_tenor_path": str(summary_by_tenor_path),
    "summary_by_bucket_path": str(summary_by_bucket_path),
    "validation_path": str(validation_path),
    "z_summary_path": str(z_summary_path),
    "prior_only_check_path": str(prior_only_check_path),
    "primary_cross_tenor_trade_quality_metric": "normalized_pnl_per_day",
    "normalized_pnl_per_day_formula": "normalized_pnl_dollars / tenor",
    "zscore_method": {
        "model_vrp_z_3m": "prior-only rolling z-score of model_vrp_log by tenor, 63-row window, shifted one row",
        "model_vrp_z_1y": "prior-only rolling z-score of model_vrp_log by tenor, 252-row window, shifted one row",
    },
    "locked_back_core_thresholds": LOCKED_BACK_CORE_THRESHOLDS,
    "notes": [
        "No threshold sweep performed.",
        "No Secondary research performed.",
        "No sizing changes performed.",
        "This panel is the clean input for Core threshold research.",
        "Total P&L retained, but average P&L per day should be used for cross-tenor trade quality comparisons.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

failures = validation[validation["status"] == "FAIL"]

print("=" * 100)
print("Cell 3 — Validate locked signal base panel")
print("=" * 100)
print(f"Signal base panel:         {base_panel_path}")
print(f"Validation file:           {validation_path}")
print(f"Z-score summary:           {z_summary_path}")
print(f"Prior-only check:          {prior_only_check_path}")
print(f"Manifest:                  {manifest_path}")
print()
print("Validation:")
display(validation)

print()
print("Z-score summary:")
display(z_summary)

print()
print("Prior-only z-score warmup check:")
display(prior_only_check)

print()
print("Latest date rows:")
display(
    latest_rows[
        [
            "trade_date",
            "tenor",
            "tenor_bucket",
            "implied_vol_pct",
            "forecast_vol_pct",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "forward_realized_complete",
            "last_forward_rv_date",
            "normalized_pnl_dollars",
            "normalized_pnl_per_day",
        ]
    ].sort_values("tenor")
)

if not failures.empty:
    raise RuntimeError("Cell 3 validation failed. Review validation output above.")

print("\nCELL 3 PASSED")

Cell 3 — Validate locked signal base panel
Signal base panel:         C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\01_unified_fds_signal_base_panel_20200102_20260701_20260704_211636.parquet
Validation file:           C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\01_signal_base_validation_20260704_211636.csv
Z-score summary:           C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\01_signal_base_zscore_summary_20260704_211636.csv
Prior-only check:          C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\01_signal_base_prior_only_zscore_check_20260704_211636.csv
Manifest:                  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\01_signal_base_manifest_20260704_211636.json

Validation:


,check,status,detail
0,locked_model_only,PASS,models_found=['unified_fds_no_min_return']
1,one_row_per_trade_date_tenor,PASS,duplicate_key_count=0
2,all_exact_tenors_present,PASS,missing=[]; extra=[]
3,bucket_mapping,PASS,bad_rows=0
4,positive_implied_and_forecast_variance,PASS,bad_rows=0
5,model_vrp_log_finite,PASS,bad_rows=0
6,z_3m_available_after_warmup,PASS,"[{'tenor': 9, 'z_3m_available_rows': 1569, 'fi..."
7,z_1y_available_after_warmup,PASS,"[{'tenor': 9, 'z_1y_available_rows': 1380, 'fi..."
8,zscore_prior_only_warmup_check,PASS,"[{'tenor': 9, 'first_z3_idx': 63, 'first_z1_id..."
9,normalized_pnl_per_day_created,PASS,bad_rows=0; formula=normalized_pnl_dollars / t...



Z-score summary:


,tenor,rows,z_3m_available_rows,z_1y_available_rows,first_date,first_z_3m_date,first_z_1y_date
0,9,1632,1569,1380,2020-01-02,2020-04-02,2020-12-31
1,12,1632,1569,1380,2020-01-02,2020-04-02,2020-12-31
2,15,1632,1569,1380,2020-01-02,2020-04-02,2020-12-31
3,18,1632,1569,1380,2020-01-02,2020-04-02,2020-12-31
4,21,1632,1569,1380,2020-01-02,2020-04-02,2020-12-31
5,24,1632,1569,1380,2020-01-02,2020-04-02,2020-12-31
6,27,1632,1569,1380,2020-01-02,2020-04-02,2020-12-31
7,30,1632,1569,1380,2020-01-02,2020-04-02,2020-12-31
8,33,1632,1569,1380,2020-01-02,2020-04-02,2020-12-31



Prior-only z-score warmup check:


,tenor,first_z3_idx,first_z1_idx,z3_prior_only_ok,z1_prior_only_ok
0,9,63,252,True,True
1,12,63,252,True,True
2,15,63,252,True,True
3,18,63,252,True,True
4,21,63,252,True,True
5,24,63,252,True,True
6,27,63,252,True,True
7,30,63,252,True,True
8,33,63,252,True,True



Latest date rows:


,trade_date,tenor,tenor_bucket,implied_vol_pct,forecast_vol_pct,model_vrp_log,model_vrp_z_3m,model_vrp_z_1y,forward_realized_complete,last_forward_rv_date,normalized_pnl_dollars,normalized_pnl_per_day
14679,2026-07-01,9,Front,12.841456,12.672383,0.026507,-1.727898,-1.529773,False,2026-07-02,NaN,NaN
14680,2026-07-01,12,Front,13.738603,12.785639,0.143774,-1.514802,-1.442833,False,2026-07-02,NaN,NaN
14681,2026-07-01,15,Front,14.249805,13.444656,0.116323,-1.437723,-1.415119,False,2026-07-02,NaN,NaN
14682,2026-07-01,18,Middle,14.767299,13.468368,0.184143,-1.338608,-1.394870,False,2026-07-02,NaN,NaN
14683,2026-07-01,21,Middle,15.175856,13.817420,0.187551,-1.272126,-1.375707,False,2026-07-02,NaN,NaN
14684,2026-07-01,24,Middle,15.570145,13.927472,0.222984,-1.233577,-1.356383,False,2026-07-02,NaN,NaN
14685,2026-07-01,27,Back,16.034296,13.951883,0.278231,-1.109388,-1.244294,False,2026-07-02,NaN,NaN
14686,2026-07-01,30,Back,16.396160,14.129525,0.297561,-1.029999,-1.140252,False,2026-07-02,NaN,NaN
14687,2026-07-01,33,Back,16.567266,14.150541,0.315352,-1.049221,-1.181808,False,2026-07-02,NaN,NaN



CELL 3 PASSED


In [6]:
# ======================================================================================
# Cell 4 — Legacy Core baseline diagnostic using P&L/day
# ======================================================================================
#
# Objective:
#   Apply the current legacy/reference Core thresholds to the clean locked signal panel
#   and produce baseline diagnostics before any threshold sweep.
#
# Uses:
#   - locked unified_fds_no_min_return signal base panel from Cell 2 / Cell 3
#   - normalized_pnl_per_day = normalized_pnl_dollars / tenor
#
# Thresholds applied:
#   Front:
#       model_vrp_log > 0.60
#       model_vrp_z_3m > 0.55
#       model_vrp_z_1y > 0.65
#       RSI14 < 70
#       forecast_vol_pct > 8.5
#
#   Middle:
#       model_vrp_log > 0.65
#       model_vrp_z_3m > 0.75
#       model_vrp_z_1y > 0.65
#       RSI14 < 68
#       forecast_vol_pct > 8.5
#
#   Back:
#       model_vrp_log > 0.70
#       model_vrp_z_3m > 0.70
#       model_vrp_z_1y > 0.70
#       RSI14 < 70
#       forecast_vol_pct > 8.5
#
# Selection universes evaluated:
#   1. all_qualified_trades
#   2. one_trade_per_bucket_per_date_best_rank
#   3. one_trade_per_date_best_rank
#   4. thirty_day_anchor_only
#
# Important:
#   The best-rank selector is diagnostic only. It is not yet the final production
#   DTE-selection rule. We will discuss and test that separately.
#
# Not in scope:
#   - No threshold sweep.
#   - No Secondary.
#   - No production lock.
#   - No sizing changes.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 320)

# ======================================================================================
# 0. Fallback config if this cell is run independently
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

if "TIMESTAMP" not in globals():
    TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "ALL_TENORS" not in globals():
    ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

if "TENOR_BUCKET_MAP" not in globals():
    TENOR_BUCKET_MAP = {
        9: "Front",
        12: "Front",
        15: "Front",
        18: "Middle",
        21: "Middle",
        24: "Middle",
        27: "Back",
        30: "Back",
        33: "Back",
    }

if "BUCKET_ORDER_MAP" not in globals():
    BUCKET_ORDER_MAP = {"Front": 1, "Middle": 2, "Back": 3}

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

if "LEGACY_CORE_REFERENCE_THRESHOLDS" not in globals():
    LEGACY_CORE_REFERENCE_THRESHOLDS = {
        "Front": {
            "model_vrp_log": 0.60,
            "model_vrp_z_3m": 0.55,
            "model_vrp_z_1y": 0.65,
            "RSI14_max": 70.0,
            "forecast_vol_pct_min": 8.5,
        },
        "Middle": {
            "model_vrp_log": 0.65,
            "model_vrp_z_3m": 0.75,
            "model_vrp_z_1y": 0.65,
            "RSI14_max": 68.0,
            "forecast_vol_pct_min": 8.5,
        },
        "Back": {
            "model_vrp_log": 0.70,
            "model_vrp_z_3m": 0.70,
            "model_vrp_z_1y": 0.70,
            "RSI14_max": 70.0,
            "forecast_vol_pct_min": 8.5,
        },
    }

BUCKET_CENTER_TENOR = {
    "Front": 12,
    "Middle": 21,
    "Back": 30,
}

BASELINE_LABEL = "legacy_core_reference_thresholds"

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def max_drawdown_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)
    if len(pnl) == 0:
        return np.nan
    cumulative = pnl.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max
    return float(drawdown.min())


def worst_rolling_sum(pnl: pd.Series, window: int) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)
    if len(pnl) == 0:
        return np.nan
    if len(pnl) < window:
        return float(pnl.sum())
    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def profit_factor_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())
    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def summarize_trade_set(df: pd.DataFrame, selection_universe: str, group_cols: list[str]) -> pd.DataFrame:
    """
    Summarize selected trade rows.

    Includes both total P&L and P&L/day diagnostics.
    P&L/day is the primary cross-tenor quality metric.
    """
    rows = []

    if df.empty:
        return pd.DataFrame()

    d = df.copy()
    d["normalized_pnl_dollars"] = pd.to_numeric(d["normalized_pnl_dollars"], errors="coerce")
    d["normalized_pnl_per_day"] = pd.to_numeric(d["normalized_pnl_per_day"], errors="coerce")
    d["win"] = pd.to_numeric(d["win"], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame()

    d = d.sort_values(["trade_date", "tenor"]).copy()

    for keys, g in d.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))

        pnl = g["normalized_pnl_dollars"]
        pnl_day = g["normalized_pnl_per_day"]
        wins = g["win"]

        y2025 = g[g["trade_date"].dt.year == 2025].copy()

        row.update({
            "baseline_label": BASELINE_LABEL,
            "selection_universe": selection_universe,
            "trades": int(len(g)),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if g["tenor"].notna().any() else np.nan,

            # Win / raw P&L.
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,
            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            # P&L/day — primary cross-tenor trade-quality metrics.
            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            # 2025 behavior.
            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_2025": float(y2025["normalized_pnl_dollars"].mean()) if len(y2025) else np.nan,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,
            "worst_trade_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].min()) if len(y2025) else np.nan,
        })

        rows.append(row)

    return pd.DataFrame(rows)


def add_selection_ranks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add diagnostic ranking fields for selecting one tenor when multiple pass.

    Diagnostic only:
      - rank z_1y descending
      - rank z_3m descending
      - rank model_vrp_log descending
      - average rank
      - tie-break by distance to bucket center
    """
    d = df.copy()

    d["bucket_center_tenor"] = d["tenor_bucket"].map(BUCKET_CENTER_TENOR)
    d["distance_to_bucket_center"] = (d["tenor"] - d["bucket_center_tenor"]).abs()

    rank_group = ["trade_date", "tenor_bucket"]

    d["rank_z_1y_in_bucket_date"] = (
        d.groupby(rank_group)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_bucket_date"] = (
        d.groupby(rank_group)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_bucket_date"] = (
        d.groupby(rank_group)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_bucket_date"] = d[
        [
            "rank_z_1y_in_bucket_date",
            "rank_z_3m_in_bucket_date",
            "rank_vrp_log_in_bucket_date",
        ]
    ].mean(axis=1)

    # Global date ranks across all passing tenors.
    d["rank_z_1y_in_date"] = (
        d.groupby("trade_date")["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_date"] = (
        d.groupby("trade_date")["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_date"] = (
        d.groupby("trade_date")["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_date"] = d[
        [
            "rank_z_1y_in_date",
            "rank_z_3m_in_date",
            "rank_vrp_log_in_date",
        ]
    ].mean(axis=1)

    return d


def apply_legacy_core_thresholds(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    d["core_model_vrp_log_threshold"] = d["tenor_bucket"].map(
        {k: v["model_vrp_log"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
    )
    d["core_model_vrp_z_3m_threshold"] = d["tenor_bucket"].map(
        {k: v["model_vrp_z_3m"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
    )
    d["core_model_vrp_z_1y_threshold"] = d["tenor_bucket"].map(
        {k: v["model_vrp_z_1y"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
    )
    d["core_RSI14_max"] = d["tenor_bucket"].map(
        {k: v["RSI14_max"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
    )
    d["core_forecast_vol_pct_min"] = d["tenor_bucket"].map(
        {k: v["forecast_vol_pct_min"] for k, v in LEGACY_CORE_REFERENCE_THRESHOLDS.items()}
    )

    d["core_threshold_inputs_available"] = (
        d["model_vrp_log"].notna()
        & d["model_vrp_z_3m"].notna()
        & d["model_vrp_z_1y"].notna()
        & d["RSI14"].notna()
        & d["forecast_vol_pct"].notna()
    )

    d["legacy_core_pass"] = (
        d["core_threshold_inputs_available"]
        & (d["model_vrp_log"] > d["core_model_vrp_log_threshold"])
        & (d["model_vrp_z_3m"] > d["core_model_vrp_z_3m_threshold"])
        & (d["model_vrp_z_1y"] > d["core_model_vrp_z_1y_threshold"])
        & (d["RSI14"] < d["core_RSI14_max"])
        & (d["forecast_vol_pct"] > d["core_forecast_vol_pct_min"])
    )

    d["trade_outcome_available"] = (
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    )

    return d


# ======================================================================================
# 2. Load clean signal base panel if needed
# ======================================================================================

if "signal" not in globals() or not isinstance(signal, pd.DataFrame) or signal.empty:
    base_panel_path = latest_file(
        OUT_PROCESSED_DIR,
        "01_unified_fds_signal_base_panel_*.parquet",
        required=True,
    )
    signal = pd.read_parquet(base_panel_path)
else:
    base_panel_path = globals().get("base_panel_path", None)

signal = signal.copy()
signal["trade_date"] = pd.to_datetime(signal["trade_date"], errors="coerce").dt.normalize()
signal["tenor"] = pd.to_numeric(signal["tenor"], errors="coerce").astype(int)

if "last_forward_rv_date" in signal.columns:
    signal["last_forward_rv_date"] = pd.to_datetime(
        signal["last_forward_rv_date"],
        errors="coerce",
    ).dt.normalize()

required_signal_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "model_spec",
    "implied_variance",
    "forecast_variance",
    "implied_vol_pct",
    "forecast_vol_pct",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_signal_cols = [c for c in required_signal_cols if c not in signal.columns]
if missing_signal_cols:
    raise KeyError(f"Cell 4 missing required signal columns: {missing_signal_cols}")

# ======================================================================================
# 3. Apply baseline Core thresholds
# ======================================================================================

baseline = apply_legacy_core_thresholds(signal)

# Add fixed baseline label for grouping / audit summaries.
baseline["baseline_label"] = BASELINE_LABEL

# Trade evaluation should only use rows with completed / available outcomes.
qualified_all = baseline[
    baseline["legacy_core_pass"]
    & baseline["trade_outcome_available"]
].copy()

# Trade evaluation should only use rows with completed / available outcomes.
qualified_all = baseline[
    baseline["legacy_core_pass"]
    & baseline["trade_outcome_available"]
].copy()

qualified_all = add_selection_ranks(qualified_all)

one_trade_per_bucket_date = (
    qualified_all
    .sort_values(
        [
            "trade_date",
            "tenor_bucket_order",
            "avg_signal_rank_in_bucket_date",
            "distance_to_bucket_center",
            "rank_z_1y_in_bucket_date",
            "rank_z_3m_in_bucket_date",
            "rank_vrp_log_in_bucket_date",
            "tenor",
        ],
        ascending=[True, True, True, True, True, True, True, True],
    )
    .groupby(["trade_date", "tenor_bucket"], as_index=False)
    .head(1)
    .copy()
)

one_trade_per_date = (
    qualified_all
    .sort_values(
        [
            "trade_date",
            "avg_signal_rank_in_date",
            "rank_z_1y_in_date",
            "rank_z_3m_in_date",
            "rank_vrp_log_in_date",
            "distance_to_bucket_center",
            "tenor_bucket_order",
            "tenor",
        ],
        ascending=[True, True, True, True, True, True, True, True],
    )
    .groupby(["trade_date"], as_index=False)
    .head(1)
    .copy()
)

thirty_day_anchor = qualified_all[qualified_all["tenor"] == 30].copy()

# Add selection flags back to full baseline panel.
baseline["selected_all_qualified"] = baseline["legacy_core_pass"] & baseline["trade_outcome_available"]

selected_bucket_keys = set(
    zip(
        one_trade_per_bucket_date["trade_date"],
        one_trade_per_bucket_date["tenor"],
    )
)
selected_date_keys = set(
    zip(
        one_trade_per_date["trade_date"],
        one_trade_per_date["tenor"],
    )
)
selected_30d_keys = set(
    zip(
        thirty_day_anchor["trade_date"],
        thirty_day_anchor["tenor"],
    )
)

baseline["_key"] = list(zip(baseline["trade_date"], baseline["tenor"]))
baseline["selected_one_trade_per_bucket_date_best_rank"] = baseline["_key"].isin(selected_bucket_keys)
baseline["selected_one_trade_per_date_best_rank"] = baseline["_key"].isin(selected_date_keys)
baseline["selected_thirty_day_anchor"] = baseline["_key"].isin(selected_30d_keys)
baseline = baseline.drop(columns=["_key"])

# ======================================================================================
# 4. Summaries
# ======================================================================================

summary_frames = []

summary_frames.append(
    summarize_trade_set(
        qualified_all,
        selection_universe="all_qualified_trades",
        group_cols=["baseline_label"],
    )
)

summary_frames.append(
    summarize_trade_set(
        one_trade_per_bucket_date,
        selection_universe="one_trade_per_bucket_per_date_best_rank",
        group_cols=["baseline_label"],
    )
)

summary_frames.append(
    summarize_trade_set(
        one_trade_per_date,
        selection_universe="one_trade_per_date_best_rank",
        group_cols=["baseline_label"],
    )
)

summary_frames.append(
    summarize_trade_set(
        thirty_day_anchor,
        selection_universe="thirty_day_anchor_only",
        group_cols=["baseline_label"],
    )
)

summary_overall = pd.concat(
    [x for x in summary_frames if x is not None and not x.empty],
    ignore_index=True,
) if any(x is not None and not x.empty for x in summary_frames) else pd.DataFrame()

by_bucket_frames = []
for universe_name, d in [
    ("all_qualified_trades", qualified_all),
    ("one_trade_per_bucket_per_date_best_rank", one_trade_per_bucket_date),
    ("one_trade_per_date_best_rank", one_trade_per_date),
    ("thirty_day_anchor_only", thirty_day_anchor),
]:
    s = summarize_trade_set(
        d,
        selection_universe=universe_name,
        group_cols=["tenor_bucket", "tenor_bucket_order"],
    )
    if not s.empty:
        by_bucket_frames.append(s)

summary_by_bucket = pd.concat(by_bucket_frames, ignore_index=True) if by_bucket_frames else pd.DataFrame()
if not summary_by_bucket.empty:
    summary_by_bucket = summary_by_bucket.sort_values(
        ["selection_universe", "tenor_bucket_order"]
    )

by_tenor_frames = []
for universe_name, d in [
    ("all_qualified_trades", qualified_all),
    ("one_trade_per_bucket_per_date_best_rank", one_trade_per_bucket_date),
    ("one_trade_per_date_best_rank", one_trade_per_date),
    ("thirty_day_anchor_only", thirty_day_anchor),
]:
    s = summarize_trade_set(
        d,
        selection_universe=universe_name,
        group_cols=["tenor", "tenor_bucket", "tenor_bucket_order"],
    )
    if not s.empty:
        by_tenor_frames.append(s)

summary_by_tenor = pd.concat(by_tenor_frames, ignore_index=True) if by_tenor_frames else pd.DataFrame()
if not summary_by_tenor.empty:
    summary_by_tenor = summary_by_tenor.sort_values(
        ["selection_universe", "tenor"]
    )

by_year_frames = []
for universe_name, d in [
    ("all_qualified_trades", qualified_all),
    ("one_trade_per_bucket_per_date_best_rank", one_trade_per_bucket_date),
    ("one_trade_per_date_best_rank", one_trade_per_date),
    ("thirty_day_anchor_only", thirty_day_anchor),
]:
    if not d.empty:
        dy = d.copy()
        dy["year"] = dy["trade_date"].dt.year.astype(int)
        s = summarize_trade_set(
            dy,
            selection_universe=universe_name,
            group_cols=["year"],
        )
        if not s.empty:
            by_year_frames.append(s)

summary_by_year = pd.concat(by_year_frames, ignore_index=True) if by_year_frames else pd.DataFrame()
if not summary_by_year.empty:
    summary_by_year = summary_by_year.sort_values(
        ["selection_universe", "year"]
    )

# Pass-rate diagnostics, including rows without trade outcomes.
pass_diagnostics_by_tenor = (
    baseline.groupby(["tenor", "tenor_bucket", "tenor_bucket_order"], as_index=False)
    .agg(
        rows=("trade_date", "size"),
        threshold_inputs_available_rows=("core_threshold_inputs_available", "sum"),
        pass_rows_all_dates=("legacy_core_pass", "sum"),
        trade_outcome_available_rows=("trade_outcome_available", "sum"),
        pass_rows_with_outcomes=("selected_all_qualified", "sum"),
        avg_model_vrp_log=("model_vrp_log", "mean"),
        avg_z_3m=("model_vrp_z_3m", "mean"),
        avg_z_1y=("model_vrp_z_1y", "mean"),
        avg_RSI14=("RSI14", "mean"),
        avg_forecast_vol_pct=("forecast_vol_pct", "mean"),
        avg_implied_vol_pct=("implied_vol_pct", "mean"),
    )
    .sort_values("tenor")
)

pass_diagnostics_by_tenor["pass_rate_all_dates"] = (
    pass_diagnostics_by_tenor["pass_rows_all_dates"]
    / pass_diagnostics_by_tenor["threshold_inputs_available_rows"].replace(0, np.nan)
)

pass_diagnostics_by_tenor["pass_rate_with_outcomes"] = (
    pass_diagnostics_by_tenor["pass_rows_with_outcomes"]
    / pass_diagnostics_by_tenor["trade_outcome_available_rows"].replace(0, np.nan)
)

pass_diagnostics_by_bucket = (
    baseline.groupby(["tenor_bucket", "tenor_bucket_order"], as_index=False)
    .agg(
        rows=("trade_date", "size"),
        threshold_inputs_available_rows=("core_threshold_inputs_available", "sum"),
        pass_rows_all_dates=("legacy_core_pass", "sum"),
        trade_outcome_available_rows=("trade_outcome_available", "sum"),
        pass_rows_with_outcomes=("selected_all_qualified", "sum"),
        avg_model_vrp_log=("model_vrp_log", "mean"),
        avg_z_3m=("model_vrp_z_3m", "mean"),
        avg_z_1y=("model_vrp_z_1y", "mean"),
        avg_RSI14=("RSI14", "mean"),
        avg_forecast_vol_pct=("forecast_vol_pct", "mean"),
        avg_implied_vol_pct=("implied_vol_pct", "mean"),
    )
    .sort_values("tenor_bucket_order")
)

pass_diagnostics_by_bucket["pass_rate_all_dates"] = (
    pass_diagnostics_by_bucket["pass_rows_all_dates"]
    / pass_diagnostics_by_bucket["threshold_inputs_available_rows"].replace(0, np.nan)
)

pass_diagnostics_by_bucket["pass_rate_with_outcomes"] = (
    pass_diagnostics_by_bucket["pass_rows_with_outcomes"]
    / pass_diagnostics_by_bucket["trade_outcome_available_rows"].replace(0, np.nan)
)

# ======================================================================================
# 5. Save outputs
# ======================================================================================

date_min = baseline["trade_date"].min()
date_max = baseline["trade_date"].max()

baseline_panel_path = OUT_PROCESSED_DIR / (
    f"02_legacy_core_baseline_trade_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)
qualified_panel_path = OUT_PROCESSED_DIR / (
    f"02_legacy_core_baseline_qualified_trade_panel_{date_min:%Y%m%d}_{date_max:%Y%m%d}_{TIMESTAMP}.parquet"
)

summary_overall_path = OUT_AUDIT_DIR / f"02_legacy_core_baseline_summary_{TIMESTAMP}.csv"
summary_by_year_path = OUT_AUDIT_DIR / f"02_legacy_core_baseline_by_year_{TIMESTAMP}.csv"
summary_by_bucket_path = OUT_AUDIT_DIR / f"02_legacy_core_baseline_by_bucket_{TIMESTAMP}.csv"
summary_by_tenor_path = OUT_AUDIT_DIR / f"02_legacy_core_baseline_by_tenor_{TIMESTAMP}.csv"
pass_diag_by_bucket_path = OUT_AUDIT_DIR / f"02_legacy_core_baseline_pass_diagnostics_by_bucket_{TIMESTAMP}.csv"
pass_diag_by_tenor_path = OUT_AUDIT_DIR / f"02_legacy_core_baseline_pass_diagnostics_by_tenor_{TIMESTAMP}.csv"
thresholds_used_path = OUT_AUDIT_DIR / f"02_legacy_core_baseline_thresholds_used_{TIMESTAMP}.csv"
validation_path = OUT_AUDIT_DIR / f"02_legacy_core_baseline_validation_{TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"02_legacy_core_baseline_manifest_{TIMESTAMP}.json"

baseline.to_parquet(baseline_panel_path, index=False)
qualified_all.to_parquet(qualified_panel_path, index=False)

summary_overall.to_csv(summary_overall_path, index=False)
summary_by_year.to_csv(summary_by_year_path, index=False)
summary_by_bucket.to_csv(summary_by_bucket_path, index=False)
summary_by_tenor.to_csv(summary_by_tenor_path, index=False)
pass_diagnostics_by_bucket.to_csv(pass_diag_by_bucket_path, index=False)
pass_diagnostics_by_tenor.to_csv(pass_diag_by_tenor_path, index=False)

threshold_rows = []
for bucket, params in LEGACY_CORE_REFERENCE_THRESHOLDS.items():
    row = {"tenor_bucket": bucket, "tenor_bucket_order": BUCKET_ORDER_MAP[bucket]}
    row.update(params)
    threshold_rows.append(row)

thresholds_used = pd.DataFrame(threshold_rows).sort_values("tenor_bucket_order")
thresholds_used.to_csv(thresholds_used_path, index=False)

# ======================================================================================
# 6. Validation
# ======================================================================================

validation_rows = []

def add_validation(check: str, status: str, detail: str):
    validation_rows.append({"check": check, "status": status, "detail": detail})

models_found = sorted(baseline["model_spec"].dropna().unique().tolist())
tenors_found = sorted(baseline["tenor"].dropna().astype(int).unique().tolist())
missing_tenors = sorted(set(ALL_TENORS) - set(tenors_found))
extra_tenors = sorted(set(tenors_found) - set(ALL_TENORS))
duplicate_key_count = int(baseline.duplicated(["trade_date", "tenor"]).sum())

bad_pnl_day = baseline[
    baseline["normalized_pnl_dollars"].notna()
    & baseline["normalized_pnl_per_day"].notna()
    & (
        (
            baseline["normalized_pnl_dollars"] / baseline["tenor"].replace(0, np.nan)
        )
        - baseline["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

add_validation(
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    "one_row_per_trade_date_tenor",
    "PASS" if duplicate_key_count == 0 else "FAIL",
    f"duplicate_key_count={duplicate_key_count}",
)

add_validation(
    "all_exact_tenors_present",
    "PASS" if not missing_tenors and not extra_tenors else "FAIL",
    f"missing={missing_tenors}; extra={extra_tenors}",
)

add_validation(
    "legacy_core_thresholds_applied",
    "PASS" if "legacy_core_pass" in baseline.columns else "FAIL",
    f"qualified_rows_with_outcomes={len(qualified_all):,}",
)

add_validation(
    "normalized_pnl_per_day_formula",
    "PASS" if bad_pnl_day.empty else "FAIL",
    f"bad_rows={len(bad_pnl_day):,}; formula=normalized_pnl_dollars / tenor",
)

add_validation(
    "selection_all_qualified_created",
    "PASS" if len(qualified_all) > 0 else "WARN",
    f"rows={len(qualified_all):,}",
)

add_validation(
    "selection_one_trade_per_bucket_date_created",
    "PASS" if len(one_trade_per_bucket_date) > 0 else "WARN",
    f"rows={len(one_trade_per_bucket_date):,}",
)

add_validation(
    "selection_one_trade_per_date_created",
    "PASS" if len(one_trade_per_date) > 0 else "WARN",
    f"rows={len(one_trade_per_date):,}",
)

add_validation(
    "selection_thirty_day_anchor_created",
    "PASS" if len(thirty_day_anchor) > 0 else "WARN",
    f"rows={len(thirty_day_anchor):,}",
)

add_validation(
    "no_threshold_sweep",
    "PASS",
    "Only one fixed legacy/reference Core threshold set was evaluated.",
)

add_validation(
    "no_secondary",
    "PASS",
    "No Secondary threshold logic evaluated.",
)

add_validation(
    "diagnostic_selection_only",
    "PASS",
    "Best-rank DTE selection is diagnostic only, not a production lock.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"] == "FAIL"]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1.ipynb",
    "cell": "Cell 4 — Legacy Core baseline diagnostic using P&L/day",
    "timestamp": TIMESTAMP,
    "base_panel_path": str(base_panel_path),
    "baseline_panel_path": str(baseline_panel_path),
    "qualified_panel_path": str(qualified_panel_path),
    "summary_overall_path": str(summary_overall_path),
    "summary_by_year_path": str(summary_by_year_path),
    "summary_by_bucket_path": str(summary_by_bucket_path),
    "summary_by_tenor_path": str(summary_by_tenor_path),
    "pass_diag_by_bucket_path": str(pass_diag_by_bucket_path),
    "pass_diag_by_tenor_path": str(pass_diag_by_tenor_path),
    "thresholds_used_path": str(thresholds_used_path),
    "validation_path": str(validation_path),
    "locked_model_spec": LOCKED_MODEL_SPEC,
    "baseline_label": BASELINE_LABEL,
    "thresholds_used": LEGACY_CORE_REFERENCE_THRESHOLDS,
    "primary_cross_tenor_trade_quality_metric": "avg_pnl_per_day",
    "pnl_per_day_formula": "normalized_pnl_dollars / tenor",
    "selection_universes": [
        "all_qualified_trades",
        "one_trade_per_bucket_per_date_best_rank",
        "one_trade_per_date_best_rank",
        "thirty_day_anchor_only",
    ],
    "notes": [
        "No threshold sweep performed.",
        "No Secondary research performed.",
        "No sizing changes performed.",
        "Best-rank DTE selection is diagnostic only.",
        "Total P&L retained, but P&L/day is the primary cross-tenor quality metric.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 4 validation failed. Review validation output above.")

# ======================================================================================
# 7. Display concise review
# ======================================================================================

print("=" * 100)
print("Cell 4 — Legacy Core baseline diagnostic using P&L/day")
print("=" * 100)
print(f"Signal base panel:             {base_panel_path}")
print(f"Baseline trade panel:          {baseline_panel_path}")
print(f"Qualified trade panel:         {qualified_panel_path}")
print(f"Date range:                    {date_min.date()} to {date_max.date()}")
print(f"Rows:                          {len(baseline):,}")
print(f"All qualified rows:            {len(qualified_all):,}")
print(f"One per bucket/date rows:      {len(one_trade_per_bucket_date):,}")
print(f"One per date rows:             {len(one_trade_per_date):,}")
print(f"30D anchor rows:               {len(thirty_day_anchor):,}")
print("Primary quality metric:        avg_pnl_per_day")
print("No threshold sweep:            True")
print("No Secondary:                  True")
print("Diagnostic DTE selector only:  True")

print()
print("Validation:")
display(validation)

print()
print("Thresholds used:")
display(thresholds_used)

print()
print("Pass diagnostics by bucket:")
display(pass_diagnostics_by_bucket)

print()
print("Pass diagnostics by tenor:")
display(pass_diagnostics_by_tenor)

print()
print("Overall baseline summary:")
overall_display_cols = [
    "selection_universe",
    "trades",
    "date_min",
    "date_max",
    "avg_tenor",
    "win_rate",
    "total_pnl",
    "profit_factor",
    "avg_pnl",
    "avg_pnl_per_day",
    "median_pnl_per_day",
    "worst_trade",
    "worst_trade_pnl_per_day",
    "selected_trade_drawdown",
    "pnl_per_day_drawdown",
    "worst_20_trade_sum",
    "worst_20_trade_pnl_per_day_sum",
    "trades_le_neg_100k",
    "pnl_2025",
    "avg_pnl_per_day_2025",
    "worst_trade_2025",
]
display(
    summary_overall[
        [c for c in overall_display_cols if c in summary_overall.columns]
    ]
)

print()
print("Baseline summary by bucket:")
bucket_display_cols = [
    "selection_universe",
    "tenor_bucket",
    "trades",
    "avg_tenor",
    "win_rate",
    "total_pnl",
    "profit_factor",
    "avg_pnl",
    "avg_pnl_per_day",
    "median_pnl_per_day",
    "worst_trade",
    "worst_trade_pnl_per_day",
    "selected_trade_drawdown",
    "pnl_per_day_drawdown",
    "trades_le_neg_100k",
    "pnl_2025",
    "avg_pnl_per_day_2025",
]
display(
    summary_by_bucket[
        [c for c in bucket_display_cols if c in summary_by_bucket.columns]
    ]
)

print()
print("Baseline summary by tenor:")
tenor_display_cols = [
    "selection_universe",
    "tenor",
    "tenor_bucket",
    "trades",
    "win_rate",
    "total_pnl",
    "profit_factor",
    "avg_pnl",
    "avg_pnl_per_day",
    "median_pnl_per_day",
    "worst_trade",
    "worst_trade_pnl_per_day",
    "trades_le_neg_100k",
    "pnl_2025",
    "avg_pnl_per_day_2025",
]
display(
    summary_by_tenor[
        [c for c in tenor_display_cols if c in summary_by_tenor.columns]
    ]
)

print()
print("Baseline summary by year:")
year_display_cols = [
    "selection_universe",
    "year",
    "trades",
    "avg_tenor",
    "win_rate",
    "total_pnl",
    "profit_factor",
    "avg_pnl",
    "avg_pnl_per_day",
    "median_pnl_per_day",
    "worst_trade",
    "worst_trade_pnl_per_day",
    "trades_le_neg_100k",
]
display(
    summary_by_year[
        [c for c in year_display_cols if c in summary_by_year.columns]
    ]
)

print()
print("Saved outputs:")
saved_paths = [
    baseline_panel_path,
    qualified_panel_path,
    summary_overall_path,
    summary_by_year_path,
    summary_by_bucket_path,
    summary_by_tenor_path,
    pass_diag_by_bucket_path,
    pass_diag_by_tenor_path,
    thresholds_used_path,
    validation_path,
    manifest_path,
]
for p in saved_paths:
    print(f"  {p}")

print("\nCELL 4 PASSED")

Cell 4 — Legacy Core baseline diagnostic using P&L/day
Signal base panel:             C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\01_unified_fds_signal_base_panel_20200102_20260701_20260704_211636.parquet
Baseline trade panel:          C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\02_legacy_core_baseline_trade_panel_20200102_20260701_20260704_211636.parquet
Qualified trade panel:         C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\02_legacy_core_baseline_qualified_trade_panel_20200102_20260701_20260704_211636.parquet
Date range:                    2020-01-02 to 2026-07-01
Rows:                          14,688
All qualified rows:            1,516
One per bucket/date rows:      572
One per date rows:             271
30D anchor rows:               149
Primary quality metric:        avg_pnl_per_day
No threshold sweep:            True
No Secondary:   

,check,status,detail
0,locked_model_only,PASS,models_found=['unified_fds_no_min_return']
1,one_row_per_trade_date_tenor,PASS,duplicate_key_count=0
2,all_exact_tenors_present,PASS,missing=[]; extra=[]
3,legacy_core_thresholds_applied,PASS,"qualified_rows_with_outcomes=1,516"
4,normalized_pnl_per_day_formula,PASS,bad_rows=0; formula=normalized_pnl_dollars / t...
5,selection_all_qualified_created,PASS,"rows=1,516"
6,selection_one_trade_per_bucket_date_created,PASS,rows=572
7,selection_one_trade_per_date_created,PASS,rows=271
8,selection_thirty_day_anchor_created,PASS,rows=149
9,no_threshold_sweep,PASS,Only one fixed legacy/reference Core threshold...



Thresholds used:


,tenor_bucket,tenor_bucket_order,model_vrp_log,model_vrp_z_3m,model_vrp_z_1y,RSI14_max,forecast_vol_pct_min
0,Front,1,0.60,0.55,0.65,70.0,8.5
1,Middle,2,0.65,0.75,0.65,68.0,8.5
2,Back,3,0.70,0.70,0.70,70.0,8.5



Pass diagnostics by bucket:


...


Baseline summary by tenor:


,selection_universe,tenor,tenor_bucket,trades,win_rate,total_pnl,profit_factor,avg_pnl,avg_pnl_per_day,median_pnl_per_day,worst_trade,worst_trade_pnl_per_day,trades_le_neg_100k,pnl_2025,avg_pnl_per_day_2025
0,all_qualified_trades,9,Front,215,0.716279,4.718107e+05,1.332341,2194.468466,243.829830,974.669129,-97591.734861,-10843.526096,0,183708.799766,537.160233
1,all_qualified_trades,12,Front,219,0.757991,9.778962e+05,1.778103,4465.279246,372.106604,927.671694,-93387.142625,-7782.261885,0,301102.221792,512.078608
2,all_qualified_trades,15,Front,192,0.807292,1.541241e+06,2.693380,8027.295586,535.153039,856.814928,-108475.796818,-7231.719788,1,454587.889537,618.486925
3,all_qualified_trades,18,Middle,150,0.840000,1.596866e+06,3.667159,10645.775517,591.431973,948.175200,-108475.796818,-6026.433157,1,306184.345567,680.409657
4,all_qualified_trades,21,Middle,145,0.868966,1.899085e+06,4.373309,13097.141061,623.673384,924.055271,-108475.796818,-5165.514134,1,371243.158060,736.593568
5,all_qualified_trades,24,Middle,146,0.856164,1.899220e+06,4.091935,13008.358471,542.014936,813.997671,-92579.572354,-3857.482181,0,401102.977593,668.504963
6,all_qualified_trades,27,Back,150,0.886667,2.261703e+06,4.622321,15078.020966,558.445221,799.790089,-92588.332880,-3429.197514,0,476139.167658,476.615783
7,all_qualified_trades,30,Back,149,0.912752,2.609075e+06,6.144710,17510.567838,583.685595,760.748549,-103670.923339,-3455.697445,1,522043.942659,470.309858
8,all_qualified_trades,33,Back,150,0.933333,2.932429e+06,8.815754,19549.526954,592.409908,707.910764,-103670.923339,-3141.543131,1,596999.171873,488.942811
9,one_trade_per_bucket_per_date_best_rank,9,Front,99,0.717172,-1.252547e+05,0.867058,-1265.199033,-140.577670,963.929498,-97591.734861,-10843.526096,0,-36177.604344,-334.977818



Baseline summary by year:


,selection_universe,year,trades,avg_tenor,win_rate,total_pnl,profit_factor,avg_pnl,avg_pnl_per_day,median_pnl_per_day,worst_trade,worst_trade_pnl_per_day,trades_le_neg_100k
0,all_qualified_trades,2020,3,14.000000,1.000000,3.205478e+04,inf,10684.926887,786.205350,809.800314,7640.965158,699.819606,0
1,all_qualified_trades,2021,44,18.409091,0.795455,5.320991e+05,5.649787,12093.161408,556.328944,919.725315,-27346.824178,-3038.536020,0
2,all_qualified_trades,2022,610,20.621311,0.863934,9.310524e+06,4.388050,15263.154751,688.171248,1070.124798,-108475.796818,-10376.349181,5
3,all_qualified_trades,2023,55,13.636364,0.854545,4.514193e+05,2.708876,8207.624361,528.665736,1067.983722,-49261.410409,-5473.490045,0
4,all_qualified_trades,2024,217,18.165899,0.898618,2.215444e+06,11.301579,10209.420327,578.955983,635.126605,-17160.183108,-1713.991609,0
5,all_qualified_trades,2025,321,20.401869,0.919003,3.613112e+06,6.499567,11255.799609,561.800760,647.312944,-97591.734861,-10843.526096,0
6,all_qualified_trades,2026,266,20.639098,0.597744,3.467304e+04,1.012079,130.349759,-101.979228,663.056652,-59939.127377,-4353.558229,0
7,one_trade_per_bucket_per_date_best_rank,2020,2,15.000000,1.000000,2.233718e+04,inf,11168.588445,774.407867,774.407867,7640.965158,699.819606,0
8,one_trade_per_bucket_per_date_best_rank,2021,19,16.894737,0.789474,1.911491e+05,4.405221,10060.477904,488.436003,899.315728,-21201.476345,-2355.719594,0
9,one_trade_per_bucket_per_date_best_rank,2022,217,21.110599,0.852535,3.190454e+06,3.906377,14702.552075,553.455733,1052.045145,-108475.796818,-10376.349181,1



Saved outputs:
  C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\02_legacy_core_baseline_trade_panel_20200102_20260701_20260704_211636.parquet
  C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\02_legacy_core_baseline_qualified_trade_panel_20200102_20260701_20260704_211636.parquet
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\02_legacy_core_baseline_summary_20260704_211636.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\02_legacy_core_baseline_by_year_20260704_211636.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\02_legacy_core_baseline_by_bucket_20260704_211636.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\02_legacy_core_baseline_by_tenor_20260704_211636.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_cor

In [7]:
# ======================================================================================
# Cell 5 — Core threshold candidate grid design
# ======================================================================================
#
# Objective:
#   Define the Core threshold candidate grid for the next sweep.
#
# Important:
#   This cell only DESIGNS and SAVES the candidate grid.
#   It does not evaluate trades.
#   It does not run a threshold sweep.
#
# Locked constraints:
#   - Back Core thresholds remain fixed.
#   - No Secondary.
#   - No sizing changes.
#   - No production lock.
#
# New z-score constraint:
#   Within each bucket:
#
#       model_vrp_z_3m threshold == model_vrp_z_1y threshold
#
#   Therefore the grid uses one bucket-level z threshold:
#
#       Front z
#       Middle z
#       Back z
#
#   and applies that same value to both 3m and 1y z-score filters.
#
# Design direction from Cell 4:
#   - Front is the weak bucket, especially 9D.
#   - Middle and Back are already strong on P&L/day.
#   - Back remains the anchor.
#   - First sweep should mostly test whether Front needs tighter log / RSI / vol-floor
#     filters while leaving Middle close to current levels.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import itertools
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 320)

# ======================================================================================
# 0. Fallback config if this cell is run independently
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL5_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

if "ALL_TENORS" not in globals():
    ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

if "BUCKET_ORDER_MAP" not in globals():
    BUCKET_ORDER_MAP = {"Front": 1, "Middle": 2, "Back": 3}

if "LOCKED_BACK_CORE_THRESHOLDS" not in globals():
    LOCKED_BACK_CORE_THRESHOLDS = {
        "model_vrp_log": 0.70,
        "model_vrp_z_3m": 0.70,
        "model_vrp_z_1y": 0.70,
        "RSI14_max": 70.0,
        "forecast_vol_pct_min": 8.5,
    }

# ======================================================================================
# 1. Candidate grid ranges
# ======================================================================================
#
# Notes:
#   - Back remains fixed.
#   - Front z and Middle z are single bucket-level z thresholds.
#   - That single z value is copied into both model_vrp_z_3m and model_vrp_z_1y.
#   - Because Back z is locked at 0.70 and z thresholds are directionally monotonic,
#     Front/Middle z candidates are constrained to <= 0.70.
#   - Front weakness should be addressed mainly by VRP log, RSI, and forecast-vol floor.
# ======================================================================================

GRID_RANGES = {
    "Front": {
        "model_vrp_log": [0.60, 0.65, 0.70],
        "model_vrp_z": [0.65, 0.70],
        "RSI14_max": [70.0, 68.0, 66.0],
        "forecast_vol_pct_min": [8.5, 10.0, 11.5],
    },
    "Middle": {
        "model_vrp_log": [0.65, 0.70],
        "model_vrp_z": [0.65, 0.70],
        "RSI14_max": [68.0, 66.0],
        "forecast_vol_pct_min": [8.5, 10.0],
    },
    "Back": {
        "model_vrp_log": [LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"]],
        "model_vrp_z": [LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"]],
        "RSI14_max": [LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"]],
        "forecast_vol_pct_min": [LOCKED_BACK_CORE_THRESHOLDS["forecast_vol_pct_min"]],
    },
}

# ======================================================================================
# 2. Constraint helpers
# ======================================================================================

def is_between(x, a, b):
    lo = min(a, b)
    hi = max(a, b)
    return (x >= lo) and (x <= hi)


def candidate_passes_design_constraints(row):
    """
    Enforce grid-design constraints only.

    This does NOT evaluate trade performance.
    """

    # Back fixed.
    back_fixed_ok = (
        row["Back_model_vrp_log"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"]
        and row["Back_model_vrp_z"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"]
        and row["Back_RSI14_max"] == LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"]
        and row["Back_forecast_vol_pct_min"] == LOCKED_BACK_CORE_THRESHOLDS["forecast_vol_pct_min"]
    )

    if not back_fixed_ok:
        return False

    # VRP log threshold:
    # Higher is stricter. Keep the standard Front <= Middle <= Back structure.
    log_ok = (
        row["Front_model_vrp_log"]
        <= row["Middle_model_vrp_log"]
        <= row["Back_model_vrp_log"]
    )

    # Z threshold:
    # One z threshold per bucket, applied equally to 3m and 1y.
    # Higher is stricter. Back is locked at 0.70, so Front/Middle cannot exceed Back.
    z_ok = (
        row["Front_model_vrp_z"]
        <= row["Middle_model_vrp_z"]
        <= row["Back_model_vrp_z"]
    )

    # Forecast-vol floor:
    # Higher is stricter. Since Back is locked at 8.5, Front may be tighter than Back.
    # Middle must sit between Front and Back.
    forecast_floor_ok = is_between(
        row["Middle_forecast_vol_pct_min"],
        row["Front_forecast_vol_pct_min"],
        row["Back_forecast_vol_pct_min"],
    )

    # RSI cap:
    # Lower is stricter. We allow Back's locked 70 as an anchor exception.
    # For Front/Middle, require Middle not looser than Front.
    # Example allowed: Front 70, Middle 68.
    rsi_ok = row["Front_RSI14_max"] >= row["Middle_RSI14_max"]

    return bool(log_ok and z_ok and forecast_floor_ok and rsi_ok)


def build_wide_candidate_grid():
    rows = []

    product_iter = itertools.product(
        GRID_RANGES["Front"]["model_vrp_log"],
        GRID_RANGES["Front"]["model_vrp_z"],
        GRID_RANGES["Front"]["RSI14_max"],
        GRID_RANGES["Front"]["forecast_vol_pct_min"],

        GRID_RANGES["Middle"]["model_vrp_log"],
        GRID_RANGES["Middle"]["model_vrp_z"],
        GRID_RANGES["Middle"]["RSI14_max"],
        GRID_RANGES["Middle"]["forecast_vol_pct_min"],

        GRID_RANGES["Back"]["model_vrp_log"],
        GRID_RANGES["Back"]["model_vrp_z"],
        GRID_RANGES["Back"]["RSI14_max"],
        GRID_RANGES["Back"]["forecast_vol_pct_min"],
    )

    for vals in product_iter:
        (
            front_log,
            front_z,
            front_rsi,
            front_fv_floor,

            middle_log,
            middle_z,
            middle_rsi,
            middle_fv_floor,

            back_log,
            back_z,
            back_rsi,
            back_fv_floor,
        ) = vals

        row = {
            "Front_model_vrp_log": float(front_log),
            "Front_model_vrp_z": float(front_z),
            "Front_RSI14_max": float(front_rsi),
            "Front_forecast_vol_pct_min": float(front_fv_floor),

            "Middle_model_vrp_log": float(middle_log),
            "Middle_model_vrp_z": float(middle_z),
            "Middle_RSI14_max": float(middle_rsi),
            "Middle_forecast_vol_pct_min": float(middle_fv_floor),

            "Back_model_vrp_log": float(back_log),
            "Back_model_vrp_z": float(back_z),
            "Back_RSI14_max": float(back_rsi),
            "Back_forecast_vol_pct_min": float(back_fv_floor),
        }

        row["passes_design_constraints"] = candidate_passes_design_constraints(row)

        if row["passes_design_constraints"]:
            rows.append(row)

    grid = pd.DataFrame(rows).reset_index(drop=True)

    if grid.empty:
        raise RuntimeError("Candidate grid is empty after applying design constraints.")

    grid.insert(0, "candidate_id", [f"core_grid_{i:05d}" for i in range(1, len(grid) + 1)])
    grid.insert(1, "candidate_family", "core_front_middle_back_locked")
    grid.insert(2, "locked_model_spec", LOCKED_MODEL_SPEC)

    # Expand the single bucket z threshold into explicit 3m and 1y threshold columns.
    for bucket in ["Front", "Middle", "Back"]:
        grid[f"{bucket}_model_vrp_z_3m"] = grid[f"{bucket}_model_vrp_z"]
        grid[f"{bucket}_model_vrp_z_1y"] = grid[f"{bucket}_model_vrp_z"]

    # Convenience diagnostics.
    grid["front_middle_log_gap"] = grid["Middle_model_vrp_log"] - grid["Front_model_vrp_log"]
    grid["middle_back_log_gap"] = grid["Back_model_vrp_log"] - grid["Middle_model_vrp_log"]

    grid["front_middle_z_gap"] = grid["Middle_model_vrp_z"] - grid["Front_model_vrp_z"]
    grid["middle_back_z_gap"] = grid["Back_model_vrp_z"] - grid["Middle_model_vrp_z"]

    grid["front_middle_RSI_gap"] = grid["Front_RSI14_max"] - grid["Middle_RSI14_max"]

    grid["front_middle_forecast_floor_gap"] = (
        grid["Front_forecast_vol_pct_min"] - grid["Middle_forecast_vol_pct_min"]
    )
    grid["middle_back_forecast_floor_gap"] = (
        grid["Middle_forecast_vol_pct_min"] - grid["Back_forecast_vol_pct_min"]
    )

    return grid


def build_long_candidate_grid(grid_wide):
    rows = []

    for _, row in grid_wide.iterrows():
        for bucket in ["Front", "Middle", "Back"]:
            rows.append({
                "candidate_id": row["candidate_id"],
                "candidate_family": row["candidate_family"],
                "locked_model_spec": row["locked_model_spec"],
                "tenor_bucket": bucket,
                "tenor_bucket_order": BUCKET_ORDER_MAP[bucket],
                "model_vrp_log": row[f"{bucket}_model_vrp_log"],
                "model_vrp_z": row[f"{bucket}_model_vrp_z"],
                "model_vrp_z_3m": row[f"{bucket}_model_vrp_z_3m"],
                "model_vrp_z_1y": row[f"{bucket}_model_vrp_z_1y"],
                "RSI14_max": row[f"{bucket}_RSI14_max"],
                "forecast_vol_pct_min": row[f"{bucket}_forecast_vol_pct_min"],
                "is_back_locked": bucket == "Back",
            })

    long_grid = pd.DataFrame(rows).sort_values(
        ["candidate_id", "tenor_bucket_order"]
    ).reset_index(drop=True)

    return long_grid


# ======================================================================================
# 3. Build candidate grid
# ======================================================================================

grid_wide = build_wide_candidate_grid()
grid_long = build_long_candidate_grid(grid_wide)

# ======================================================================================
# 4. Save outputs
# ======================================================================================

grid_wide_csv_path = OUT_AUDIT_DIR / f"03_core_threshold_candidate_grid_wide_{CELL5_TIMESTAMP}.csv"
grid_wide_parquet_path = OUT_PROCESSED_DIR / f"03_core_threshold_candidate_grid_wide_{CELL5_TIMESTAMP}.parquet"

grid_long_csv_path = OUT_AUDIT_DIR / f"03_core_threshold_candidate_grid_long_{CELL5_TIMESTAMP}.csv"
grid_long_parquet_path = OUT_PROCESSED_DIR / f"03_core_threshold_candidate_grid_long_{CELL5_TIMESTAMP}.parquet"

grid_ranges_path = OUT_AUDIT_DIR / f"03_core_threshold_candidate_grid_ranges_{CELL5_TIMESTAMP}.json"
validation_path = OUT_AUDIT_DIR / f"03_core_threshold_candidate_grid_validation_{CELL5_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"03_core_threshold_candidate_grid_manifest_{CELL5_TIMESTAMP}.json"

grid_wide.to_csv(grid_wide_csv_path, index=False)
grid_wide.to_parquet(grid_wide_parquet_path, index=False)

grid_long.to_csv(grid_long_csv_path, index=False)
grid_long.to_parquet(grid_long_parquet_path, index=False)

with open(grid_ranges_path, "w", encoding="utf-8") as f:
    json.dump(GRID_RANGES, f, indent=2)

# ======================================================================================
# 5. Validation
# ======================================================================================

validation_rows = []

def add_validation(check, status, detail):
    validation_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })

# Candidate count.
add_validation(
    "candidate_grid_non_empty",
    "PASS" if len(grid_wide) > 0 else "FAIL",
    f"candidate_count={len(grid_wide):,}",
)

# Back fixed.
back_fixed_bad = grid_wide[
    (grid_wide["Back_model_vrp_log"] != LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"])
    | (grid_wide["Back_model_vrp_z"] != LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"])
    | (grid_wide["Back_RSI14_max"] != LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"])
    | (grid_wide["Back_forecast_vol_pct_min"] != LOCKED_BACK_CORE_THRESHOLDS["forecast_vol_pct_min"])
]

add_validation(
    "back_thresholds_locked",
    "PASS" if back_fixed_bad.empty else "FAIL",
    f"bad_rows={len(back_fixed_bad):,}; locked_back={LOCKED_BACK_CORE_THRESHOLDS}",
)

# Z equality within bucket.
z_equality_bad_rows = []

for bucket in ["Front", "Middle", "Back"]:
    bad = grid_wide[
        grid_wide[f"{bucket}_model_vrp_z_3m"] != grid_wide[f"{bucket}_model_vrp_z_1y"]
    ]
    if not bad.empty:
        z_equality_bad_rows.append((bucket, len(bad)))

add_validation(
    "z_3m_equals_z_1y_within_each_bucket",
    "PASS" if not z_equality_bad_rows else "FAIL",
    f"bad_bucket_counts={z_equality_bad_rows}",
)

# Log monotonicity.
log_bad = grid_wide[
    ~(
        (grid_wide["Front_model_vrp_log"] <= grid_wide["Middle_model_vrp_log"])
        & (grid_wide["Middle_model_vrp_log"] <= grid_wide["Back_model_vrp_log"])
    )
]

add_validation(
    "vrp_log_front_middle_back_monotonic",
    "PASS" if log_bad.empty else "FAIL",
    f"bad_rows={len(log_bad):,}; rule=Front <= Middle <= Back",
)

# Z monotonicity.
z_bad = grid_wide[
    ~(
        (grid_wide["Front_model_vrp_z"] <= grid_wide["Middle_model_vrp_z"])
        & (grid_wide["Middle_model_vrp_z"] <= grid_wide["Back_model_vrp_z"])
    )
]

add_validation(
    "z_threshold_front_middle_back_monotonic",
    "PASS" if z_bad.empty else "FAIL",
    f"bad_rows={len(z_bad):,}; rule=Front <= Middle <= Back",
)

# Forecast floor middle-between-front-and-back.
forecast_floor_bad = grid_wide[
    ~grid_wide.apply(
        lambda r: is_between(
            r["Middle_forecast_vol_pct_min"],
            r["Front_forecast_vol_pct_min"],
            r["Back_forecast_vol_pct_min"],
        ),
        axis=1,
    )
]

add_validation(
    "forecast_vol_floor_middle_between_front_and_back",
    "PASS" if forecast_floor_bad.empty else "FAIL",
    f"bad_rows={len(forecast_floor_bad):,}; Back fixed at 8.5, Front may be tighter",
)

# RSI Front/Middle only, Back exception.
rsi_bad = grid_wide[
    ~(grid_wide["Front_RSI14_max"] >= grid_wide["Middle_RSI14_max"])
]

add_validation(
    "rsi_front_middle_constraint",
    "PASS" if rsi_bad.empty else "FAIL",
    f"bad_rows={len(rsi_bad):,}; rule=Front RSI cap >= Middle RSI cap; Back locked exception",
)

# Long grid row count should be 3 per candidate.
long_count_ok = len(grid_long) == 3 * len(grid_wide)

add_validation(
    "long_grid_has_three_rows_per_candidate",
    "PASS" if long_count_ok else "FAIL",
    f"long_rows={len(grid_long):,}; expected={3 * len(grid_wide):,}",
)

# Explicit no-evaluation checks.
trade_metric_cols = [
    "trades",
    "win_rate",
    "total_pnl",
    "avg_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "worst_trade",
]

unexpected_trade_cols = [c for c in trade_metric_cols if c in grid_wide.columns]

add_validation(
    "no_trade_evaluation_performed",
    "PASS" if not unexpected_trade_cols else "FAIL",
    f"unexpected_trade_metric_cols={unexpected_trade_cols}",
)

add_validation(
    "no_secondary",
    "PASS",
    "Candidate grid is Core only.",
)

add_validation(
    "no_sizing_changes",
    "PASS",
    "Candidate grid contains thresholds only, no sizing fields.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"] == "FAIL"]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1.ipynb",
    "cell": "Cell 5 — Core threshold candidate grid design",
    "timestamp": CELL5_TIMESTAMP,
    "locked_model_spec": LOCKED_MODEL_SPEC,
    "candidate_count": int(len(grid_wide)),
    "long_grid_rows": int(len(grid_long)),
    "grid_wide_csv_path": str(grid_wide_csv_path),
    "grid_wide_parquet_path": str(grid_wide_parquet_path),
    "grid_long_csv_path": str(grid_long_csv_path),
    "grid_long_parquet_path": str(grid_long_parquet_path),
    "grid_ranges_path": str(grid_ranges_path),
    "validation_path": str(validation_path),
    "grid_ranges": GRID_RANGES,
    "locked_back_core_thresholds": LOCKED_BACK_CORE_THRESHOLDS,
    "hard_constraints": [
        "Back Core thresholds fixed.",
        "Within each bucket, model_vrp_z_3m threshold equals model_vrp_z_1y threshold.",
        "VRP log threshold: Front <= Middle <= Back.",
        "Z threshold: Front <= Middle <= Back.",
        "Forecast-vol floor: Middle must sit between Front and Back; Back fixed at 8.5; Front may be tighter.",
        "RSI cap: Front >= Middle; Back RSI fixed exception.",
        "No trade evaluation performed.",
        "No threshold sweep performed.",
        "No Secondary.",
        "No sizing changes.",
    ],
    "notes": [
        "This cell defines threshold candidates only.",
        "Cell 6 should evaluate this candidate grid against the clean signal base panel.",
        "P&L/day remains the primary cross-tenor quality metric for Cell 6.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 5 validation failed. Review validation output above.")

# ======================================================================================
# 6. Display concise review
# ======================================================================================

print("=" * 100)
print("Cell 5 — Core threshold candidate grid design")
print("=" * 100)
print(f"Locked model spec:              {LOCKED_MODEL_SPEC}")
print(f"Candidate count:                {len(grid_wide):,}")
print(f"Long grid rows:                 {len(grid_long):,}")
print(f"Back thresholds locked:         True")
print(f"Z 3m == Z 1y within bucket:     True")
print(f"No trade evaluation:            True")
print(f"No threshold sweep:             True")
print(f"No Secondary:                   True")
print(f"No sizing changes:              True")
print()
print("Saved outputs:")
for p in [
    grid_wide_csv_path,
    grid_wide_parquet_path,
    grid_long_csv_path,
    grid_long_parquet_path,
    grid_ranges_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print()
print("Validation:")
display(validation)

print()
print("Grid ranges:")
display(
    pd.DataFrame(
        [
            {
                "bucket": bucket,
                "model_vrp_log": GRID_RANGES[bucket]["model_vrp_log"],
                "model_vrp_z_applied_to_3m_and_1y": GRID_RANGES[bucket]["model_vrp_z"],
                "RSI14_max": GRID_RANGES[bucket]["RSI14_max"],
                "forecast_vol_pct_min": GRID_RANGES[bucket]["forecast_vol_pct_min"],
            }
            for bucket in ["Front", "Middle", "Back"]
        ]
    )
)

print()
print("Candidate grid sample:")
display(grid_wide.head(20))

print()
print("Long-format sample:")
display(grid_long.head(30))

print("\nCELL 5 PASSED")

Cell 5 — Core threshold candidate grid design
Locked model spec:              unified_fds_no_min_return
Candidate count:                375
Long grid rows:                 1,125
Back thresholds locked:         True
Z 3m == Z 1y within bucket:     True
No trade evaluation:            True
No threshold sweep:             True
No Secondary:                   True
No sizing changes:              True

Saved outputs:
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\03_core_threshold_candidate_grid_wide_20260704_213318.csv
  C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\03_core_threshold_candidate_grid_wide_20260704_213318.parquet
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\03_core_threshold_candidate_grid_long_20260704_213318.csv
  C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\03_core_threshold_candidate_grid_long_

,check,status,detail
0,candidate_grid_non_empty,PASS,candidate_count=375
1,back_thresholds_locked,PASS,"bad_rows=0; locked_back={'model_vrp_log': 0.7,..."
2,z_3m_equals_z_1y_within_each_bucket,PASS,bad_bucket_counts=[]
3,vrp_log_front_middle_back_monotonic,PASS,bad_rows=0; rule=Front <= Middle <= Back
4,z_threshold_front_middle_back_monotonic,PASS,bad_rows=0; rule=Front <= Middle <= Back
5,forecast_vol_floor_middle_between_front_and_back,PASS,"bad_rows=0; Back fixed at 8.5, Front may be ti..."
6,rsi_front_middle_constraint,PASS,bad_rows=0; rule=Front RSI cap >= Middle RSI c...
7,long_grid_has_three_rows_per_candidate,PASS,"long_rows=1,125; expected=1,125"
8,no_trade_evaluation_performed,PASS,unexpected_trade_metric_cols=[]
9,no_secondary,PASS,Candidate grid is Core only.



Grid ranges:


,bucket,model_vrp_log,model_vrp_z_applied_to_3m_and_1y,RSI14_max,forecast_vol_pct_min
0,Front,"[0.6, 0.65, 0.7]","[0.65, 0.7]","[70.0, 68.0, 66.0]","[8.5, 10.0, 11.5]"
1,Middle,"[0.65, 0.7]","[0.65, 0.7]","[68.0, 66.0]","[8.5, 10.0]"
2,Back,[0.7],[0.7],[70.0],[8.5]



Candidate grid sample:


,candidate_id,candidate_family,locked_model_spec,Front_model_vrp_log,Front_model_vrp_z,Front_RSI14_max,Front_forecast_vol_pct_min,Middle_model_vrp_log,Middle_model_vrp_z,Middle_RSI14_max,Middle_forecast_vol_pct_min,Back_model_vrp_log,Back_model_vrp_z,Back_RSI14_max,Back_forecast_vol_pct_min,passes_design_constraints,Front_model_vrp_z_3m,Front_model_vrp_z_1y,Middle_model_vrp_z_3m,Middle_model_vrp_z_1y,Back_model_vrp_z_3m,Back_model_vrp_z_1y,front_middle_log_gap,middle_back_log_gap,front_middle_z_gap,middle_back_z_gap,front_middle_RSI_gap,front_middle_forecast_floor_gap,middle_back_forecast_floor_gap
0,core_grid_00001,core_front_middle_back_locked,unified_fds_no_min_return,0.6,0.65,70.0,8.5,0.65,0.65,68.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.65,0.65,0.7,0.7,0.05,0.05,0.00,0.05,2.0,0.0,0.0
1,core_grid_00002,core_front_middle_back_locked,unified_fds_no_min_return,0.6,0.65,70.0,8.5,0.65,0.65,66.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.65,0.65,0.7,0.7,0.05,0.05,0.00,0.05,4.0,0.0,0.0
2,core_grid_00003,core_front_middle_back_locked,unified_fds_no_min_return,0.6,0.65,70.0,8.5,0.65,0.70,68.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.70,0.70,0.7,0.7,0.05,0.05,0.05,0.00,2.0,0.0,0.0
3,core_grid_00004,core_front_middle_back_locked,unified_fds_no_min_return,0.6,0.65,70.0,8.5,0.65,0.70,66.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.70,0.70,0.7,0.7,0.05,0.05,0.05,0.00,4.0,0.0,0.0
4,core_grid_00005,core_front_middle_back_locked,unified_fds_no_min_return,0.6,0.65,70.0,8.5,0.70,0.65,68.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.65,0.65,0.7,0.7,0.10,0.00,0.00,0.05,2.0,0.0,0.0
5,core_grid_00006,core_front_middle_back_locked,unified_fds_no_min_return,0.6,0.65,70.0,8.5,0.70,0.65,66.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.65,0.65,0.7,0.7,0.10,0.00,0.00,0.05,4.0,0.0,0.0
6,core_grid_00007,core_front_middle_back_locked,unified_fds_no_min_return,0.6,0.65,70.0,8.5,0.70,0.70,68.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.70,0.70,0.7,0.7,0.10,0.00,0.05,0.00,2.0,0.0,0.0
7,core_grid_00008,core_front_middle_back_locked,unified_fds_no_min_return,0.6,0.65,70.0,8.5,0.70,0.70,66.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.70,0.70,0.7,0.7,0.10,0.00,0.05,0.00,4.0,0.0,0.0
8,core_grid_00009,core_front_middle_back_locked,unified_fds_no_min_return,0.6,0.65,70.0,10.0,0.65,0.65,68.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.65,0.65,0.7,0.7,0.05,0.05,0.00,0.05,2.0,1.5,0.0
9,core_grid_00010,core_front_middle_back_locked,unified_fds_no_min_return,0.6,0.65,70.0,10.0,0.65,0.65,68.0,10.0,0.7,0.7,70.0,8.5,True,0.65,0.65,0.65,0.65,0.7,0.7,0.05,0.05,0.00,0.05,2.0,0.0,1.5



Long-format sample:


,candidate_id,candidate_family,locked_model_spec,tenor_bucket,tenor_bucket_order,model_vrp_log,model_vrp_z,model_vrp_z_3m,model_vrp_z_1y,RSI14_max,forecast_vol_pct_min,is_back_locked
0,core_grid_00001,core_front_middle_back_locked,unified_fds_no_min_return,Front,1,0.60,0.65,0.65,0.65,70.0,8.5,False
1,core_grid_00001,core_front_middle_back_locked,unified_fds_no_min_return,Middle,2,0.65,0.65,0.65,0.65,68.0,8.5,False
2,core_grid_00001,core_front_middle_back_locked,unified_fds_no_min_return,Back,3,0.70,0.70,0.70,0.70,70.0,8.5,True
3,core_grid_00002,core_front_middle_back_locked,unified_fds_no_min_return,Front,1,0.60,0.65,0.65,0.65,70.0,8.5,False
4,core_grid_00002,core_front_middle_back_locked,unified_fds_no_min_return,Middle,2,0.65,0.65,0.65,0.65,66.0,8.5,False
5,core_grid_00002,core_front_middle_back_locked,unified_fds_no_min_return,Back,3,0.70,0.70,0.70,0.70,70.0,8.5,True
6,core_grid_00003,core_front_middle_back_locked,unified_fds_no_min_return,Front,1,0.60,0.65,0.65,0.65,70.0,8.5,False
7,core_grid_00003,core_front_middle_back_locked,unified_fds_no_min_return,Middle,2,0.65,0.70,0.70,0.70,68.0,8.5,False
8,core_grid_00003,core_front_middle_back_locked,unified_fds_no_min_return,Back,3,0.70,0.70,0.70,0.70,70.0,8.5,True
9,core_grid_00004,core_front_middle_back_locked,unified_fds_no_min_return,Front,1,0.60,0.65,0.65,0.65,70.0,8.5,False



CELL 5 PASSED


In [8]:
# ======================================================================================
# Cell 6 — Core threshold candidate sweep using P&L/day
# ======================================================================================
#
# Objective:
#   Evaluate the Cell 5 Core threshold candidate grid against the clean locked signal panel.
#
# Inputs:
#   01_unified_fds_signal_base_panel_*.parquet
#   03_core_threshold_candidate_grid_wide_*.parquet
#   03_core_threshold_candidate_grid_long_*.parquet
#
# Outputs:
#   04_core_threshold_sweep_candidate_summary_*.csv
#   04_core_threshold_sweep_by_bucket_*.csv
#   04_core_threshold_sweep_by_tenor_*.csv
#   04_core_threshold_sweep_by_year_*.csv
#   04_core_threshold_sweep_candidate_pass_diagnostics_*.csv
#   04_core_threshold_sweep_selected_trade_panel_*.parquet
#   04_core_threshold_sweep_validation_*.csv
#   04_core_threshold_sweep_manifest_*.json
#
# Selection universes evaluated:
#   1. all_qualified_trades
#   2. one_trade_per_bucket_per_date_best_rank
#   3. one_trade_per_date_best_rank
#   4. thirty_day_anchor_only
#
# Primary cross-tenor quality metric:
#   avg_pnl_per_day
#
# Important:
#   This is the first-pass sweep only.
#   It does not choose a final threshold set.
#   If best candidates cluster at grid boundaries, we will expand the grid later.
#
# Not in scope:
#   - No final candidate selection.
#   - No Secondary.
#   - No sizing changes.
#   - No production lock.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 360)

# ======================================================================================
# 0. Fallback config if this cell is run independently
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL6_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

if "ALL_TENORS" not in globals():
    ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

if "BUCKET_ORDER_MAP" not in globals():
    BUCKET_ORDER_MAP = {"Front": 1, "Middle": 2, "Back": 3}

if "BUCKET_CENTER_TENOR" not in globals():
    BUCKET_CENTER_TENOR = {
        "Front": 12,
        "Middle": 21,
        "Back": 30,
    }

if "LOCKED_BACK_CORE_THRESHOLDS" not in globals():
    LOCKED_BACK_CORE_THRESHOLDS = {
        "model_vrp_log": 0.70,
        "model_vrp_z_3m": 0.70,
        "model_vrp_z_1y": 0.70,
        "RSI14_max": 70.0,
        "forecast_vol_pct_min": 8.5,
    }

SELECTION_UNIVERSES = [
    "all_qualified_trades",
    "one_trade_per_bucket_per_date_best_rank",
    "one_trade_per_date_best_rank",
    "thirty_day_anchor_only",
]

PRIMARY_RANKING_METRIC = "avg_pnl_per_day"

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def profit_factor_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max

    return float(drawdown.min())


def worst_rolling_sum(pnl: pd.Series, window: int) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def add_selection_ranks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add diagnostic best-rank fields for selected-tenor views.

    This is still not a final production DTE selector.
    It is a diagnostic selector to make threshold comparisons less noisy.
    """
    d = df.copy()

    if d.empty:
        return d

    d["bucket_center_tenor"] = d["tenor_bucket"].map(BUCKET_CENTER_TENOR)
    d["distance_to_bucket_center"] = (d["tenor"] - d["bucket_center_tenor"]).abs()

    # Within bucket/date.
    rank_group = ["trade_date", "tenor_bucket"]

    d["rank_z_1y_in_bucket_date"] = (
        d.groupby(rank_group)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_bucket_date"] = (
        d.groupby(rank_group)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_bucket_date"] = (
        d.groupby(rank_group)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_bucket_date"] = d[
        [
            "rank_z_1y_in_bucket_date",
            "rank_z_3m_in_bucket_date",
            "rank_vrp_log_in_bucket_date",
        ]
    ].mean(axis=1)

    # Across full date.
    d["rank_z_1y_in_date"] = (
        d.groupby("trade_date")["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_date"] = (
        d.groupby("trade_date")["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_date"] = (
        d.groupby("trade_date")["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_date"] = d[
        [
            "rank_z_1y_in_date",
            "rank_z_3m_in_date",
            "rank_vrp_log_in_date",
        ]
    ].mean(axis=1)

    return d


def select_trade_universes(qualified: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """
    Given qualified trades for one candidate, return all selection universes.
    """
    if qualified.empty:
        return {
            "all_qualified_trades": qualified.copy(),
            "one_trade_per_bucket_per_date_best_rank": qualified.copy(),
            "one_trade_per_date_best_rank": qualified.copy(),
            "thirty_day_anchor_only": qualified.copy(),
        }

    q = add_selection_ranks(qualified)

    one_trade_per_bucket_date = (
        q.sort_values(
            [
                "trade_date",
                "tenor_bucket_order",
                "avg_signal_rank_in_bucket_date",
                "distance_to_bucket_center",
                "rank_z_1y_in_bucket_date",
                "rank_z_3m_in_bucket_date",
                "rank_vrp_log_in_bucket_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date", "tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    one_trade_per_date = (
        q.sort_values(
            [
                "trade_date",
                "avg_signal_rank_in_date",
                "rank_z_1y_in_date",
                "rank_z_3m_in_date",
                "rank_vrp_log_in_date",
                "distance_to_bucket_center",
                "tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    thirty_day_anchor = q[q["tenor"] == 30].copy()

    return {
        "all_qualified_trades": q.copy(),
        "one_trade_per_bucket_per_date_best_rank": one_trade_per_bucket_date,
        "one_trade_per_date_best_rank": one_trade_per_date,
        "thirty_day_anchor_only": thirty_day_anchor,
    }


def summarize_trade_set(
    df: pd.DataFrame,
    candidate_id: str,
    selection_universe: str,
    group_cols: list[str] | None = None,
    include_empty_overall: bool = False,
) -> pd.DataFrame:
    """
    Summarize one selected trade set.

    For overall candidate-level summaries, use group_cols=[] and include_empty_overall=True.
    """
    if group_cols is None:
        group_cols = []

    rows = []

    d = df.copy()

    if not d.empty:
        d["normalized_pnl_dollars"] = pd.to_numeric(d["normalized_pnl_dollars"], errors="coerce")
        d["normalized_pnl_per_day"] = pd.to_numeric(d["normalized_pnl_per_day"], errors="coerce")
        d["win"] = pd.to_numeric(d["win"], errors="coerce")
        d = d[
            d["normalized_pnl_dollars"].notna()
            & d["normalized_pnl_per_day"].notna()
        ].copy()
        d = d.sort_values(["trade_date", "tenor"]).copy()

    if d.empty:
        if include_empty_overall and len(group_cols) == 0:
            return pd.DataFrame([{
                "candidate_id": candidate_id,
                "selection_universe": selection_universe,
                "trades": 0,
                "unique_trade_dates": 0,
                "date_min": pd.NaT,
                "date_max": pd.NaT,
                "avg_tenor": np.nan,
                "win_rate": np.nan,
                "total_pnl": 0.0,
                "avg_pnl": np.nan,
                "median_pnl": np.nan,
                "worst_trade": np.nan,
                "best_trade": np.nan,
                "profit_factor": np.nan,
                "selected_trade_drawdown": np.nan,
                "worst_5_trade_sum": np.nan,
                "worst_10_trade_sum": np.nan,
                "worst_20_trade_sum": np.nan,
                "trades_le_neg_50k": 0,
                "trades_le_neg_100k": 0,
                "trades_le_neg_150k": 0,
                "total_pnl_per_day_sum": 0.0,
                "avg_pnl_per_day": np.nan,
                "median_pnl_per_day": np.nan,
                "worst_trade_pnl_per_day": np.nan,
                "best_trade_pnl_per_day": np.nan,
                "profit_factor_pnl_per_day": np.nan,
                "pnl_per_day_drawdown": np.nan,
                "worst_5_trade_pnl_per_day_sum": np.nan,
                "worst_10_trade_pnl_per_day_sum": np.nan,
                "worst_20_trade_pnl_per_day_sum": np.nan,
                "trades_2025": 0,
                "pnl_2025": 0.0,
                "avg_pnl_per_day_2025": np.nan,
                "worst_trade_2025": np.nan,
                "trades_2026": 0,
                "pnl_2026": 0.0,
                "avg_pnl_per_day_2026": np.nan,
                "worst_trade_2026": np.nan,
            }])
        return pd.DataFrame()

    if len(group_cols) == 0:
        grouped = [((), d)]
    else:
        grouped = d.groupby(group_cols, dropna=False)

    for keys, g in grouped:
        if len(group_cols) == 0:
            keys = ()
        elif not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            "candidate_id": candidate_id,
            "selection_universe": selection_universe,
        }

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = g["normalized_pnl_dollars"]
        pnl_day = g["normalized_pnl_per_day"]
        wins = g["win"]

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": int(g["trade_date"].nunique()),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if g["tenor"].notna().any() else np.nan,

            # Total-dollar P&L.
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,
            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            # P&L/day — primary cross-tenor quality.
            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            # Recent year diagnostics.
            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        rows.append(row)

    return pd.DataFrame(rows)


def apply_candidate_thresholds(signal_df: pd.DataFrame, candidate: pd.Series) -> tuple[pd.DataFrame, dict]:
    """
    Apply one candidate's bucket thresholds to the signal panel.
    """
    d = signal_df.copy()

    required_inputs = (
        d["model_vrp_log"].notna()
        & d["model_vrp_z_3m"].notna()
        & d["model_vrp_z_1y"].notna()
        & d["RSI14"].notna()
        & d["forecast_vol_pct"].notna()
    )

    pass_mask = pd.Series(False, index=d.index)

    bucket_pass_counts = {}

    for bucket in ["Front", "Middle", "Back"]:
        bucket_mask = d["tenor_bucket"].eq(bucket)

        log_threshold = candidate[f"{bucket}_model_vrp_log"]
        z_threshold = candidate[f"{bucket}_model_vrp_z"]
        rsi_max = candidate[f"{bucket}_RSI14_max"]
        fv_floor = candidate[f"{bucket}_forecast_vol_pct_min"]

        bucket_pass = (
            required_inputs
            & bucket_mask
            & (d["model_vrp_log"] > log_threshold)
            & (d["model_vrp_z_3m"] > z_threshold)
            & (d["model_vrp_z_1y"] > z_threshold)
            & (d["RSI14"] < rsi_max)
            & (d["forecast_vol_pct"] > fv_floor)
        )

        pass_mask = pass_mask | bucket_pass
        bucket_pass_counts[f"{bucket}_pass_rows_all_dates"] = int(bucket_pass.sum())

    d["candidate_core_pass"] = pass_mask

    outcome_available = (
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    )

    qualified = d[
        d["candidate_core_pass"]
        & outcome_available
    ].copy()

    diagnostics = {
        "candidate_id": candidate["candidate_id"],
        "pass_rows_all_dates": int(pass_mask.sum()),
        "pass_rows_with_outcomes": int(len(qualified)),
        "threshold_input_available_rows": int(required_inputs.sum()),
        "trade_outcome_available_rows": int(outcome_available.sum()),
    }

    diagnostics.update(bucket_pass_counts)

    return qualified, diagnostics


def attach_candidate_parameters(summary: pd.DataFrame, grid_wide: pd.DataFrame) -> pd.DataFrame:
    if summary.empty:
        return summary

    param_cols = [
        c for c in grid_wide.columns
        if c == "candidate_id"
        or c.endswith("_model_vrp_log")
        or c.endswith("_model_vrp_z")
        or c.endswith("_RSI14_max")
        or c.endswith("_forecast_vol_pct_min")
        or c in [
            "front_middle_log_gap",
            "middle_back_log_gap",
            "front_middle_z_gap",
            "middle_back_z_gap",
            "front_middle_RSI_gap",
            "front_middle_forecast_floor_gap",
            "middle_back_forecast_floor_gap",
        ]
    ]

    params = grid_wide[param_cols].copy()

    out = summary.merge(params, on="candidate_id", how="left", validate="many_to_one")

    return out


# ======================================================================================
# 2. Load signal panel and candidate grid
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01_unified_fds_signal_base_panel_*.parquet",
    required=True,
)

grid_wide_path = latest_file(
    OUT_PROCESSED_DIR,
    "03_core_threshold_candidate_grid_wide_*.parquet",
    required=True,
)

grid_long_path = latest_file(
    OUT_PROCESSED_DIR,
    "03_core_threshold_candidate_grid_long_*.parquet",
    required=True,
)

signal = pd.read_parquet(base_panel_path)
grid_wide = pd.read_parquet(grid_wide_path)
grid_long = pd.read_parquet(grid_long_path)

# Normalize dates / numeric fields.
signal["trade_date"] = pd.to_datetime(signal["trade_date"], errors="coerce").dt.normalize()
signal["tenor"] = pd.to_numeric(signal["tenor"], errors="coerce").astype(int)

if "last_forward_rv_date" in signal.columns:
    signal["last_forward_rv_date"] = pd.to_datetime(
        signal["last_forward_rv_date"],
        errors="coerce",
    ).dt.normalize()

required_signal_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_signal_cols = [c for c in required_signal_cols if c not in signal.columns]
if missing_signal_cols:
    raise KeyError(f"Cell 6 missing required signal columns: {missing_signal_cols}")

required_grid_cols = [
    "candidate_id",
    "Front_model_vrp_log",
    "Front_model_vrp_z",
    "Front_RSI14_max",
    "Front_forecast_vol_pct_min",
    "Middle_model_vrp_log",
    "Middle_model_vrp_z",
    "Middle_RSI14_max",
    "Middle_forecast_vol_pct_min",
    "Back_model_vrp_log",
    "Back_model_vrp_z",
    "Back_RSI14_max",
    "Back_forecast_vol_pct_min",
]

missing_grid_cols = [c for c in required_grid_cols if c not in grid_wide.columns]
if missing_grid_cols:
    raise KeyError(f"Cell 6 missing required grid columns: {missing_grid_cols}")

# Ensure numeric fields.
for c in [
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    signal[c] = pd.to_numeric(signal[c], errors="coerce")

for c in [c for c in grid_wide.columns if c.endswith("_model_vrp_log") or c.endswith("_model_vrp_z") or c.endswith("_RSI14_max") or c.endswith("_forecast_vol_pct_min")]:
    grid_wide[c] = pd.to_numeric(grid_wide[c], errors="coerce")

grid_wide = grid_wide.sort_values("candidate_id").reset_index(drop=True)

# ======================================================================================
# 3. Sweep candidates
# ======================================================================================

start_time = time.time()

overall_summary_rows = []
bucket_summary_rows = []
tenor_summary_rows = []
year_summary_rows = []
pass_diagnostic_rows = []
selected_trade_frames = []

candidate_count = len(grid_wide)

print("=" * 100)
print("Cell 6 — Core threshold candidate sweep using P&L/day")
print("=" * 100)
print(f"Signal panel:             {base_panel_path}")
print(f"Wide candidate grid:      {grid_wide_path}")
print(f"Long candidate grid:      {grid_long_path}")
print(f"Candidates to evaluate:   {candidate_count:,}")
print(f"Selection universes:      {SELECTION_UNIVERSES}")
print(f"Primary metric:           {PRIMARY_RANKING_METRIC}")
print()

for i, candidate in grid_wide.iterrows():
    candidate_id = candidate["candidate_id"]

    qualified, pass_diag = apply_candidate_thresholds(signal, candidate)
    pass_diagnostic_rows.append(pass_diag)

    universes = select_trade_universes(qualified)

    for universe_name, selected in universes.items():
        if not selected.empty:
            selected = selected.copy()
            selected.insert(0, "selection_universe", universe_name)
            selected.insert(0, "candidate_id", candidate_id)

            # Attach the candidate's exact thresholds to selected rows for audit traceability.
            for bucket in ["Front", "Middle", "Back"]:
                selected[f"{bucket}_model_vrp_log_threshold"] = candidate[f"{bucket}_model_vrp_log"]
                selected[f"{bucket}_model_vrp_z_threshold"] = candidate[f"{bucket}_model_vrp_z"]
                selected[f"{bucket}_RSI14_max"] = candidate[f"{bucket}_RSI14_max"]
                selected[f"{bucket}_forecast_vol_pct_min"] = candidate[f"{bucket}_forecast_vol_pct_min"]

            selected_trade_frames.append(selected)

        overall_summary_rows.append(
            summarize_trade_set(
                selected,
                candidate_id=candidate_id,
                selection_universe=universe_name,
                group_cols=[],
                include_empty_overall=True,
            )
        )

        bucket_summary_rows.append(
            summarize_trade_set(
                selected,
                candidate_id=candidate_id,
                selection_universe=universe_name,
                group_cols=["tenor_bucket", "tenor_bucket_order"],
                include_empty_overall=False,
            )
        )

        tenor_summary_rows.append(
            summarize_trade_set(
                selected,
                candidate_id=candidate_id,
                selection_universe=universe_name,
                group_cols=["tenor", "tenor_bucket", "tenor_bucket_order"],
                include_empty_overall=False,
            )
        )

        if not selected.empty:
            selected_year = selected.copy()
            selected_year["year"] = selected_year["trade_date"].dt.year.astype(int)

            year_summary_rows.append(
                summarize_trade_set(
                    selected_year,
                    candidate_id=candidate_id,
                    selection_universe=universe_name,
                    group_cols=["year"],
                    include_empty_overall=False,
                )
            )

    if (i + 1) % 50 == 0 or (i + 1) == candidate_count:
        elapsed = time.time() - start_time
        print(f"  Evaluated {i + 1:,} / {candidate_count:,} candidates | elapsed {elapsed:,.1f}s")

# Combine.
summary_overall = pd.concat(overall_summary_rows, ignore_index=True) if overall_summary_rows else pd.DataFrame()
summary_by_bucket = pd.concat(bucket_summary_rows, ignore_index=True) if bucket_summary_rows else pd.DataFrame()
summary_by_tenor = pd.concat(tenor_summary_rows, ignore_index=True) if tenor_summary_rows else pd.DataFrame()
summary_by_year = pd.concat(year_summary_rows, ignore_index=True) if year_summary_rows else pd.DataFrame()
pass_diagnostics = pd.DataFrame(pass_diagnostic_rows)

selected_trade_panel = (
    pd.concat(selected_trade_frames, ignore_index=True)
    if selected_trade_frames
    else pd.DataFrame()
)

# Attach candidate thresholds to summary tables.
summary_overall = attach_candidate_parameters(summary_overall, grid_wide)
summary_by_bucket = attach_candidate_parameters(summary_by_bucket, grid_wide)
summary_by_tenor = attach_candidate_parameters(summary_by_tenor, grid_wide)
summary_by_year = attach_candidate_parameters(summary_by_year, grid_wide)
pass_diagnostics = attach_candidate_parameters(pass_diagnostics, grid_wide)

# Sort useful outputs.
if not summary_overall.empty:
    summary_overall = summary_overall.sort_values(
        ["selection_universe", PRIMARY_RANKING_METRIC, "profit_factor_pnl_per_day", "trades"],
        ascending=[True, False, False, False],
    )

if not summary_by_bucket.empty:
    summary_by_bucket = summary_by_bucket.sort_values(
        ["selection_universe", "candidate_id", "tenor_bucket_order"]
    )

if not summary_by_tenor.empty:
    summary_by_tenor = summary_by_tenor.sort_values(
        ["selection_universe", "candidate_id", "tenor"]
    )

if not summary_by_year.empty:
    summary_by_year = summary_by_year.sort_values(
        ["selection_universe", "candidate_id", "year"]
    )

elapsed_seconds = time.time() - start_time

# ======================================================================================
# 4. Save outputs
# ======================================================================================

candidate_summary_path = OUT_AUDIT_DIR / f"04_core_threshold_sweep_candidate_summary_{CELL6_TIMESTAMP}.csv"
by_bucket_path = OUT_AUDIT_DIR / f"04_core_threshold_sweep_by_bucket_{CELL6_TIMESTAMP}.csv"
by_tenor_path = OUT_AUDIT_DIR / f"04_core_threshold_sweep_by_tenor_{CELL6_TIMESTAMP}.csv"
by_year_path = OUT_AUDIT_DIR / f"04_core_threshold_sweep_by_year_{CELL6_TIMESTAMP}.csv"
pass_diagnostics_path = OUT_AUDIT_DIR / f"04_core_threshold_sweep_candidate_pass_diagnostics_{CELL6_TIMESTAMP}.csv"
selected_trade_panel_path = OUT_PROCESSED_DIR / f"04_core_threshold_sweep_selected_trade_panel_{CELL6_TIMESTAMP}.parquet"
validation_path = OUT_AUDIT_DIR / f"04_core_threshold_sweep_validation_{CELL6_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"04_core_threshold_sweep_manifest_{CELL6_TIMESTAMP}.json"

summary_overall.to_csv(candidate_summary_path, index=False)
summary_by_bucket.to_csv(by_bucket_path, index=False)
summary_by_tenor.to_csv(by_tenor_path, index=False)
summary_by_year.to_csv(by_year_path, index=False)
pass_diagnostics.to_csv(pass_diagnostics_path, index=False)

selected_trade_panel.to_parquet(selected_trade_panel_path, index=False)

# ======================================================================================
# 5. Validation
# ======================================================================================

validation_rows = []

def add_validation(check, status, detail):
    validation_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })

models_found = sorted(signal["model_spec"].dropna().unique().tolist())
candidate_ids = grid_wide["candidate_id"].dropna().tolist()
candidate_id_unique = len(candidate_ids) == len(set(candidate_ids))

expected_overall_rows = len(grid_wide) * len(SELECTION_UNIVERSES)
actual_overall_rows = len(summary_overall)

back_bad = grid_wide[
    (grid_wide["Back_model_vrp_log"] != LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"])
    | (grid_wide["Back_model_vrp_z"] != LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"])
    | (grid_wide["Back_RSI14_max"] != LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"])
    | (grid_wide["Back_forecast_vol_pct_min"] != LOCKED_BACK_CORE_THRESHOLDS["forecast_vol_pct_min"])
]

z_equality_bad = grid_wide[
    ("Front_model_vrp_z_3m" in grid_wide.columns)
    & ("Front_model_vrp_z_1y" in grid_wide.columns)
] if False else pd.DataFrame()

if {"Front_model_vrp_z_3m", "Front_model_vrp_z_1y", "Middle_model_vrp_z_3m", "Middle_model_vrp_z_1y", "Back_model_vrp_z_3m", "Back_model_vrp_z_1y"}.issubset(set(grid_wide.columns)):
    z_equality_bad = grid_wide[
        (grid_wide["Front_model_vrp_z_3m"] != grid_wide["Front_model_vrp_z_1y"])
        | (grid_wide["Middle_model_vrp_z_3m"] != grid_wide["Middle_model_vrp_z_1y"])
        | (grid_wide["Back_model_vrp_z_3m"] != grid_wide["Back_model_vrp_z_1y"])
    ]

bad_pnl_day = signal[
    signal["normalized_pnl_dollars"].notna()
    & signal["normalized_pnl_per_day"].notna()
    & (
        (
            signal["normalized_pnl_dollars"] / signal["tenor"].replace(0, np.nan)
        )
        - signal["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

selection_universes_found = sorted(summary_overall["selection_universe"].dropna().unique().tolist())

selected_trade_universes_found = (
    sorted(selected_trade_panel["selection_universe"].dropna().unique().tolist())
    if not selected_trade_panel.empty and "selection_universe" in selected_trade_panel.columns
    else []
)

add_validation(
    "signal_panel_loaded",
    "PASS" if len(signal) > 0 else "FAIL",
    f"rows={len(signal):,}; path={base_panel_path}",
)

add_validation(
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    "candidate_grid_loaded",
    "PASS" if len(grid_wide) > 0 else "FAIL",
    f"candidate_count={len(grid_wide):,}; path={grid_wide_path}",
)

add_validation(
    "candidate_id_unique",
    "PASS" if candidate_id_unique else "FAIL",
    f"candidate_ids={len(candidate_ids):,}; unique={len(set(candidate_ids)):,}",
)

add_validation(
    "back_thresholds_locked",
    "PASS" if back_bad.empty else "FAIL",
    f"bad_rows={len(back_bad):,}; locked_back={LOCKED_BACK_CORE_THRESHOLDS}",
)

add_validation(
    "z_3m_equals_z_1y_within_bucket",
    "PASS" if z_equality_bad.empty else "FAIL",
    f"bad_rows={len(z_equality_bad):,}",
)

add_validation(
    "normalized_pnl_per_day_formula",
    "PASS" if bad_pnl_day.empty else "FAIL",
    f"bad_rows={len(bad_pnl_day):,}; formula=normalized_pnl_dollars / tenor",
)

add_validation(
    "overall_summary_rows_complete",
    "PASS" if actual_overall_rows == expected_overall_rows else "FAIL",
    f"actual={actual_overall_rows:,}; expected={expected_overall_rows:,}",
)

add_validation(
    "all_selection_universes_evaluated",
    "PASS" if selection_universes_found == sorted(SELECTION_UNIVERSES) else "FAIL",
    f"found={selection_universes_found}",
)

add_validation(
    "selected_trade_panel_saved",
    "PASS" if len(selected_trade_panel) > 0 else "WARN",
    f"rows={len(selected_trade_panel):,}; universes={selected_trade_universes_found}",
)

add_validation(
    "candidate_pass_diagnostics_created",
    "PASS" if len(pass_diagnostics) == len(grid_wide) else "FAIL",
    f"rows={len(pass_diagnostics):,}; expected={len(grid_wide):,}",
)

add_validation(
    "no_final_candidate_selection",
    "PASS",
    "Cell 6 is raw sweep only. Final ranking/selection deferred to Cell 7.",
)

add_validation(
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    "no_sizing_changes",
    "PASS",
    "Threshold sweep only. No sizing fields changed.",
)

add_validation(
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"] == "FAIL"]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1.ipynb",
    "cell": "Cell 6 — Core threshold candidate sweep using P&L/day",
    "timestamp": CELL6_TIMESTAMP,
    "elapsed_seconds": elapsed_seconds,
    "locked_model_spec": LOCKED_MODEL_SPEC,
    "base_panel_path": str(base_panel_path),
    "grid_wide_path": str(grid_wide_path),
    "grid_long_path": str(grid_long_path),
    "candidate_count": int(len(grid_wide)),
    "selection_universes": SELECTION_UNIVERSES,
    "primary_ranking_metric": PRIMARY_RANKING_METRIC,
    "candidate_summary_path": str(candidate_summary_path),
    "by_bucket_path": str(by_bucket_path),
    "by_tenor_path": str(by_tenor_path),
    "by_year_path": str(by_year_path),
    "pass_diagnostics_path": str(pass_diagnostics_path),
    "selected_trade_panel_path": str(selected_trade_panel_path),
    "validation_path": str(validation_path),
    "notes": [
        "Raw sweep only.",
        "No final candidate selected.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
        "If top candidates cluster at parameter boundaries, expand the grid in a later cell.",
        "P&L/day is the primary cross-tenor quality metric.",
        "Best-rank DTE selection remains diagnostic only.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 6 validation failed. Review validation output above.")

# ======================================================================================
# 6. Display concise review
# ======================================================================================

print("=" * 100)
print("Cell 6 — Core threshold candidate sweep using P&L/day")
print("=" * 100)
print(f"Signal panel:                  {base_panel_path}")
print(f"Candidate grid:                {grid_wide_path}")
print(f"Candidates evaluated:          {len(grid_wide):,}")
print(f"Overall summary rows:          {len(summary_overall):,}")
print(f"By-bucket rows:                {len(summary_by_bucket):,}")
print(f"By-tenor rows:                 {len(summary_by_tenor):,}")
print(f"By-year rows:                  {len(summary_by_year):,}")
print(f"Selected trade panel rows:     {len(selected_trade_panel):,}")
print(f"Elapsed seconds:               {elapsed_seconds:,.1f}")
print(f"Primary ranking metric:        {PRIMARY_RANKING_METRIC}")
print("No final candidate selection:  True")
print("No Secondary:                  True")
print("No sizing changes:             True")
print("No production lock:            True")

print()
print("Validation:")
display(validation)

print()
print("Top 20 by avg P&L/day — one trade per bucket/date:")
top_bucket = summary_overall[
    summary_overall["selection_universe"].eq("one_trade_per_bucket_per_date_best_rank")
].sort_values(
    ["avg_pnl_per_day", "profit_factor_pnl_per_day", "trades"],
    ascending=[False, False, False],
).head(20)

top_display_cols = [
    "candidate_id",
    "selection_universe",
    "trades",
    "unique_trade_dates",
    "avg_tenor",
    "win_rate",
    "total_pnl",
    "profit_factor",
    "avg_pnl_per_day",
    "median_pnl_per_day",
    "worst_trade_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "trades_le_neg_100k",
    "pnl_2025",
    "avg_pnl_per_day_2025",
    "pnl_2026",
    "avg_pnl_per_day_2026",
    "Front_model_vrp_log",
    "Front_model_vrp_z",
    "Front_RSI14_max",
    "Front_forecast_vol_pct_min",
    "Middle_model_vrp_log",
    "Middle_model_vrp_z",
    "Middle_RSI14_max",
    "Middle_forecast_vol_pct_min",
    "Back_model_vrp_log",
    "Back_model_vrp_z",
]
display(top_bucket[[c for c in top_display_cols if c in top_bucket.columns]])

print()
print("Top 20 by avg P&L/day — one trade per date:")
top_date = summary_overall[
    summary_overall["selection_universe"].eq("one_trade_per_date_best_rank")
].sort_values(
    ["avg_pnl_per_day", "profit_factor_pnl_per_day", "trades"],
    ascending=[False, False, False],
).head(20)

display(top_date[[c for c in top_display_cols if c in top_date.columns]])

print()
print("Top 20 by avg P&L/day — all qualified trades:")
top_all = summary_overall[
    summary_overall["selection_universe"].eq("all_qualified_trades")
].sort_values(
    ["avg_pnl_per_day", "profit_factor_pnl_per_day", "trades"],
    ascending=[False, False, False],
).head(20)

display(top_all[[c for c in top_display_cols if c in top_all.columns]])

print()
print("30D anchor reference rows:")
anchor = summary_overall[
    summary_overall["selection_universe"].eq("thirty_day_anchor_only")
].sort_values("candidate_id").head(10)

display(anchor[[c for c in top_display_cols if c in anchor.columns]])

print()
print("Pass diagnostics sample:")
pass_diag_display_cols = [
    "candidate_id",
    "pass_rows_all_dates",
    "pass_rows_with_outcomes",
    "Front_pass_rows_all_dates",
    "Middle_pass_rows_all_dates",
    "Back_pass_rows_all_dates",
    "Front_model_vrp_log",
    "Front_model_vrp_z",
    "Front_RSI14_max",
    "Front_forecast_vol_pct_min",
    "Middle_model_vrp_log",
    "Middle_model_vrp_z",
    "Middle_RSI14_max",
    "Middle_forecast_vol_pct_min",
]
display(pass_diagnostics[[c for c in pass_diag_display_cols if c in pass_diagnostics.columns]].head(20))

print()
print("Saved outputs:")
for p in [
    candidate_summary_path,
    by_bucket_path,
    by_tenor_path,
    by_year_path,
    pass_diagnostics_path,
    selected_trade_panel_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 6 PASSED")

Cell 6 — Core threshold candidate sweep using P&L/day
Signal panel:             C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\01_unified_fds_signal_base_panel_20200102_20260701_20260704_211636.parquet
Wide candidate grid:      C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\03_core_threshold_candidate_grid_wide_20260704_213318.parquet
Long candidate grid:      C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\03_core_threshold_candidate_grid_long_20260704_213318.parquet
Candidates to evaluate:   375
Selection universes:      ['all_qualified_trades', 'one_trade_per_bucket_per_date_best_rank', 'one_trade_per_date_best_rank', 'thirty_day_anchor_only']
Primary metric:           avg_pnl_per_day

  Evaluated 50 / 375 candidates | elapsed 23.2s
  Evaluated 100 / 375 candidates | elapsed 45.6s
  Evaluated 150 / 375 candidates | elapsed 67.2s
  Evaluated 200 / 375 

,check,status,detail
0,signal_panel_loaded,PASS,"rows=14,688; path=C:\Users\patri\vrp_project\d..."
1,locked_model_only,PASS,models_found=['unified_fds_no_min_return']
2,candidate_grid_loaded,PASS,candidate_count=375; path=C:\Users\patri\vrp_p...
3,candidate_id_unique,PASS,candidate_ids=375; unique=375
4,back_thresholds_locked,PASS,"bad_rows=0; locked_back={'model_vrp_log': 0.7,..."
5,z_3m_equals_z_1y_within_bucket,PASS,bad_rows=0
6,normalized_pnl_per_day_formula,PASS,bad_rows=0; formula=normalized_pnl_dollars / t...
7,overall_summary_rows_complete,PASS,"actual=1,500; expected=1,500"
8,all_selection_universes_evaluated,PASS,"found=['all_qualified_trades', 'one_trade_per_..."
9,selected_trade_panel_saved,PASS,"rows=834,098; universes=['all_qualified_trades..."



Top 20 by avg P&L/day — one trade per bucket/date:


,candidate_id,selection_universe,trades,unique_trade_dates,avg_tenor,win_rate,total_pnl,profit_factor,avg_pnl_per_day,median_pnl_per_day,worst_trade_pnl_per_day,pnl_per_day_drawdown,worst_20_trade_pnl_per_day_sum,trades_le_neg_100k,pnl_2025,avg_pnl_per_day_2025,pnl_2026,avg_pnl_per_day_2026,Front_model_vrp_log,Front_model_vrp_z,Front_RSI14_max,Front_forecast_vol_pct_min,Middle_model_vrp_log,Middle_model_vrp_z,Middle_RSI14_max,Middle_forecast_vol_pct_min,Back_model_vrp_log,Back_model_vrp_z
1313,core_grid_00329,one_trade_per_bucket_per_date_best_rank,492,224,20.579268,0.839431,5.556302e+06,3.408855,478.215063,867.741336,-10376.349181,-54837.874446,-30554.551981,1,1.250250e+06,663.667312,-9141.293529,-154.274083,0.70,0.65,68.0,10.0,0.70,0.70,68.0,8.5,0.7,0.7
1317,core_grid_00330,one_trade_per_bucket_per_date_best_rank,488,224,20.581967,0.838115,5.515712e+06,3.391258,478.046934,873.480210,-10376.349181,-54837.874446,-30554.551981,1,1.209660e+06,671.521620,-9141.293529,-154.274083,0.70,0.65,68.0,10.0,0.70,0.70,68.0,10.0,0.7,0.7
1297,core_grid_00325,one_trade_per_bucket_per_date_best_rank,501,227,20.598802,0.840319,5.629937e+06,3.389343,477.255445,865.753788,-10376.349181,-54837.874446,-30554.551981,1,1.275860e+06,662.989656,-9141.293529,-154.274083,0.70,0.65,68.0,10.0,0.70,0.65,68.0,8.5,0.7,0.7
1321,core_grid_00331,one_trade_per_bucket_per_date_best_rank,484,224,20.559917,0.836777,5.461740e+06,3.367859,477.160908,879.606050,-10376.349181,-54837.874446,-30554.551981,1,1.198626e+06,675.033211,-38017.482311,-173.190990,0.70,0.65,68.0,10.0,0.70,0.70,66.0,8.5,0.7,0.7
1325,core_grid_00332,one_trade_per_bucket_per_date_best_rank,484,224,20.578512,0.836777,5.465876e+06,3.369652,477.154319,879.606050,-10376.349181,-54837.874446,-30554.551981,1,1.202761e+06,674.994787,-38017.482311,-173.190990,0.70,0.65,68.0,10.0,0.70,0.70,66.0,10.0,0.7,0.7
809,core_grid_00203,one_trade_per_bucket_per_date_best_rank,509,232,20.416503,0.838900,5.678573e+06,3.424063,476.858038,845.092627,-10376.349181,-54837.874446,-30554.551981,1,1.260338e+06,636.992355,8259.859683,-144.716233,0.65,0.65,68.0,10.0,0.65,0.70,68.0,8.5,0.7,0.7
813,core_grid_00204,one_trade_per_bucket_per_date_best_rank,505,232,20.417822,0.837624,5.637983e+06,3.406736,476.684820,851.275886,-10376.349181,-54837.874446,-30554.551981,1,1.219748e+06,643.277152,8259.859683,-144.716233,0.65,0.65,68.0,10.0,0.65,0.70,68.0,10.0,0.7,0.7
1301,core_grid_00326,one_trade_per_bucket_per_date_best_rank,496,226,20.600806,0.838710,5.574651e+06,3.365880,476.633571,870.479355,-10376.349181,-54837.874446,-30554.551981,1,1.235270e+06,670.629787,-9141.293529,-154.274083,0.70,0.65,68.0,10.0,0.70,0.65,68.0,10.0,0.7,0.7
793,core_grid_00199,one_trade_per_bucket_per_date_best_rank,518,235,20.438224,0.839768,5.752208e+06,3.404529,475.953491,843.286845,-10376.349181,-54837.874446,-30554.551981,1,1.285949e+06,636.911088,8259.859683,-144.716233,0.65,0.65,68.0,10.0,0.65,0.65,68.0,8.5,0.7,0.7
841,core_grid_00211,one_trade_per_bucket_per_date_best_rank,502,232,20.408367,0.838645,5.595748e+06,3.390586,475.946264,857.744702,-10376.349181,-54837.874446,-30554.551981,1,1.245589e+06,644.603693,-9141.293529,-154.274083,0.65,0.65,68.0,10.0,0.70,0.70,68.0,8.5,0.7,0.7



Top 20 by avg P&L/day — one trade per date:


...


30D anchor reference rows:


,candidate_id,selection_universe,trades,unique_trade_dates,avg_tenor,win_rate,total_pnl,profit_factor,avg_pnl_per_day,median_pnl_per_day,worst_trade_pnl_per_day,pnl_per_day_drawdown,worst_20_trade_pnl_per_day_sum,trades_le_neg_100k,pnl_2025,avg_pnl_per_day_2025,pnl_2026,avg_pnl_per_day_2026,Front_model_vrp_log,Front_model_vrp_z,Front_RSI14_max,Front_forecast_vol_pct_min,Middle_model_vrp_log,Middle_model_vrp_z,Middle_RSI14_max,Middle_forecast_vol_pct_min,Back_model_vrp_log,Back_model_vrp_z
3,core_grid_00001,thirty_day_anchor_only,149,149,30.0,0.912752,2.609075e+06,6.14471,583.685595,760.748549,-3455.697445,-7548.166806,-1363.370357,1,522043.942659,470.309858,206722.762055,255.213286,0.6,0.65,70.0,8.5,0.65,0.65,68.0,8.5,0.7,0.7
7,core_grid_00002,thirty_day_anchor_only,149,149,30.0,0.912752,2.609075e+06,6.14471,583.685595,760.748549,-3455.697445,-7548.166806,-1363.370357,1,522043.942659,470.309858,206722.762055,255.213286,0.6,0.65,70.0,8.5,0.65,0.65,66.0,8.5,0.7,0.7
11,core_grid_00003,thirty_day_anchor_only,149,149,30.0,0.912752,2.609075e+06,6.14471,583.685595,760.748549,-3455.697445,-7548.166806,-1363.370357,1,522043.942659,470.309858,206722.762055,255.213286,0.6,0.65,70.0,8.5,0.65,0.70,68.0,8.5,0.7,0.7
15,core_grid_00004,thirty_day_anchor_only,149,149,30.0,0.912752,2.609075e+06,6.14471,583.685595,760.748549,-3455.697445,-7548.166806,-1363.370357,1,522043.942659,470.309858,206722.762055,255.213286,0.6,0.65,70.0,8.5,0.65,0.70,66.0,8.5,0.7,0.7
19,core_grid_00005,thirty_day_anchor_only,149,149,30.0,0.912752,2.609075e+06,6.14471,583.685595,760.748549,-3455.697445,-7548.166806,-1363.370357,1,522043.942659,470.309858,206722.762055,255.213286,0.6,0.65,70.0,8.5,0.70,0.65,68.0,8.5,0.7,0.7
23,core_grid_00006,thirty_day_anchor_only,149,149,30.0,0.912752,2.609075e+06,6.14471,583.685595,760.748549,-3455.697445,-7548.166806,-1363.370357,1,522043.942659,470.309858,206722.762055,255.213286,0.6,0.65,70.0,8.5,0.70,0.65,66.0,8.5,0.7,0.7
27,core_grid_00007,thirty_day_anchor_only,149,149,30.0,0.912752,2.609075e+06,6.14471,583.685595,760.748549,-3455.697445,-7548.166806,-1363.370357,1,522043.942659,470.309858,206722.762055,255.213286,0.6,0.65,70.0,8.5,0.70,0.70,68.0,8.5,0.7,0.7
31,core_grid_00008,thirty_day_anchor_only,149,149,30.0,0.912752,2.609075e+06,6.14471,583.685595,760.748549,-3455.697445,-7548.166806,-1363.370357,1,522043.942659,470.309858,206722.762055,255.213286,0.6,0.65,70.0,8.5,0.70,0.70,66.0,8.5,0.7,0.7
35,core_grid_00009,thirty_day_anchor_only,149,149,30.0,0.912752,2.609075e+06,6.14471,583.685595,760.748549,-3455.697445,-7548.166806,-1363.370357,1,522043.942659,470.309858,206722.762055,255.213286,0.6,0.65,70.0,10.0,0.65,0.65,68.0,8.5,0.7,0.7
39,core_grid_00010,thirty_day_anchor_only,149,149,30.0,0.912752,2.609075e+06,6.14471,583.685595,760.748549,-3455.697445,-7548.166806,-1363.370357,1,522043.942659,470.309858,206722.762055,255.213286,0.6,0.65,70.0,10.0,0.65,0.65,68.0,10.0,0.7,0.7



Pass diagnostics sample:


,candidate_id,pass_rows_all_dates,pass_rows_with_outcomes,Front_pass_rows_all_dates,Middle_pass_rows_all_dates,Back_pass_rows_all_dates,Front_model_vrp_log,Front_model_vrp_z,Front_RSI14_max,Front_forecast_vol_pct_min,Middle_model_vrp_log,Middle_model_vrp_z,Middle_RSI14_max,Middle_forecast_vol_pct_min
0,core_grid_00001,1505,1505,592,464,449,0.6,0.65,70.0,8.5,0.65,0.65,68.0,8.5
1,core_grid_00002,1480,1480,592,439,449,0.6,0.65,70.0,8.5,0.65,0.65,66.0,8.5
2,core_grid_00003,1486,1486,592,445,449,0.6,0.65,70.0,8.5,0.65,0.70,68.0,8.5
3,core_grid_00004,1462,1462,592,421,449,0.6,0.65,70.0,8.5,0.65,0.70,66.0,8.5
4,core_grid_00005,1479,1479,592,438,449,0.6,0.65,70.0,8.5,0.70,0.65,68.0,8.5
5,core_grid_00006,1454,1454,592,413,449,0.6,0.65,70.0,8.5,0.70,0.65,66.0,8.5
6,core_grid_00007,1463,1463,592,422,449,0.6,0.65,70.0,8.5,0.70,0.70,68.0,8.5
7,core_grid_00008,1439,1439,592,398,449,0.6,0.65,70.0,8.5,0.70,0.70,66.0,8.5
8,core_grid_00009,1446,1446,533,464,449,0.6,0.65,70.0,10.0,0.65,0.65,68.0,8.5
9,core_grid_00010,1429,1429,533,447,449,0.6,0.65,70.0,10.0,0.65,0.65,68.0,10.0



Saved outputs:
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\04_core_threshold_sweep_candidate_summary_20260704_214008.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\04_core_threshold_sweep_by_bucket_20260704_214008.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\04_core_threshold_sweep_by_tenor_20260704_214008.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\04_core_threshold_sweep_by_year_20260704_214008.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\04_core_threshold_sweep_candidate_pass_diagnostics_20260704_214008.csv
  C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\04_core_threshold_sweep_selected_trade_panel_20260704_214008.parquet
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_r

In [9]:
# ======================================================================================
# Cell 7 — Front-focused expanded Core threshold grid design
# ======================================================================================
#
# Objective:
#   Create a second-pass expanded Core threshold grid focused on the Front bucket.
#
# Why:
#   Cell 6 showed that the first-pass grid improved P&L/day, but:
#     - 2026 remained weak.
#     - Top all-qualified candidates clustered near the Front forecast-vol-floor boundary.
#     - Front remains the weak bucket.
#
# This cell only designs and saves the expanded grid.
#
# Important:
#   This cell does NOT evaluate trades.
#   This cell does NOT run a threshold sweep.
#   This cell does NOT select final thresholds.
#
# Design choices:
#   - Back remains locked.
#   - Middle remains narrow.
#   - Front is expanded tighter.
#   - Within each bucket:
#         model_vrp_z_3m threshold == model_vrp_z_1y threshold
#   - Middle must sit between Front and Back for each parameter.
#   - Front is allowed to be tighter than Back for this expanded research grid.
#
# Not in scope:
#   - No trade evaluation.
#   - No final candidate ranking.
#   - No Secondary.
#   - No sizing changes.
#   - No production lock.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import itertools
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 360)

# ======================================================================================
# 0. Fallback config if this cell is run independently
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL7_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

if "BUCKET_ORDER_MAP" not in globals():
    BUCKET_ORDER_MAP = {"Front": 1, "Middle": 2, "Back": 3}

if "LOCKED_BACK_CORE_THRESHOLDS" not in globals():
    LOCKED_BACK_CORE_THRESHOLDS = {
        "model_vrp_log": 0.70,
        "model_vrp_z_3m": 0.70,
        "model_vrp_z_1y": 0.70,
        "RSI14_max": 70.0,
        "forecast_vol_pct_min": 8.5,
    }

EXPANDED_GRID_FAMILY = "core_front_expanded_back_locked"

# ======================================================================================
# 1. Helper functions
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def is_between_inclusive(x, a, b, tol=1e-12):
    lo = min(float(a), float(b)) - tol
    hi = max(float(a), float(b)) + tol
    return lo <= float(x) <= hi


def candidate_passes_design_constraints(row):
    """
    Design constraints only. No trade evaluation.

    Key difference versus Cell 5:
      - Front may be tighter than Back.
      - Middle must sit between Front and Back.
    """

    # Back fixed.
    back_fixed_ok = (
        row["Back_model_vrp_log"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"]
        and row["Back_model_vrp_z"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"]
        and row["Back_RSI14_max"] == LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"]
        and row["Back_forecast_vol_pct_min"] == LOCKED_BACK_CORE_THRESHOLDS["forecast_vol_pct_min"]
    )

    if not back_fixed_ok:
        return False

    # Within-bucket z equality is handled by construction, but require the bucket-level z
    # value to be a valid scalar.
    z_scalar_ok = (
        np.isfinite(row["Front_model_vrp_z"])
        and np.isfinite(row["Middle_model_vrp_z"])
        and np.isfinite(row["Back_model_vrp_z"])
    )

    if not z_scalar_ok:
        return False

    # Middle must sit between Front and Back.
    #
    # For log, z, and forecast-vol floor:
    #   Higher number is stricter.
    #
    # For RSI cap:
    #   Lower number is stricter.
    #
    # Numeric "between Front and Back" handles both cases.
    middle_between_log_ok = is_between_inclusive(
        row["Middle_model_vrp_log"],
        row["Front_model_vrp_log"],
        row["Back_model_vrp_log"],
    )

    middle_between_z_ok = is_between_inclusive(
        row["Middle_model_vrp_z"],
        row["Front_model_vrp_z"],
        row["Back_model_vrp_z"],
    )

    middle_between_rsi_ok = is_between_inclusive(
        row["Middle_RSI14_max"],
        row["Front_RSI14_max"],
        row["Back_RSI14_max"],
    )

    middle_between_fv_floor_ok = is_between_inclusive(
        row["Middle_forecast_vol_pct_min"],
        row["Front_forecast_vol_pct_min"],
        row["Back_forecast_vol_pct_min"],
    )

    return bool(
        middle_between_log_ok
        and middle_between_z_ok
        and middle_between_rsi_ok
        and middle_between_fv_floor_ok
    )


# ======================================================================================
# 2. Expanded candidate grid ranges
# ======================================================================================
#
# Front is expanded tighter.
# Middle stays narrow.
# Back stays locked.
#
# Note:
#   Front_model_vrp_log includes 0.75, which is tighter than locked Back 0.70.
#   This is intentional for Front-focused expansion.
# ======================================================================================

EXPANDED_GRID_RANGES = {
    "Front": {
        "model_vrp_log": [0.65, 0.70, 0.75],
        "model_vrp_z": [0.65, 0.70],
        "RSI14_max": [68.0, 66.0, 64.0],
        "forecast_vol_pct_min": [10.0, 11.5, 13.0, 14.5],
    },
    "Middle": {
        "model_vrp_log": [0.65, 0.70],
        "model_vrp_z": [0.70],
        "RSI14_max": [68.0, 66.0],
        "forecast_vol_pct_min": [8.5, 10.0],
    },
    "Back": {
        "model_vrp_log": [LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"]],
        "model_vrp_z": [LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"]],
        "RSI14_max": [LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"]],
        "forecast_vol_pct_min": [LOCKED_BACK_CORE_THRESHOLDS["forecast_vol_pct_min"]],
    },
}

# ======================================================================================
# 3. Optional: read first-pass sweep output for boundary-review context
# ======================================================================================

first_pass_summary_path = latest_file(
    OUT_AUDIT_DIR,
    "04_core_threshold_sweep_candidate_summary_*.csv",
    required=False,
)

first_pass_boundary_review = pd.DataFrame()

if first_pass_summary_path is not None:
    first_pass = pd.read_csv(first_pass_summary_path)

    first_pass_required_cols = [
        "candidate_id",
        "selection_universe",
        "trades",
        "avg_pnl_per_day",
        "profit_factor",
        "pnl_per_day_drawdown",
        "avg_pnl_per_day_2026",
        "Front_model_vrp_log",
        "Front_model_vrp_z",
        "Front_RSI14_max",
        "Front_forecast_vol_pct_min",
        "Middle_model_vrp_log",
        "Middle_model_vrp_z",
        "Middle_RSI14_max",
        "Middle_forecast_vol_pct_min",
    ]

    usable_cols = [c for c in first_pass_required_cols if c in first_pass.columns]

    review_frames = []

    for universe in [
        "all_qualified_trades",
        "one_trade_per_bucket_per_date_best_rank",
        "one_trade_per_date_best_rank",
    ]:
        u = first_pass[first_pass["selection_universe"].eq(universe)].copy()

        if not u.empty and "avg_pnl_per_day" in u.columns:
            review_frames.append(
                u.sort_values(
                    ["avg_pnl_per_day", "profit_factor", "trades"],
                    ascending=[False, False, False],
                )
                .head(25)
                .loc[:, usable_cols]
            )

    if review_frames:
        first_pass_boundary_review = pd.concat(review_frames, ignore_index=True)

# ======================================================================================
# 4. Build expanded wide candidate grid
# ======================================================================================

rows = []

product_iter = itertools.product(
    EXPANDED_GRID_RANGES["Front"]["model_vrp_log"],
    EXPANDED_GRID_RANGES["Front"]["model_vrp_z"],
    EXPANDED_GRID_RANGES["Front"]["RSI14_max"],
    EXPANDED_GRID_RANGES["Front"]["forecast_vol_pct_min"],

    EXPANDED_GRID_RANGES["Middle"]["model_vrp_log"],
    EXPANDED_GRID_RANGES["Middle"]["model_vrp_z"],
    EXPANDED_GRID_RANGES["Middle"]["RSI14_max"],
    EXPANDED_GRID_RANGES["Middle"]["forecast_vol_pct_min"],

    EXPANDED_GRID_RANGES["Back"]["model_vrp_log"],
    EXPANDED_GRID_RANGES["Back"]["model_vrp_z"],
    EXPANDED_GRID_RANGES["Back"]["RSI14_max"],
    EXPANDED_GRID_RANGES["Back"]["forecast_vol_pct_min"],
)

for vals in product_iter:
    (
        front_log,
        front_z,
        front_rsi,
        front_fv_floor,

        middle_log,
        middle_z,
        middle_rsi,
        middle_fv_floor,

        back_log,
        back_z,
        back_rsi,
        back_fv_floor,
    ) = vals

    row = {
        "Front_model_vrp_log": float(front_log),
        "Front_model_vrp_z": float(front_z),
        "Front_RSI14_max": float(front_rsi),
        "Front_forecast_vol_pct_min": float(front_fv_floor),

        "Middle_model_vrp_log": float(middle_log),
        "Middle_model_vrp_z": float(middle_z),
        "Middle_RSI14_max": float(middle_rsi),
        "Middle_forecast_vol_pct_min": float(middle_fv_floor),

        "Back_model_vrp_log": float(back_log),
        "Back_model_vrp_z": float(back_z),
        "Back_RSI14_max": float(back_rsi),
        "Back_forecast_vol_pct_min": float(back_fv_floor),
    }

    row["passes_design_constraints"] = candidate_passes_design_constraints(row)

    if row["passes_design_constraints"]:
        rows.append(row)

expanded_grid_wide = pd.DataFrame(rows).reset_index(drop=True)

if expanded_grid_wide.empty:
    raise RuntimeError("Expanded candidate grid is empty after applying design constraints.")

expanded_grid_wide.insert(
    0,
    "candidate_id",
    [f"core_front_expanded_{i:05d}" for i in range(1, len(expanded_grid_wide) + 1)],
)
expanded_grid_wide.insert(1, "candidate_family", EXPANDED_GRID_FAMILY)
expanded_grid_wide.insert(2, "locked_model_spec", LOCKED_MODEL_SPEC)

# Expand bucket-level z into explicit 3m and 1y thresholds.
for bucket in ["Front", "Middle", "Back"]:
    expanded_grid_wide[f"{bucket}_model_vrp_z_3m"] = expanded_grid_wide[f"{bucket}_model_vrp_z"]
    expanded_grid_wide[f"{bucket}_model_vrp_z_1y"] = expanded_grid_wide[f"{bucket}_model_vrp_z"]

# Diagnostics showing how Front compares to locked Back and Middle.
expanded_grid_wide["front_minus_back_log"] = (
    expanded_grid_wide["Front_model_vrp_log"] - expanded_grid_wide["Back_model_vrp_log"]
)
expanded_grid_wide["front_minus_back_z"] = (
    expanded_grid_wide["Front_model_vrp_z"] - expanded_grid_wide["Back_model_vrp_z"]
)
expanded_grid_wide["front_minus_back_RSI_cap"] = (
    expanded_grid_wide["Front_RSI14_max"] - expanded_grid_wide["Back_RSI14_max"]
)
expanded_grid_wide["front_minus_back_forecast_floor"] = (
    expanded_grid_wide["Front_forecast_vol_pct_min"] - expanded_grid_wide["Back_forecast_vol_pct_min"]
)

expanded_grid_wide["middle_between_front_back_log"] = expanded_grid_wide.apply(
    lambda r: is_between_inclusive(r["Middle_model_vrp_log"], r["Front_model_vrp_log"], r["Back_model_vrp_log"]),
    axis=1,
)
expanded_grid_wide["middle_between_front_back_z"] = expanded_grid_wide.apply(
    lambda r: is_between_inclusive(r["Middle_model_vrp_z"], r["Front_model_vrp_z"], r["Back_model_vrp_z"]),
    axis=1,
)
expanded_grid_wide["middle_between_front_back_RSI"] = expanded_grid_wide.apply(
    lambda r: is_between_inclusive(r["Middle_RSI14_max"], r["Front_RSI14_max"], r["Back_RSI14_max"]),
    axis=1,
)
expanded_grid_wide["middle_between_front_back_forecast_floor"] = expanded_grid_wide.apply(
    lambda r: is_between_inclusive(
        r["Middle_forecast_vol_pct_min"],
        r["Front_forecast_vol_pct_min"],
        r["Back_forecast_vol_pct_min"],
    ),
    axis=1,
)

expanded_grid_wide["front_log_tighter_than_back"] = (
    expanded_grid_wide["Front_model_vrp_log"] > expanded_grid_wide["Back_model_vrp_log"]
)
expanded_grid_wide["front_RSI_tighter_than_back"] = (
    expanded_grid_wide["Front_RSI14_max"] < expanded_grid_wide["Back_RSI14_max"]
)
expanded_grid_wide["front_forecast_floor_tighter_than_back"] = (
    expanded_grid_wide["Front_forecast_vol_pct_min"] > expanded_grid_wide["Back_forecast_vol_pct_min"]
)

# ======================================================================================
# 5. Build expanded long candidate grid
# ======================================================================================

long_rows = []

for _, row in expanded_grid_wide.iterrows():
    for bucket in ["Front", "Middle", "Back"]:
        long_rows.append({
            "candidate_id": row["candidate_id"],
            "candidate_family": row["candidate_family"],
            "locked_model_spec": row["locked_model_spec"],
            "tenor_bucket": bucket,
            "tenor_bucket_order": BUCKET_ORDER_MAP[bucket],
            "model_vrp_log": row[f"{bucket}_model_vrp_log"],
            "model_vrp_z": row[f"{bucket}_model_vrp_z"],
            "model_vrp_z_3m": row[f"{bucket}_model_vrp_z_3m"],
            "model_vrp_z_1y": row[f"{bucket}_model_vrp_z_1y"],
            "RSI14_max": row[f"{bucket}_RSI14_max"],
            "forecast_vol_pct_min": row[f"{bucket}_forecast_vol_pct_min"],
            "is_back_locked": bucket == "Back",
        })

expanded_grid_long = (
    pd.DataFrame(long_rows)
    .sort_values(["candidate_id", "tenor_bucket_order"])
    .reset_index(drop=True)
)

# ======================================================================================
# 6. Save outputs
# ======================================================================================

grid_wide_csv_path = OUT_AUDIT_DIR / f"05_core_front_expanded_candidate_grid_wide_{CELL7_TIMESTAMP}.csv"
grid_wide_parquet_path = OUT_PROCESSED_DIR / f"05_core_front_expanded_candidate_grid_wide_{CELL7_TIMESTAMP}.parquet"

grid_long_csv_path = OUT_AUDIT_DIR / f"05_core_front_expanded_candidate_grid_long_{CELL7_TIMESTAMP}.csv"
grid_long_parquet_path = OUT_PROCESSED_DIR / f"05_core_front_expanded_candidate_grid_long_{CELL7_TIMESTAMP}.parquet"

grid_ranges_path = OUT_AUDIT_DIR / f"05_core_front_expanded_candidate_grid_ranges_{CELL7_TIMESTAMP}.json"
first_pass_review_path = OUT_AUDIT_DIR / f"05_core_front_expanded_first_pass_boundary_review_{CELL7_TIMESTAMP}.csv"
validation_path = OUT_AUDIT_DIR / f"05_core_front_expanded_candidate_grid_validation_{CELL7_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"05_core_front_expanded_candidate_grid_manifest_{CELL7_TIMESTAMP}.json"

expanded_grid_wide.to_csv(grid_wide_csv_path, index=False)
expanded_grid_wide.to_parquet(grid_wide_parquet_path, index=False)

expanded_grid_long.to_csv(grid_long_csv_path, index=False)
expanded_grid_long.to_parquet(grid_long_parquet_path, index=False)

with open(grid_ranges_path, "w", encoding="utf-8") as f:
    json.dump(EXPANDED_GRID_RANGES, f, indent=2, default=str)

if not first_pass_boundary_review.empty:
    first_pass_boundary_review.to_csv(first_pass_review_path, index=False)
else:
    pd.DataFrame().to_csv(first_pass_review_path, index=False)

# ======================================================================================
# 7. Validation
# ======================================================================================

validation_rows = []

def add_validation(check, status, detail):
    validation_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })

# Candidate count.
add_validation(
    "expanded_candidate_grid_non_empty",
    "PASS" if len(expanded_grid_wide) > 0 else "FAIL",
    f"candidate_count={len(expanded_grid_wide):,}",
)

# Long grid row count.
add_validation(
    "long_grid_has_three_rows_per_candidate",
    "PASS" if len(expanded_grid_long) == 3 * len(expanded_grid_wide) else "FAIL",
    f"long_rows={len(expanded_grid_long):,}; expected={3 * len(expanded_grid_wide):,}",
)

# Back fixed.
back_bad = expanded_grid_wide[
    (expanded_grid_wide["Back_model_vrp_log"] != LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"])
    | (expanded_grid_wide["Back_model_vrp_z"] != LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"])
    | (expanded_grid_wide["Back_RSI14_max"] != LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"])
    | (expanded_grid_wide["Back_forecast_vol_pct_min"] != LOCKED_BACK_CORE_THRESHOLDS["forecast_vol_pct_min"])
]

add_validation(
    "back_thresholds_locked",
    "PASS" if back_bad.empty else "FAIL",
    f"bad_rows={len(back_bad):,}; locked_back={LOCKED_BACK_CORE_THRESHOLDS}",
)

# Z equality within bucket.
z_equality_bad = expanded_grid_wide[
    (expanded_grid_wide["Front_model_vrp_z_3m"] != expanded_grid_wide["Front_model_vrp_z_1y"])
    | (expanded_grid_wide["Middle_model_vrp_z_3m"] != expanded_grid_wide["Middle_model_vrp_z_1y"])
    | (expanded_grid_wide["Back_model_vrp_z_3m"] != expanded_grid_wide["Back_model_vrp_z_1y"])
]

add_validation(
    "z_3m_equals_z_1y_within_each_bucket",
    "PASS" if z_equality_bad.empty else "FAIL",
    f"bad_rows={len(z_equality_bad):,}",
)

# Middle between Front and Back.
middle_between_bad = expanded_grid_wide[
    ~(
        expanded_grid_wide["middle_between_front_back_log"]
        & expanded_grid_wide["middle_between_front_back_z"]
        & expanded_grid_wide["middle_between_front_back_RSI"]
        & expanded_grid_wide["middle_between_front_back_forecast_floor"]
    )
]

add_validation(
    "middle_between_front_and_back_all_parameters",
    "PASS" if middle_between_bad.empty else "FAIL",
    f"bad_rows={len(middle_between_bad):,}",
)

# Front-focused expansion should include some candidates tighter than Back.
front_log_tighter_count = int(expanded_grid_wide["front_log_tighter_than_back"].sum())
front_RSI_tighter_count = int(expanded_grid_wide["front_RSI_tighter_than_back"].sum())
front_fv_tighter_count = int(expanded_grid_wide["front_forecast_floor_tighter_than_back"].sum())

add_validation(
    "front_log_can_be_tighter_than_back",
    "PASS" if front_log_tighter_count > 0 else "WARN",
    f"front_log_tighter_candidates={front_log_tighter_count:,}",
)

add_validation(
    "front_RSI_can_be_tighter_than_back",
    "PASS" if front_RSI_tighter_count > 0 else "WARN",
    f"front_RSI_tighter_candidates={front_RSI_tighter_count:,}",
)

add_validation(
    "front_forecast_floor_can_be_tighter_than_back",
    "PASS" if front_fv_tighter_count > 0 else "WARN",
    f"front_forecast_floor_tighter_candidates={front_fv_tighter_count:,}",
)

# First-pass review loaded if available.
add_validation(
    "first_pass_sweep_summary_loaded_for_context",
    "PASS" if first_pass_summary_path is not None else "WARN",
    f"path={first_pass_summary_path}",
)

# No trade-evaluation metric columns in grid itself.
trade_metric_cols = [
    "trades",
    "win_rate",
    "total_pnl",
    "avg_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "worst_trade",
    "pnl_per_day_drawdown",
]

unexpected_trade_cols = [c for c in trade_metric_cols if c in expanded_grid_wide.columns]

add_validation(
    "no_trade_evaluation_performed",
    "PASS" if not unexpected_trade_cols else "FAIL",
    f"unexpected_trade_metric_cols={unexpected_trade_cols}",
)

add_validation(
    "no_secondary",
    "PASS",
    "Expanded grid is Core only.",
)

add_validation(
    "no_sizing_changes",
    "PASS",
    "Expanded grid contains thresholds only, no sizing fields.",
)

add_validation(
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"] == "FAIL"]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1.ipynb",
    "cell": "Cell 7 — Front-focused expanded Core threshold grid design",
    "timestamp": CELL7_TIMESTAMP,
    "locked_model_spec": LOCKED_MODEL_SPEC,
    "candidate_family": EXPANDED_GRID_FAMILY,
    "candidate_count": int(len(expanded_grid_wide)),
    "long_grid_rows": int(len(expanded_grid_long)),
    "grid_wide_csv_path": str(grid_wide_csv_path),
    "grid_wide_parquet_path": str(grid_wide_parquet_path),
    "grid_long_csv_path": str(grid_long_csv_path),
    "grid_long_parquet_path": str(grid_long_parquet_path),
    "grid_ranges_path": str(grid_ranges_path),
    "first_pass_summary_path": str(first_pass_summary_path) if first_pass_summary_path is not None else None,
    "first_pass_review_path": str(first_pass_review_path),
    "validation_path": str(validation_path),
    "expanded_grid_ranges": EXPANDED_GRID_RANGES,
    "locked_back_core_thresholds": LOCKED_BACK_CORE_THRESHOLDS,
    "hard_constraints": [
        "Back Core thresholds fixed.",
        "Within each bucket, model_vrp_z_3m threshold equals model_vrp_z_1y threshold.",
        "Middle must sit between Front and Back for model_vrp_log.",
        "Middle must sit between Front and Back for model_vrp_z.",
        "Middle must sit between Front and Back for RSI14_max.",
        "Middle must sit between Front and Back for forecast_vol_pct_min.",
        "Front may be tighter than Back in the expanded Front-focused grid.",
        "No trade evaluation performed.",
        "No threshold sweep performed.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
    ],
    "notes": [
        "This cell defines the expanded threshold candidates only.",
        "Cell 8 should sweep this expanded grid.",
        "P&L/day remains the primary cross-tenor trade-quality metric for Cell 8.",
        "If expanded winners still cluster at boundaries, do not lock; expand again or reconsider selection rules.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 7 validation failed. Review validation output above.")

# ======================================================================================
# 8. Display concise review
# ======================================================================================

print("=" * 100)
print("Cell 7 — Front-focused expanded Core threshold grid design")
print("=" * 100)
print(f"Locked model spec:                  {LOCKED_MODEL_SPEC}")
print(f"Candidate family:                   {EXPANDED_GRID_FAMILY}")
print(f"Expanded candidate count:           {len(expanded_grid_wide):,}")
print(f"Long grid rows:                     {len(expanded_grid_long):,}")
print(f"Back thresholds locked:             True")
print(f"Z 3m == Z 1y within bucket:         True")
print(f"Middle between Front and Back:      True")
print(f"Front can be tighter than Back:     True")
print(f"No trade evaluation:                True")
print(f"No threshold sweep:                 True")
print(f"No Secondary:                       True")
print(f"No sizing changes:                  True")
print(f"No production lock:                 True")

print()
print("Saved outputs:")
for p in [
    grid_wide_csv_path,
    grid_wide_parquet_path,
    grid_long_csv_path,
    grid_long_parquet_path,
    grid_ranges_path,
    first_pass_review_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print()
print("Validation:")
display(validation)

print()
print("Expanded grid ranges:")
display(
    pd.DataFrame(
        [
            {
                "bucket": bucket,
                "model_vrp_log": EXPANDED_GRID_RANGES[bucket]["model_vrp_log"],
                "model_vrp_z_applied_to_3m_and_1y": EXPANDED_GRID_RANGES[bucket]["model_vrp_z"],
                "RSI14_max": EXPANDED_GRID_RANGES[bucket]["RSI14_max"],
                "forecast_vol_pct_min": EXPANDED_GRID_RANGES[bucket]["forecast_vol_pct_min"],
            }
            for bucket in ["Front", "Middle", "Back"]
        ]
    )
)

print()
print("Candidate counts by Front parameters:")
display(
    expanded_grid_wide
    .groupby(
        [
            "Front_model_vrp_log",
            "Front_model_vrp_z",
            "Front_RSI14_max",
            "Front_forecast_vol_pct_min",
        ],
        as_index=False,
    )
    .agg(candidates=("candidate_id", "nunique"))
    .sort_values(
        [
            "Front_model_vrp_log",
            "Front_model_vrp_z",
            "Front_RSI14_max",
            "Front_forecast_vol_pct_min",
        ]
    )
)

print()
print("Candidate counts by Middle parameters:")
display(
    expanded_grid_wide
    .groupby(
        [
            "Middle_model_vrp_log",
            "Middle_model_vrp_z",
            "Middle_RSI14_max",
            "Middle_forecast_vol_pct_min",
        ],
        as_index=False,
    )
    .agg(candidates=("candidate_id", "nunique"))
    .sort_values(
        [
            "Middle_model_vrp_log",
            "Middle_model_vrp_z",
            "Middle_RSI14_max",
            "Middle_forecast_vol_pct_min",
        ]
    )
)

if not first_pass_boundary_review.empty:
    print()
    print("First-pass top-candidate review used as context:")
    display(first_pass_boundary_review.head(50))

print()
print("Expanded wide grid sample:")
display(expanded_grid_wide.head(25))

print()
print("Expanded long grid sample:")
display(expanded_grid_long.head(30))

print("\nCELL 7 PASSED")

Cell 7 — Front-focused expanded Core threshold grid design
Locked model spec:                  unified_fds_no_min_return
Candidate family:                   core_front_expanded_back_locked
Expanded candidate count:           320
Long grid rows:                     960
Back thresholds locked:             True
Z 3m == Z 1y within bucket:         True
Middle between Front and Back:      True
Front can be tighter than Back:     True
No trade evaluation:                True
No threshold sweep:                 True
No Secondary:                       True
No sizing changes:                  True
No production lock:                 True

Saved outputs:
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\05_core_front_expanded_candidate_grid_wide_20260704_215243.csv
  C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\05_core_front_expanded_candidate_grid_wide_20260704_215243.parquet
  C:\Users\patri\vrp_project\dat

,check,status,detail
0,expanded_candidate_grid_non_empty,PASS,candidate_count=320
1,long_grid_has_three_rows_per_candidate,PASS,long_rows=960; expected=960
2,back_thresholds_locked,PASS,"bad_rows=0; locked_back={'model_vrp_log': 0.7,..."
3,z_3m_equals_z_1y_within_each_bucket,PASS,bad_rows=0
4,middle_between_front_and_back_all_parameters,PASS,bad_rows=0
5,front_log_can_be_tighter_than_back,PASS,front_log_tighter_candidates=80
6,front_RSI_can_be_tighter_than_back,PASS,front_RSI_tighter_candidates=320
7,front_forecast_floor_can_be_tighter_than_back,PASS,front_forecast_floor_tighter_candidates=320
8,first_pass_sweep_summary_loaded_for_context,PASS,path=C:\Users\patri\vrp_project\data\audit\vrp...
9,no_trade_evaluation_performed,PASS,unexpected_trade_metric_cols=[]



Expanded grid ranges:


,bucket,model_vrp_log,model_vrp_z_applied_to_3m_and_1y,RSI14_max,forecast_vol_pct_min
0,Front,"[0.65, 0.7, 0.75]","[0.65, 0.7]","[68.0, 66.0, 64.0]","[10.0, 11.5, 13.0, 14.5]"
1,Middle,"[0.65, 0.7]",[0.7],"[68.0, 66.0]","[8.5, 10.0]"
2,Back,[0.7],[0.7],[70.0],[8.5]



Candidate counts by Front parameters:


...


Expanded wide grid sample:


,candidate_id,candidate_family,locked_model_spec,Front_model_vrp_log,Front_model_vrp_z,Front_RSI14_max,Front_forecast_vol_pct_min,Middle_model_vrp_log,Middle_model_vrp_z,Middle_RSI14_max,Middle_forecast_vol_pct_min,Back_model_vrp_log,Back_model_vrp_z,Back_RSI14_max,Back_forecast_vol_pct_min,passes_design_constraints,Front_model_vrp_z_3m,Front_model_vrp_z_1y,Middle_model_vrp_z_3m,Middle_model_vrp_z_1y,Back_model_vrp_z_3m,Back_model_vrp_z_1y,front_minus_back_log,front_minus_back_z,front_minus_back_RSI_cap,front_minus_back_forecast_floor,middle_between_front_back_log,middle_between_front_back_z,middle_between_front_back_RSI,middle_between_front_back_forecast_floor,front_log_tighter_than_back,front_RSI_tighter_than_back,front_forecast_floor_tighter_than_back
0,core_front_expanded_00001,core_front_expanded_back_locked,unified_fds_no_min_return,0.65,0.65,68.0,10.0,0.65,0.7,68.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.7,0.7,0.7,0.7,-0.05,-0.05,-2.0,1.5,True,True,True,True,False,True,True
1,core_front_expanded_00002,core_front_expanded_back_locked,unified_fds_no_min_return,0.65,0.65,68.0,10.0,0.65,0.7,68.0,10.0,0.7,0.7,70.0,8.5,True,0.65,0.65,0.7,0.7,0.7,0.7,-0.05,-0.05,-2.0,1.5,True,True,True,True,False,True,True
2,core_front_expanded_00003,core_front_expanded_back_locked,unified_fds_no_min_return,0.65,0.65,68.0,10.0,0.70,0.7,68.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.7,0.7,0.7,0.7,-0.05,-0.05,-2.0,1.5,True,True,True,True,False,True,True
3,core_front_expanded_00004,core_front_expanded_back_locked,unified_fds_no_min_return,0.65,0.65,68.0,10.0,0.70,0.7,68.0,10.0,0.7,0.7,70.0,8.5,True,0.65,0.65,0.7,0.7,0.7,0.7,-0.05,-0.05,-2.0,1.5,True,True,True,True,False,True,True
4,core_front_expanded_00005,core_front_expanded_back_locked,unified_fds_no_min_return,0.65,0.65,68.0,11.5,0.65,0.7,68.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.7,0.7,0.7,0.7,-0.05,-0.05,-2.0,3.0,True,True,True,True,False,True,True
5,core_front_expanded_00006,core_front_expanded_back_locked,unified_fds_no_min_return,0.65,0.65,68.0,11.5,0.65,0.7,68.0,10.0,0.7,0.7,70.0,8.5,True,0.65,0.65,0.7,0.7,0.7,0.7,-0.05,-0.05,-2.0,3.0,True,True,True,True,False,True,True
6,core_front_expanded_00007,core_front_expanded_back_locked,unified_fds_no_min_return,0.65,0.65,68.0,11.5,0.70,0.7,68.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.7,0.7,0.7,0.7,-0.05,-0.05,-2.0,3.0,True,True,True,True,False,True,True
7,core_front_expanded_00008,core_front_expanded_back_locked,unified_fds_no_min_return,0.65,0.65,68.0,11.5,0.70,0.7,68.0,10.0,0.7,0.7,70.0,8.5,True,0.65,0.65,0.7,0.7,0.7,0.7,-0.05,-0.05,-2.0,3.0,True,True,True,True,False,True,True
8,core_front_expanded_00009,core_front_expanded_back_locked,unified_fds_no_min_return,0.65,0.65,68.0,13.0,0.65,0.7,68.0,8.5,0.7,0.7,70.0,8.5,True,0.65,0.65,0.7,0.7,0.7,0.7,-0.05,-0.05,-2.0,4.5,True,True,True,True,False,True,True
9,core_front_expanded_00010,core_front_expanded_back_locked,unified_fds_no_min_return,0.65,0.65,68.0,13.0,0.65,0.7,68.0,10.0,0.7,0.7,70.0,8.5,True,0.65,0.65,0.7,0.7,0.7,0.7,-0.05,-0.05,-2.0,4.5,True,True,True,True,False,True,True



Expanded long grid sample:


,candidate_id,candidate_family,locked_model_spec,tenor_bucket,tenor_bucket_order,model_vrp_log,model_vrp_z,model_vrp_z_3m,model_vrp_z_1y,RSI14_max,forecast_vol_pct_min,is_back_locked
0,core_front_expanded_00001,core_front_expanded_back_locked,unified_fds_no_min_return,Front,1,0.65,0.65,0.65,0.65,68.0,10.0,False
1,core_front_expanded_00001,core_front_expanded_back_locked,unified_fds_no_min_return,Middle,2,0.65,0.70,0.70,0.70,68.0,8.5,False
2,core_front_expanded_00001,core_front_expanded_back_locked,unified_fds_no_min_return,Back,3,0.70,0.70,0.70,0.70,70.0,8.5,True
3,core_front_expanded_00002,core_front_expanded_back_locked,unified_fds_no_min_return,Front,1,0.65,0.65,0.65,0.65,68.0,10.0,False
4,core_front_expanded_00002,core_front_expanded_back_locked,unified_fds_no_min_return,Middle,2,0.65,0.70,0.70,0.70,68.0,10.0,False
5,core_front_expanded_00002,core_front_expanded_back_locked,unified_fds_no_min_return,Back,3,0.70,0.70,0.70,0.70,70.0,8.5,True
6,core_front_expanded_00003,core_front_expanded_back_locked,unified_fds_no_min_return,Front,1,0.65,0.65,0.65,0.65,68.0,10.0,False
7,core_front_expanded_00003,core_front_expanded_back_locked,unified_fds_no_min_return,Middle,2,0.70,0.70,0.70,0.70,68.0,8.5,False
8,core_front_expanded_00003,core_front_expanded_back_locked,unified_fds_no_min_return,Back,3,0.70,0.70,0.70,0.70,70.0,8.5,True
9,core_front_expanded_00004,core_front_expanded_back_locked,unified_fds_no_min_return,Front,1,0.65,0.65,0.65,0.65,68.0,10.0,False



CELL 7 PASSED


In [10]:
# ======================================================================================
# Cell 8 — Front-focused expanded Core threshold sweep using P&L/day
# ======================================================================================
#
# Objective:
#   Evaluate the Cell 7 Front-focused expanded Core threshold grid against the clean
#   locked signal panel.
#
# Inputs:
#   01_unified_fds_signal_base_panel_*.parquet
#   05_core_front_expanded_candidate_grid_wide_*.parquet
#   05_core_front_expanded_candidate_grid_long_*.parquet
#
# Outputs:
#   06_core_front_expanded_sweep_candidate_summary_*.csv
#   06_core_front_expanded_sweep_by_bucket_*.csv
#   06_core_front_expanded_sweep_by_tenor_*.csv
#   06_core_front_expanded_sweep_by_year_*.csv
#   06_core_front_expanded_sweep_candidate_pass_diagnostics_*.csv
#   06_core_front_expanded_sweep_boundary_diagnostics_*.csv
#   06_core_front_expanded_sweep_selected_trade_panel_*.parquet
#   06_core_front_expanded_sweep_validation_*.csv
#   06_core_front_expanded_sweep_manifest_*.json
#
# Selection universes evaluated:
#   1. all_qualified_trades
#   2. one_trade_per_bucket_per_date_best_rank
#   3. one_trade_per_date_best_rank
#   4. thirty_day_anchor_only
#
# Primary cross-tenor quality metric:
#   avg_pnl_per_day
#
# Important:
#   This is an expanded sweep only.
#   It does not choose a final threshold set.
#   It explicitly checks whether top candidates are still sitting at parameter boundaries.
#
# Not in scope:
#   - No final candidate selection.
#   - No Secondary.
#   - No sizing changes.
#   - No production lock.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 380)

# ======================================================================================
# 0. Fallback config if this cell is run independently
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL8_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

if "ALL_TENORS" not in globals():
    ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

if "BUCKET_ORDER_MAP" not in globals():
    BUCKET_ORDER_MAP = {"Front": 1, "Middle": 2, "Back": 3}

if "BUCKET_CENTER_TENOR" not in globals():
    BUCKET_CENTER_TENOR = {
        "Front": 12,
        "Middle": 21,
        "Back": 30,
    }

if "LOCKED_BACK_CORE_THRESHOLDS" not in globals():
    LOCKED_BACK_CORE_THRESHOLDS = {
        "model_vrp_log": 0.70,
        "model_vrp_z_3m": 0.70,
        "model_vrp_z_1y": 0.70,
        "RSI14_max": 70.0,
        "forecast_vol_pct_min": 8.5,
    }

SELECTION_UNIVERSES = [
    "all_qualified_trades",
    "one_trade_per_bucket_per_date_best_rank",
    "one_trade_per_date_best_rank",
    "thirty_day_anchor_only",
]

PRIMARY_RANKING_METRIC = "avg_pnl_per_day"
TOP_N_BOUNDARY_REVIEW = 50

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def profit_factor_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max

    return float(drawdown.min())


def worst_rolling_sum(pnl: pd.Series, window: int) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def add_selection_ranks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add diagnostic best-rank fields for selected-tenor views.

    This remains a diagnostic DTE selector only.
    """
    d = df.copy()

    if d.empty:
        return d

    d["bucket_center_tenor"] = d["tenor_bucket"].map(BUCKET_CENTER_TENOR)
    d["distance_to_bucket_center"] = (d["tenor"] - d["bucket_center_tenor"]).abs()

    # Within bucket/date.
    rank_group = ["trade_date", "tenor_bucket"]

    d["rank_z_1y_in_bucket_date"] = (
        d.groupby(rank_group)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_bucket_date"] = (
        d.groupby(rank_group)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_bucket_date"] = (
        d.groupby(rank_group)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_bucket_date"] = d[
        [
            "rank_z_1y_in_bucket_date",
            "rank_z_3m_in_bucket_date",
            "rank_vrp_log_in_bucket_date",
        ]
    ].mean(axis=1)

    # Across full date.
    d["rank_z_1y_in_date"] = (
        d.groupby("trade_date")["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_date"] = (
        d.groupby("trade_date")["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_date"] = (
        d.groupby("trade_date")["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_date"] = d[
        [
            "rank_z_1y_in_date",
            "rank_z_3m_in_date",
            "rank_vrp_log_in_date",
        ]
    ].mean(axis=1)

    return d


def select_trade_universes(qualified: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """
    Given qualified trades for one candidate, return all selection universes.
    """
    if qualified.empty:
        return {
            "all_qualified_trades": qualified.copy(),
            "one_trade_per_bucket_per_date_best_rank": qualified.copy(),
            "one_trade_per_date_best_rank": qualified.copy(),
            "thirty_day_anchor_only": qualified.copy(),
        }

    q = add_selection_ranks(qualified)

    one_trade_per_bucket_date = (
        q.sort_values(
            [
                "trade_date",
                "tenor_bucket_order",
                "avg_signal_rank_in_bucket_date",
                "distance_to_bucket_center",
                "rank_z_1y_in_bucket_date",
                "rank_z_3m_in_bucket_date",
                "rank_vrp_log_in_bucket_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date", "tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    one_trade_per_date = (
        q.sort_values(
            [
                "trade_date",
                "avg_signal_rank_in_date",
                "rank_z_1y_in_date",
                "rank_z_3m_in_date",
                "rank_vrp_log_in_date",
                "distance_to_bucket_center",
                "tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    thirty_day_anchor = q[q["tenor"] == 30].copy()

    return {
        "all_qualified_trades": q.copy(),
        "one_trade_per_bucket_per_date_best_rank": one_trade_per_bucket_date,
        "one_trade_per_date_best_rank": one_trade_per_date,
        "thirty_day_anchor_only": thirty_day_anchor,
    }


def summarize_trade_set(
    df: pd.DataFrame,
    candidate_id: str,
    selection_universe: str,
    group_cols: list[str] | None = None,
    include_empty_overall: bool = False,
) -> pd.DataFrame:
    """
    Summarize one selected trade set.

    For overall candidate-level summaries, use group_cols=[] and include_empty_overall=True.
    """
    if group_cols is None:
        group_cols = []

    rows = []
    d = df.copy()

    if not d.empty:
        d["normalized_pnl_dollars"] = pd.to_numeric(d["normalized_pnl_dollars"], errors="coerce")
        d["normalized_pnl_per_day"] = pd.to_numeric(d["normalized_pnl_per_day"], errors="coerce")
        d["win"] = pd.to_numeric(d["win"], errors="coerce")
        d = d[
            d["normalized_pnl_dollars"].notna()
            & d["normalized_pnl_per_day"].notna()
        ].copy()
        d = d.sort_values(["trade_date", "tenor"]).copy()

    if d.empty:
        if include_empty_overall and len(group_cols) == 0:
            return pd.DataFrame([{
                "candidate_id": candidate_id,
                "selection_universe": selection_universe,
                "trades": 0,
                "unique_trade_dates": 0,
                "date_min": pd.NaT,
                "date_max": pd.NaT,
                "avg_tenor": np.nan,
                "win_rate": np.nan,
                "total_pnl": 0.0,
                "avg_pnl": np.nan,
                "median_pnl": np.nan,
                "worst_trade": np.nan,
                "best_trade": np.nan,
                "profit_factor": np.nan,
                "selected_trade_drawdown": np.nan,
                "worst_5_trade_sum": np.nan,
                "worst_10_trade_sum": np.nan,
                "worst_20_trade_sum": np.nan,
                "trades_le_neg_50k": 0,
                "trades_le_neg_100k": 0,
                "trades_le_neg_150k": 0,
                "total_pnl_per_day_sum": 0.0,
                "avg_pnl_per_day": np.nan,
                "median_pnl_per_day": np.nan,
                "worst_trade_pnl_per_day": np.nan,
                "best_trade_pnl_per_day": np.nan,
                "profit_factor_pnl_per_day": np.nan,
                "pnl_per_day_drawdown": np.nan,
                "worst_5_trade_pnl_per_day_sum": np.nan,
                "worst_10_trade_pnl_per_day_sum": np.nan,
                "worst_20_trade_pnl_per_day_sum": np.nan,
                "trades_2025": 0,
                "pnl_2025": 0.0,
                "avg_pnl_per_day_2025": np.nan,
                "worst_trade_2025": np.nan,
                "trades_2026": 0,
                "pnl_2026": 0.0,
                "avg_pnl_per_day_2026": np.nan,
                "worst_trade_2026": np.nan,
            }])
        return pd.DataFrame()

    if len(group_cols) == 0:
        grouped = [((), d)]
    else:
        grouped = d.groupby(group_cols, dropna=False)

    for keys, g in grouped:
        if len(group_cols) == 0:
            keys = ()
        elif not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            "candidate_id": candidate_id,
            "selection_universe": selection_universe,
        }

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = g["normalized_pnl_dollars"]
        pnl_day = g["normalized_pnl_per_day"]
        wins = g["win"]

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": int(g["trade_date"].nunique()),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if g["tenor"].notna().any() else np.nan,

            # Total-dollar P&L.
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,
            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            # P&L/day — primary cross-tenor quality.
            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            # Recent year diagnostics.
            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        rows.append(row)

    return pd.DataFrame(rows)


def apply_candidate_thresholds(signal_df: pd.DataFrame, candidate: pd.Series) -> tuple[pd.DataFrame, dict]:
    """
    Apply one expanded-grid candidate's bucket thresholds to the signal panel.
    """
    d = signal_df.copy()

    required_inputs = (
        d["model_vrp_log"].notna()
        & d["model_vrp_z_3m"].notna()
        & d["model_vrp_z_1y"].notna()
        & d["RSI14"].notna()
        & d["forecast_vol_pct"].notna()
    )

    pass_mask = pd.Series(False, index=d.index)

    diagnostics = {
        "candidate_id": candidate["candidate_id"],
        "threshold_input_available_rows": int(required_inputs.sum()),
    }

    for bucket in ["Front", "Middle", "Back"]:
        bucket_mask = d["tenor_bucket"].eq(bucket)

        log_threshold = candidate[f"{bucket}_model_vrp_log"]
        z_threshold = candidate[f"{bucket}_model_vrp_z"]
        rsi_max = candidate[f"{bucket}_RSI14_max"]
        fv_floor = candidate[f"{bucket}_forecast_vol_pct_min"]

        bucket_pass = (
            required_inputs
            & bucket_mask
            & (d["model_vrp_log"] > log_threshold)
            & (d["model_vrp_z_3m"] > z_threshold)
            & (d["model_vrp_z_1y"] > z_threshold)
            & (d["RSI14"] < rsi_max)
            & (d["forecast_vol_pct"] > fv_floor)
        )

        pass_mask = pass_mask | bucket_pass

        diagnostics[f"{bucket}_pass_rows_all_dates"] = int(bucket_pass.sum())

    d["candidate_core_pass"] = pass_mask

    outcome_available = (
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    )

    qualified = d[
        d["candidate_core_pass"]
        & outcome_available
    ].copy()

    diagnostics["pass_rows_all_dates"] = int(pass_mask.sum())
    diagnostics["pass_rows_with_outcomes"] = int(len(qualified))
    diagnostics["trade_outcome_available_rows"] = int(outcome_available.sum())

    return qualified, diagnostics


def attach_candidate_parameters(summary: pd.DataFrame, grid_wide: pd.DataFrame) -> pd.DataFrame:
    if summary.empty:
        return summary

    param_cols = [
        c for c in grid_wide.columns
        if c == "candidate_id"
        or c.endswith("_model_vrp_log")
        or c.endswith("_model_vrp_z")
        or c.endswith("_RSI14_max")
        or c.endswith("_forecast_vol_pct_min")
        or c.startswith("front_minus_back_")
        or c.startswith("middle_between_front_back_")
        or c in [
            "front_log_tighter_than_back",
            "front_RSI_tighter_than_back",
            "front_forecast_floor_tighter_than_back",
            "candidate_family",
            "locked_model_spec",
        ]
    ]

    params = grid_wide[param_cols].copy()

    return summary.merge(params, on="candidate_id", how="left", validate="many_to_one")


def build_boundary_diagnostics(summary_overall: pd.DataFrame, grid_wide: pd.DataFrame, top_n: int = 50) -> pd.DataFrame:
    """
    Identify whether top candidates are sitting at parameter boundaries.

    This is diagnostic only; it does not select final thresholds.
    """
    if summary_overall.empty:
        return pd.DataFrame()

    boundary_specs = []

    for bucket in ["Front", "Middle", "Back"]:
        for metric in [
            "model_vrp_log",
            "model_vrp_z",
            "RSI14_max",
            "forecast_vol_pct_min",
        ]:
            col = f"{bucket}_{metric}"
            if col in grid_wide.columns:
                vals = sorted(pd.to_numeric(grid_wide[col], errors="coerce").dropna().unique().tolist())
                if vals:
                    boundary_specs.append({
                        "parameter": col,
                        "grid_min": min(vals),
                        "grid_max": max(vals),
                    })

    boundary_spec_df = pd.DataFrame(boundary_specs)

    rows = []

    for universe in sorted(summary_overall["selection_universe"].dropna().unique().tolist()):
        u = summary_overall[summary_overall["selection_universe"].eq(universe)].copy()

        if u.empty or PRIMARY_RANKING_METRIC not in u.columns:
            continue

        top = (
            u.sort_values(
                [PRIMARY_RANKING_METRIC, "profit_factor_pnl_per_day", "trades"],
                ascending=[False, False, False],
            )
            .head(top_n)
            .copy()
        )

        for _, spec in boundary_spec_df.iterrows():
            param = spec["parameter"]
            grid_min = spec["grid_min"]
            grid_max = spec["grid_max"]

            if param not in top.columns:
                continue

            vals = pd.to_numeric(top[param], errors="coerce")

            rows.append({
                "selection_universe": universe,
                "top_n": int(len(top)),
                "parameter": param,
                "grid_min": grid_min,
                "grid_max": grid_max,
                "top_min": float(vals.min()) if vals.notna().any() else np.nan,
                "top_max": float(vals.max()) if vals.notna().any() else np.nan,
                "top_mode": float(vals.mode().iloc[0]) if vals.notna().any() and not vals.mode().empty else np.nan,
                "share_top_at_grid_min": float((vals == grid_min).mean()) if len(vals) else np.nan,
                "share_top_at_grid_max": float((vals == grid_max).mean()) if len(vals) else np.nan,
                "count_top_at_grid_min": int((vals == grid_min).sum()) if len(vals) else 0,
                "count_top_at_grid_max": int((vals == grid_max).sum()) if len(vals) else 0,
                "boundary_warning": bool(
                    ((vals == grid_min).mean() >= 0.50)
                    or ((vals == grid_max).mean() >= 0.50)
                ) if len(vals) else False,
            })

    return pd.DataFrame(rows)


# ======================================================================================
# 2. Load signal panel and expanded candidate grid
# ======================================================================================

base_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01_unified_fds_signal_base_panel_*.parquet",
    required=True,
)

grid_wide_path = latest_file(
    OUT_PROCESSED_DIR,
    "05_core_front_expanded_candidate_grid_wide_*.parquet",
    required=True,
)

grid_long_path = latest_file(
    OUT_PROCESSED_DIR,
    "05_core_front_expanded_candidate_grid_long_*.parquet",
    required=True,
)

signal = pd.read_parquet(base_panel_path)
grid_wide = pd.read_parquet(grid_wide_path)
grid_long = pd.read_parquet(grid_long_path)

# Normalize dates / numeric fields.
signal["trade_date"] = pd.to_datetime(signal["trade_date"], errors="coerce").dt.normalize()
signal["tenor"] = pd.to_numeric(signal["tenor"], errors="coerce").astype(int)

if "last_forward_rv_date" in signal.columns:
    signal["last_forward_rv_date"] = pd.to_datetime(
        signal["last_forward_rv_date"],
        errors="coerce",
    ).dt.normalize()

required_signal_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

missing_signal_cols = [c for c in required_signal_cols if c not in signal.columns]
if missing_signal_cols:
    raise KeyError(f"Cell 8 missing required signal columns: {missing_signal_cols}")

required_grid_cols = [
    "candidate_id",
    "Front_model_vrp_log",
    "Front_model_vrp_z",
    "Front_RSI14_max",
    "Front_forecast_vol_pct_min",
    "Middle_model_vrp_log",
    "Middle_model_vrp_z",
    "Middle_RSI14_max",
    "Middle_forecast_vol_pct_min",
    "Back_model_vrp_log",
    "Back_model_vrp_z",
    "Back_RSI14_max",
    "Back_forecast_vol_pct_min",
]

missing_grid_cols = [c for c in required_grid_cols if c not in grid_wide.columns]
if missing_grid_cols:
    raise KeyError(f"Cell 8 missing required grid columns: {missing_grid_cols}")

# Ensure numeric fields.
for c in [
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    signal[c] = pd.to_numeric(signal[c], errors="coerce")

for c in [
    c for c in grid_wide.columns
    if c.endswith("_model_vrp_log")
    or c.endswith("_model_vrp_z")
    or c.endswith("_RSI14_max")
    or c.endswith("_forecast_vol_pct_min")
]:
    grid_wide[c] = pd.to_numeric(grid_wide[c], errors="coerce")

grid_wide = grid_wide.sort_values("candidate_id").reset_index(drop=True)

# ======================================================================================
# 3. Sweep expanded candidates
# ======================================================================================

start_time = time.time()

overall_summary_rows = []
bucket_summary_rows = []
tenor_summary_rows = []
year_summary_rows = []
pass_diagnostic_rows = []
selected_trade_frames = []

candidate_count = len(grid_wide)

print("=" * 100)
print("Cell 8 — Front-focused expanded Core threshold sweep using P&L/day")
print("=" * 100)
print(f"Signal panel:             {base_panel_path}")
print(f"Expanded wide grid:       {grid_wide_path}")
print(f"Expanded long grid:       {grid_long_path}")
print(f"Candidates to evaluate:   {candidate_count:,}")
print(f"Selection universes:      {SELECTION_UNIVERSES}")
print(f"Primary metric:           {PRIMARY_RANKING_METRIC}")
print()

for i, candidate in grid_wide.iterrows():
    candidate_id = candidate["candidate_id"]

    qualified, pass_diag = apply_candidate_thresholds(signal, candidate)
    pass_diagnostic_rows.append(pass_diag)

    universes = select_trade_universes(qualified)

    for universe_name, selected in universes.items():
        if not selected.empty:
            selected = selected.copy()
            selected.insert(0, "selection_universe", universe_name)
            selected.insert(0, "candidate_id", candidate_id)

            # Attach exact candidate thresholds for audit traceability.
            for bucket in ["Front", "Middle", "Back"]:
                selected[f"{bucket}_model_vrp_log_threshold"] = candidate[f"{bucket}_model_vrp_log"]
                selected[f"{bucket}_model_vrp_z_threshold"] = candidate[f"{bucket}_model_vrp_z"]
                selected[f"{bucket}_RSI14_max"] = candidate[f"{bucket}_RSI14_max"]
                selected[f"{bucket}_forecast_vol_pct_min"] = candidate[f"{bucket}_forecast_vol_pct_min"]

            selected_trade_frames.append(selected)

        overall_summary_rows.append(
            summarize_trade_set(
                selected,
                candidate_id=candidate_id,
                selection_universe=universe_name,
                group_cols=[],
                include_empty_overall=True,
            )
        )

        bucket_summary_rows.append(
            summarize_trade_set(
                selected,
                candidate_id=candidate_id,
                selection_universe=universe_name,
                group_cols=["tenor_bucket", "tenor_bucket_order"],
                include_empty_overall=False,
            )
        )

        tenor_summary_rows.append(
            summarize_trade_set(
                selected,
                candidate_id=candidate_id,
                selection_universe=universe_name,
                group_cols=["tenor", "tenor_bucket", "tenor_bucket_order"],
                include_empty_overall=False,
            )
        )

        if not selected.empty:
            selected_year = selected.copy()
            selected_year["year"] = selected_year["trade_date"].dt.year.astype(int)

            year_summary_rows.append(
                summarize_trade_set(
                    selected_year,
                    candidate_id=candidate_id,
                    selection_universe=universe_name,
                    group_cols=["year"],
                    include_empty_overall=False,
                )
            )

    if (i + 1) % 50 == 0 or (i + 1) == candidate_count:
        elapsed = time.time() - start_time
        print(f"  Evaluated {i + 1:,} / {candidate_count:,} candidates | elapsed {elapsed:,.1f}s")

# Combine.
summary_overall = pd.concat(overall_summary_rows, ignore_index=True) if overall_summary_rows else pd.DataFrame()
summary_by_bucket = pd.concat(bucket_summary_rows, ignore_index=True) if bucket_summary_rows else pd.DataFrame()
summary_by_tenor = pd.concat(tenor_summary_rows, ignore_index=True) if tenor_summary_rows else pd.DataFrame()
summary_by_year = pd.concat(year_summary_rows, ignore_index=True) if year_summary_rows else pd.DataFrame()
pass_diagnostics = pd.DataFrame(pass_diagnostic_rows)

selected_trade_panel = (
    pd.concat(selected_trade_frames, ignore_index=True)
    if selected_trade_frames
    else pd.DataFrame()
)

# Attach candidate thresholds and grid diagnostics.
summary_overall = attach_candidate_parameters(summary_overall, grid_wide)
summary_by_bucket = attach_candidate_parameters(summary_by_bucket, grid_wide)
summary_by_tenor = attach_candidate_parameters(summary_by_tenor, grid_wide)
summary_by_year = attach_candidate_parameters(summary_by_year, grid_wide)
pass_diagnostics = attach_candidate_parameters(pass_diagnostics, grid_wide)

# Boundary diagnostics after parameters are attached.
boundary_diagnostics = build_boundary_diagnostics(
    summary_overall=summary_overall,
    grid_wide=grid_wide,
    top_n=TOP_N_BOUNDARY_REVIEW,
)

# Sort useful outputs.
if not summary_overall.empty:
    summary_overall = summary_overall.sort_values(
        ["selection_universe", PRIMARY_RANKING_METRIC, "profit_factor_pnl_per_day", "trades"],
        ascending=[True, False, False, False],
    )

if not summary_by_bucket.empty:
    summary_by_bucket = summary_by_bucket.sort_values(
        ["selection_universe", "candidate_id", "tenor_bucket_order"]
    )

if not summary_by_tenor.empty:
    summary_by_tenor = summary_by_tenor.sort_values(
        ["selection_universe", "candidate_id", "tenor"]
    )

if not summary_by_year.empty:
    summary_by_year = summary_by_year.sort_values(
        ["selection_universe", "candidate_id", "year"]
    )

elapsed_seconds = time.time() - start_time

# ======================================================================================
# 4. Save outputs
# ======================================================================================

candidate_summary_path = OUT_AUDIT_DIR / f"06_core_front_expanded_sweep_candidate_summary_{CELL8_TIMESTAMP}.csv"
by_bucket_path = OUT_AUDIT_DIR / f"06_core_front_expanded_sweep_by_bucket_{CELL8_TIMESTAMP}.csv"
by_tenor_path = OUT_AUDIT_DIR / f"06_core_front_expanded_sweep_by_tenor_{CELL8_TIMESTAMP}.csv"
by_year_path = OUT_AUDIT_DIR / f"06_core_front_expanded_sweep_by_year_{CELL8_TIMESTAMP}.csv"
pass_diagnostics_path = OUT_AUDIT_DIR / f"06_core_front_expanded_sweep_candidate_pass_diagnostics_{CELL8_TIMESTAMP}.csv"
boundary_diagnostics_path = OUT_AUDIT_DIR / f"06_core_front_expanded_sweep_boundary_diagnostics_{CELL8_TIMESTAMP}.csv"
selected_trade_panel_path = OUT_PROCESSED_DIR / f"06_core_front_expanded_sweep_selected_trade_panel_{CELL8_TIMESTAMP}.parquet"
validation_path = OUT_AUDIT_DIR / f"06_core_front_expanded_sweep_validation_{CELL8_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"06_core_front_expanded_sweep_manifest_{CELL8_TIMESTAMP}.json"

summary_overall.to_csv(candidate_summary_path, index=False)
summary_by_bucket.to_csv(by_bucket_path, index=False)
summary_by_tenor.to_csv(by_tenor_path, index=False)
summary_by_year.to_csv(by_year_path, index=False)
pass_diagnostics.to_csv(pass_diagnostics_path, index=False)
boundary_diagnostics.to_csv(boundary_diagnostics_path, index=False)

selected_trade_panel.to_parquet(selected_trade_panel_path, index=False)

# ======================================================================================
# 5. Validation
# ======================================================================================

validation_rows = []

def add_validation(check, status, detail):
    validation_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })

models_found = sorted(signal["model_spec"].dropna().unique().tolist())
candidate_ids = grid_wide["candidate_id"].dropna().tolist()
candidate_id_unique = len(candidate_ids) == len(set(candidate_ids))

expected_overall_rows = len(grid_wide) * len(SELECTION_UNIVERSES)
actual_overall_rows = len(summary_overall)

back_bad = grid_wide[
    (grid_wide["Back_model_vrp_log"] != LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"])
    | (grid_wide["Back_model_vrp_z"] != LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"])
    | (grid_wide["Back_RSI14_max"] != LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"])
    | (grid_wide["Back_forecast_vol_pct_min"] != LOCKED_BACK_CORE_THRESHOLDS["forecast_vol_pct_min"])
]

z_equality_required_cols = {
    "Front_model_vrp_z_3m",
    "Front_model_vrp_z_1y",
    "Middle_model_vrp_z_3m",
    "Middle_model_vrp_z_1y",
    "Back_model_vrp_z_3m",
    "Back_model_vrp_z_1y",
}

if z_equality_required_cols.issubset(set(grid_wide.columns)):
    z_equality_bad = grid_wide[
        (grid_wide["Front_model_vrp_z_3m"] != grid_wide["Front_model_vrp_z_1y"])
        | (grid_wide["Middle_model_vrp_z_3m"] != grid_wide["Middle_model_vrp_z_1y"])
        | (grid_wide["Back_model_vrp_z_3m"] != grid_wide["Back_model_vrp_z_1y"])
    ]
else:
    z_equality_bad = pd.DataFrame({"missing_z_equality_cols": list(z_equality_required_cols - set(grid_wide.columns))})

bad_pnl_day = signal[
    signal["normalized_pnl_dollars"].notna()
    & signal["normalized_pnl_per_day"].notna()
    & (
        (
            signal["normalized_pnl_dollars"] / signal["tenor"].replace(0, np.nan)
        )
        - signal["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

selection_universes_found = sorted(summary_overall["selection_universe"].dropna().unique().tolist())

selected_trade_universes_found = (
    sorted(selected_trade_panel["selection_universe"].dropna().unique().tolist())
    if not selected_trade_panel.empty and "selection_universe" in selected_trade_panel.columns
    else []
)

boundary_warning_count = (
    int(boundary_diagnostics["boundary_warning"].sum())
    if not boundary_diagnostics.empty and "boundary_warning" in boundary_diagnostics.columns
    else 0
)

add_validation(
    "signal_panel_loaded",
    "PASS" if len(signal) > 0 else "FAIL",
    f"rows={len(signal):,}; path={base_panel_path}",
)

add_validation(
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    "expanded_candidate_grid_loaded",
    "PASS" if len(grid_wide) > 0 else "FAIL",
    f"candidate_count={len(grid_wide):,}; path={grid_wide_path}",
)

add_validation(
    "candidate_id_unique",
    "PASS" if candidate_id_unique else "FAIL",
    f"candidate_ids={len(candidate_ids):,}; unique={len(set(candidate_ids)):,}",
)

add_validation(
    "back_thresholds_locked",
    "PASS" if back_bad.empty else "FAIL",
    f"bad_rows={len(back_bad):,}; locked_back={LOCKED_BACK_CORE_THRESHOLDS}",
)

add_validation(
    "z_3m_equals_z_1y_within_bucket",
    "PASS" if z_equality_bad.empty else "FAIL",
    f"bad_rows={len(z_equality_bad):,}",
)

add_validation(
    "normalized_pnl_per_day_formula",
    "PASS" if bad_pnl_day.empty else "FAIL",
    f"bad_rows={len(bad_pnl_day):,}; formula=normalized_pnl_dollars / tenor",
)

add_validation(
    "overall_summary_rows_complete",
    "PASS" if actual_overall_rows == expected_overall_rows else "FAIL",
    f"actual={actual_overall_rows:,}; expected={expected_overall_rows:,}",
)

add_validation(
    "all_selection_universes_evaluated",
    "PASS" if selection_universes_found == sorted(SELECTION_UNIVERSES) else "FAIL",
    f"found={selection_universes_found}",
)

add_validation(
    "selected_trade_panel_saved",
    "PASS" if len(selected_trade_panel) > 0 else "WARN",
    f"rows={len(selected_trade_panel):,}; universes={selected_trade_universes_found}",
)

add_validation(
    "candidate_pass_diagnostics_created",
    "PASS" if len(pass_diagnostics) == len(grid_wide) else "FAIL",
    f"rows={len(pass_diagnostics):,}; expected={len(grid_wide):,}",
)

add_validation(
    "boundary_diagnostics_created",
    "PASS" if len(boundary_diagnostics) > 0 else "WARN",
    f"rows={len(boundary_diagnostics):,}; warnings={boundary_warning_count:,}",
)

add_validation(
    "no_final_candidate_selection",
    "PASS",
    "Cell 8 is expanded sweep only. Final ranking/selection deferred.",
)

add_validation(
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    "no_sizing_changes",
    "PASS",
    "Threshold sweep only. No sizing fields changed.",
)

add_validation(
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"] == "FAIL"]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1.ipynb",
    "cell": "Cell 8 — Front-focused expanded Core threshold sweep using P&L/day",
    "timestamp": CELL8_TIMESTAMP,
    "elapsed_seconds": elapsed_seconds,
    "locked_model_spec": LOCKED_MODEL_SPEC,
    "base_panel_path": str(base_panel_path),
    "grid_wide_path": str(grid_wide_path),
    "grid_long_path": str(grid_long_path),
    "candidate_count": int(len(grid_wide)),
    "selection_universes": SELECTION_UNIVERSES,
    "primary_ranking_metric": PRIMARY_RANKING_METRIC,
    "top_n_boundary_review": TOP_N_BOUNDARY_REVIEW,
    "candidate_summary_path": str(candidate_summary_path),
    "by_bucket_path": str(by_bucket_path),
    "by_tenor_path": str(by_tenor_path),
    "by_year_path": str(by_year_path),
    "pass_diagnostics_path": str(pass_diagnostics_path),
    "boundary_diagnostics_path": str(boundary_diagnostics_path),
    "selected_trade_panel_path": str(selected_trade_panel_path),
    "validation_path": str(validation_path),
    "boundary_warning_count": boundary_warning_count,
    "notes": [
        "Expanded sweep only.",
        "No final candidate selected.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
        "If top candidates cluster at parameter boundaries, do not lock; expand again or reconsider selection rules.",
        "P&L/day is the primary cross-tenor quality metric.",
        "Best-rank DTE selection remains diagnostic only.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 8 validation failed. Review validation output above.")

# ======================================================================================
# 6. Display concise review
# ======================================================================================

print("=" * 100)
print("Cell 8 — Front-focused expanded Core threshold sweep using P&L/day")
print("=" * 100)
print(f"Signal panel:                  {base_panel_path}")
print(f"Expanded candidate grid:       {grid_wide_path}")
print(f"Candidates evaluated:          {len(grid_wide):,}")
print(f"Overall summary rows:          {len(summary_overall):,}")
print(f"By-bucket rows:                {len(summary_by_bucket):,}")
print(f"By-tenor rows:                 {len(summary_by_tenor):,}")
print(f"By-year rows:                  {len(summary_by_year):,}")
print(f"Pass diagnostic rows:          {len(pass_diagnostics):,}")
print(f"Boundary diagnostic rows:      {len(boundary_diagnostics):,}")
print(f"Boundary warnings:             {boundary_warning_count:,}")
print(f"Selected trade panel rows:     {len(selected_trade_panel):,}")
print(f"Elapsed seconds:               {elapsed_seconds:,.1f}")
print(f"Primary ranking metric:        {PRIMARY_RANKING_METRIC}")
print("No final candidate selection:  True")
print("No Secondary:                  True")
print("No sizing changes:             True")
print("No production lock:            True")

print()
print("Validation:")
display(validation)

top_display_cols = [
    "candidate_id",
    "selection_universe",
    "trades",
    "unique_trade_dates",
    "avg_tenor",
    "win_rate",
    "total_pnl",
    "profit_factor",
    "avg_pnl_per_day",
    "median_pnl_per_day",
    "worst_trade_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "trades_le_neg_100k",
    "pnl_2025",
    "avg_pnl_per_day_2025",
    "pnl_2026",
    "avg_pnl_per_day_2026",
    "Front_model_vrp_log",
    "Front_model_vrp_z",
    "Front_RSI14_max",
    "Front_forecast_vol_pct_min",
    "Middle_model_vrp_log",
    "Middle_model_vrp_z",
    "Middle_RSI14_max",
    "Middle_forecast_vol_pct_min",
    "Back_model_vrp_log",
    "Back_model_vrp_z",
    "front_log_tighter_than_back",
    "front_RSI_tighter_than_back",
    "front_forecast_floor_tighter_than_back",
]

print()
print("Top 25 by avg P&L/day — one trade per bucket/date:")
top_bucket = summary_overall[
    summary_overall["selection_universe"].eq("one_trade_per_bucket_per_date_best_rank")
].sort_values(
    ["avg_pnl_per_day", "profit_factor_pnl_per_day", "trades"],
    ascending=[False, False, False],
).head(25)

display(top_bucket[[c for c in top_display_cols if c in top_bucket.columns]])

print()
print("Top 25 by avg P&L/day — one trade per date:")
top_date = summary_overall[
    summary_overall["selection_universe"].eq("one_trade_per_date_best_rank")
].sort_values(
    ["avg_pnl_per_day", "profit_factor_pnl_per_day", "trades"],
    ascending=[False, False, False],
).head(25)

display(top_date[[c for c in top_display_cols if c in top_date.columns]])

print()
print("Top 25 by avg P&L/day — all qualified trades:")
top_all = summary_overall[
    summary_overall["selection_universe"].eq("all_qualified_trades")
].sort_values(
    ["avg_pnl_per_day", "profit_factor_pnl_per_day", "trades"],
    ascending=[False, False, False],
).head(25)

display(top_all[[c for c in top_display_cols if c in top_all.columns]])

print()
print("30D anchor reference rows:")
anchor = summary_overall[
    summary_overall["selection_universe"].eq("thirty_day_anchor_only")
].sort_values("candidate_id").head(10)

display(anchor[[c for c in top_display_cols if c in anchor.columns]])

print()
print("Boundary diagnostics — parameters where >=50% of top candidates sit at min or max:")
if not boundary_diagnostics.empty:
    display(
        boundary_diagnostics[
            boundary_diagnostics["boundary_warning"].eq(True)
        ].sort_values(
            ["selection_universe", "parameter"]
        )
    )
else:
    print("No boundary diagnostics available.")

print()
print("Pass diagnostics sample:")
pass_diag_display_cols = [
    "candidate_id",
    "pass_rows_all_dates",
    "pass_rows_with_outcomes",
    "Front_pass_rows_all_dates",
    "Middle_pass_rows_all_dates",
    "Back_pass_rows_all_dates",
    "Front_model_vrp_log",
    "Front_model_vrp_z",
    "Front_RSI14_max",
    "Front_forecast_vol_pct_min",
    "Middle_model_vrp_log",
    "Middle_model_vrp_z",
    "Middle_RSI14_max",
    "Middle_forecast_vol_pct_min",
]
display(pass_diagnostics[[c for c in pass_diag_display_cols if c in pass_diagnostics.columns]].head(25))

print()
print("Saved outputs:")
for p in [
    candidate_summary_path,
    by_bucket_path,
    by_tenor_path,
    by_year_path,
    pass_diagnostics_path,
    boundary_diagnostics_path,
    selected_trade_panel_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 8 PASSED")

Cell 8 — Front-focused expanded Core threshold sweep using P&L/day
Signal panel:             C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\01_unified_fds_signal_base_panel_20200102_20260701_20260704_211636.parquet
Expanded wide grid:       C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\05_core_front_expanded_candidate_grid_wide_20260704_215243.parquet
Expanded long grid:       C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\05_core_front_expanded_candidate_grid_long_20260704_215243.parquet
Candidates to evaluate:   320
Selection universes:      ['all_qualified_trades', 'one_trade_per_bucket_per_date_best_rank', 'one_trade_per_date_best_rank', 'thirty_day_anchor_only']
Primary metric:           avg_pnl_per_day

  Evaluated 50 / 320 candidates | elapsed 21.8s
  Evaluated 100 / 320 candidates | elapsed 43.4s
  Evaluated 150 / 320 candidates | elapsed 65.1s

,check,status,detail
0,signal_panel_loaded,PASS,"rows=14,688; path=C:\Users\patri\vrp_project\d..."
1,locked_model_only,PASS,models_found=['unified_fds_no_min_return']
2,expanded_candidate_grid_loaded,PASS,candidate_count=320; path=C:\Users\patri\vrp_p...
3,candidate_id_unique,PASS,candidate_ids=320; unique=320
4,back_thresholds_locked,PASS,"bad_rows=0; locked_back={'model_vrp_log': 0.7,..."
5,z_3m_equals_z_1y_within_bucket,PASS,bad_rows=0
6,normalized_pnl_per_day_formula,PASS,bad_rows=0; formula=normalized_pnl_dollars / t...
7,overall_summary_rows_complete,PASS,"actual=1,280; expected=1,280"
8,all_selection_universes_evaluated,PASS,"found=['all_qualified_trades', 'one_trade_per_..."
9,selected_trade_panel_saved,PASS,"rows=659,356; universes=['all_qualified_trades..."



Top 25 by avg P&L/day — one trade per bucket/date:


,candidate_id,selection_universe,trades,unique_trade_dates,avg_tenor,win_rate,total_pnl,profit_factor,avg_pnl_per_day,median_pnl_per_day,worst_trade_pnl_per_day,pnl_per_day_drawdown,worst_20_trade_pnl_per_day_sum,trades_le_neg_100k,pnl_2025,avg_pnl_per_day_2025,pnl_2026,avg_pnl_per_day_2026,Front_model_vrp_log,Front_model_vrp_z,Front_RSI14_max,Front_forecast_vol_pct_min,Middle_model_vrp_log,Middle_model_vrp_z,Middle_RSI14_max,Middle_forecast_vol_pct_min,Back_model_vrp_log,Back_model_vrp_z,front_log_tighter_than_back,front_RSI_tighter_than_back,front_forecast_floor_tighter_than_back
669,core_front_expanded_00168,one_trade_per_bucket_per_date_best_rank,428,197,21.995327,0.855140,5.479290e+06,3.881266,547.859866,885.630340,-10376.349181,-30723.516018,-24889.323006,2,1.139874e+06,661.947436,204439.021480,77.113098,0.70,0.65,68.0,14.5,0.70,0.7,68.0,10.0,0.7,0.7,False,True,True
725,core_front_expanded_00182,one_trade_per_bucket_per_date_best_rank,428,197,21.995327,0.855140,5.479290e+06,3.881266,547.859866,885.630340,-10376.349181,-30723.516018,-24889.323006,2,1.139874e+06,661.947436,204439.021480,77.113098,0.70,0.65,66.0,14.5,0.70,0.7,68.0,10.0,0.7,0.7,False,True,True
789,core_front_expanded_00198,one_trade_per_bucket_per_date_best_rank,428,197,21.995327,0.855140,5.479290e+06,3.881266,547.859866,885.630340,-10376.349181,-30723.516018,-24889.323006,2,1.139874e+06,661.947436,204439.021480,77.113098,0.70,0.65,64.0,14.5,0.70,0.7,68.0,10.0,0.7,0.7,False,True,True
729,core_front_expanded_00183,one_trade_per_bucket_per_date_best_rank,424,196,21.983491,0.853774,5.425318e+06,3.852885,547.507072,890.768149,-10376.349181,-30723.516018,-24889.323006,2,1.128840e+06,665.656487,175562.832699,60.157120,0.70,0.65,66.0,14.5,0.70,0.7,66.0,8.5,0.7,0.7,False,True,True
793,core_front_expanded_00199,one_trade_per_bucket_per_date_best_rank,424,196,21.983491,0.853774,5.425318e+06,3.852885,547.507072,890.768149,-10376.349181,-30723.516018,-24889.323006,2,1.128840e+06,665.656487,175562.832699,60.157120,0.70,0.65,64.0,14.5,0.70,0.7,66.0,8.5,0.7,0.7,False,True,True
733,core_front_expanded_00184,one_trade_per_bucket_per_date_best_rank,424,196,22.004717,0.853774,5.429453e+06,3.855060,547.499550,890.768149,-10376.349181,-30723.516018,-24889.323006,2,1.132976e+06,665.614524,175562.832699,60.157120,0.70,0.65,66.0,14.5,0.70,0.7,66.0,10.0,0.7,0.7,False,True,True
797,core_front_expanded_00200,one_trade_per_bucket_per_date_best_rank,424,196,22.004717,0.853774,5.429453e+06,3.855060,547.499550,890.768149,-10376.349181,-30723.516018,-24889.323006,2,1.132976e+06,665.614524,175562.832699,60.157120,0.70,0.65,64.0,14.5,0.70,0.7,66.0,10.0,0.7,0.7,False,True,True
665,core_front_expanded_00167,one_trade_per_bucket_per_date_best_rank,432,197,21.979167,0.856481,5.519880e+06,3.902610,547.404931,879.606050,-10376.349181,-30723.516018,-24889.323006,2,1.180464e+06,653.887160,204439.021480,77.113098,0.70,0.65,68.0,14.5,0.70,0.7,68.0,8.5,0.7,0.7,False,True,True
721,core_front_expanded_00181,one_trade_per_bucket_per_date_best_rank,432,197,21.979167,0.856481,5.519880e+06,3.902610,547.404931,879.606050,-10376.349181,-30723.516018,-24889.323006,2,1.180464e+06,653.887160,204439.021480,77.113098,0.70,0.65,66.0,14.5,0.70,0.7,68.0,8.5,0.7,0.7,False,True,True
785,core_front_expanded_00197,one_trade_per_bucket_per_date_best_rank,432,197,21.979167,0.856481,5.519880e+06,3.902610,547.404931,879.606050,-10376.349181,-30723.516018,-24889.323006,2,1.180464e+06,653.887160,204439.021480,77.113098,0.70,0.65,64.0,14.5,0.70,0.7,68.0,8.5,0.7,0.7,False,True,True



Top 25 by avg P&L/day — one trade per date:


...


Boundary diagnostics — parameters where >=50% of top candidates sit at min or max:


,selection_universe,top_n,parameter,grid_min,grid_max,top_min,top_max,top_mode,share_top_at_grid_min,share_top_at_grid_max,count_top_at_grid_min,count_top_at_grid_max,boundary_warning
10,all_qualified_trades,50,Back_RSI14_max,70.00,70.00,70.00,70.00,70.00,1.00,1.00,50,50,True
11,all_qualified_trades,50,Back_forecast_vol_pct_min,8.50,8.50,8.50,8.50,8.50,1.00,1.00,50,50,True
8,all_qualified_trades,50,Back_model_vrp_log,0.70,0.70,0.70,0.70,0.70,1.00,1.00,50,50,True
9,all_qualified_trades,50,Back_model_vrp_z,0.70,0.70,0.70,0.70,0.70,1.00,1.00,50,50,True
3,all_qualified_trades,50,Front_forecast_vol_pct_min,10.00,14.50,14.50,14.50,14.50,0.00,1.00,0,50,True
1,all_qualified_trades,50,Front_model_vrp_z,0.65,0.70,0.65,0.70,0.65,0.62,0.38,31,19,True
6,all_qualified_trades,50,Middle_RSI14_max,66.00,68.00,66.00,68.00,66.00,0.64,0.36,32,18,True
7,all_qualified_trades,50,Middle_forecast_vol_pct_min,8.50,10.00,8.50,10.00,10.00,0.38,0.62,19,31,True
4,all_qualified_trades,50,Middle_model_vrp_log,0.65,0.70,0.65,0.70,0.70,0.22,0.78,11,39,True
5,all_qualified_trades,50,Middle_model_vrp_z,0.70,0.70,0.70,0.70,0.70,1.00,1.00,50,50,True



Pass diagnostics sample:


,candidate_id,pass_rows_all_dates,pass_rows_with_outcomes,Front_pass_rows_all_dates,Middle_pass_rows_all_dates,Back_pass_rows_all_dates,Front_model_vrp_log,Front_model_vrp_z,Front_RSI14_max,Front_forecast_vol_pct_min,Middle_model_vrp_log,Middle_model_vrp_z,Middle_RSI14_max,Middle_forecast_vol_pct_min
0,core_front_expanded_00001,1376,1376,482,445,449,0.65,0.65,68.0,10.0,0.65,0.7,68.0,8.5
1,core_front_expanded_00002,1360,1360,482,429,449,0.65,0.65,68.0,10.0,0.65,0.7,68.0,10.0
2,core_front_expanded_00003,1353,1353,482,422,449,0.65,0.65,68.0,10.0,0.70,0.7,68.0,8.5
3,core_front_expanded_00004,1337,1337,482,406,449,0.65,0.65,68.0,10.0,0.70,0.7,68.0,10.0
4,core_front_expanded_00005,1341,1341,447,445,449,0.65,0.65,68.0,11.5,0.65,0.7,68.0,8.5
5,core_front_expanded_00006,1325,1325,447,429,449,0.65,0.65,68.0,11.5,0.65,0.7,68.0,10.0
6,core_front_expanded_00007,1318,1318,447,422,449,0.65,0.65,68.0,11.5,0.70,0.7,68.0,8.5
7,core_front_expanded_00008,1302,1302,447,406,449,0.65,0.65,68.0,11.5,0.70,0.7,68.0,10.0
8,core_front_expanded_00009,1301,1301,407,445,449,0.65,0.65,68.0,13.0,0.65,0.7,68.0,8.5
9,core_front_expanded_00010,1285,1285,407,429,449,0.65,0.65,68.0,13.0,0.65,0.7,68.0,10.0



Saved outputs:
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\06_core_front_expanded_sweep_candidate_summary_20260704_215631.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\06_core_front_expanded_sweep_by_bucket_20260704_215631.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\06_core_front_expanded_sweep_by_tenor_20260704_215631.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\06_core_front_expanded_sweep_by_year_20260704_215631.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\06_core_front_expanded_sweep_candidate_pass_diagnostics_20260704_215631.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\06_core_front_expanded_sweep_boundary_diagnostics_20260704_215631.csv
  C:\Users\patri\vrp_project\data\processed\vrp_unified_f

In [11]:
# ======================================================================================
# Cell 9 — Candidate ranking and robustness diagnostics
# ======================================================================================
#
# Objective:
#   Rank the expanded Cell 8 Core candidates using robustness diagnostics, not pure
#   avg P&L/day alone.
#
# Inputs:
#   06_core_front_expanded_sweep_candidate_summary_*.csv
#   06_core_front_expanded_sweep_by_bucket_*.csv
#   06_core_front_expanded_sweep_by_tenor_*.csv
#   06_core_front_expanded_sweep_by_year_*.csv
#   06_core_front_expanded_sweep_boundary_diagnostics_*.csv
#
# Outputs:
#   07_core_candidate_ranked_summary_*.csv
#   07_core_candidate_ranked_by_bucket_*.csv
#   07_core_candidate_ranked_by_tenor_*.csv
#   07_core_candidate_ranked_by_year_*.csv
#   07_core_candidate_boundary_review_*.csv
#   07_core_candidate_ranking_validation_*.csv
#   07_core_candidate_ranking_manifest_*.json
#
# Primary selection universe for candidate ranking:
#   one_trade_per_bucket_per_date_best_rank
#
# Primary metric remains:
#   avg_pnl_per_day
#
# Robustness screens:
#   - enough trades
#   - positive 2025 avg P&L/day
#   - positive 2026 avg P&L/day
#   - acceptable P&L/day drawdown
#   - acceptable worst 20-trade P&L/day sum
#   - no increase in large-loss count beyond tolerance
#   - positive bucket-level P&L/day across Front/Middle/Back
#   - year/bucket/tenor diagnostics retained for review
#
# Important:
#   This cell does NOT select final thresholds.
#   This cell does NOT run another sweep.
#   This cell does NOT expand the grid.
#   This cell does NOT change sizing.
#   This cell does NOT lock production.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 420)

# ======================================================================================
# 0. Fallback config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL9_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

PRIMARY_SELECTION_UNIVERSE = "one_trade_per_bucket_per_date_best_rank"
PRIMARY_RANKING_METRIC = "avg_pnl_per_day"

# These are ranking/review screens, not production locks.
ROBUSTNESS_SCREEN_CONFIG = {
    "min_trades": 375,
    "min_unique_trade_dates": 175,
    "min_avg_pnl_per_day": 450.0,
    "min_avg_pnl_per_day_2025": 0.0,
    "min_avg_pnl_per_day_2026": 0.0,
    "min_pnl_per_day_drawdown": -40_000.0,
    "min_worst_20_trade_pnl_per_day_sum": -35_000.0,
    "max_trades_le_neg_100k": 2,
    "min_bucket_avg_pnl_per_day": 0.0,
    "min_positive_bucket_count": 3,
}

TOP_N_REVIEW = 50

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def coerce_numeric(df: pd.DataFrame, exclude_cols=None) -> pd.DataFrame:
    if exclude_cols is None:
        exclude_cols = set()
    else:
        exclude_cols = set(exclude_cols)

    out = df.copy()

    for c in out.columns:
        if c in exclude_cols:
            continue
        if out[c].dtype == "object":
            continue
        out[c] = pd.to_numeric(out[c], errors="ignore")

    return out


def safe_numeric(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)

    if x.notna().any():
        return x

    return pd.Series(np.nan, index=s.index)


def percentile_score(series: pd.Series, higher_is_better: bool = True) -> pd.Series:
    """
    Return 0-1 percentile scores.

    higher_is_better=True:
        highest value gets highest score.

    higher_is_better=False:
        lowest value gets highest score.
    """
    x = safe_numeric(series)

    if x.notna().sum() <= 1:
        return pd.Series(0.5, index=series.index)

    fill_value = x.median()
    x = x.fillna(fill_value)

    if higher_is_better:
        return x.rank(pct=True, method="average", ascending=True)

    return x.rank(pct=True, method="average", ascending=False)


def add_param_boundary_flags(df: pd.DataFrame, param_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Add per-candidate min/max boundary flags based on values present in the candidate universe.
    """
    out = df.copy()
    rows = []

    for col in param_cols:
        if col not in out.columns:
            continue

        vals = pd.to_numeric(out[col], errors="coerce").dropna().unique().tolist()

        if not vals:
            continue

        grid_min = min(vals)
        grid_max = max(vals)

        min_flag = f"{col}_at_grid_min"
        max_flag = f"{col}_at_grid_max"

        out[min_flag] = pd.to_numeric(out[col], errors="coerce").eq(grid_min)
        out[max_flag] = pd.to_numeric(out[col], errors="coerce").eq(grid_max)

        rows.append({
            "parameter": col,
            "grid_min": grid_min,
            "grid_max": grid_max,
            "candidate_count_at_min": int(out[min_flag].sum()),
            "candidate_count_at_max": int(out[max_flag].sum()),
            "candidate_share_at_min": float(out[min_flag].mean()),
            "candidate_share_at_max": float(out[max_flag].mean()),
        })

    return out, pd.DataFrame(rows)


def flatten_pivot_columns(pivot_df: pd.DataFrame) -> pd.DataFrame:
    pivot_df = pivot_df.copy()
    pivot_df.columns = [
        "_".join([str(x) for x in col if str(x) != ""])
        if isinstance(col, tuple)
        else str(col)
        for col in pivot_df.columns
    ]
    return pivot_df.reset_index()


def build_bucket_diagnostics(by_bucket: pd.DataFrame, universe: str) -> pd.DataFrame:
    d = by_bucket[by_bucket["selection_universe"].eq(universe)].copy()

    if d.empty:
        return pd.DataFrame()

    metrics = [
        "trades",
        "unique_trade_dates",
        "avg_pnl_per_day",
        "median_pnl_per_day",
        "profit_factor_pnl_per_day",
        "pnl_per_day_drawdown",
        "worst_20_trade_pnl_per_day_sum",
        "avg_pnl_per_day_2025",
        "avg_pnl_per_day_2026",
        "trades_le_neg_100k",
    ]

    available_metrics = [m for m in metrics if m in d.columns]

    pivot = d.pivot_table(
        index="candidate_id",
        columns="tenor_bucket",
        values=available_metrics,
        aggfunc="first",
    )

    out = flatten_pivot_columns(pivot)

    # Rename from metric_bucket to Bucket_metric for easier reading.
    rename_map = {}
    for c in out.columns:
        if "_" not in c or c == "candidate_id":
            continue
        parts = c.split("_")
        bucket = parts[-1]
        metric = "_".join(parts[:-1])
        if bucket in ["Front", "Middle", "Back"]:
            rename_map[c] = f"{bucket}_{metric}"

    out = out.rename(columns=rename_map)

    bucket_avg_cols = [
        c for c in [
            "Front_avg_pnl_per_day",
            "Middle_avg_pnl_per_day",
            "Back_avg_pnl_per_day",
        ]
        if c in out.columns
    ]

    if bucket_avg_cols:
        out["min_bucket_avg_pnl_per_day"] = out[bucket_avg_cols].min(axis=1)
        out["max_bucket_avg_pnl_per_day"] = out[bucket_avg_cols].max(axis=1)
        out["positive_bucket_count"] = out[bucket_avg_cols].gt(0).sum(axis=1)
        out["all_buckets_positive_avg_pnl_per_day"] = out["positive_bucket_count"].eq(len(bucket_avg_cols))

    return out


def build_tenor_diagnostics(by_tenor: pd.DataFrame, universe: str) -> pd.DataFrame:
    d = by_tenor[by_tenor["selection_universe"].eq(universe)].copy()

    if d.empty:
        return pd.DataFrame()

    # Candidate-level tenor stability diagnostics.
    g = d.groupby("candidate_id", as_index=False).agg(
        tenors_active=("tenor", "nunique"),
        min_tenor_avg_pnl_per_day=("avg_pnl_per_day", "min"),
        max_tenor_avg_pnl_per_day=("avg_pnl_per_day", "max"),
        positive_tenor_count=("avg_pnl_per_day", lambda x: int((pd.to_numeric(x, errors="coerce") > 0).sum())),
        negative_tenor_count=("avg_pnl_per_day", lambda x: int((pd.to_numeric(x, errors="coerce") <= 0).sum())),
    )

    front = d[d["tenor_bucket"].eq("Front")].copy()

    if not front.empty:
        front_g = front.groupby("candidate_id", as_index=False).agg(
            front_tenors_active=("tenor", "nunique"),
            front_min_tenor_avg_pnl_per_day=("avg_pnl_per_day", "min"),
            front_positive_tenor_count=("avg_pnl_per_day", lambda x: int((pd.to_numeric(x, errors="coerce") > 0).sum())),
            front_negative_tenor_count=("avg_pnl_per_day", lambda x: int((pd.to_numeric(x, errors="coerce") <= 0).sum())),
        )

        g = g.merge(front_g, on="candidate_id", how="left", validate="one_to_one")

    # Add explicit 9/12/15 avg P&L/day where present.
    front_tenor_values = d[d["tenor"].isin([9, 12, 15])].pivot_table(
        index="candidate_id",
        columns="tenor",
        values="avg_pnl_per_day",
        aggfunc="first",
    )

    if not front_tenor_values.empty:
        front_tenor_values = front_tenor_values.rename(
            columns={
                9: "tenor_9_avg_pnl_per_day",
                12: "tenor_12_avg_pnl_per_day",
                15: "tenor_15_avg_pnl_per_day",
            }
        ).reset_index()

        g = g.merge(front_tenor_values, on="candidate_id", how="left", validate="one_to_one")

    return g


def build_year_diagnostics(by_year: pd.DataFrame, universe: str) -> pd.DataFrame:
    d = by_year[by_year["selection_universe"].eq(universe)].copy()

    if d.empty:
        return pd.DataFrame()

    d["year"] = pd.to_numeric(d["year"], errors="coerce").astype("Int64")

    g = d.groupby("candidate_id", as_index=False).agg(
        years_active=("year", "nunique"),
        min_year_avg_pnl_per_day=("avg_pnl_per_day", "min"),
        max_year_avg_pnl_per_day=("avg_pnl_per_day", "max"),
        positive_year_count=("avg_pnl_per_day", lambda x: int((pd.to_numeric(x, errors="coerce") > 0).sum())),
        negative_year_count=("avg_pnl_per_day", lambda x: int((pd.to_numeric(x, errors="coerce") <= 0).sum())),
        min_year_trade_count=("trades", "min"),
    )

    year_pivot = d.pivot_table(
        index="candidate_id",
        columns="year",
        values="avg_pnl_per_day",
        aggfunc="first",
    )

    if not year_pivot.empty:
        year_pivot = year_pivot.rename(
            columns={year: f"year_{int(year)}_avg_pnl_per_day" for year in year_pivot.columns if pd.notna(year)}
        ).reset_index()

        g = g.merge(year_pivot, on="candidate_id", how="left", validate="one_to_one")

    return g


def add_robustness_screens(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    cfg = ROBUSTNESS_SCREEN_CONFIG

    d["screen_min_trades"] = pd.to_numeric(d["trades"], errors="coerce") >= cfg["min_trades"]
    d["screen_min_unique_trade_dates"] = pd.to_numeric(d["unique_trade_dates"], errors="coerce") >= cfg["min_unique_trade_dates"]
    d["screen_avg_pnl_per_day"] = pd.to_numeric(d["avg_pnl_per_day"], errors="coerce") >= cfg["min_avg_pnl_per_day"]
    d["screen_2025_positive"] = pd.to_numeric(d["avg_pnl_per_day_2025"], errors="coerce") >= cfg["min_avg_pnl_per_day_2025"]
    d["screen_2026_positive"] = pd.to_numeric(d["avg_pnl_per_day_2026"], errors="coerce") >= cfg["min_avg_pnl_per_day_2026"]
    d["screen_drawdown"] = pd.to_numeric(d["pnl_per_day_drawdown"], errors="coerce") >= cfg["min_pnl_per_day_drawdown"]
    d["screen_worst20"] = pd.to_numeric(d["worst_20_trade_pnl_per_day_sum"], errors="coerce") >= cfg["min_worst_20_trade_pnl_per_day_sum"]
    d["screen_large_loss_count"] = pd.to_numeric(d["trades_le_neg_100k"], errors="coerce") <= cfg["max_trades_le_neg_100k"]

    if "min_bucket_avg_pnl_per_day" in d.columns:
        d["screen_bucket_min_positive"] = pd.to_numeric(d["min_bucket_avg_pnl_per_day"], errors="coerce") >= cfg["min_bucket_avg_pnl_per_day"]
    else:
        d["screen_bucket_min_positive"] = False

    if "positive_bucket_count" in d.columns:
        d["screen_all_buckets_positive"] = pd.to_numeric(d["positive_bucket_count"], errors="coerce") >= cfg["min_positive_bucket_count"]
    else:
        d["screen_all_buckets_positive"] = False

    screen_cols = [
        "screen_min_trades",
        "screen_min_unique_trade_dates",
        "screen_avg_pnl_per_day",
        "screen_2025_positive",
        "screen_2026_positive",
        "screen_drawdown",
        "screen_worst20",
        "screen_large_loss_count",
        "screen_bucket_min_positive",
        "screen_all_buckets_positive",
    ]

    d["robustness_screen_pass"] = d[screen_cols].all(axis=1)
    d["robustness_screen_pass_count"] = d[screen_cols].sum(axis=1)
    d["robustness_screen_fail_count"] = len(screen_cols) - d["robustness_screen_pass_count"]

    failed_items = []
    for _, row in d.iterrows():
        failed = [c.replace("screen_", "") for c in screen_cols if not bool(row[c])]
        failed_items.append(", ".join(failed))

    d["robustness_failed_screens"] = failed_items

    return d


def add_composite_score(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    score_specs = {
        "avg_pnl_per_day": {"weight": 0.25, "higher_is_better": True},
        "profit_factor_pnl_per_day": {"weight": 0.12, "higher_is_better": True},
        "pnl_per_day_drawdown": {"weight": 0.14, "higher_is_better": True},
        "worst_20_trade_pnl_per_day_sum": {"weight": 0.12, "higher_is_better": True},
        "avg_pnl_per_day_2026": {"weight": 0.16, "higher_is_better": True},
        "avg_pnl_per_day_2025": {"weight": 0.06, "higher_is_better": True},
        "min_bucket_avg_pnl_per_day": {"weight": 0.08, "higher_is_better": True},
        "trades": {"weight": 0.04, "higher_is_better": True},
        "trades_le_neg_100k": {"weight": 0.03, "higher_is_better": False},
    }

    # If profit_factor_pnl_per_day is missing, fall back to profit_factor.
    if "profit_factor_pnl_per_day" not in d.columns and "profit_factor" in d.columns:
        d["profit_factor_pnl_per_day"] = d["profit_factor"]

    raw_score_cols = []

    for metric, spec in score_specs.items():
        if metric not in d.columns:
            continue

        score_col = f"score_component_{metric}"
        d[score_col] = percentile_score(d[metric], higher_is_better=spec["higher_is_better"])
        d[score_col] = d[score_col] * spec["weight"]
        raw_score_cols.append(score_col)

    d["robustness_score"] = d[raw_score_cols].sum(axis=1) if raw_score_cols else np.nan

    # Small penalty for sitting at expanded-grid max on Front forecast vol floor.
    # This is not a disqualifier; it is a warning that more expansion may be needed.
    if "Front_forecast_vol_pct_min_at_grid_max" in d.columns:
        d["front_forecast_floor_boundary_penalty"] = np.where(
            d["Front_forecast_vol_pct_min_at_grid_max"].astype(bool),
            0.015,
            0.0,
        )
    else:
        d["front_forecast_floor_boundary_penalty"] = 0.0

    d["robustness_score_after_boundary_penalty"] = (
        d["robustness_score"] - d["front_forecast_floor_boundary_penalty"]
    )

    d["robustness_rank"] = (
        d["robustness_score_after_boundary_penalty"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

    d["avg_pnl_per_day_rank"] = (
        pd.to_numeric(d["avg_pnl_per_day"], errors="coerce")
        .rank(method="min", ascending=False)
        .astype(int)
    )

    return d


# ======================================================================================
# 2. Load Cell 8 outputs
# ======================================================================================

candidate_summary_path = latest_file(
    OUT_AUDIT_DIR,
    "06_core_front_expanded_sweep_candidate_summary_*.csv",
    required=True,
)

by_bucket_path = latest_file(
    OUT_AUDIT_DIR,
    "06_core_front_expanded_sweep_by_bucket_*.csv",
    required=True,
)

by_tenor_path = latest_file(
    OUT_AUDIT_DIR,
    "06_core_front_expanded_sweep_by_tenor_*.csv",
    required=True,
)

by_year_path = latest_file(
    OUT_AUDIT_DIR,
    "06_core_front_expanded_sweep_by_year_*.csv",
    required=True,
)

boundary_diagnostics_path = latest_file(
    OUT_AUDIT_DIR,
    "06_core_front_expanded_sweep_boundary_diagnostics_*.csv",
    required=True,
)

summary = pd.read_csv(candidate_summary_path)
by_bucket = pd.read_csv(by_bucket_path)
by_tenor = pd.read_csv(by_tenor_path)
by_year = pd.read_csv(by_year_path)
boundary_diagnostics = pd.read_csv(boundary_diagnostics_path)

# Parse dates where available.
for df in [summary, by_bucket, by_tenor, by_year]:
    for c in ["date_min", "date_max"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")

# ======================================================================================
# 3. Build ranking table
# ======================================================================================

primary = summary[summary["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)].copy()

if primary.empty:
    raise RuntimeError(f"No rows found for primary selection universe: {PRIMARY_SELECTION_UNIVERSE}")

param_cols = [
    c for c in primary.columns
    if c.startswith("Front_")
    or c.startswith("Middle_")
    or c.startswith("Back_")
]

primary, param_boundary_summary = add_param_boundary_flags(primary, param_cols)

bucket_diag = build_bucket_diagnostics(by_bucket, PRIMARY_SELECTION_UNIVERSE)
tenor_diag = build_tenor_diagnostics(by_tenor, PRIMARY_SELECTION_UNIVERSE)
year_diag = build_year_diagnostics(by_year, PRIMARY_SELECTION_UNIVERSE)

ranked = primary.copy()

for diag_name, diag in [
    ("bucket", bucket_diag),
    ("tenor", tenor_diag),
    ("year", year_diag),
]:
    if not diag.empty:
        ranked = ranked.merge(
            diag,
            on="candidate_id",
            how="left",
            validate="one_to_one",
            suffixes=("", f"_{diag_name}"),
        )

ranked = add_robustness_screens(ranked)
ranked = add_composite_score(ranked)

ranked = ranked.sort_values(
    [
        "robustness_screen_pass",
        "robustness_score_after_boundary_penalty",
        "avg_pnl_per_day",
        "avg_pnl_per_day_2026",
        "pnl_per_day_drawdown",
        "trades",
    ],
    ascending=[False, False, False, False, False, False],
).reset_index(drop=True)

# Re-number display rank after sorting.
ranked.insert(0, "review_rank", np.arange(1, len(ranked) + 1))

# ======================================================================================
# 4. Build review subsets
# ======================================================================================

top_candidate_ids = ranked.head(TOP_N_REVIEW)["candidate_id"].tolist()
screen_pass_candidate_ids = ranked[ranked["robustness_screen_pass"]]["candidate_id"].tolist()

ranked_by_bucket = by_bucket[
    by_bucket["candidate_id"].isin(top_candidate_ids)
    & by_bucket["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

ranked_by_tenor = by_tenor[
    by_tenor["candidate_id"].isin(top_candidate_ids)
    & by_tenor["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

ranked_by_year = by_year[
    by_year["candidate_id"].isin(top_candidate_ids)
    & by_year["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

# Add review rank to detail tables.
rank_map = ranked.set_index("candidate_id")["review_rank"].to_dict()

for df in [ranked_by_bucket, ranked_by_tenor, ranked_by_year]:
    df.insert(0, "review_rank", df["candidate_id"].map(rank_map))
    df.sort_values(
        [c for c in ["review_rank", "tenor_bucket_order", "tenor", "year"] if c in df.columns],
        inplace=True,
    )

# Boundary review specific to expanded sweep top candidates.
boundary_review = boundary_diagnostics.copy()

if "boundary_warning" in boundary_review.columns:
    boundary_review["boundary_warning"] = boundary_review["boundary_warning"].astype(bool)

boundary_review = boundary_review.sort_values(
    [c for c in ["selection_universe", "boundary_warning", "parameter"] if c in boundary_review.columns],
    ascending=[True, False, True][:len([c for c in ["selection_universe", "boundary_warning", "parameter"] if c in boundary_review.columns])],
).reset_index(drop=True)

# ======================================================================================
# 5. Save outputs
# ======================================================================================

ranked_summary_path = OUT_AUDIT_DIR / f"07_core_candidate_ranked_summary_{CELL9_TIMESTAMP}.csv"
ranked_by_bucket_path = OUT_AUDIT_DIR / f"07_core_candidate_ranked_by_bucket_{CELL9_TIMESTAMP}.csv"
ranked_by_tenor_path = OUT_AUDIT_DIR / f"07_core_candidate_ranked_by_tenor_{CELL9_TIMESTAMP}.csv"
ranked_by_year_path = OUT_AUDIT_DIR / f"07_core_candidate_ranked_by_year_{CELL9_TIMESTAMP}.csv"
boundary_review_path = OUT_AUDIT_DIR / f"07_core_candidate_boundary_review_{CELL9_TIMESTAMP}.csv"
param_boundary_summary_path = OUT_AUDIT_DIR / f"07_core_candidate_parameter_boundary_summary_{CELL9_TIMESTAMP}.csv"
validation_path = OUT_AUDIT_DIR / f"07_core_candidate_ranking_validation_{CELL9_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"07_core_candidate_ranking_manifest_{CELL9_TIMESTAMP}.json"

ranked.to_csv(ranked_summary_path, index=False)
ranked_by_bucket.to_csv(ranked_by_bucket_path, index=False)
ranked_by_tenor.to_csv(ranked_by_tenor_path, index=False)
ranked_by_year.to_csv(ranked_by_year_path, index=False)
boundary_review.to_csv(boundary_review_path, index=False)
param_boundary_summary.to_csv(param_boundary_summary_path, index=False)

# ======================================================================================
# 6. Validation
# ======================================================================================

validation_rows = []

def add_validation(check, status, detail):
    validation_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })

required_summary_cols = [
    "candidate_id",
    "selection_universe",
    "trades",
    "unique_trade_dates",
    "avg_pnl_per_day",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "trades_le_neg_100k",
]

missing_required_summary_cols = [c for c in required_summary_cols if c not in summary.columns]

add_validation(
    "cell8_candidate_summary_loaded",
    "PASS" if not summary.empty else "FAIL",
    f"rows={len(summary):,}; path={candidate_summary_path}",
)

add_validation(
    "required_summary_columns_available",
    "PASS" if not missing_required_summary_cols else "FAIL",
    f"missing={missing_required_summary_cols}",
)

add_validation(
    "primary_selection_universe_available",
    "PASS" if not primary.empty else "FAIL",
    f"primary_selection_universe={PRIMARY_SELECTION_UNIVERSE}; rows={len(primary):,}",
)

add_validation(
    "ranked_candidates_created",
    "PASS" if not ranked.empty else "FAIL",
    f"rows={len(ranked):,}",
)

add_validation(
    "candidate_ids_unique_in_ranked_summary",
    "PASS" if ranked["candidate_id"].is_unique else "FAIL",
    f"candidate_rows={len(ranked):,}; unique_candidates={ranked['candidate_id'].nunique():,}",
)

add_validation(
    "robustness_screen_created",
    "PASS" if "robustness_screen_pass" in ranked.columns else "FAIL",
    f"screen_pass_count={int(ranked['robustness_screen_pass'].sum()) if 'robustness_screen_pass' in ranked.columns else 'NA'}",
)

add_validation(
    "robustness_score_created",
    "PASS" if "robustness_score_after_boundary_penalty" in ranked.columns else "FAIL",
    f"non_null_scores={int(ranked['robustness_score_after_boundary_penalty'].notna().sum()) if 'robustness_score_after_boundary_penalty' in ranked.columns else 'NA'}",
)

add_validation(
    "bucket_diagnostics_created",
    "PASS" if not bucket_diag.empty else "WARN",
    f"rows={len(bucket_diag):,}",
)

add_validation(
    "tenor_diagnostics_created",
    "PASS" if not tenor_diag.empty else "WARN",
    f"rows={len(tenor_diag):,}",
)

add_validation(
    "year_diagnostics_created",
    "PASS" if not year_diag.empty else "WARN",
    f"rows={len(year_diag):,}",
)

boundary_warning_count = (
    int(boundary_review["boundary_warning"].sum())
    if "boundary_warning" in boundary_review.columns
    else 0
)

add_validation(
    "boundary_review_loaded",
    "PASS" if not boundary_review.empty else "WARN",
    f"rows={len(boundary_review):,}; boundary_warnings={boundary_warning_count:,}; path={boundary_diagnostics_path}",
)

front_fv_max_count = (
    int(ranked.get("Front_forecast_vol_pct_min_at_grid_max", pd.Series(False, index=ranked.index)).sum())
)

top25_front_fv_max_count = (
    int(
        ranked.head(25)
        .get("Front_forecast_vol_pct_min_at_grid_max", pd.Series(False, index=ranked.head(25).index))
        .sum()
    )
)

add_validation(
    "front_forecast_vol_floor_boundary_flag_created",
    "PASS" if "Front_forecast_vol_pct_min_at_grid_max" in ranked.columns else "WARN",
    f"all_ranked_at_max={front_fv_max_count:,}; top25_at_max={top25_front_fv_max_count:,}",
)

add_validation(
    "no_new_sweep",
    "PASS",
    "Cell 9 only ranks existing Cell 8 outputs.",
)

add_validation(
    "no_final_candidate_selection",
    "PASS",
    "Cell 9 creates ranked diagnostics only; no final lock.",
)

add_validation(
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed.",
)

add_validation(
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"] == "FAIL"]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1.ipynb",
    "cell": "Cell 9 — Candidate ranking and robustness diagnostics",
    "timestamp": CELL9_TIMESTAMP,
    "primary_selection_universe": PRIMARY_SELECTION_UNIVERSE,
    "primary_ranking_metric": PRIMARY_RANKING_METRIC,
    "robustness_screen_config": ROBUSTNESS_SCREEN_CONFIG,
    "top_n_review": TOP_N_REVIEW,
    "candidate_summary_path": str(candidate_summary_path),
    "by_bucket_path": str(by_bucket_path),
    "by_tenor_path": str(by_tenor_path),
    "by_year_path": str(by_year_path),
    "boundary_diagnostics_path": str(boundary_diagnostics_path),
    "ranked_summary_path": str(ranked_summary_path),
    "ranked_by_bucket_path": str(ranked_by_bucket_path),
    "ranked_by_tenor_path": str(ranked_by_tenor_path),
    "ranked_by_year_path": str(ranked_by_year_path),
    "boundary_review_path": str(boundary_review_path),
    "param_boundary_summary_path": str(param_boundary_summary_path),
    "validation_path": str(validation_path),
    "ranked_candidate_count": int(len(ranked)),
    "robustness_screen_pass_count": int(ranked["robustness_screen_pass"].sum()),
    "boundary_warning_count": int(boundary_warning_count),
    "top25_front_forecast_floor_at_grid_max_count": int(top25_front_fv_max_count),
    "notes": [
        "Ranking diagnostics only.",
        "No new sweep.",
        "No final threshold selected.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
        "If top-ranked candidates remain at Front forecast-vol-floor max, consider another Front forecast-vol-floor expansion before locking.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 9 validation failed. Review validation output above.")

# ======================================================================================
# 7. Display review
# ======================================================================================

print("=" * 100)
print("Cell 9 — Candidate ranking and robustness diagnostics")
print("=" * 100)
print(f"Candidate summary:             {candidate_summary_path}")
print(f"Primary selection universe:    {PRIMARY_SELECTION_UNIVERSE}")
print(f"Ranked candidates:             {len(ranked):,}")
print(f"Robustness screen pass count:  {int(ranked['robustness_screen_pass'].sum()):,}")
print(f"Boundary warnings loaded:      {boundary_warning_count:,}")
print(f"Top 25 Front FV floor at max:  {top25_front_fv_max_count:,} / 25")
print("No new sweep:                  True")
print("No final candidate selection:  True")
print("No Secondary:                  True")
print("No sizing changes:             True")
print("No production lock:            True")

print()
print("Validation:")
display(validation)

main_display_cols = [
    "review_rank",
    "candidate_id",
    "robustness_screen_pass",
    "robustness_score_after_boundary_penalty",
    "avg_pnl_per_day_rank",
    "trades",
    "unique_trade_dates",
    "avg_tenor",
    "win_rate",
    "total_pnl",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "avg_pnl_per_day",
    "median_pnl_per_day",
    "worst_trade_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "trades_le_neg_100k",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
    "min_bucket_avg_pnl_per_day",
    "positive_bucket_count",
    "min_tenor_avg_pnl_per_day",
    "front_min_tenor_avg_pnl_per_day",
    "positive_year_count",
    "negative_year_count",
    "Front_model_vrp_log",
    "Front_model_vrp_z",
    "Front_RSI14_max",
    "Front_forecast_vol_pct_min",
    "Middle_model_vrp_log",
    "Middle_model_vrp_z",
    "Middle_RSI14_max",
    "Middle_forecast_vol_pct_min",
    "Back_model_vrp_log",
    "Back_model_vrp_z",
    "Front_forecast_vol_pct_min_at_grid_max",
    "Front_model_vrp_log_at_grid_max",
    "robustness_failed_screens",
]

print()
print("Top 30 ranked candidates — robustness score:")
display(ranked[[c for c in main_display_cols if c in ranked.columns]].head(30))

print()
print("Candidates passing all robustness screens:")
screen_pass = ranked[ranked["robustness_screen_pass"]].copy()
display(screen_pass[[c for c in main_display_cols if c in screen_pass.columns]].head(50))

print()
print("Top 30 by raw avg P&L/day:")
raw_top = ranked.sort_values(
    ["avg_pnl_per_day", "profit_factor_pnl_per_day", "avg_pnl_per_day_2026", "trades"],
    ascending=[False, False, False, False],
).head(30)
display(raw_top[[c for c in main_display_cols if c in raw_top.columns]])

print()
print("Bucket detail for top 20 ranked candidates:")
display(ranked_by_bucket[ranked_by_bucket["review_rank"].le(20)])

print()
print("Tenor detail for top 20 ranked candidates:")
display(ranked_by_tenor[ranked_by_tenor["review_rank"].le(20)])

print()
print("Year detail for top 20 ranked candidates:")
display(ranked_by_year[ranked_by_year["review_rank"].le(20)])

print()
print("Boundary warnings from Cell 8:")
if "boundary_warning" in boundary_review.columns:
    display(
        boundary_review[boundary_review["boundary_warning"]]
        .sort_values(["selection_universe", "parameter"])
    )
else:
    display(boundary_review)

print()
print("Parameter boundary summary across ranked candidates:")
display(param_boundary_summary)

print()
print("Saved outputs:")
for p in [
    ranked_summary_path,
    ranked_by_bucket_path,
    ranked_by_tenor_path,
    ranked_by_year_path,
    boundary_review_path,
    param_boundary_summary_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 9 PASSED")

Cell 9 — Candidate ranking and robustness diagnostics
Candidate summary:             C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\06_core_front_expanded_sweep_candidate_summary_20260704_215631.csv
Primary selection universe:    one_trade_per_bucket_per_date_best_rank
Ranked candidates:             320
Robustness screen pass count:  80
Boundary warnings loaded:      41
Top 25 Front FV floor at max:  25 / 25
No new sweep:                  True
No final candidate selection:  True
No Secondary:                  True
No sizing changes:             True
No production lock:            True

Validation:


,check,status,detail
0,cell8_candidate_summary_loaded,PASS,"rows=1,280; path=C:\Users\patri\vrp_project\da..."
1,required_summary_columns_available,PASS,missing=[]
2,primary_selection_universe_available,PASS,primary_selection_universe=one_trade_per_bucke...
3,ranked_candidates_created,PASS,rows=320
4,candidate_ids_unique_in_ranked_summary,PASS,candidate_rows=320; unique_candidates=320
5,robustness_screen_created,PASS,screen_pass_count=80
6,robustness_score_created,PASS,non_null_scores=320
7,bucket_diagnostics_created,PASS,rows=320
8,tenor_diagnostics_created,PASS,rows=320
9,year_diagnostics_created,PASS,rows=320



Top 30 ranked candidates — robustness score:


,review_rank,candidate_id,robustness_screen_pass,robustness_score_after_boundary_penalty,avg_pnl_per_day_rank,trades,unique_trade_dates,avg_tenor,win_rate,total_pnl,profit_factor,profit_factor_pnl_per_day,avg_pnl_per_day,median_pnl_per_day,worst_trade_pnl_per_day,pnl_per_day_drawdown,worst_20_trade_pnl_per_day_sum,trades_le_neg_100k,avg_pnl_per_day_2025,avg_pnl_per_day_2026,min_bucket_avg_pnl_per_day,positive_bucket_count,min_tenor_avg_pnl_per_day,front_min_tenor_avg_pnl_per_day,positive_year_count,negative_year_count,Front_model_vrp_log,Front_model_vrp_z,Front_RSI14_max,Front_forecast_vol_pct_min,Middle_model_vrp_log,Middle_model_vrp_z,Middle_RSI14_max,Middle_forecast_vol_pct_min,Back_model_vrp_log,Back_model_vrp_z,Front_forecast_vol_pct_min_at_grid_max,Front_model_vrp_log_at_grid_max,robustness_failed_screens
0,1,core_front_expanded_00168,True,0.846453,1,428,197,21.995327,0.855140,5.479290e+06,3.881266,2.897659,547.859866,885.630340,-10376.349181,-30723.516018,-24889.323006,2,661.947436,77.113098,520.383465,3,-129.435164,-129.435164,6,0,0.70,0.65,68.0,14.5,0.70,0.7,68.0,10.0,0.7,0.7,True,False,
1,2,core_front_expanded_00182,True,0.846453,1,428,197,21.995327,0.855140,5.479290e+06,3.881266,2.897659,547.859866,885.630340,-10376.349181,-30723.516018,-24889.323006,2,661.947436,77.113098,520.383465,3,-129.435164,-129.435164,6,0,0.70,0.65,66.0,14.5,0.70,0.7,68.0,10.0,0.7,0.7,True,False,
2,3,core_front_expanded_00198,True,0.846453,1,428,197,21.995327,0.855140,5.479290e+06,3.881266,2.897659,547.859866,885.630340,-10376.349181,-30723.516018,-24889.323006,2,661.947436,77.113098,520.383465,3,-129.435164,-129.435164,6,0,0.70,0.65,64.0,14.5,0.70,0.7,68.0,10.0,0.7,0.7,True,False,
3,4,core_front_expanded_00167,True,0.839922,8,432,197,21.979167,0.856481,5.519880e+06,3.902610,2.913804,547.404931,879.606050,-10376.349181,-30723.516018,-24889.323006,2,653.887160,77.113098,520.383465,3,-129.435164,-129.435164,6,0,0.70,0.65,68.0,14.5,0.70,0.7,68.0,8.5,0.7,0.7,True,False,
4,5,core_front_expanded_00181,True,0.839922,8,432,197,21.979167,0.856481,5.519880e+06,3.902610,2.913804,547.404931,879.606050,-10376.349181,-30723.516018,-24889.323006,2,653.887160,77.113098,520.383465,3,-129.435164,-129.435164,6,0,0.70,0.65,66.0,14.5,0.70,0.7,68.0,8.5,0.7,0.7,True,False,
5,6,core_front_expanded_00197,True,0.839922,8,432,197,21.979167,0.856481,5.519880e+06,3.902610,2.913804,547.404931,879.606050,-10376.349181,-30723.516018,-24889.323006,2,653.887160,77.113098,520.383465,3,-129.435164,-129.435164,6,0,0.70,0.65,64.0,14.5,0.70,0.7,68.0,8.5,0.7,0.7,True,False,
6,7,core_front_expanded_00248,True,0.835891,11,423,194,22.113475,0.855792,5.434757e+06,3.872530,2.879166,545.335363,881.455029,-10376.349181,-30723.516018,-24889.323006,2,670.977756,77.113098,510.082211,3,-169.827553,-169.827553,6,0,0.75,0.65,68.0,14.5,0.70,0.7,68.0,10.0,0.7,0.7,True,True,
7,8,core_front_expanded_00262,True,0.835891,11,423,194,22.113475,0.855792,5.434757e+06,3.872530,2.879166,545.335363,881.455029,-10376.349181,-30723.516018,-24889.323006,2,670.977756,77.113098,510.082211,3,-169.827553,-169.827553,6,0,0.75,0.65,66.0,14.5,0.70,0.7,68.0,10.0,0.7,0.7,True,True,
8,9,core_front_expanded_00278,True,0.835891,11,423,194,22.113475,0.855792,5.434757e+06,3.872530,2.879166,545.335363,881.455029,-10376.349181,-30723.516018,-24889.323006,2,670.977756,77.113098,510.082211,3,-169.827553,-169.827553,6,0,0.75,0.65,64.0,14.5,0.70,0.7,68.0,10.0,0.7,0.7,True,True,
9,10,core_front_expanded_00183,True,0.820937,4,424,196,21.983491,0.853774,5.425318e+06,3.852885,2.878714,547.507072,890.768149,-10376.349181,-30723.516018,-24889.323006,2,665.656487,60.157120,520.383465,3,-129.435164,-129.435164,6,0,0.70,0.65,66.0,14.5,0.70,0.7,66.0,8.5,0.7,0.7,True,False,



Candidates passing all robustness screens:


...


Boundary warnings from Cell 8:


,selection_universe,top_n,parameter,grid_min,grid_max,top_min,top_max,top_mode,share_top_at_grid_min,share_top_at_grid_max,count_top_at_grid_min,count_top_at_grid_max,boundary_warning
0,all_qualified_trades,50,Back_RSI14_max,70.00,70.00,70.00,70.00,70.00,1.00,1.00,50,50,True
1,all_qualified_trades,50,Back_forecast_vol_pct_min,8.50,8.50,8.50,8.50,8.50,1.00,1.00,50,50,True
2,all_qualified_trades,50,Back_model_vrp_log,0.70,0.70,0.70,0.70,0.70,1.00,1.00,50,50,True
3,all_qualified_trades,50,Back_model_vrp_z,0.70,0.70,0.70,0.70,0.70,1.00,1.00,50,50,True
4,all_qualified_trades,50,Front_forecast_vol_pct_min,10.00,14.50,14.50,14.50,14.50,0.00,1.00,0,50,True
5,all_qualified_trades,50,Front_model_vrp_z,0.65,0.70,0.65,0.70,0.65,0.62,0.38,31,19,True
6,all_qualified_trades,50,Middle_RSI14_max,66.00,68.00,66.00,68.00,66.00,0.64,0.36,32,18,True
7,all_qualified_trades,50,Middle_forecast_vol_pct_min,8.50,10.00,8.50,10.00,10.00,0.38,0.62,19,31,True
8,all_qualified_trades,50,Middle_model_vrp_log,0.65,0.70,0.65,0.70,0.70,0.22,0.78,11,39,True
9,all_qualified_trades,50,Middle_model_vrp_z,0.70,0.70,0.70,0.70,0.70,1.00,1.00,50,50,True



Parameter boundary summary across ranked candidates:


,parameter,grid_min,grid_max,candidate_count_at_min,candidate_count_at_max,candidate_share_at_min,candidate_share_at_max
0,Front_model_vrp_log,0.65,0.75,160,80,0.50,0.25
1,Front_model_vrp_z,0.65,0.70,160,160,0.50,0.50
2,Front_RSI14_max,64.00,68.00,128,64,0.40,0.20
3,Front_forecast_vol_pct_min,10.00,14.50,80,80,0.25,0.25
4,Middle_model_vrp_log,0.65,0.70,80,240,0.25,0.75
5,Middle_model_vrp_z,0.70,0.70,320,320,1.00,1.00
6,Middle_RSI14_max,66.00,68.00,128,192,0.40,0.60
7,Middle_forecast_vol_pct_min,8.50,10.00,160,160,0.50,0.50
8,Back_model_vrp_log,0.70,0.70,320,320,1.00,1.00
9,Back_model_vrp_z,0.70,0.70,320,320,1.00,1.00



Saved outputs:
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\07_core_candidate_ranked_summary_20260704_220821.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\07_core_candidate_ranked_by_bucket_20260704_220821.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\07_core_candidate_ranked_by_tenor_20260704_220821.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\07_core_candidate_ranked_by_year_20260704_220821.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\07_core_candidate_boundary_review_20260704_220821.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\07_core_candidate_parameter_boundary_summary_20260704_220821.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\07_core_candidate_ra

In [12]:
# ======================================================================================
# Cell 10 — Front tenor and boundary diagnostics
# ======================================================================================
#
# Objective:
#   Diagnose the remaining Front-bucket issues after Cell 9 ranking.
#
# Cell 9 showed:
#   - 80 candidates passed robustness screens.
#   - Top 25 candidates all used Front forecast_vol_pct_min = 14.5.
#   - Top candidate family remained strong overall.
#   - But 9D remained negative while 12D and 15D were strong.
#
# This cell tests whether the remaining issue is:
#   A. Front forecast-vol floor boundary,
#   B. 9D-specific weakness,
#   C. the current DTE-selection rule,
#   D. or Middle / 2026 instability.
#
# Tests:
#   1. Compare actual Front selected trades by 9D / 12D / 15D.
#   2. Compare Front all-tenor selected vs Front excluding 9D.
#   3. Recompute a diagnostic one-trade-per-bucket/date selector that excludes 9D
#      from Front, while keeping Middle and Back logic unchanged.
#   4. Show Front all-qualified P&L/day by forecast-vol state.
#   5. Check whether Front RSI is non-binding among top candidates.
#
# Important:
#   - No new threshold grid.
#   - No sweep expansion.
#   - No final threshold selection.
#   - No production lock.
#   - No Secondary.
#   - No sizing changes.
#
# This is diagnostic only.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 280)
pd.set_option("display.width", 460)

# ======================================================================================
# 0. Fallback config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL10_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

PRIMARY_SELECTION_UNIVERSE = "one_trade_per_bucket_per_date_best_rank"
ALL_QUALIFIED_UNIVERSE = "all_qualified_trades"

TOP_N_CANDIDATES = 50

BUCKET_CENTER_TENOR = {
    "Front": 12,
    "Middle": 21,
    "Back": 30,
}

FORECAST_VOL_BINS = [-np.inf, 14.5, 16.0, 18.0, 20.0, np.inf]
FORECAST_VOL_BIN_LABELS = [
    "<=14.5",
    "14.5-16.0",
    "16.0-18.0",
    "18.0-20.0",
    ">20.0",
]

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def profit_factor_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max

    return float(drawdown.min())


def worst_rolling_sum(pnl: pd.Series, window: int) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def summarize_trades(
    df: pd.DataFrame,
    group_cols: list[str],
    diagnostic_label: str,
) -> pd.DataFrame:
    d = df.copy()

    if d.empty:
        return pd.DataFrame()

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    for c in [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "forecast_vol_pct",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
    ]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame()

    d = d.sort_values(["candidate_id", "trade_date", "tenor"]).copy()

    rows = []

    for keys, g in d.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            "diagnostic_label": diagnostic_label,
        }

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": int(g["trade_date"].nunique()),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        if "forecast_vol_pct" in g.columns:
            row.update({
                "avg_forecast_vol_pct": float(g["forecast_vol_pct"].mean()),
                "min_forecast_vol_pct": float(g["forecast_vol_pct"].min()),
                "max_forecast_vol_pct": float(g["forecast_vol_pct"].max()),
            })

        if "model_vrp_log" in g.columns:
            row["avg_model_vrp_log"] = float(g["model_vrp_log"].mean())

        if "model_vrp_z_3m" in g.columns:
            row["avg_model_vrp_z_3m"] = float(g["model_vrp_z_3m"].mean())

        if "model_vrp_z_1y" in g.columns:
            row["avg_model_vrp_z_1y"] = float(g["model_vrp_z_1y"].mean())

        rows.append(row)

    return pd.DataFrame(rows)


def add_diagnostic_selection_ranks(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    if d.empty:
        return d

    d["bucket_center_tenor"] = d["tenor_bucket"].map(BUCKET_CENTER_TENOR)
    d["distance_to_bucket_center"] = (d["tenor"] - d["bucket_center_tenor"]).abs()

    group_cols = ["candidate_id", "trade_date", "tenor_bucket"]

    d["rank_z_1y_in_bucket_date_diag"] = (
        d.groupby(group_cols)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )

    d["rank_z_3m_in_bucket_date_diag"] = (
        d.groupby(group_cols)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )

    d["rank_vrp_log_in_bucket_date_diag"] = (
        d.groupby(group_cols)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_bucket_date_diag"] = d[
        [
            "rank_z_1y_in_bucket_date_diag",
            "rank_z_3m_in_bucket_date_diag",
            "rank_vrp_log_in_bucket_date_diag",
        ]
    ].mean(axis=1)

    return d


def diagnostic_one_trade_per_bucket_excluding_9d(all_qualified: pd.DataFrame) -> pd.DataFrame:
    """
    Recompute a diagnostic one-trade-per-bucket/date selector, excluding 9D from Front.

    This is not a new threshold sweep. It uses the exact trades that already passed
    the existing Cell 8 candidate thresholds.
    """
    d = all_qualified.copy()

    if d.empty:
        return d

    # Exclude 9D only from Front.
    d = d[
        ~(
            d["tenor_bucket"].eq("Front")
            & d["tenor"].eq(9)
        )
    ].copy()

    if d.empty:
        return d

    d = add_diagnostic_selection_ranks(d)

    selected = (
        d.sort_values(
            [
                "candidate_id",
                "trade_date",
                "tenor_bucket_order",
                "avg_signal_rank_in_bucket_date_diag",
                "distance_to_bucket_center",
                "rank_z_1y_in_bucket_date_diag",
                "rank_z_3m_in_bucket_date_diag",
                "rank_vrp_log_in_bucket_date_diag",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True, True],
        )
        .groupby(["candidate_id", "trade_date", "tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    selected["diagnostic_selection_universe"] = "one_trade_per_bucket_per_date_excluding_front_9d"

    return selected


def make_trade_set_hash(df: pd.DataFrame) -> str:
    if df.empty:
        return "EMPTY"

    d = df.copy()
    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.strftime("%Y-%m-%d")
    d["tenor"] = pd.to_numeric(d["tenor"], errors="coerce").astype("Int64").astype(str)

    if "tenor_bucket" not in d.columns:
        d["tenor_bucket"] = ""

    keys = (
        d[["trade_date", "tenor_bucket", "tenor"]]
        .astype(str)
        .sort_values(["trade_date", "tenor_bucket", "tenor"])
        .agg("|".join, axis=1)
        .tolist()
    )

    joined = "\n".join(keys).encode("utf-8")

    return hashlib.md5(joined).hexdigest()


def attach_ranked_params(df: pd.DataFrame, ranked: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    param_cols = [
        "candidate_id",
        "review_rank",
        "robustness_screen_pass",
        "robustness_score_after_boundary_penalty",
        "Front_model_vrp_log",
        "Front_model_vrp_z",
        "Front_RSI14_max",
        "Front_forecast_vol_pct_min",
        "Middle_model_vrp_log",
        "Middle_model_vrp_z",
        "Middle_RSI14_max",
        "Middle_forecast_vol_pct_min",
        "Back_model_vrp_log",
        "Back_model_vrp_z",
        "Back_RSI14_max",
        "Back_forecast_vol_pct_min",
        "Front_forecast_vol_pct_min_at_grid_max",
        "Front_model_vrp_log_at_grid_max",
    ]

    available = [c for c in param_cols if c in ranked.columns]

    return df.merge(
        ranked[available].drop_duplicates("candidate_id"),
        on="candidate_id",
        how="left",
        validate="many_to_one",
    )


# ======================================================================================
# 2. Load inputs
# ======================================================================================

selected_trade_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "06_core_front_expanded_sweep_selected_trade_panel_*.parquet",
    required=True,
)

ranked_summary_path = latest_file(
    OUT_AUDIT_DIR,
    "07_core_candidate_ranked_summary_*.csv",
    required=True,
)

ranked_by_tenor_path = latest_file(
    OUT_AUDIT_DIR,
    "07_core_candidate_ranked_by_tenor_*.csv",
    required=True,
)

ranked_by_bucket_path = latest_file(
    OUT_AUDIT_DIR,
    "07_core_candidate_ranked_by_bucket_*.csv",
    required=True,
)

ranked_by_year_path = latest_file(
    OUT_AUDIT_DIR,
    "07_core_candidate_ranked_by_year_*.csv",
    required=True,
)

selected_panel = pd.read_parquet(selected_trade_panel_path)
ranked = pd.read_csv(ranked_summary_path)
ranked_by_tenor = pd.read_csv(ranked_by_tenor_path)
ranked_by_bucket = pd.read_csv(ranked_by_bucket_path)
ranked_by_year = pd.read_csv(ranked_by_year_path)

selected_panel["trade_date"] = pd.to_datetime(selected_panel["trade_date"], errors="coerce").dt.normalize()

for c in [
    "tenor",
    "tenor_bucket_order",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
    "forecast_vol_pct",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
]:
    if c in selected_panel.columns:
        selected_panel[c] = pd.to_numeric(selected_panel[c], errors="coerce")

ranked = ranked.sort_values("review_rank").reset_index(drop=True)

top_candidate_ids = ranked.head(TOP_N_CANDIDATES)["candidate_id"].tolist()

top_panel = selected_panel[
    selected_panel["candidate_id"].isin(top_candidate_ids)
].copy()

actual_primary = top_panel[
    top_panel["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

all_qualified = top_panel[
    top_panel["selection_universe"].eq(ALL_QUALIFIED_UNIVERSE)
].copy()

# ======================================================================================
# 3. Actual Front tenor diagnostics
# ======================================================================================

actual_front = actual_primary[
    actual_primary["tenor_bucket"].eq("Front")
].copy()

actual_front_ex_9d_drop_only = actual_front[
    actual_front["tenor"].ne(9)
].copy()

actual_front_all_summary = summarize_trades(
    actual_front,
    group_cols=["candidate_id"],
    diagnostic_label="actual_selected_front_all_tenors",
)

actual_front_ex_9d_drop_summary = summarize_trades(
    actual_front_ex_9d_drop_only,
    group_cols=["candidate_id"],
    diagnostic_label="actual_selected_front_drop_9d_no_reselection",
)

actual_front_by_tenor = summarize_trades(
    actual_front,
    group_cols=["candidate_id", "tenor"],
    diagnostic_label="actual_selected_front_by_tenor",
)

actual_front_by_tenor = attach_ranked_params(actual_front_by_tenor, ranked)

# ======================================================================================
# 4. Diagnostic re-selection excluding Front 9D
# ======================================================================================

diagnostic_reselected = diagnostic_one_trade_per_bucket_excluding_9d(all_qualified)

diagnostic_reselected_summary = summarize_trades(
    diagnostic_reselected,
    group_cols=["candidate_id"],
    diagnostic_label="diagnostic_one_trade_per_bucket_excluding_front_9d",
)

diagnostic_reselected_by_bucket = summarize_trades(
    diagnostic_reselected,
    group_cols=["candidate_id", "tenor_bucket", "tenor_bucket_order"],
    diagnostic_label="diagnostic_one_trade_per_bucket_excluding_front_9d_by_bucket",
)

diagnostic_reselected_by_tenor = summarize_trades(
    diagnostic_reselected,
    group_cols=["candidate_id", "tenor", "tenor_bucket", "tenor_bucket_order"],
    diagnostic_label="diagnostic_one_trade_per_bucket_excluding_front_9d_by_tenor",
)

diagnostic_reselected_summary = attach_ranked_params(diagnostic_reselected_summary, ranked)
diagnostic_reselected_by_bucket = attach_ranked_params(diagnostic_reselected_by_bucket, ranked)
diagnostic_reselected_by_tenor = attach_ranked_params(diagnostic_reselected_by_tenor, ranked)

# Actual primary summary for comparison.
actual_primary_summary = summarize_trades(
    actual_primary,
    group_cols=["candidate_id"],
    diagnostic_label="actual_one_trade_per_bucket_per_date",
)

actual_primary_summary = attach_ranked_params(actual_primary_summary, ranked)

# Compare actual vs diagnostic excluding 9D.
comparison_cols = [
    "candidate_id",
    "trades",
    "unique_trade_dates",
    "avg_tenor",
    "win_rate",
    "total_pnl",
    "profit_factor",
    "avg_pnl_per_day",
    "median_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "trades_le_neg_100k",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
]

actual_compare = actual_primary_summary[[c for c in comparison_cols if c in actual_primary_summary.columns]].copy()
diagnostic_compare = diagnostic_reselected_summary[[c for c in comparison_cols if c in diagnostic_reselected_summary.columns]].copy()

actual_compare = actual_compare.add_prefix("actual_").rename(columns={"actual_candidate_id": "candidate_id"})
diagnostic_compare = diagnostic_compare.add_prefix("ex9d_").rename(columns={"ex9d_candidate_id": "candidate_id"})

front_ex9d_comparison = actual_compare.merge(
    diagnostic_compare,
    on="candidate_id",
    how="inner",
    validate="one_to_one",
)

front_ex9d_comparison = attach_ranked_params(front_ex9d_comparison, ranked)

for metric in [
    "avg_pnl_per_day",
    "total_pnl",
    "profit_factor",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
    "trades",
]:
    actual_col = f"actual_{metric}"
    ex9d_col = f"ex9d_{metric}"
    delta_col = f"delta_ex9d_minus_actual_{metric}"

    if actual_col in front_ex9d_comparison.columns and ex9d_col in front_ex9d_comparison.columns:
        front_ex9d_comparison[delta_col] = (
            front_ex9d_comparison[ex9d_col] - front_ex9d_comparison[actual_col]
        )

front_ex9d_comparison = front_ex9d_comparison.sort_values(
    [
        "review_rank",
        "delta_ex9d_minus_actual_avg_pnl_per_day",
    ],
    ascending=[True, False],
).reset_index(drop=True)

# ======================================================================================
# 5. Forecast-vol state diagnostics for Front all-qualified trades
# ======================================================================================

front_all_qualified = all_qualified[
    all_qualified["tenor_bucket"].eq("Front")
].copy()

front_all_qualified["front_forecast_vol_bin"] = pd.cut(
    front_all_qualified["forecast_vol_pct"],
    bins=FORECAST_VOL_BINS,
    labels=FORECAST_VOL_BIN_LABELS,
    include_lowest=True,
)

front_forecast_vol_state = summarize_trades(
    front_all_qualified,
    group_cols=["candidate_id", "tenor", "front_forecast_vol_bin"],
    diagnostic_label="front_all_qualified_by_tenor_and_forecast_vol_state",
)

front_forecast_vol_state = attach_ranked_params(front_forecast_vol_state, ranked)

front_forecast_vol_state_summary = summarize_trades(
    front_all_qualified,
    group_cols=["candidate_id", "front_forecast_vol_bin"],
    diagnostic_label="front_all_qualified_by_forecast_vol_state",
)

front_forecast_vol_state_summary = attach_ranked_params(front_forecast_vol_state_summary, ranked)

# ======================================================================================
# 6. Front RSI binding diagnostics
# ======================================================================================

primary_actual_hash_rows = []

for candidate_id, g in actual_primary.groupby("candidate_id"):
    primary_actual_hash_rows.append({
        "candidate_id": candidate_id,
        "actual_primary_trade_set_hash": make_trade_set_hash(g),
        "actual_primary_selected_rows": int(len(g)),
    })

actual_hash = pd.DataFrame(primary_actual_hash_rows)

front_hash_rows = []

for candidate_id, g in actual_front.groupby("candidate_id"):
    front_hash_rows.append({
        "candidate_id": candidate_id,
        "actual_front_trade_set_hash": make_trade_set_hash(g),
        "actual_front_selected_rows": int(len(g)),
    })

front_hash = pd.DataFrame(front_hash_rows)

rsi_diag_base = ranked[
    ranked["candidate_id"].isin(top_candidate_ids)
].copy()

rsi_diag_base = rsi_diag_base.merge(actual_hash, on="candidate_id", how="left")
rsi_diag_base = rsi_diag_base.merge(front_hash, on="candidate_id", how="left")

signature_cols_without_front_rsi = [
    "Front_model_vrp_log",
    "Front_model_vrp_z",
    "Front_forecast_vol_pct_min",
    "Middle_model_vrp_log",
    "Middle_model_vrp_z",
    "Middle_RSI14_max",
    "Middle_forecast_vol_pct_min",
    "Back_model_vrp_log",
    "Back_model_vrp_z",
    "Back_RSI14_max",
    "Back_forecast_vol_pct_min",
]

signature_cols_without_front_rsi = [
    c for c in signature_cols_without_front_rsi
    if c in rsi_diag_base.columns
]

rsi_binding_diagnostics = (
    rsi_diag_base
    .groupby(signature_cols_without_front_rsi, dropna=False)
    .agg(
        candidate_count=("candidate_id", "nunique"),
        candidate_ids=("candidate_id", lambda x: ", ".join(map(str, x))),
        review_ranks=("review_rank", lambda x: ", ".join(map(str, x))),
        front_rsi_values=("Front_RSI14_max", lambda x: ", ".join(map(lambda y: str(float(y)), sorted(pd.to_numeric(x, errors="coerce").dropna().unique())))),
        unique_front_rsi_values=("Front_RSI14_max", lambda x: int(pd.to_numeric(x, errors="coerce").nunique())),
        unique_actual_primary_trade_sets=("actual_primary_trade_set_hash", "nunique"),
        unique_actual_front_trade_sets=("actual_front_trade_set_hash", "nunique"),
        min_avg_pnl_per_day=("avg_pnl_per_day", "min"),
        max_avg_pnl_per_day=("avg_pnl_per_day", "max"),
        min_2026_avg_pnl_per_day=("avg_pnl_per_day_2026", "min"),
        max_2026_avg_pnl_per_day=("avg_pnl_per_day_2026", "max"),
        min_trades=("trades", "min"),
        max_trades=("trades", "max"),
    )
    .reset_index()
)

rsi_binding_diagnostics["front_rsi_nonbinding_trade_set_test"] = (
    (rsi_binding_diagnostics["unique_front_rsi_values"] > 1)
    & (rsi_binding_diagnostics["unique_actual_primary_trade_sets"] == 1)
    & (rsi_binding_diagnostics["unique_actual_front_trade_sets"] == 1)
)

rsi_binding_diagnostics["front_rsi_nonbinding_metric_test"] = (
    (rsi_binding_diagnostics["unique_front_rsi_values"] > 1)
    & np.isclose(
        rsi_binding_diagnostics["min_avg_pnl_per_day"],
        rsi_binding_diagnostics["max_avg_pnl_per_day"],
        rtol=0,
        atol=1e-9,
    )
    & np.isclose(
        rsi_binding_diagnostics["min_2026_avg_pnl_per_day"],
        rsi_binding_diagnostics["max_2026_avg_pnl_per_day"],
        rtol=0,
        atol=1e-9,
    )
)

rsi_binding_diagnostics = rsi_binding_diagnostics.sort_values(
    [
        "front_rsi_nonbinding_trade_set_test",
        "candidate_count",
        "max_avg_pnl_per_day",
    ],
    ascending=[False, False, False],
).reset_index(drop=True)

# ======================================================================================
# 7. Diagnostic conclusion flags
# ======================================================================================

top25 = ranked.head(25).copy()

top25_front_fv_all_at_max = bool(
    "Front_forecast_vol_pct_min_at_grid_max" in top25.columns
    and top25["Front_forecast_vol_pct_min_at_grid_max"].astype(bool).all()
)

top_candidate_id = ranked.iloc[0]["candidate_id"]

top_front_tenor = actual_front_by_tenor[
    actual_front_by_tenor["candidate_id"].eq(top_candidate_id)
].copy()

top_9d_avg = np.nan
top_12d_avg = np.nan
top_15d_avg = np.nan

if not top_front_tenor.empty:
    if (top_front_tenor["tenor"] == 9).any():
        top_9d_avg = float(top_front_tenor.loc[top_front_tenor["tenor"].eq(9), "avg_pnl_per_day"].iloc[0])
    if (top_front_tenor["tenor"] == 12).any():
        top_12d_avg = float(top_front_tenor.loc[top_front_tenor["tenor"].eq(12), "avg_pnl_per_day"].iloc[0])
    if (top_front_tenor["tenor"] == 15).any():
        top_15d_avg = float(top_front_tenor.loc[top_front_tenor["tenor"].eq(15), "avg_pnl_per_day"].iloc[0])

top_9d_negative = bool(pd.notna(top_9d_avg) and top_9d_avg < 0)
top_12d_positive = bool(pd.notna(top_12d_avg) and top_12d_avg > 0)
top_15d_positive = bool(pd.notna(top_15d_avg) and top_15d_avg > 0)

top_ex9d_delta = np.nan

if not front_ex9d_comparison.empty:
    row = front_ex9d_comparison[front_ex9d_comparison["candidate_id"].eq(top_candidate_id)]
    if not row.empty and "delta_ex9d_minus_actual_avg_pnl_per_day" in row.columns:
        top_ex9d_delta = float(row["delta_ex9d_minus_actual_avg_pnl_per_day"].iloc[0])

diagnostic_conclusion = pd.DataFrame([
    {
        "diagnostic": "top25_front_forecast_vol_floor_all_at_grid_max",
        "status": "WARN" if top25_front_fv_all_at_max else "PASS",
        "detail": f"top25_all_at_front_fv_floor_max={top25_front_fv_all_at_max}",
    },
    {
        "diagnostic": "top_candidate_9d_avg_pnl_per_day_negative",
        "status": "WARN" if top_9d_negative else "PASS",
        "detail": f"top_candidate={top_candidate_id}; 9D_avg_pnl_per_day={top_9d_avg}",
    },
    {
        "diagnostic": "top_candidate_12d_and_15d_positive",
        "status": "PASS" if top_12d_positive and top_15d_positive else "WARN",
        "detail": f"top_candidate={top_candidate_id}; 12D={top_12d_avg}; 15D={top_15d_avg}",
    },
    {
        "diagnostic": "diagnostic_excluding_9d_improves_top_candidate_avg_pnl_per_day",
        "status": "INFO",
        "detail": f"top_candidate={top_candidate_id}; ex9d_minus_actual_avg_pnl_per_day={top_ex9d_delta}",
    },
    {
        "diagnostic": "front_rsi_nonbinding_groups_found",
        "status": "INFO",
        "detail": f"nonbinding_trade_set_groups={int(rsi_binding_diagnostics['front_rsi_nonbinding_trade_set_test'].sum()) if not rsi_binding_diagnostics.empty else 0}",
    },
    {
        "diagnostic": "no_new_grid_or_sweep",
        "status": "PASS",
        "detail": "Cell 10 uses existing Cell 8/9 outputs only.",
    },
    {
        "diagnostic": "no_final_lock",
        "status": "PASS",
        "detail": "Cell 10 is diagnostic only.",
    },
])

# ======================================================================================
# 8. Save outputs
# ======================================================================================

front_tenor_summary_path = OUT_AUDIT_DIR / f"08_front_tenor_boundary_actual_front_by_tenor_{CELL10_TIMESTAMP}.csv"
front_summary_path = OUT_AUDIT_DIR / f"08_front_tenor_boundary_front_summary_{CELL10_TIMESTAMP}.csv"
front_ex9d_comparison_path = OUT_AUDIT_DIR / f"08_front_tenor_boundary_ex9d_comparison_{CELL10_TIMESTAMP}.csv"
reselected_summary_path = OUT_AUDIT_DIR / f"08_front_tenor_boundary_ex9d_reselected_summary_{CELL10_TIMESTAMP}.csv"
reselected_by_bucket_path = OUT_AUDIT_DIR / f"08_front_tenor_boundary_ex9d_reselected_by_bucket_{CELL10_TIMESTAMP}.csv"
reselected_by_tenor_path = OUT_AUDIT_DIR / f"08_front_tenor_boundary_ex9d_reselected_by_tenor_{CELL10_TIMESTAMP}.csv"
front_forecast_state_path = OUT_AUDIT_DIR / f"08_front_tenor_boundary_forecast_vol_state_by_tenor_{CELL10_TIMESTAMP}.csv"
front_forecast_state_summary_path = OUT_AUDIT_DIR / f"08_front_tenor_boundary_forecast_vol_state_summary_{CELL10_TIMESTAMP}.csv"
rsi_binding_path = OUT_AUDIT_DIR / f"08_front_tenor_boundary_front_rsi_binding_diagnostics_{CELL10_TIMESTAMP}.csv"
diagnostic_conclusion_path = OUT_AUDIT_DIR / f"08_front_tenor_boundary_diagnostic_conclusion_{CELL10_TIMESTAMP}.csv"
reselected_trade_panel_path = OUT_PROCESSED_DIR / f"08_front_tenor_boundary_ex9d_reselected_trade_panel_{CELL10_TIMESTAMP}.parquet"
validation_path = OUT_AUDIT_DIR / f"08_front_tenor_boundary_validation_{CELL10_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"08_front_tenor_boundary_manifest_{CELL10_TIMESTAMP}.json"

front_summary_combined = pd.concat(
    [
        actual_front_all_summary,
        actual_front_ex_9d_drop_summary,
    ],
    ignore_index=True,
)

front_summary_combined = attach_ranked_params(front_summary_combined, ranked)

actual_front_by_tenor.to_csv(front_tenor_summary_path, index=False)
front_summary_combined.to_csv(front_summary_path, index=False)
front_ex9d_comparison.to_csv(front_ex9d_comparison_path, index=False)
diagnostic_reselected_summary.to_csv(reselected_summary_path, index=False)
diagnostic_reselected_by_bucket.to_csv(reselected_by_bucket_path, index=False)
diagnostic_reselected_by_tenor.to_csv(reselected_by_tenor_path, index=False)
front_forecast_vol_state.to_csv(front_forecast_state_path, index=False)
front_forecast_vol_state_summary.to_csv(front_forecast_state_summary_path, index=False)
rsi_binding_diagnostics.to_csv(rsi_binding_path, index=False)
diagnostic_conclusion.to_csv(diagnostic_conclusion_path, index=False)
diagnostic_reselected.to_parquet(reselected_trade_panel_path, index=False)

# ======================================================================================
# 9. Validation
# ======================================================================================

validation_rows = []

def add_validation(check, status, detail):
    validation_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })

required_selected_cols = [
    "candidate_id",
    "selection_universe",
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
    "forecast_vol_pct",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
]

missing_selected_cols = [c for c in required_selected_cols if c not in selected_panel.columns]

add_validation(
    "selected_trade_panel_loaded",
    "PASS" if not selected_panel.empty else "FAIL",
    f"rows={len(selected_panel):,}; path={selected_trade_panel_path}",
)

add_validation(
    "required_selected_columns_available",
    "PASS" if not missing_selected_cols else "FAIL",
    f"missing={missing_selected_cols}",
)

add_validation(
    "ranked_summary_loaded",
    "PASS" if not ranked.empty else "FAIL",
    f"rows={len(ranked):,}; path={ranked_summary_path}",
)

add_validation(
    "top_candidate_set_created",
    "PASS" if len(top_candidate_ids) > 0 else "FAIL",
    f"top_n={TOP_N_CANDIDATES}; actual={len(top_candidate_ids)}",
)

add_validation(
    "actual_primary_universe_available",
    "PASS" if not actual_primary.empty else "FAIL",
    f"rows={len(actual_primary):,}; universe={PRIMARY_SELECTION_UNIVERSE}",
)

add_validation(
    "all_qualified_universe_available",
    "PASS" if not all_qualified.empty else "FAIL",
    f"rows={len(all_qualified):,}; universe={ALL_QUALIFIED_UNIVERSE}",
)

add_validation(
    "front_actual_by_tenor_created",
    "PASS" if not actual_front_by_tenor.empty else "FAIL",
    f"rows={len(actual_front_by_tenor):,}",
)

add_validation(
    "diagnostic_ex9d_reselection_created",
    "PASS" if not diagnostic_reselected.empty else "FAIL",
    f"rows={len(diagnostic_reselected):,}",
)

add_validation(
    "front_ex9d_comparison_created",
    "PASS" if not front_ex9d_comparison.empty else "FAIL",
    f"rows={len(front_ex9d_comparison):,}",
)

add_validation(
    "forecast_vol_state_diagnostics_created",
    "PASS" if not front_forecast_vol_state.empty else "WARN",
    f"rows={len(front_forecast_vol_state):,}",
)

add_validation(
    "front_rsi_binding_diagnostics_created",
    "PASS" if not rsi_binding_diagnostics.empty else "WARN",
    f"rows={len(rsi_binding_diagnostics):,}",
)

add_validation(
    "top25_front_forecast_vol_floor_boundary_confirmed",
    "WARN" if top25_front_fv_all_at_max else "PASS",
    f"top25_all_at_front_fv_floor_max={top25_front_fv_all_at_max}",
)

add_validation(
    "top_candidate_9d_negative_check",
    "WARN" if top_9d_negative else "PASS",
    f"top_candidate={top_candidate_id}; 9D_avg_pnl_per_day={top_9d_avg}",
)

add_validation(
    "no_new_grid",
    "PASS",
    "No candidate grid created.",
)

add_validation(
    "no_new_sweep",
    "PASS",
    "Only existing selected trades were recombined diagnostically.",
)

add_validation(
    "no_final_candidate_selection",
    "PASS",
    "Diagnostic only.",
)

add_validation(
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed.",
)

add_validation(
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"] == "FAIL"]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1.ipynb",
    "cell": "Cell 10 — Front tenor and boundary diagnostics",
    "timestamp": CELL10_TIMESTAMP,
    "top_n_candidates": TOP_N_CANDIDATES,
    "primary_selection_universe": PRIMARY_SELECTION_UNIVERSE,
    "all_qualified_universe": ALL_QUALIFIED_UNIVERSE,
    "selected_trade_panel_path": str(selected_trade_panel_path),
    "ranked_summary_path": str(ranked_summary_path),
    "ranked_by_tenor_path": str(ranked_by_tenor_path),
    "ranked_by_bucket_path": str(ranked_by_bucket_path),
    "ranked_by_year_path": str(ranked_by_year_path),
    "front_tenor_summary_path": str(front_tenor_summary_path),
    "front_summary_path": str(front_summary_path),
    "front_ex9d_comparison_path": str(front_ex9d_comparison_path),
    "reselected_summary_path": str(reselected_summary_path),
    "reselected_by_bucket_path": str(reselected_by_bucket_path),
    "reselected_by_tenor_path": str(reselected_by_tenor_path),
    "front_forecast_state_path": str(front_forecast_state_path),
    "front_forecast_state_summary_path": str(front_forecast_state_summary_path),
    "rsi_binding_path": str(rsi_binding_path),
    "diagnostic_conclusion_path": str(diagnostic_conclusion_path),
    "reselected_trade_panel_path": str(reselected_trade_panel_path),
    "validation_path": str(validation_path),
    "top_candidate_id": str(top_candidate_id),
    "top_candidate_9d_avg_pnl_per_day": top_9d_avg,
    "top_candidate_12d_avg_pnl_per_day": top_12d_avg,
    "top_candidate_15d_avg_pnl_per_day": top_15d_avg,
    "top25_front_forecast_vol_floor_all_at_grid_max": top25_front_fv_all_at_max,
    "notes": [
        "Diagnostic only.",
        "No new threshold grid.",
        "No new sweep expansion.",
        "No final candidate selection.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
        "Diagnostic ex-9D selection recombines already-qualified trades only.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 10 validation failed. Review validation output above.")

# ======================================================================================
# 10. Display review
# ======================================================================================

print("=" * 100)
print("Cell 10 — Front tenor and boundary diagnostics")
print("=" * 100)
print(f"Selected trade panel:                   {selected_trade_panel_path}")
print(f"Ranked summary:                         {ranked_summary_path}")
print(f"Top candidates reviewed:                {len(top_candidate_ids):,}")
print(f"Actual primary rows reviewed:           {len(actual_primary):,}")
print(f"All-qualified rows reviewed:            {len(all_qualified):,}")
print(f"Diagnostic ex-9D reselected rows:        {len(diagnostic_reselected):,}")
print(f"Top candidate:                          {top_candidate_id}")
print(f"Top candidate 9D avg P&L/day:            {top_9d_avg:,.2f}")
print(f"Top candidate 12D avg P&L/day:           {top_12d_avg:,.2f}")
print(f"Top candidate 15D avg P&L/day:           {top_15d_avg:,.2f}")
print(f"Top 25 Front FV floor all at grid max:   {top25_front_fv_all_at_max}")
print("No new grid:                            True")
print("No new sweep:                           True")
print("No final candidate selection:           True")
print("No Secondary:                           True")
print("No sizing changes:                      True")
print("No production lock:                     True")

print()
print("Validation:")
display(validation)

print()
print("Diagnostic conclusions:")
display(diagnostic_conclusion)

print()
print("Top 20 — actual Front selected by tenor:")
display(
    actual_front_by_tenor[
        actual_front_by_tenor["review_rank"].le(20)
    ].sort_values(["review_rank", "tenor"])
)

print()
print("Top 20 — actual vs diagnostic one-trade-per-bucket/date excluding Front 9D:")
compare_display_cols = [
    "review_rank",
    "candidate_id",
    "actual_trades",
    "ex9d_trades",
    "delta_ex9d_minus_actual_trades",
    "actual_avg_pnl_per_day",
    "ex9d_avg_pnl_per_day",
    "delta_ex9d_minus_actual_avg_pnl_per_day",
    "actual_total_pnl",
    "ex9d_total_pnl",
    "delta_ex9d_minus_actual_total_pnl",
    "actual_profit_factor",
    "ex9d_profit_factor",
    "delta_ex9d_minus_actual_profit_factor",
    "actual_pnl_per_day_drawdown",
    "ex9d_pnl_per_day_drawdown",
    "delta_ex9d_minus_actual_pnl_per_day_drawdown",
    "actual_avg_pnl_per_day_2026",
    "ex9d_avg_pnl_per_day_2026",
    "delta_ex9d_minus_actual_avg_pnl_per_day_2026",
    "Front_model_vrp_log",
    "Front_model_vrp_z",
    "Front_RSI14_max",
    "Front_forecast_vol_pct_min",
    "Middle_model_vrp_log",
    "Middle_model_vrp_z",
    "Middle_RSI14_max",
    "Middle_forecast_vol_pct_min",
]
display(front_ex9d_comparison[[c for c in compare_display_cols if c in front_ex9d_comparison.columns]].head(20))

print()
print("Top 20 — diagnostic ex-9D reselected by bucket:")
display(
    diagnostic_reselected_by_bucket[
        diagnostic_reselected_by_bucket["review_rank"].le(20)
    ].sort_values(["review_rank", "tenor_bucket_order"])
)

print()
print("Top 20 — diagnostic ex-9D reselected by tenor:")
display(
    diagnostic_reselected_by_tenor[
        diagnostic_reselected_by_tenor["review_rank"].le(20)
    ].sort_values(["review_rank", "tenor_bucket_order", "tenor"])
)

print()
print("Top 20 — Front all-qualified by forecast-vol state and tenor:")
display(
    front_forecast_vol_state[
        front_forecast_vol_state["review_rank"].le(20)
    ].sort_values(["review_rank", "tenor", "front_forecast_vol_bin"])
)

print()
print("Front RSI binding diagnostics:")
display(rsi_binding_diagnostics.head(30))

print()
print("Saved outputs:")
for p in [
    front_tenor_summary_path,
    front_summary_path,
    front_ex9d_comparison_path,
    reselected_summary_path,
    reselected_by_bucket_path,
    reselected_by_tenor_path,
    front_forecast_state_path,
    front_forecast_state_summary_path,
    rsi_binding_path,
    diagnostic_conclusion_path,
    reselected_trade_panel_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 10 PASSED")

C:\Users\patri\AppData\Local\Temp\ipykernel_21560\1391526047.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for keys, g in d.groupby(group_cols, dropna=False):
C:\Users\patri\AppData\Local\Temp\ipykernel_21560\1391526047.py:178: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for keys, g in d.groupby(group_cols, dropna=False):


Cell 10 — Front tenor and boundary diagnostics
Selected trade panel:                   C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\06_core_front_expanded_sweep_selected_trade_panel_20260704_215631.parquet
Ranked summary:                         C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\07_core_candidate_ranked_summary_20260704_220821.csv
Top candidates reviewed:                50
Actual primary rows reviewed:           21,411
All-qualified rows reviewed:            58,599
Diagnostic ex-9D reselected rows:        21,049
Top candidate:                          core_front_expanded_00168
Top candidate 9D avg P&L/day:            -129.44
Top candidate 12D avg P&L/day:           691.72
Top candidate 15D avg P&L/day:           907.17
Top 25 Front FV floor all at grid max:   True
No new grid:                            True
No new sweep:                           True
No final candidate selection:     

,check,status,detail
0,selected_trade_panel_loaded,PASS,"rows=659,356; path=C:\Users\patri\vrp_project\..."
1,required_selected_columns_available,PASS,missing=[]
2,ranked_summary_loaded,PASS,rows=320; path=C:\Users\patri\vrp_project\data...
3,top_candidate_set_created,PASS,top_n=50; actual=50
4,actual_primary_universe_available,PASS,"rows=21,411; universe=one_trade_per_bucket_per..."
5,all_qualified_universe_available,PASS,"rows=58,599; universe=all_qualified_trades"
6,front_actual_by_tenor_created,PASS,rows=150
7,diagnostic_ex9d_reselection_created,PASS,"rows=21,049"
8,front_ex9d_comparison_created,PASS,rows=50
9,forecast_vol_state_diagnostics_created,PASS,rows=600



Diagnostic conclusions:


,diagnostic,status,detail
0,top25_front_forecast_vol_floor_all_at_grid_max,WARN,top25_all_at_front_fv_floor_max=True
1,top_candidate_9d_avg_pnl_per_day_negative,WARN,top_candidate=core_front_expanded_00168; 9D_av...
2,top_candidate_12d_and_15d_positive,PASS,top_candidate=core_front_expanded_00168; 12D=6...
3,diagnostic_excluding_9d_improves_top_candidate...,INFO,top_candidate=core_front_expanded_00168; ex9d_...
4,front_rsi_nonbinding_groups_found,INFO,nonbinding_trade_set_groups=19
5,no_new_grid_or_sweep,PASS,Cell 10 uses existing Cell 8/9 outputs only.
6,no_final_lock,PASS,Cell 10 is diagnostic only.


...


Top 20 — Front all-qualified by forecast-vol state and tenor:


,diagnostic_label,candidate_id,tenor,front_forecast_vol_bin,trades,unique_trade_dates,date_min,date_max,tenor_min,tenor_max,avg_tenor,win_rate,total_pnl,avg_pnl,median_pnl,worst_trade,best_trade,profit_factor,selected_trade_drawdown,worst_5_trade_sum,worst_10_trade_sum,worst_20_trade_sum,trades_le_neg_50k,trades_le_neg_100k,trades_le_neg_150k,total_pnl_per_day_sum,avg_pnl_per_day,median_pnl_per_day,worst_trade_pnl_per_day,best_trade_pnl_per_day,profit_factor_pnl_per_day,pnl_per_day_drawdown,worst_5_trade_pnl_per_day_sum,worst_10_trade_pnl_per_day_sum,worst_20_trade_pnl_per_day_sum,trades_2025,pnl_2025,avg_pnl_per_day_2025,worst_trade_2025,trades_2026,pnl_2026,avg_pnl_per_day_2026,worst_trade_2026,avg_forecast_vol_pct,min_forecast_vol_pct,max_forecast_vol_pct,avg_model_vrp_log,avg_model_vrp_z_3m,avg_model_vrp_z_1y,review_rank,robustness_screen_pass,robustness_score_after_boundary_penalty,Front_model_vrp_log,Front_model_vrp_z,Front_RSI14_max,Front_forecast_vol_pct_min,Middle_model_vrp_log,Middle_model_vrp_z,Middle_RSI14_max,Middle_forecast_vol_pct_min,Back_model_vrp_log,Back_model_vrp_z,Back_RSI14_max,Back_forecast_vol_pct_min,Front_forecast_vol_pct_min_at_grid_max,Front_model_vrp_log_at_grid_max
264,front_all_qualified_by_tenor_and_forecast_vol_...,core_front_expanded_00168,9,14.5-16.0,27,27,2021-09-20,2026-03-25,9,9,9.0,0.592593,-48888.317355,-1810.678421,11929.277477,-93387.142625,19447.414501,0.807317,-114818.822346,-97275.376201,-96978.617527,-37285.054805,1,0,0,-5432.035262,-201.186491,1325.475275,-10376.349181,2160.823833,0.807317,-12757.646927,-10808.375133,-10775.401947,-4142.783867,3,37442.662838,1386.765290,12324.018696,10,-44064.330492,-489.603672,-34533.046342,15.415351,14.849697,15.922100,1.056021,1.444606,1.449487,1,True,0.846453,0.70,0.65,68.0,14.5,0.7,0.7,68.0,10.0,0.7,0.7,70.0,8.5,True,False
265,front_all_qualified_by_tenor_and_forecast_vol_...,core_front_expanded_00168,9,16.0-18.0,38,38,2021-12-03,2026-03-27,9,9,9.0,0.710526,145677.702150,3833.623741,13744.511376,-91511.926686,20272.650373,1.611993,-139236.750214,-86725.249685,-139236.750214,14560.733750,1,0,0,16186.411350,425.958193,1527.167931,-10167.991854,2252.516708,1.611993,-15470.750024,-9636.138854,-15470.750024,1617.859306,7,1703.884091,27.045779,-40441.053294,3,20420.243016,756.305297,-7630.854164,16.791175,16.010776,17.679317,0.991156,1.442374,1.512091,1,True,0.846453,0.70,0.65,68.0,14.5,0.7,0.7,68.0,10.0,0.7,0.7,70.0,8.5,True,False
266,front_all_qualified_by_tenor_and_forecast_vol_...,core_front_expanded_00168,9,18.0-20.0,26,26,2021-11-26,2026-03-30,9,9,9.0,0.692308,147582.539325,5676.251513,16309.807045,-51422.095318,20796.749647,1.874678,-156274.990752,-131403.979248,-90995.981143,45315.327066,1,0,0,16398.059925,630.694613,1812.200783,-5713.566146,2310.749961,1.874678,-17363.887861,-14600.442139,-10110.664571,5035.036341,0,0.000000,NaN,NaN,1,17607.965043,1956.440560,17607.965043,19.103491,18.156373,19.982162,0.983056,1.249816,1.257193,1,True,0.846453,0.70,0.65,68.0,14.5,0.7,0.7,68.0,10.0,0.7,0.7,70.0,8.5,True,False
267,front_all_qualified_by_tenor_and_forecast_vol_...,core_front_expanded_00168,9,>20.0,11,11,2022-03-07,2025-04-08,9,9,9.0,1.000000,217909.736059,19809.976005,19811.415388,1145.711413,34077.430827,inf,0.000000,68402.991829,183832.305233,217909.736059,0,0,0,24212.192895,2201.108445,2201.268376,127.301268,3786.381203,inf,0.000000,7600.332425,20425.811693,24212.192895,2,68133.433592,3785.190755,34056.002766,0,0.000000,NaN,NaN,25.431652,20.093698,44.221370,0.953874,1.554130,1.574822,1,True,0.846453,0.70,0.65,68.0,14.5,0.7,0.7,68.0,10.0,0.7,0.7,70.0,8.5,True,False
268,front_all_qualified_by_tenor_and_forecast_vol_...,core_front_expanded_00168,12,14.5-16.0,37,37,2022-01-31,2026-03-25,12,12,12.0,0.729730,154992.434469,4188.984715,13970.305530,-93387.142625,22246.736532,1.615540,-105596.559441,-111434.165049,-69242.240846,62727.148829,1,0,0,12916.036206,349.082060,1164.192127,-7782.261885,1853.894711,1.615540,-8799.713287,-9286.180421,-5770.1


Front RSI binding diagnostics:


,Front_model_vrp_log,Front_model_vrp_z,Front_forecast_vol_pct_min,Middle_model_vrp_log,Middle_model_vrp_z,Middle_RSI14_max,Middle_forecast_vol_pct_min,Back_model_vrp_log,Back_model_vrp_z,Back_RSI14_max,Back_forecast_vol_pct_min,candidate_count,candidate_ids,review_ranks,front_rsi_values,unique_front_rsi_values,unique_actual_primary_trade_sets,unique_actual_front_trade_sets,min_avg_pnl_per_day,max_avg_pnl_per_day,min_2026_avg_pnl_per_day,max_2026_avg_pnl_per_day,min_trades,max_trades,front_rsi_nonbinding_trade_set_test,front_rsi_nonbinding_metric_test
0,0.70,0.65,14.5,0.70,0.7,68.0,10.0,0.7,0.7,70.0,8.5,3,"core_front_expanded_00168, core_front_expanded...","1, 2, 3","64.0, 66.0, 68.0",3,1,1,547.859866,547.859866,77.113098,77.113098,428,428,True,True
1,0.70,0.65,14.5,0.70,0.7,68.0,8.5,0.7,0.7,70.0,8.5,3,"core_front_expanded_00167, core_front_expanded...","4, 5, 6","64.0, 66.0, 68.0",3,1,1,547.404931,547.404931,77.113098,77.113098,432,432,True,True
2,0.75,0.65,14.5,0.70,0.7,68.0,10.0,0.7,0.7,70.0,8.5,3,"core_front_expanded_00248, core_front_expanded...","7, 8, 9","64.0, 66.0, 68.0",3,1,1,545.335363,545.335363,77.113098,77.113098,423,423,True,True
3,0.65,0.65,14.5,0.70,0.7,68.0,10.0,0.7,0.7,70.0,8.5,3,"core_front_expanded_00016, core_front_expanded...","22, 23, 24","64.0, 66.0, 68.0",3,1,1,545.126315,545.126315,77.113098,77.113098,431,431,True,True
4,0.65,0.65,14.5,0.65,0.7,68.0,10.0,0.7,0.7,70.0,8.5,3,"core_front_expanded_00014, core_front_expanded...","17, 18, 19","64.0, 66.0, 68.0",3,1,1,545.080270,545.080270,85.752231,85.752231,438,438,True,True
5,0.75,0.65,14.5,0.70,0.7,68.0,8.5,0.7,0.7,70.0,8.5,3,"core_front_expanded_00247, core_front_expanded...","12, 13, 14","64.0, 66.0, 68.0",3,1,1,544.898750,544.898750,77.113098,77.113098,427,427,True,True
6,0.65,0.65,14.5,0.70,0.7,68.0,8.5,0.7,0.7,70.0,8.5,3,"core_front_expanded_00015, core_front_expanded...","30, 31, 32","64.0, 66.0, 68.0",3,1,1,544.699653,544.699653,77.113098,77.113098,435,435,True,True
7,0.65,0.65,14.5,0.65,0.7,68.0,8.5,0.7,0.7,70.0,8.5,3,"core_front_expanded_00013, core_front_expanded...","27, 28, 29","64.0, 66.0, 68.0",3,1,1,544.660782,544.660782,85.752231,85.752231,442,442,True,True
8,0.70,0.70,14.5,0.70,0.7,68.0,10.0,0.7,0.7,70.0,8.5,3,"core_front_expanded_00208, core_front_expanded...","33, 34, 35","64.0, 66.0, 68.0",3,1,1,536.425019,536.425019,77.113098,77.113098,422,422,True,True
9,0.70,0.70,14.5,0.70,0.7,68.0,8.5,0.7,0.7,70.0,8.5,3,"core_front_expanded_00207, core_front_expanded...","36, 37, 38","64.0, 66.0, 68.0",3,1,1,536.071045,536.071045,77.113098,77.113098,426,426,True,True



Saved outputs:
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\08_front_tenor_boundary_actual_front_by_tenor_20260704_221323.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\08_front_tenor_boundary_front_summary_20260704_221323.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\08_front_tenor_boundary_ex9d_comparison_20260704_221323.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\08_front_tenor_boundary_ex9d_reselected_summary_20260704_221323.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\08_front_tenor_boundary_ex9d_reselected_by_bucket_20260704_221323.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\08_front_tenor_boundary_ex9d_reselected_by_tenor_20260704_221323.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_f

In [13]:
# ======================================================================================
# Cell 11 — 9D-specific Front rule candidate design
# ======================================================================================
#
# Objective:
#   Create a small diagnostic candidate grid to compare:
#       1. Core Front excluding 9D entirely.
#       2. Core Front allowing 9D only with a higher 9D-specific forecast-vol floor.
#
# Why:
#   Cell 10 showed:
#       - Top-ranked expanded candidate still had Front forecast-vol floor at grid max 14.5.
#       - 9D remained negative under the shared Front rule.
#       - 12D and 15D were strong.
#       - 9D improved materially as forecast-vol state increased.
#
# This cell only designs and saves candidate rules.
#
# Important:
#   This cell does NOT evaluate trades.
#   This cell does NOT run a sweep.
#   This cell does NOT select final thresholds.
#   This cell does NOT lock production.
#
# Candidate families:
#   A. front_ex_9d:
#        - 9D excluded from Core Front.
#        - 12D and 15D use the selected Front rule.
#
#   B. front_conditional_9d:
#        - 12D and 15D use the selected Front rule.
#        - 9D is allowed only with a higher 9D-specific forecast-vol floor:
#              16.0, 18.0, 20.0
#
# Fixed reference rule used for this diagnostic design:
#   Front 12D/15D:
#       model_vrp_log > 0.70
#       model_vrp_z   > 0.65
#       RSI14         < 68
#       forecast vol  > 14.5
#
#   Middle:
#       model_vrp_log > 0.70
#       model_vrp_z   > 0.70
#       RSI14         < 68
#       forecast vol  > 10.0
#
#   Back:
#       locked unchanged
#
# Not in scope:
#   - No trade evaluation.
#   - No parameter expansion beyond the 9D forecast-vol floor test.
#   - No Secondary.
#   - No sizing changes.
#   - No production lock.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 420)

# ======================================================================================
# 0. Fallback config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL11_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

if "ALL_TENORS" not in globals():
    ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

if "TENOR_BUCKET_MAP" not in globals():
    TENOR_BUCKET_MAP = {
        9: "Front",
        12: "Front",
        15: "Front",
        18: "Middle",
        21: "Middle",
        24: "Middle",
        27: "Back",
        30: "Back",
        33: "Back",
    }

if "BUCKET_ORDER_MAP" not in globals():
    BUCKET_ORDER_MAP = {
        "Front": 1,
        "Middle": 2,
        "Back": 3,
    }

if "LOCKED_BACK_CORE_THRESHOLDS" not in globals():
    LOCKED_BACK_CORE_THRESHOLDS = {
        "model_vrp_log": 0.70,
        "model_vrp_z_3m": 0.70,
        "model_vrp_z_1y": 0.70,
        "RSI14_max": 70.0,
        "forecast_vol_pct_min": 8.5,
    }

CANDIDATE_FAMILY = "core_front_9d_rule_test"

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def tenor_bucket(tenor: int) -> str:
    return TENOR_BUCKET_MAP[int(tenor)]


def threshold_dict(
    model_vrp_log: float,
    model_vrp_z: float,
    rsi_max: float,
    forecast_vol_pct_min: float,
) -> dict:
    return {
        "model_vrp_log": float(model_vrp_log),
        "model_vrp_z": float(model_vrp_z),
        "model_vrp_z_3m": float(model_vrp_z),
        "model_vrp_z_1y": float(model_vrp_z),
        "RSI14_max": float(rsi_max),
        "forecast_vol_pct_min": float(forecast_vol_pct_min),
    }


# ======================================================================================
# 2. Load context inputs
# ======================================================================================

signal_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01_unified_fds_signal_base_panel_*.parquet",
    required=True,
)

ranked_summary_path = latest_file(
    OUT_AUDIT_DIR,
    "07_core_candidate_ranked_summary_*.csv",
    required=True,
)

front_tenor_diag_path = latest_file(
    OUT_AUDIT_DIR,
    "08_front_tenor_boundary_actual_front_by_tenor_*.csv",
    required=False,
)

front_ex9d_comparison_path = latest_file(
    OUT_AUDIT_DIR,
    "08_front_tenor_boundary_ex9d_comparison_*.csv",
    required=False,
)

front_forecast_state_path = latest_file(
    OUT_AUDIT_DIR,
    "08_front_tenor_boundary_forecast_vol_state_by_tenor_*.csv",
    required=False,
)

signal = pd.read_parquet(signal_panel_path)
ranked_summary = pd.read_csv(ranked_summary_path)

signal["trade_date"] = pd.to_datetime(signal["trade_date"], errors="coerce").dt.normalize()
signal["tenor"] = pd.to_numeric(signal["tenor"], errors="coerce").astype(int)

# Context review only. Not used to evaluate or select trades.
context_rows = []

context_rows.append({
    "context_item": "signal_panel_path",
    "value": str(signal_panel_path),
})

context_rows.append({
    "context_item": "ranked_summary_path",
    "value": str(ranked_summary_path),
})

context_rows.append({
    "context_item": "front_tenor_diag_path",
    "value": str(front_tenor_diag_path) if front_tenor_diag_path is not None else None,
})

context_rows.append({
    "context_item": "front_ex9d_comparison_path",
    "value": str(front_ex9d_comparison_path) if front_ex9d_comparison_path is not None else None,
})

context_rows.append({
    "context_item": "front_forecast_state_path",
    "value": str(front_forecast_state_path) if front_forecast_state_path is not None else None,
})

context_rows.append({
    "context_item": "signal_rows",
    "value": int(len(signal)),
})

context_rows.append({
    "context_item": "signal_date_min",
    "value": str(signal["trade_date"].min().date()),
})

context_rows.append({
    "context_item": "signal_date_max",
    "value": str(signal["trade_date"].max().date()),
})

context_rows.append({
    "context_item": "signal_tenors",
    "value": str(sorted(signal["tenor"].dropna().unique().tolist())),
})

if not ranked_summary.empty and "review_rank" in ranked_summary.columns:
    top_ranked = ranked_summary.sort_values("review_rank").head(10).copy()

    for _, row in top_ranked.iterrows():
        context_rows.append({
            "context_item": f"top_ranked_candidate_{int(row['review_rank'])}",
            "value": str(row.get("candidate_id")),
        })

context_review = pd.DataFrame(context_rows)

# ======================================================================================
# 3. Fixed reference rules for the 9D-specific candidate design
# ======================================================================================

FRONT_12_15_RULE = threshold_dict(
    model_vrp_log=0.70,
    model_vrp_z=0.65,
    rsi_max=68.0,
    forecast_vol_pct_min=14.5,
)

# For conditional 9D candidates, keep log/z/RSI same as Front rule, but raise FV floor.
NINE_D_BASE_RULE = threshold_dict(
    model_vrp_log=0.70,
    model_vrp_z=0.65,
    rsi_max=68.0,
    forecast_vol_pct_min=14.5,
)

MIDDLE_RULE = threshold_dict(
    model_vrp_log=0.70,
    model_vrp_z=0.70,
    rsi_max=68.0,
    forecast_vol_pct_min=10.0,
)

BACK_RULE = threshold_dict(
    model_vrp_log=LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"],
    model_vrp_z=LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"],
    rsi_max=LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"],
    forecast_vol_pct_min=LOCKED_BACK_CORE_THRESHOLDS["forecast_vol_pct_min"],
)

NINE_D_CONDITIONAL_FORECAST_VOL_FLOORS = [16.0, 18.0, 20.0]

# ======================================================================================
# 4. Build wide candidate design
# ======================================================================================

wide_rows = []

# Candidate A: exclude 9D.
wide_rows.append({
    "candidate_id": "core_front_9d_rule_0001",
    "candidate_family": CANDIDATE_FAMILY,
    "candidate_subfamily": "front_ex_9d",
    "candidate_description": "Exclude 9D from Core Front; use Front rule for 12D/15D only.",
    "locked_model_spec": LOCKED_MODEL_SPEC,

    "front_9d_include": False,
    "front_9d_rule_type": "excluded",
    "front_9d_model_vrp_log": NINE_D_BASE_RULE["model_vrp_log"],
    "front_9d_model_vrp_z": NINE_D_BASE_RULE["model_vrp_z"],
    "front_9d_RSI14_max": NINE_D_BASE_RULE["RSI14_max"],
    "front_9d_forecast_vol_pct_min": np.nan,

    "front_12_15_include": True,
    "front_12_15_model_vrp_log": FRONT_12_15_RULE["model_vrp_log"],
    "front_12_15_model_vrp_z": FRONT_12_15_RULE["model_vrp_z"],
    "front_12_15_RSI14_max": FRONT_12_15_RULE["RSI14_max"],
    "front_12_15_forecast_vol_pct_min": FRONT_12_15_RULE["forecast_vol_pct_min"],

    "middle_include": True,
    "middle_model_vrp_log": MIDDLE_RULE["model_vrp_log"],
    "middle_model_vrp_z": MIDDLE_RULE["model_vrp_z"],
    "middle_RSI14_max": MIDDLE_RULE["RSI14_max"],
    "middle_forecast_vol_pct_min": MIDDLE_RULE["forecast_vol_pct_min"],

    "back_include": True,
    "back_model_vrp_log": BACK_RULE["model_vrp_log"],
    "back_model_vrp_z": BACK_RULE["model_vrp_z"],
    "back_RSI14_max": BACK_RULE["RSI14_max"],
    "back_forecast_vol_pct_min": BACK_RULE["forecast_vol_pct_min"],
})

# Candidate B family: conditional 9D with higher FV floor.
for i, nine_d_fv_floor in enumerate(NINE_D_CONDITIONAL_FORECAST_VOL_FLOORS, start=2):
    wide_rows.append({
        "candidate_id": f"core_front_9d_rule_{i:04d}",
        "candidate_family": CANDIDATE_FAMILY,
        "candidate_subfamily": "front_conditional_9d",
        "candidate_description": f"Allow 9D only if 9D forecast vol > {nine_d_fv_floor}; use Front rule for 12D/15D.",
        "locked_model_spec": LOCKED_MODEL_SPEC,

        "front_9d_include": True,
        "front_9d_rule_type": "conditional_higher_forecast_vol_floor",
        "front_9d_model_vrp_log": NINE_D_BASE_RULE["model_vrp_log"],
        "front_9d_model_vrp_z": NINE_D_BASE_RULE["model_vrp_z"],
        "front_9d_RSI14_max": NINE_D_BASE_RULE["RSI14_max"],
        "front_9d_forecast_vol_pct_min": float(nine_d_fv_floor),

        "front_12_15_include": True,
        "front_12_15_model_vrp_log": FRONT_12_15_RULE["model_vrp_log"],
        "front_12_15_model_vrp_z": FRONT_12_15_RULE["model_vrp_z"],
        "front_12_15_RSI14_max": FRONT_12_15_RULE["RSI14_max"],
        "front_12_15_forecast_vol_pct_min": FRONT_12_15_RULE["forecast_vol_pct_min"],

        "middle_include": True,
        "middle_model_vrp_log": MIDDLE_RULE["model_vrp_log"],
        "middle_model_vrp_z": MIDDLE_RULE["model_vrp_z"],
        "middle_RSI14_max": MIDDLE_RULE["RSI14_max"],
        "middle_forecast_vol_pct_min": MIDDLE_RULE["forecast_vol_pct_min"],

        "back_include": True,
        "back_model_vrp_log": BACK_RULE["model_vrp_log"],
        "back_model_vrp_z": BACK_RULE["model_vrp_z"],
        "back_RSI14_max": BACK_RULE["RSI14_max"],
        "back_forecast_vol_pct_min": BACK_RULE["forecast_vol_pct_min"],
    })

candidate_grid_wide = pd.DataFrame(wide_rows)

# Convenience diagnostics.
candidate_grid_wide["z_3m_equals_z_1y_within_rule"] = True
candidate_grid_wide["back_locked"] = (
    (candidate_grid_wide["back_model_vrp_log"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"])
    & (candidate_grid_wide["back_model_vrp_z"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"])
    & (candidate_grid_wide["back_RSI14_max"] == LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"])
    & (candidate_grid_wide["back_forecast_vol_pct_min"] == LOCKED_BACK_CORE_THRESHOLDS["forecast_vol_pct_min"])
)
candidate_grid_wide["front_12_15_same_as_top_reference"] = (
    (candidate_grid_wide["front_12_15_model_vrp_log"] == 0.70)
    & (candidate_grid_wide["front_12_15_model_vrp_z"] == 0.65)
    & (candidate_grid_wide["front_12_15_RSI14_max"] == 68.0)
    & (candidate_grid_wide["front_12_15_forecast_vol_pct_min"] == 14.5)
)
candidate_grid_wide["middle_same_as_top_reference"] = (
    (candidate_grid_wide["middle_model_vrp_log"] == 0.70)
    & (candidate_grid_wide["middle_model_vrp_z"] == 0.70)
    & (candidate_grid_wide["middle_RSI14_max"] == 68.0)
    & (candidate_grid_wide["middle_forecast_vol_pct_min"] == 10.0)
)

# ======================================================================================
# 5. Build tenor-level long candidate design
# ======================================================================================
#
# This is the practical input for Cell 12.
# Each row is candidate_id × tenor.
#
# include_tenor=False means the tenor is excluded from that candidate's Core rules.
# ======================================================================================

long_rows = []

for _, row in candidate_grid_wide.iterrows():
    candidate_id = row["candidate_id"]

    for tenor in ALL_TENORS:
        bucket = tenor_bucket(tenor)

        if bucket == "Front" and tenor == 9:
            include_tenor = bool(row["front_9d_include"])
            rule_scope = "front_9d"
            rule_type = row["front_9d_rule_type"]
            model_vrp_log = row["front_9d_model_vrp_log"]
            model_vrp_z = row["front_9d_model_vrp_z"]
            rsi_max = row["front_9d_RSI14_max"]
            fv_floor = row["front_9d_forecast_vol_pct_min"]

        elif bucket == "Front" and tenor in [12, 15]:
            include_tenor = bool(row["front_12_15_include"])
            rule_scope = "front_12_15"
            rule_type = "shared_front_12_15_rule"
            model_vrp_log = row["front_12_15_model_vrp_log"]
            model_vrp_z = row["front_12_15_model_vrp_z"]
            rsi_max = row["front_12_15_RSI14_max"]
            fv_floor = row["front_12_15_forecast_vol_pct_min"]

        elif bucket == "Middle":
            include_tenor = bool(row["middle_include"])
            rule_scope = "middle"
            rule_type = "shared_middle_rule"
            model_vrp_log = row["middle_model_vrp_log"]
            model_vrp_z = row["middle_model_vrp_z"]
            rsi_max = row["middle_RSI14_max"]
            fv_floor = row["middle_forecast_vol_pct_min"]

        elif bucket == "Back":
            include_tenor = bool(row["back_include"])
            rule_scope = "back"
            rule_type = "locked_back_rule"
            model_vrp_log = row["back_model_vrp_log"]
            model_vrp_z = row["back_model_vrp_z"]
            rsi_max = row["back_RSI14_max"]
            fv_floor = row["back_forecast_vol_pct_min"]

        else:
            raise ValueError(f"Unexpected tenor/bucket combination: tenor={tenor}, bucket={bucket}")

        long_rows.append({
            "candidate_id": candidate_id,
            "candidate_family": row["candidate_family"],
            "candidate_subfamily": row["candidate_subfamily"],
            "locked_model_spec": row["locked_model_spec"],
            "tenor": int(tenor),
            "tenor_bucket": bucket,
            "tenor_bucket_order": BUCKET_ORDER_MAP[bucket],
            "rule_scope": rule_scope,
            "rule_type": rule_type,
            "include_tenor": include_tenor,

            "model_vrp_log": float(model_vrp_log) if pd.notna(model_vrp_log) else np.nan,
            "model_vrp_z": float(model_vrp_z) if pd.notna(model_vrp_z) else np.nan,
            "model_vrp_z_3m": float(model_vrp_z) if pd.notna(model_vrp_z) else np.nan,
            "model_vrp_z_1y": float(model_vrp_z) if pd.notna(model_vrp_z) else np.nan,
            "RSI14_max": float(rsi_max) if pd.notna(rsi_max) else np.nan,
            "forecast_vol_pct_min": float(fv_floor) if pd.notna(fv_floor) else np.nan,

            "is_front_9d": bucket == "Front" and tenor == 9,
            "is_front_12_15": bucket == "Front" and tenor in [12, 15],
            "is_back_locked": bucket == "Back",
        })

candidate_grid_long = (
    pd.DataFrame(long_rows)
    .sort_values(["candidate_id", "tenor"])
    .reset_index(drop=True)
)

# ======================================================================================
# 6. Save outputs
# ======================================================================================

wide_csv_path = OUT_AUDIT_DIR / f"09_core_front_9d_rule_candidate_grid_wide_{CELL11_TIMESTAMP}.csv"
wide_parquet_path = OUT_PROCESSED_DIR / f"09_core_front_9d_rule_candidate_grid_wide_{CELL11_TIMESTAMP}.parquet"

long_csv_path = OUT_AUDIT_DIR / f"09_core_front_9d_rule_candidate_grid_long_{CELL11_TIMESTAMP}.csv"
long_parquet_path = OUT_PROCESSED_DIR / f"09_core_front_9d_rule_candidate_grid_long_{CELL11_TIMESTAMP}.parquet"

context_path = OUT_AUDIT_DIR / f"09_core_front_9d_rule_candidate_grid_context_{CELL11_TIMESTAMP}.csv"
ranges_path = OUT_AUDIT_DIR / f"09_core_front_9d_rule_candidate_grid_ranges_{CELL11_TIMESTAMP}.json"
validation_path = OUT_AUDIT_DIR / f"09_core_front_9d_rule_candidate_grid_validation_{CELL11_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"09_core_front_9d_rule_candidate_grid_manifest_{CELL11_TIMESTAMP}.json"

candidate_grid_wide.to_csv(wide_csv_path, index=False)
candidate_grid_wide.to_parquet(wide_parquet_path, index=False)

candidate_grid_long.to_csv(long_csv_path, index=False)
candidate_grid_long.to_parquet(long_parquet_path, index=False)

context_review.to_csv(context_path, index=False)

grid_ranges = {
    "candidate_family": CANDIDATE_FAMILY,
    "front_12_15_rule": FRONT_12_15_RULE,
    "nine_d_base_rule": NINE_D_BASE_RULE,
    "nine_d_candidate_forecast_vol_pct_min": NINE_D_CONDITIONAL_FORECAST_VOL_FLOORS,
    "middle_rule": MIDDLE_RULE,
    "back_rule": BACK_RULE,
    "candidate_subfamilies": {
        "front_ex_9d": {
            "description": "9D excluded from Core Front. 12D/15D use Front rule.",
            "candidate_count": int(candidate_grid_wide["candidate_subfamily"].eq("front_ex_9d").sum()),
        },
        "front_conditional_9d": {
            "description": "9D included only with higher 9D-specific forecast-vol floor.",
            "candidate_count": int(candidate_grid_wide["candidate_subfamily"].eq("front_conditional_9d").sum()),
        },
    },
}

with open(ranges_path, "w", encoding="utf-8") as f:
    json.dump(grid_ranges, f, indent=2, default=str)

# ======================================================================================
# 7. Validation
# ======================================================================================

validation_rows = []

def add_validation(check, status, detail):
    validation_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })

candidate_count = len(candidate_grid_wide)
expected_candidate_count = 1 + len(NINE_D_CONDITIONAL_FORECAST_VOL_FLOORS)

long_row_count = len(candidate_grid_long)
expected_long_row_count = candidate_count * len(ALL_TENORS)

missing_tenors_by_candidate = (
    candidate_grid_long
    .groupby("candidate_id")["tenor"]
    .apply(lambda x: sorted(set(ALL_TENORS) - set(x.tolist())))
    .reset_index(name="missing_tenors")
)

bad_missing = missing_tenors_by_candidate[
    missing_tenors_by_candidate["missing_tenors"].apply(len).gt(0)
]

bad_z_equality = candidate_grid_long[
    candidate_grid_long["include_tenor"]
    & (
        candidate_grid_long["model_vrp_z_3m"].ne(candidate_grid_long["model_vrp_z_1y"])
    )
]

bad_ex9d = candidate_grid_long[
    candidate_grid_long["candidate_subfamily"].eq("front_ex_9d")
    & candidate_grid_long["is_front_9d"]
    & candidate_grid_long["include_tenor"]
]

bad_conditional_9d = candidate_grid_long[
    candidate_grid_long["candidate_subfamily"].eq("front_conditional_9d")
    & candidate_grid_long["is_front_9d"]
    & (
        ~candidate_grid_long["include_tenor"]
        | ~candidate_grid_long["forecast_vol_pct_min"].isin(NINE_D_CONDITIONAL_FORECAST_VOL_FLOORS)
    )
]

bad_12_15 = candidate_grid_long[
    candidate_grid_long["is_front_12_15"]
    & (
        ~candidate_grid_long["include_tenor"]
        | candidate_grid_long["model_vrp_log"].ne(FRONT_12_15_RULE["model_vrp_log"])
        | candidate_grid_long["model_vrp_z"].ne(FRONT_12_15_RULE["model_vrp_z"])
        | candidate_grid_long["RSI14_max"].ne(FRONT_12_15_RULE["RSI14_max"])
        | candidate_grid_long["forecast_vol_pct_min"].ne(FRONT_12_15_RULE["forecast_vol_pct_min"])
    )
]

bad_middle = candidate_grid_long[
    candidate_grid_long["tenor_bucket"].eq("Middle")
    & (
        ~candidate_grid_long["include_tenor"]
        | candidate_grid_long["model_vrp_log"].ne(MIDDLE_RULE["model_vrp_log"])
        | candidate_grid_long["model_vrp_z"].ne(MIDDLE_RULE["model_vrp_z"])
        | candidate_grid_long["RSI14_max"].ne(MIDDLE_RULE["RSI14_max"])
        | candidate_grid_long["forecast_vol_pct_min"].ne(MIDDLE_RULE["forecast_vol_pct_min"])
    )
]

bad_back = candidate_grid_long[
    candidate_grid_long["tenor_bucket"].eq("Back")
    & (
        ~candidate_grid_long["include_tenor"]
        | candidate_grid_long["model_vrp_log"].ne(BACK_RULE["model_vrp_log"])
        | candidate_grid_long["model_vrp_z"].ne(BACK_RULE["model_vrp_z"])
        | candidate_grid_long["RSI14_max"].ne(BACK_RULE["RSI14_max"])
        | candidate_grid_long["forecast_vol_pct_min"].ne(BACK_RULE["forecast_vol_pct_min"])
    )
]

trade_metric_cols = [
    "trades",
    "win_rate",
    "total_pnl",
    "avg_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "worst_trade",
    "pnl_per_day_drawdown",
]

unexpected_trade_cols = [
    c for c in trade_metric_cols
    if c in candidate_grid_wide.columns or c in candidate_grid_long.columns
]

add_validation(
    "signal_panel_loaded_for_context",
    "PASS" if len(signal) > 0 else "FAIL",
    f"rows={len(signal):,}; path={signal_panel_path}",
)

add_validation(
    "ranked_summary_loaded_for_context",
    "PASS" if len(ranked_summary) > 0 else "FAIL",
    f"rows={len(ranked_summary):,}; path={ranked_summary_path}",
)

add_validation(
    "candidate_grid_count",
    "PASS" if candidate_count == expected_candidate_count else "FAIL",
    f"candidate_count={candidate_count:,}; expected={expected_candidate_count:,}",
)

add_validation(
    "long_grid_has_all_tenors_per_candidate",
    "PASS" if long_row_count == expected_long_row_count and bad_missing.empty else "FAIL",
    f"long_rows={long_row_count:,}; expected={expected_long_row_count:,}; candidates_missing_tenors={len(bad_missing):,}",
)

add_validation(
    "front_ex_9d_candidate_exists",
    "PASS" if candidate_grid_wide["candidate_subfamily"].eq("front_ex_9d").sum() == 1 else "FAIL",
    f"count={int(candidate_grid_wide['candidate_subfamily'].eq('front_ex_9d').sum())}",
)

add_validation(
    "conditional_9d_candidates_exist",
    "PASS" if candidate_grid_wide["candidate_subfamily"].eq("front_conditional_9d").sum() == len(NINE_D_CONDITIONAL_FORECAST_VOL_FLOORS) else "FAIL",
    f"count={int(candidate_grid_wide['candidate_subfamily'].eq('front_conditional_9d').sum())}; expected={len(NINE_D_CONDITIONAL_FORECAST_VOL_FLOORS)}",
)

add_validation(
    "front_ex_9d_excludes_9d",
    "PASS" if bad_ex9d.empty else "FAIL",
    f"bad_rows={len(bad_ex9d):,}",
)

add_validation(
    "conditional_9d_uses_higher_9d_forecast_vol_floors",
    "PASS" if bad_conditional_9d.empty else "FAIL",
    f"bad_rows={len(bad_conditional_9d):,}; allowed_floors={NINE_D_CONDITIONAL_FORECAST_VOL_FLOORS}",
)

add_validation(
    "front_12_15_rule_fixed",
    "PASS" if bad_12_15.empty else "FAIL",
    f"bad_rows={len(bad_12_15):,}; rule={FRONT_12_15_RULE}",
)

add_validation(
    "middle_rule_fixed",
    "PASS" if bad_middle.empty else "FAIL",
    f"bad_rows={len(bad_middle):,}; rule={MIDDLE_RULE}",
)

add_validation(
    "back_rule_locked",
    "PASS" if bad_back.empty else "FAIL",
    f"bad_rows={len(bad_back):,}; rule={BACK_RULE}",
)

add_validation(
    "z_3m_equals_z_1y_within_included_rules",
    "PASS" if bad_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_z_equality):,}",
)

add_validation(
    "no_trade_evaluation_performed",
    "PASS" if not unexpected_trade_cols else "FAIL",
    f"unexpected_trade_metric_cols={unexpected_trade_cols}",
)

add_validation(
    "no_new_sweep",
    "PASS",
    "Cell 11 creates candidate design only.",
)

add_validation(
    "no_final_candidate_selection",
    "PASS",
    "No final thresholds selected.",
)

add_validation(
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    "no_sizing_changes",
    "PASS",
    "No sizing fields included.",
)

add_validation(
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"] == "FAIL"]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1.ipynb",
    "cell": "Cell 11 — 9D-specific Front rule candidate design",
    "timestamp": CELL11_TIMESTAMP,
    "locked_model_spec": LOCKED_MODEL_SPEC,
    "candidate_family": CANDIDATE_FAMILY,
    "candidate_count": int(candidate_count),
    "long_grid_rows": int(long_row_count),
    "signal_panel_path": str(signal_panel_path),
    "ranked_summary_path": str(ranked_summary_path),
    "front_tenor_diag_path": str(front_tenor_diag_path) if front_tenor_diag_path is not None else None,
    "front_ex9d_comparison_path": str(front_ex9d_comparison_path) if front_ex9d_comparison_path is not None else None,
    "front_forecast_state_path": str(front_forecast_state_path) if front_forecast_state_path is not None else None,
    "wide_csv_path": str(wide_csv_path),
    "wide_parquet_path": str(wide_parquet_path),
    "long_csv_path": str(long_csv_path),
    "long_parquet_path": str(long_parquet_path),
    "context_path": str(context_path),
    "ranges_path": str(ranges_path),
    "validation_path": str(validation_path),
    "candidate_subfamilies": sorted(candidate_grid_wide["candidate_subfamily"].unique().tolist()),
    "front_12_15_rule": FRONT_12_15_RULE,
    "nine_d_conditional_forecast_vol_floors": NINE_D_CONDITIONAL_FORECAST_VOL_FLOORS,
    "middle_rule": MIDDLE_RULE,
    "back_rule": BACK_RULE,
    "hard_constraints": [
        "No trade evaluation.",
        "No sweep.",
        "No final threshold selection.",
        "9D exclusion candidate included.",
        "Conditional 9D candidates use forecast-vol floors 16.0, 18.0, 20.0.",
        "12D/15D use fixed Front rule.",
        "Middle uses fixed rule.",
        "Back remains locked.",
        "Within each included rule, model_vrp_z_3m equals model_vrp_z_1y.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
    ],
    "notes": [
        "Cell 12 should sweep these four candidate rules.",
        "Cell 12 should compare front_ex_9d versus front_conditional_9d directly.",
        "This design intentionally does not broaden the full parameter grid.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 11 validation failed. Review validation output above.")

# ======================================================================================
# 8. Display concise review
# ======================================================================================

print("=" * 100)
print("Cell 11 — 9D-specific Front rule candidate design")
print("=" * 100)
print(f"Signal panel:                       {signal_panel_path}")
print(f"Ranked summary:                     {ranked_summary_path}")
print(f"Candidate family:                   {CANDIDATE_FAMILY}")
print(f"Candidate count:                    {candidate_count:,}")
print(f"Long grid rows:                     {long_row_count:,}")
print(f"9D exclusion candidates:            {int(candidate_grid_wide['candidate_subfamily'].eq('front_ex_9d').sum()):,}")
print(f"Conditional 9D candidates:          {int(candidate_grid_wide['candidate_subfamily'].eq('front_conditional_9d').sum()):,}")
print(f"9D conditional FV floors:           {NINE_D_CONDITIONAL_FORECAST_VOL_FLOORS}")
print("No trade evaluation:                True")
print("No sweep:                           True")
print("No final candidate selection:       True")
print("No Secondary:                       True")
print("No sizing changes:                  True")
print("No production lock:                 True")

print()
print("Validation:")
display(validation)

print()
print("Wide candidate design:")
display(candidate_grid_wide)

print()
print("Long tenor-level candidate design:")
display(candidate_grid_long)

print()
print("Context review:")
display(context_review)

print()
print("Saved outputs:")
for p in [
    wide_csv_path,
    wide_parquet_path,
    long_csv_path,
    long_parquet_path,
    context_path,
    ranges_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 11 PASSED")

Cell 11 — 9D-specific Front rule candidate design
Signal panel:                       C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\01_unified_fds_signal_base_panel_20200102_20260701_20260704_211636.parquet
Ranked summary:                     C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\07_core_candidate_ranked_summary_20260704_220821.csv
Candidate family:                   core_front_9d_rule_test
Candidate count:                    4
Long grid rows:                     36
9D exclusion candidates:            1
Conditional 9D candidates:          3
9D conditional FV floors:           [16.0, 18.0, 20.0]
No trade evaluation:                True
No sweep:                           True
No final candidate selection:       True
No Secondary:                       True
No sizing changes:                  True
No production lock:                 True

Validation:


,check,status,detail
0,signal_panel_loaded_for_context,PASS,"rows=14,688; path=C:\Users\patri\vrp_project\d..."
1,ranked_summary_loaded_for_context,PASS,rows=320; path=C:\Users\patri\vrp_project\data...
2,candidate_grid_count,PASS,candidate_count=4; expected=4
3,long_grid_has_all_tenors_per_candidate,PASS,long_rows=36; expected=36; candidates_missing_...
4,front_ex_9d_candidate_exists,PASS,count=1
5,conditional_9d_candidates_exist,PASS,count=3; expected=3
6,front_ex_9d_excludes_9d,PASS,bad_rows=0
7,conditional_9d_uses_higher_9d_forecast_vol_floors,PASS,"bad_rows=0; allowed_floors=[16.0, 18.0, 20.0]"
8,front_12_15_rule_fixed,PASS,"bad_rows=0; rule={'model_vrp_log': 0.7, 'model..."
9,middle_rule_fixed,PASS,"bad_rows=0; rule={'model_vrp_log': 0.7, 'model..."



Wide candidate design:


,candidate_id,candidate_family,candidate_subfamily,candidate_description,locked_model_spec,front_9d_include,front_9d_rule_type,front_9d_model_vrp_log,front_9d_model_vrp_z,front_9d_RSI14_max,front_9d_forecast_vol_pct_min,front_12_15_include,front_12_15_model_vrp_log,front_12_15_model_vrp_z,front_12_15_RSI14_max,front_12_15_forecast_vol_pct_min,middle_include,middle_model_vrp_log,middle_model_vrp_z,middle_RSI14_max,middle_forecast_vol_pct_min,back_include,back_model_vrp_log,back_model_vrp_z,back_RSI14_max,back_forecast_vol_pct_min,z_3m_equals_z_1y_within_rule,back_locked,front_12_15_same_as_top_reference,middle_same_as_top_reference
0,core_front_9d_rule_0001,core_front_9d_rule_test,front_ex_9d,Exclude 9D from Core Front; use Front rule for...,unified_fds_no_min_return,False,excluded,0.7,0.65,68.0,NaN,True,0.7,0.65,68.0,14.5,True,0.7,0.7,68.0,10.0,True,0.7,0.7,70.0,8.5,True,True,True,True
1,core_front_9d_rule_0002,core_front_9d_rule_test,front_conditional_9d,Allow 9D only if 9D forecast vol > 16.0; use F...,unified_fds_no_min_return,True,conditional_higher_forecast_vol_floor,0.7,0.65,68.0,16.0,True,0.7,0.65,68.0,14.5,True,0.7,0.7,68.0,10.0,True,0.7,0.7,70.0,8.5,True,True,True,True
2,core_front_9d_rule_0003,core_front_9d_rule_test,front_conditional_9d,Allow 9D only if 9D forecast vol > 18.0; use F...,unified_fds_no_min_return,True,conditional_higher_forecast_vol_floor,0.7,0.65,68.0,18.0,True,0.7,0.65,68.0,14.5,True,0.7,0.7,68.0,10.0,True,0.7,0.7,70.0,8.5,True,True,True,True
3,core_front_9d_rule_0004,core_front_9d_rule_test,front_conditional_9d,Allow 9D only if 9D forecast vol > 20.0; use F...,unified_fds_no_min_return,True,conditional_higher_forecast_vol_floor,0.7,0.65,68.0,20.0,True,0.7,0.65,68.0,14.5,True,0.7,0.7,68.0,10.0,True,0.7,0.7,70.0,8.5,True,True,True,True



Long tenor-level candidate design:


,candidate_id,candidate_family,candidate_subfamily,locked_model_spec,tenor,tenor_bucket,tenor_bucket_order,rule_scope,rule_type,include_tenor,model_vrp_log,model_vrp_z,model_vrp_z_3m,model_vrp_z_1y,RSI14_max,forecast_vol_pct_min,is_front_9d,is_front_12_15,is_back_locked
0,core_front_9d_rule_0001,core_front_9d_rule_test,front_ex_9d,unified_fds_no_min_return,9,Front,1,front_9d,excluded,False,0.7,0.65,0.65,0.65,68.0,NaN,True,False,False
1,core_front_9d_rule_0001,core_front_9d_rule_test,front_ex_9d,unified_fds_no_min_return,12,Front,1,front_12_15,shared_front_12_15_rule,True,0.7,0.65,0.65,0.65,68.0,14.5,False,True,False
2,core_front_9d_rule_0001,core_front_9d_rule_test,front_ex_9d,unified_fds_no_min_return,15,Front,1,front_12_15,shared_front_12_15_rule,True,0.7,0.65,0.65,0.65,68.0,14.5,False,True,False
3,core_front_9d_rule_0001,core_front_9d_rule_test,front_ex_9d,unified_fds_no_min_return,18,Middle,2,middle,shared_middle_rule,True,0.7,0.70,0.70,0.70,68.0,10.0,False,False,False
4,core_front_9d_rule_0001,core_front_9d_rule_test,front_ex_9d,unified_fds_no_min_return,21,Middle,2,middle,shared_middle_rule,True,0.7,0.70,0.70,0.70,68.0,10.0,False,False,False
5,core_front_9d_rule_0001,core_front_9d_rule_test,front_ex_9d,unified_fds_no_min_return,24,Middle,2,middle,shared_middle_rule,True,0.7,0.70,0.70,0.70,68.0,10.0,False,False,False
6,core_front_9d_rule_0001,core_front_9d_rule_test,front_ex_9d,unified_fds_no_min_return,27,Back,3,back,locked_back_rule,True,0.7,0.70,0.70,0.70,70.0,8.5,False,False,True
7,core_front_9d_rule_0001,core_front_9d_rule_test,front_ex_9d,unified_fds_no_min_return,30,Back,3,back,locked_back_rule,True,0.7,0.70,0.70,0.70,70.0,8.5,False,False,True
8,core_front_9d_rule_0001,core_front_9d_rule_test,front_ex_9d,unified_fds_no_min_return,33,Back,3,back,locked_back_rule,True,0.7,0.70,0.70,0.70,70.0,8.5,False,False,True
9,core_front_9d_rule_0002,core_front_9d_rule_test,front_conditional_9d,unified_fds_no_min_return,9,Front,1,front_9d,conditional_higher_forecast_vol_floor,True,0.7,0.65,0.65,0.65,68.0,16.0,True,False,False



Context review:


,context_item,value
0,signal_panel_path,C:\Users\patri\vrp_project\data\processed\vrp_...
1,ranked_summary_path,C:\Users\patri\vrp_project\data\audit\vrp_unif...
2,front_tenor_diag_path,C:\Users\patri\vrp_project\data\audit\vrp_unif...
3,front_ex9d_comparison_path,C:\Users\patri\vrp_project\data\audit\vrp_unif...
4,front_forecast_state_path,C:\Users\patri\vrp_project\data\audit\vrp_unif...
5,signal_rows,14688
6,signal_date_min,2020-01-02
7,signal_date_max,2026-07-01
8,signal_tenors,"[9, 12, 15, 18, 21, 24, 27, 30, 33]"
9,top_ranked_candidate_1,core_front_expanded_00168



Saved outputs:
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\09_core_front_9d_rule_candidate_grid_wide_20260704_221815.csv
  C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\09_core_front_9d_rule_candidate_grid_wide_20260704_221815.parquet
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\09_core_front_9d_rule_candidate_grid_long_20260704_221815.csv
  C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\09_core_front_9d_rule_candidate_grid_long_20260704_221815.parquet
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\09_core_front_9d_rule_candidate_grid_context_20260704_221815.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\09_core_front_9d_rule_candidate_grid_ranges_20260704_221815.json
  C:\Users\patri\vrp_project\data\audit\vrp_unifie

In [14]:
# ======================================================================================
# Cell 12 — 9D-specific Front rule sweep
# ======================================================================================
#
# Objective:
#   Evaluate the four Cell 11 9D-specific Front rule candidates:
#
#       1. Exclude 9D from Core Front.
#       2. Allow 9D only if 9D forecast vol > 16.0.
#       3. Allow 9D only if 9D forecast vol > 18.0.
#       4. Allow 9D only if 9D forecast vol > 20.0.
#
# Inputs:
#   01_unified_fds_signal_base_panel_*.parquet
#   09_core_front_9d_rule_candidate_grid_wide_*.parquet
#   09_core_front_9d_rule_candidate_grid_long_*.parquet
#
# Outputs:
#   10_core_front_9d_rule_sweep_candidate_summary_*.csv
#   10_core_front_9d_rule_sweep_by_bucket_*.csv
#   10_core_front_9d_rule_sweep_by_tenor_*.csv
#   10_core_front_9d_rule_sweep_by_year_*.csv
#   10_core_front_9d_rule_sweep_candidate_pass_diagnostics_*.csv
#   10_core_front_9d_rule_sweep_selected_trade_panel_*.parquet
#   10_core_front_9d_rule_sweep_validation_*.csv
#   10_core_front_9d_rule_sweep_manifest_*.json
#
# Selection universes:
#   1. all_qualified_trades
#   2. one_trade_per_bucket_per_date_best_rank
#   3. one_trade_per_date_best_rank
#
# Primary metric:
#   avg_pnl_per_day
#
# Important:
#   This is a focused 9D-rule sweep only.
#   This cell does NOT select final thresholds.
#   This cell does NOT run a broad optimization.
#   This cell does NOT change sizing.
#   This cell does NOT lock production.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

pd.set_option("display.max_columns", 280)
pd.set_option("display.width", 460)

# ======================================================================================
# 0. Fallback config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL12_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

if "ALL_TENORS" not in globals():
    ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

if "BUCKET_CENTER_TENOR" not in globals():
    BUCKET_CENTER_TENOR = {
        "Front": 12,
        "Middle": 21,
        "Back": 30,
    }

SELECTION_UNIVERSES = [
    "all_qualified_trades",
    "one_trade_per_bucket_per_date_best_rank",
    "one_trade_per_date_best_rank",
]

PRIMARY_RANKING_METRIC = "avg_pnl_per_day"
CANDIDATE_FAMILY = "core_front_9d_rule_test"

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def profit_factor_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max

    return float(drawdown.min())


def worst_rolling_sum(pnl: pd.Series, window: int) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def add_selection_ranks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Diagnostic DTE selector:
      - Within bucket/date: rank by z_1y, z_3m, model_vrp_log.
      - Tie-break closest to bucket center.
      - Does not use realized P&L.
    """
    d = df.copy()

    if d.empty:
        return d

    d["bucket_center_tenor"] = d["tenor_bucket"].map(BUCKET_CENTER_TENOR)
    d["distance_to_bucket_center"] = (d["tenor"] - d["bucket_center_tenor"]).abs()

    bucket_group = ["trade_date", "tenor_bucket"]

    d["rank_z_1y_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_bucket_date"] = (
        d.groupby(bucket_group)["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_bucket_date"] = d[
        [
            "rank_z_1y_in_bucket_date",
            "rank_z_3m_in_bucket_date",
            "rank_vrp_log_in_bucket_date",
        ]
    ].mean(axis=1)

    d["rank_z_1y_in_date"] = (
        d.groupby("trade_date")["model_vrp_z_1y"]
        .rank(method="min", ascending=False)
    )
    d["rank_z_3m_in_date"] = (
        d.groupby("trade_date")["model_vrp_z_3m"]
        .rank(method="min", ascending=False)
    )
    d["rank_vrp_log_in_date"] = (
        d.groupby("trade_date")["model_vrp_log"]
        .rank(method="min", ascending=False)
    )

    d["avg_signal_rank_in_date"] = d[
        [
            "rank_z_1y_in_date",
            "rank_z_3m_in_date",
            "rank_vrp_log_in_date",
        ]
    ].mean(axis=1)

    return d


def select_trade_universes(qualified: pd.DataFrame) -> dict[str, pd.DataFrame]:
    if qualified.empty:
        return {
            "all_qualified_trades": qualified.copy(),
            "one_trade_per_bucket_per_date_best_rank": qualified.copy(),
            "one_trade_per_date_best_rank": qualified.copy(),
        }

    q = add_selection_ranks(qualified)

    one_trade_per_bucket_date = (
        q.sort_values(
            [
                "trade_date",
                "tenor_bucket_order",
                "avg_signal_rank_in_bucket_date",
                "distance_to_bucket_center",
                "rank_z_1y_in_bucket_date",
                "rank_z_3m_in_bucket_date",
                "rank_vrp_log_in_bucket_date",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date", "tenor_bucket"], as_index=False)
        .head(1)
        .copy()
    )

    one_trade_per_date = (
        q.sort_values(
            [
                "trade_date",
                "avg_signal_rank_in_date",
                "rank_z_1y_in_date",
                "rank_z_3m_in_date",
                "rank_vrp_log_in_date",
                "distance_to_bucket_center",
                "tenor_bucket_order",
                "tenor",
            ],
            ascending=[True, True, True, True, True, True, True, True],
        )
        .groupby(["trade_date"], as_index=False)
        .head(1)
        .copy()
    )

    return {
        "all_qualified_trades": q.copy(),
        "one_trade_per_bucket_per_date_best_rank": one_trade_per_bucket_date,
        "one_trade_per_date_best_rank": one_trade_per_date,
    }


def summarize_trade_set(
    df: pd.DataFrame,
    candidate_id: str,
    selection_universe: str,
    group_cols: list[str] | None = None,
    include_empty_overall: bool = False,
) -> pd.DataFrame:
    if group_cols is None:
        group_cols = []

    d = df.copy()

    if not d.empty:
        d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

        for c in [
            "normalized_pnl_dollars",
            "normalized_pnl_per_day",
            "win",
            "tenor",
        ]:
            if c in d.columns:
                d[c] = pd.to_numeric(d[c], errors="coerce")

        d = d[
            d["normalized_pnl_dollars"].notna()
            & d["normalized_pnl_per_day"].notna()
        ].copy()

        d = d.sort_values(["trade_date", "tenor"]).copy()

    if d.empty:
        if include_empty_overall and len(group_cols) == 0:
            return pd.DataFrame([{
                "candidate_id": candidate_id,
                "selection_universe": selection_universe,
                "trades": 0,
                "unique_trade_dates": 0,
                "date_min": pd.NaT,
                "date_max": pd.NaT,
                "tenor_min": np.nan,
                "tenor_max": np.nan,
                "avg_tenor": np.nan,
                "win_rate": np.nan,
                "total_pnl": 0.0,
                "avg_pnl": np.nan,
                "median_pnl": np.nan,
                "worst_trade": np.nan,
                "best_trade": np.nan,
                "profit_factor": np.nan,
                "selected_trade_drawdown": np.nan,
                "worst_5_trade_sum": np.nan,
                "worst_10_trade_sum": np.nan,
                "worst_20_trade_sum": np.nan,
                "trades_le_neg_50k": 0,
                "trades_le_neg_100k": 0,
                "trades_le_neg_150k": 0,
                "total_pnl_per_day_sum": 0.0,
                "avg_pnl_per_day": np.nan,
                "median_pnl_per_day": np.nan,
                "worst_trade_pnl_per_day": np.nan,
                "best_trade_pnl_per_day": np.nan,
                "profit_factor_pnl_per_day": np.nan,
                "pnl_per_day_drawdown": np.nan,
                "worst_5_trade_pnl_per_day_sum": np.nan,
                "worst_10_trade_pnl_per_day_sum": np.nan,
                "worst_20_trade_pnl_per_day_sum": np.nan,
                "trades_2025": 0,
                "pnl_2025": 0.0,
                "avg_pnl_per_day_2025": np.nan,
                "worst_trade_2025": np.nan,
                "trades_2026": 0,
                "pnl_2026": 0.0,
                "avg_pnl_per_day_2026": np.nan,
                "worst_trade_2026": np.nan,
            }])
        return pd.DataFrame()

    rows = []

    if len(group_cols) == 0:
        grouped = [((), d)]
    else:
        grouped = d.groupby(group_cols, dropna=False)

    for keys, g in grouped:
        if len(group_cols) == 0:
            keys = ()
        elif not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            "candidate_id": candidate_id,
            "selection_universe": selection_universe,
        }

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": int(g["trade_date"].nunique()),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,

            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,
            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        rows.append(row)

    return pd.DataFrame(rows)


def apply_tenor_level_candidate_thresholds(signal_df: pd.DataFrame, candidate_rules: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Apply one Cell 11 tenor-level candidate rule set.

    candidate_rules contains one row per tenor.
    include_tenor=False means the tenor is excluded from the candidate's Core rules.
    """
    candidate_id = candidate_rules["candidate_id"].iloc[0]

    threshold_cols = [
        "candidate_id",
        "candidate_family",
        "candidate_subfamily",
        "tenor",
        "rule_scope",
        "rule_type",
        "include_tenor",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14_max",
        "forecast_vol_pct_min",
    ]

    rules = candidate_rules[threshold_cols].copy()

    rules = rules.rename(columns={
        "model_vrp_log": "threshold_model_vrp_log",
        "model_vrp_z_3m": "threshold_model_vrp_z_3m",
        "model_vrp_z_1y": "threshold_model_vrp_z_1y",
        "RSI14_max": "threshold_RSI14_max",
        "forecast_vol_pct_min": "threshold_forecast_vol_pct_min",
    })

    d = signal_df.merge(
        rules,
        on="tenor",
        how="left",
        validate="many_to_one",
    )

    required_inputs = (
        d["include_tenor"].fillna(False).astype(bool)
        & d["model_vrp_log"].notna()
        & d["model_vrp_z_3m"].notna()
        & d["model_vrp_z_1y"].notna()
        & d["RSI14"].notna()
        & d["forecast_vol_pct"].notna()
        & d["threshold_model_vrp_log"].notna()
        & d["threshold_model_vrp_z_3m"].notna()
        & d["threshold_model_vrp_z_1y"].notna()
        & d["threshold_RSI14_max"].notna()
        & d["threshold_forecast_vol_pct_min"].notna()
    )

    pass_mask = (
        required_inputs
        & (d["model_vrp_log"] > d["threshold_model_vrp_log"])
        & (d["model_vrp_z_3m"] > d["threshold_model_vrp_z_3m"])
        & (d["model_vrp_z_1y"] > d["threshold_model_vrp_z_1y"])
        & (d["RSI14"] < d["threshold_RSI14_max"])
        & (d["forecast_vol_pct"] > d["threshold_forecast_vol_pct_min"])
    )

    d["candidate_core_pass"] = pass_mask

    outcome_available = (
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    )

    qualified = d[
        d["candidate_core_pass"]
        & outcome_available
    ].copy()

    diagnostics = {
        "candidate_id": candidate_id,
        "candidate_subfamily": candidate_rules["candidate_subfamily"].iloc[0],
        "threshold_input_available_rows": int(required_inputs.sum()),
        "pass_rows_all_dates": int(pass_mask.sum()),
        "pass_rows_with_outcomes": int(len(qualified)),
        "trade_outcome_available_rows": int(outcome_available.sum()),
    }

    for tenor in ALL_TENORS:
        tenor_mask = d["tenor"].eq(tenor)
        diagnostics[f"tenor_{tenor}_included"] = bool(
            candidate_rules.loc[candidate_rules["tenor"].eq(tenor), "include_tenor"].iloc[0]
        )
        diagnostics[f"tenor_{tenor}_pass_rows_all_dates"] = int((pass_mask & tenor_mask).sum())
        diagnostics[f"tenor_{tenor}_pass_rows_with_outcomes"] = int(
            (pass_mask & tenor_mask & outcome_available).sum()
        )

    for bucket in ["Front", "Middle", "Back"]:
        bucket_mask = d["tenor_bucket"].eq(bucket)
        diagnostics[f"{bucket}_pass_rows_all_dates"] = int((pass_mask & bucket_mask).sum())
        diagnostics[f"{bucket}_pass_rows_with_outcomes"] = int(
            (pass_mask & bucket_mask & outcome_available).sum()
        )

    return qualified, diagnostics


def attach_candidate_parameters(summary_df: pd.DataFrame, wide_grid: pd.DataFrame) -> pd.DataFrame:
    if summary_df.empty:
        return summary_df

    param_cols = [
        "candidate_id",
        "candidate_family",
        "candidate_subfamily",
        "candidate_description",
        "front_9d_include",
        "front_9d_rule_type",
        "front_9d_model_vrp_log",
        "front_9d_model_vrp_z",
        "front_9d_RSI14_max",
        "front_9d_forecast_vol_pct_min",
        "front_12_15_model_vrp_log",
        "front_12_15_model_vrp_z",
        "front_12_15_RSI14_max",
        "front_12_15_forecast_vol_pct_min",
        "middle_model_vrp_log",
        "middle_model_vrp_z",
        "middle_RSI14_max",
        "middle_forecast_vol_pct_min",
        "back_model_vrp_log",
        "back_model_vrp_z",
        "back_RSI14_max",
        "back_forecast_vol_pct_min",
    ]

    available = [c for c in param_cols if c in wide_grid.columns]

    return summary_df.merge(
        wide_grid[available].drop_duplicates("candidate_id"),
        on="candidate_id",
        how="left",
        validate="many_to_one",
    )


# ======================================================================================
# 2. Load inputs
# ======================================================================================

signal_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01_unified_fds_signal_base_panel_*.parquet",
    required=True,
)

candidate_grid_wide_path = latest_file(
    OUT_PROCESSED_DIR,
    "09_core_front_9d_rule_candidate_grid_wide_*.parquet",
    required=True,
)

candidate_grid_long_path = latest_file(
    OUT_PROCESSED_DIR,
    "09_core_front_9d_rule_candidate_grid_long_*.parquet",
    required=True,
)

signal = pd.read_parquet(signal_panel_path)
grid_wide = pd.read_parquet(candidate_grid_wide_path)
grid_long = pd.read_parquet(candidate_grid_long_path)

signal["trade_date"] = pd.to_datetime(signal["trade_date"], errors="coerce").dt.normalize()
signal["tenor"] = pd.to_numeric(signal["tenor"], errors="coerce").astype(int)

required_signal_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "model_spec",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]

required_long_cols = [
    "candidate_id",
    "candidate_family",
    "candidate_subfamily",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "rule_scope",
    "rule_type",
    "include_tenor",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14_max",
    "forecast_vol_pct_min",
]

missing_signal_cols = [c for c in required_signal_cols if c not in signal.columns]
missing_long_cols = [c for c in required_long_cols if c not in grid_long.columns]

if missing_signal_cols:
    raise KeyError(f"Cell 12 missing signal columns: {missing_signal_cols}")

if missing_long_cols:
    raise KeyError(f"Cell 12 missing long-grid columns: {missing_long_cols}")

for c in [
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14",
    "forecast_vol_pct",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
]:
    signal[c] = pd.to_numeric(signal[c], errors="coerce")

for c in [
    "tenor",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
    "RSI14_max",
    "forecast_vol_pct_min",
]:
    grid_long[c] = pd.to_numeric(grid_long[c], errors="coerce")

grid_long["include_tenor"] = grid_long["include_tenor"].astype(bool)
grid_long["tenor"] = grid_long["tenor"].astype(int)

grid_wide = grid_wide.sort_values("candidate_id").reset_index(drop=True)
grid_long = grid_long.sort_values(["candidate_id", "tenor"]).reset_index(drop=True)

candidate_ids = grid_wide["candidate_id"].dropna().tolist()

# ======================================================================================
# 3. Sweep candidates
# ======================================================================================

start_time = time.time()

overall_summary_rows = []
bucket_summary_rows = []
tenor_summary_rows = []
year_summary_rows = []
pass_diagnostic_rows = []
selected_trade_frames = []

print("=" * 100)
print("Cell 12 — 9D-specific Front rule sweep")
print("=" * 100)
print(f"Signal panel:             {signal_panel_path}")
print(f"Wide candidate grid:      {candidate_grid_wide_path}")
print(f"Long candidate grid:      {candidate_grid_long_path}")
print(f"Candidates to evaluate:   {len(candidate_ids):,}")
print(f"Selection universes:      {SELECTION_UNIVERSES}")
print(f"Primary metric:           {PRIMARY_RANKING_METRIC}")
print()

for i, candidate_id in enumerate(candidate_ids, start=1):
    rules = grid_long[grid_long["candidate_id"].eq(candidate_id)].copy()

    if rules.empty:
        raise RuntimeError(f"No long-grid rules found for candidate_id={candidate_id}")

    qualified, pass_diag = apply_tenor_level_candidate_thresholds(signal, rules)
    pass_diagnostic_rows.append(pass_diag)

    universes = select_trade_universes(qualified)

    for universe_name, selected in universes.items():
        if not selected.empty:
            selected = selected.copy()
            selected.insert(0, "selection_universe", universe_name)

            selected_trade_frames.append(selected)

        overall_summary_rows.append(
            summarize_trade_set(
                selected,
                candidate_id=candidate_id,
                selection_universe=universe_name,
                group_cols=[],
                include_empty_overall=True,
            )
        )

        bucket_summary_rows.append(
            summarize_trade_set(
                selected,
                candidate_id=candidate_id,
                selection_universe=universe_name,
                group_cols=["tenor_bucket", "tenor_bucket_order"],
                include_empty_overall=False,
            )
        )

        tenor_summary_rows.append(
            summarize_trade_set(
                selected,
                candidate_id=candidate_id,
                selection_universe=universe_name,
                group_cols=["tenor", "tenor_bucket", "tenor_bucket_order"],
                include_empty_overall=False,
            )
        )

        if not selected.empty:
            y = selected.copy()
            y["year"] = y["trade_date"].dt.year.astype(int)

            year_summary_rows.append(
                summarize_trade_set(
                    y,
                    candidate_id=candidate_id,
                    selection_universe=universe_name,
                    group_cols=["year"],
                    include_empty_overall=False,
                )
            )

    print(f"  Evaluated {i:,} / {len(candidate_ids):,}: {candidate_id}")

summary_overall = pd.concat(overall_summary_rows, ignore_index=True) if overall_summary_rows else pd.DataFrame()
summary_by_bucket = pd.concat(bucket_summary_rows, ignore_index=True) if bucket_summary_rows else pd.DataFrame()
summary_by_tenor = pd.concat(tenor_summary_rows, ignore_index=True) if tenor_summary_rows else pd.DataFrame()
summary_by_year = pd.concat(year_summary_rows, ignore_index=True) if year_summary_rows else pd.DataFrame()
pass_diagnostics = pd.DataFrame(pass_diagnostic_rows)

selected_trade_panel = (
    pd.concat(selected_trade_frames, ignore_index=True)
    if selected_trade_frames
    else pd.DataFrame()
)

# Attach candidate parameters.
summary_overall = attach_candidate_parameters(summary_overall, grid_wide)
summary_by_bucket = attach_candidate_parameters(summary_by_bucket, grid_wide)
summary_by_tenor = attach_candidate_parameters(summary_by_tenor, grid_wide)
summary_by_year = attach_candidate_parameters(summary_by_year, grid_wide)
pass_diagnostics = attach_candidate_parameters(pass_diagnostics, grid_wide)

# Sort outputs.
if not summary_overall.empty:
    summary_overall = summary_overall.sort_values(
        ["selection_universe", PRIMARY_RANKING_METRIC, "profit_factor_pnl_per_day", "trades"],
        ascending=[True, False, False, False],
    ).reset_index(drop=True)

if not summary_by_bucket.empty:
    summary_by_bucket = summary_by_bucket.sort_values(
        ["selection_universe", "candidate_id", "tenor_bucket_order"],
    ).reset_index(drop=True)

if not summary_by_tenor.empty:
    summary_by_tenor = summary_by_tenor.sort_values(
        ["selection_universe", "candidate_id", "tenor"],
    ).reset_index(drop=True)

if not summary_by_year.empty:
    summary_by_year = summary_by_year.sort_values(
        ["selection_universe", "candidate_id", "year"],
    ).reset_index(drop=True)

elapsed_seconds = time.time() - start_time

# ======================================================================================
# 4. Direct comparison table
# ======================================================================================

primary = summary_overall[
    summary_overall["selection_universe"].eq("one_trade_per_bucket_per_date_best_rank")
].copy()

primary = primary.sort_values(
    ["avg_pnl_per_day", "profit_factor_pnl_per_day", "trades"],
    ascending=[False, False, False],
).reset_index(drop=True)

primary.insert(0, "comparison_rank", np.arange(1, len(primary) + 1))

baseline_ex9d = primary[primary["candidate_subfamily"].eq("front_ex_9d")].copy()

comparison_rows = []

if not baseline_ex9d.empty:
    base = baseline_ex9d.iloc[0]

    for _, row in primary.iterrows():
        comparison_rows.append({
            "candidate_id": row["candidate_id"],
            "candidate_subfamily": row["candidate_subfamily"],
            "front_9d_include": row["front_9d_include"],
            "front_9d_forecast_vol_pct_min": row["front_9d_forecast_vol_pct_min"],
            "trades": row["trades"],
            "avg_pnl_per_day": row["avg_pnl_per_day"],
            "total_pnl": row["total_pnl"],
            "profit_factor": row["profit_factor"],
            "profit_factor_pnl_per_day": row["profit_factor_pnl_per_day"],
            "pnl_per_day_drawdown": row["pnl_per_day_drawdown"],
            "worst_20_trade_pnl_per_day_sum": row["worst_20_trade_pnl_per_day_sum"],
            "avg_pnl_per_day_2025": row["avg_pnl_per_day_2025"],
            "avg_pnl_per_day_2026": row["avg_pnl_per_day_2026"],
            "trades_le_neg_100k": row["trades_le_neg_100k"],

            "delta_vs_ex9d_trades": row["trades"] - base["trades"],
            "delta_vs_ex9d_avg_pnl_per_day": row["avg_pnl_per_day"] - base["avg_pnl_per_day"],
            "delta_vs_ex9d_total_pnl": row["total_pnl"] - base["total_pnl"],
            "delta_vs_ex9d_profit_factor": row["profit_factor"] - base["profit_factor"],
            "delta_vs_ex9d_pnl_per_day_drawdown": row["pnl_per_day_drawdown"] - base["pnl_per_day_drawdown"],
            "delta_vs_ex9d_worst_20_trade_pnl_per_day_sum": row["worst_20_trade_pnl_per_day_sum"] - base["worst_20_trade_pnl_per_day_sum"],
            "delta_vs_ex9d_avg_pnl_per_day_2025": row["avg_pnl_per_day_2025"] - base["avg_pnl_per_day_2025"],
            "delta_vs_ex9d_avg_pnl_per_day_2026": row["avg_pnl_per_day_2026"] - base["avg_pnl_per_day_2026"],
        })

comparison_table = pd.DataFrame(comparison_rows)

if not comparison_table.empty:
    comparison_table = comparison_table.sort_values(
        ["avg_pnl_per_day", "profit_factor_pnl_per_day", "trades"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

# ======================================================================================
# 5. Save outputs
# ======================================================================================

candidate_summary_path = OUT_AUDIT_DIR / f"10_core_front_9d_rule_sweep_candidate_summary_{CELL12_TIMESTAMP}.csv"
by_bucket_path = OUT_AUDIT_DIR / f"10_core_front_9d_rule_sweep_by_bucket_{CELL12_TIMESTAMP}.csv"
by_tenor_path = OUT_AUDIT_DIR / f"10_core_front_9d_rule_sweep_by_tenor_{CELL12_TIMESTAMP}.csv"
by_year_path = OUT_AUDIT_DIR / f"10_core_front_9d_rule_sweep_by_year_{CELL12_TIMESTAMP}.csv"
pass_diagnostics_path = OUT_AUDIT_DIR / f"10_core_front_9d_rule_sweep_candidate_pass_diagnostics_{CELL12_TIMESTAMP}.csv"
comparison_path = OUT_AUDIT_DIR / f"10_core_front_9d_rule_sweep_ex9d_vs_conditional_comparison_{CELL12_TIMESTAMP}.csv"
selected_trade_panel_path = OUT_PROCESSED_DIR / f"10_core_front_9d_rule_sweep_selected_trade_panel_{CELL12_TIMESTAMP}.parquet"
validation_path = OUT_AUDIT_DIR / f"10_core_front_9d_rule_sweep_validation_{CELL12_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"10_core_front_9d_rule_sweep_manifest_{CELL12_TIMESTAMP}.json"

summary_overall.to_csv(candidate_summary_path, index=False)
summary_by_bucket.to_csv(by_bucket_path, index=False)
summary_by_tenor.to_csv(by_tenor_path, index=False)
summary_by_year.to_csv(by_year_path, index=False)
pass_diagnostics.to_csv(pass_diagnostics_path, index=False)
comparison_table.to_csv(comparison_path, index=False)
selected_trade_panel.to_parquet(selected_trade_panel_path, index=False)

# ======================================================================================
# 6. Validation
# ======================================================================================

validation_rows = []

def add_validation(check, status, detail):
    validation_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })

models_found = sorted(signal["model_spec"].dropna().unique().tolist())

expected_candidate_count = 4
actual_candidate_count = len(candidate_ids)

expected_overall_rows = actual_candidate_count * len(SELECTION_UNIVERSES)
actual_overall_rows = len(summary_overall)

selection_universes_found = sorted(summary_overall["selection_universe"].dropna().unique().tolist())

bad_long_grid_candidate_tenors = (
    grid_long
    .groupby("candidate_id")["tenor"]
    .apply(lambda x: sorted(set(ALL_TENORS) - set(x.tolist())))
    .reset_index(name="missing_tenors")
)
bad_long_grid_candidate_tenors = bad_long_grid_candidate_tenors[
    bad_long_grid_candidate_tenors["missing_tenors"].apply(len).gt(0)
]

bad_ex9d_inclusion = grid_long[
    grid_long["candidate_subfamily"].eq("front_ex_9d")
    & grid_long["tenor"].eq(9)
    & grid_long["include_tenor"].astype(bool)
]

conditional_9d_floors_found = sorted(
    grid_long[
        grid_long["candidate_subfamily"].eq("front_conditional_9d")
        & grid_long["tenor"].eq(9)
    ]["forecast_vol_pct_min"].dropna().unique().tolist()
)

bad_pnl_day = signal[
    signal["normalized_pnl_dollars"].notna()
    & signal["normalized_pnl_per_day"].notna()
    & (
        (
            signal["normalized_pnl_dollars"] / signal["tenor"].replace(0, np.nan)
        )
        - signal["normalized_pnl_per_day"]
    ).abs().gt(1e-8)
]

add_validation(
    "signal_panel_loaded",
    "PASS" if len(signal) > 0 else "FAIL",
    f"rows={len(signal):,}; path={signal_panel_path}",
)

add_validation(
    "locked_model_only",
    "PASS" if models_found == [LOCKED_MODEL_SPEC] else "FAIL",
    f"models_found={models_found}",
)

add_validation(
    "candidate_grid_loaded",
    "PASS" if actual_candidate_count == expected_candidate_count else "FAIL",
    f"candidate_count={actual_candidate_count:,}; expected={expected_candidate_count:,}",
)

add_validation(
    "long_grid_has_all_tenors_per_candidate",
    "PASS" if bad_long_grid_candidate_tenors.empty else "FAIL",
    f"bad_candidates={len(bad_long_grid_candidate_tenors):,}",
)

add_validation(
    "front_ex_9d_candidate_excludes_9d",
    "PASS" if bad_ex9d_inclusion.empty else "FAIL",
    f"bad_rows={len(bad_ex9d_inclusion):,}",
)

add_validation(
    "conditional_9d_floors_present",
    "PASS" if conditional_9d_floors_found == [16.0, 18.0, 20.0] else "FAIL",
    f"found={conditional_9d_floors_found}",
)

add_validation(
    "normalized_pnl_per_day_formula",
    "PASS" if bad_pnl_day.empty else "FAIL",
    f"bad_rows={len(bad_pnl_day):,}",
)

add_validation(
    "overall_summary_rows_complete",
    "PASS" if actual_overall_rows == expected_overall_rows else "FAIL",
    f"actual={actual_overall_rows:,}; expected={expected_overall_rows:,}",
)

add_validation(
    "all_selection_universes_evaluated",
    "PASS" if selection_universes_found == sorted(SELECTION_UNIVERSES) else "FAIL",
    f"found={selection_universes_found}",
)

add_validation(
    "selected_trade_panel_saved",
    "PASS" if len(selected_trade_panel) > 0 else "WARN",
    f"rows={len(selected_trade_panel):,}; path={selected_trade_panel_path}",
)

add_validation(
    "pass_diagnostics_created",
    "PASS" if len(pass_diagnostics) == actual_candidate_count else "FAIL",
    f"rows={len(pass_diagnostics):,}; expected={actual_candidate_count:,}",
)

add_validation(
    "comparison_table_created",
    "PASS" if not comparison_table.empty else "WARN",
    f"rows={len(comparison_table):,}; path={comparison_path}",
)

add_validation(
    "no_broad_optimization",
    "PASS",
    "Only four Cell 11 9D-rule candidates evaluated.",
)

add_validation(
    "no_final_candidate_selection",
    "PASS",
    "Cell 12 is a focused comparison sweep only.",
)

add_validation(
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    "no_sizing_changes",
    "PASS",
    "No sizing fields changed.",
)

add_validation(
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"] == "FAIL"]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1.ipynb",
    "cell": "Cell 12 — 9D-specific Front rule sweep",
    "timestamp": CELL12_TIMESTAMP,
    "elapsed_seconds": elapsed_seconds,
    "locked_model_spec": LOCKED_MODEL_SPEC,
    "candidate_family": CANDIDATE_FAMILY,
    "signal_panel_path": str(signal_panel_path),
    "candidate_grid_wide_path": str(candidate_grid_wide_path),
    "candidate_grid_long_path": str(candidate_grid_long_path),
    "candidate_count": int(actual_candidate_count),
    "selection_universes": SELECTION_UNIVERSES,
    "primary_ranking_metric": PRIMARY_RANKING_METRIC,
    "candidate_summary_path": str(candidate_summary_path),
    "by_bucket_path": str(by_bucket_path),
    "by_tenor_path": str(by_tenor_path),
    "by_year_path": str(by_year_path),
    "pass_diagnostics_path": str(pass_diagnostics_path),
    "comparison_path": str(comparison_path),
    "selected_trade_panel_path": str(selected_trade_panel_path),
    "validation_path": str(validation_path),
    "notes": [
        "Focused 9D-rule sweep only.",
        "No broad threshold optimization.",
        "No final threshold selected.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
        "The key comparison is front_ex_9d versus front_conditional_9d with 9D forecast-vol floors 16.0, 18.0, and 20.0.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 12 validation failed. Review validation output above.")

# ======================================================================================
# 7. Display concise review
# ======================================================================================

print("=" * 100)
print("Cell 12 — 9D-specific Front rule sweep")
print("=" * 100)
print(f"Signal panel:                  {signal_panel_path}")
print(f"Wide candidate grid:           {candidate_grid_wide_path}")
print(f"Long candidate grid:           {candidate_grid_long_path}")
print(f"Candidates evaluated:          {actual_candidate_count:,}")
print(f"Overall summary rows:          {len(summary_overall):,}")
print(f"By-bucket rows:                {len(summary_by_bucket):,}")
print(f"By-tenor rows:                 {len(summary_by_tenor):,}")
print(f"By-year rows:                  {len(summary_by_year):,}")
print(f"Pass diagnostic rows:          {len(pass_diagnostics):,}")
print(f"Selected trade panel rows:     {len(selected_trade_panel):,}")
print(f"Elapsed seconds:               {elapsed_seconds:,.1f}")
print(f"Primary ranking metric:        {PRIMARY_RANKING_METRIC}")
print("No broad optimization:         True")
print("No final candidate selection:  True")
print("No Secondary:                  True")
print("No sizing changes:             True")
print("No production lock:            True")

print()
print("Validation:")
display(validation)

main_display_cols = [
    "comparison_rank",
    "candidate_id",
    "candidate_subfamily",
    "front_9d_include",
    "front_9d_forecast_vol_pct_min",
    "trades",
    "unique_trade_dates",
    "avg_tenor",
    "win_rate",
    "total_pnl",
    "profit_factor",
    "profit_factor_pnl_per_day",
    "avg_pnl_per_day",
    "median_pnl_per_day",
    "worst_trade_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "trades_le_neg_100k",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
    "delta_vs_ex9d_trades",
    "delta_vs_ex9d_avg_pnl_per_day",
    "delta_vs_ex9d_total_pnl",
    "delta_vs_ex9d_profit_factor",
    "delta_vs_ex9d_pnl_per_day_drawdown",
    "delta_vs_ex9d_avg_pnl_per_day_2026",
]

print()
print("Direct comparison — one trade per bucket/date:")
display(comparison_table[[c for c in main_display_cols if c in comparison_table.columns]])

top_display_cols = [
    "candidate_id",
    "candidate_subfamily",
    "selection_universe",
    "front_9d_include",
    "front_9d_forecast_vol_pct_min",
    "trades",
    "unique_trade_dates",
    "avg_tenor",
    "win_rate",
    "total_pnl",
    "profit_factor",
    "avg_pnl_per_day",
    "median_pnl_per_day",
    "pnl_per_day_drawdown",
    "worst_20_trade_pnl_per_day_sum",
    "trades_le_neg_100k",
    "avg_pnl_per_day_2025",
    "avg_pnl_per_day_2026",
]

print()
print("Candidate summary — all selection universes:")
display(summary_overall[[c for c in top_display_cols if c in summary_overall.columns]])

print()
print("By-bucket detail:")
display(summary_by_bucket)

print()
print("By-tenor detail:")
display(summary_by_tenor)

print()
print("By-year detail:")
display(summary_by_year)

print()
print("Pass diagnostics:")
pass_diag_cols = [
    "candidate_id",
    "candidate_subfamily",
    "pass_rows_all_dates",
    "pass_rows_with_outcomes",
    "Front_pass_rows_with_outcomes",
    "Middle_pass_rows_with_outcomes",
    "Back_pass_rows_with_outcomes",
    "tenor_9_included",
    "tenor_9_pass_rows_with_outcomes",
    "tenor_12_pass_rows_with_outcomes",
    "tenor_15_pass_rows_with_outcomes",
    "front_9d_forecast_vol_pct_min",
]
display(pass_diagnostics[[c for c in pass_diag_cols if c in pass_diagnostics.columns]])

print()
print("Saved outputs:")
for p in [
    candidate_summary_path,
    by_bucket_path,
    by_tenor_path,
    by_year_path,
    pass_diagnostics_path,
    comparison_path,
    selected_trade_panel_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 12 PASSED")

Cell 12 — 9D-specific Front rule sweep
Signal panel:             C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\01_unified_fds_signal_base_panel_20200102_20260701_20260704_211636.parquet
Wide candidate grid:      C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\09_core_front_9d_rule_candidate_grid_wide_20260704_221815.parquet
Long candidate grid:      C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\09_core_front_9d_rule_candidate_grid_long_20260704_221815.parquet
Candidates to evaluate:   4
Selection universes:      ['all_qualified_trades', 'one_trade_per_bucket_per_date_best_rank', 'one_trade_per_date_best_rank']
Primary metric:           avg_pnl_per_day

  Evaluated 1 / 4: core_front_9d_rule_0001
  Evaluated 2 / 4: core_front_9d_rule_0002
  Evaluated 3 / 4: core_front_9d_rule_0003
  Evaluated 4 / 4: core_front_9d_rule_0004
Cell 12 — 9D-specific Front rul

,check,status,detail
0,signal_panel_loaded,PASS,"rows=14,688; path=C:\Users\patri\vrp_project\d..."
1,locked_model_only,PASS,models_found=['unified_fds_no_min_return']
2,candidate_grid_loaded,PASS,candidate_count=4; expected=4
3,long_grid_has_all_tenors_per_candidate,PASS,bad_candidates=0
4,front_ex_9d_candidate_excludes_9d,PASS,bad_rows=0
5,conditional_9d_floors_present,PASS,"found=[16.0, 18.0, 20.0]"
6,normalized_pnl_per_day_formula,PASS,bad_rows=0
7,overall_summary_rows_complete,PASS,actual=12; expected=12
8,all_selection_universes_evaluated,PASS,"found=['all_qualified_trades', 'one_trade_per_..."
9,selected_trade_panel_saved,PASS,"rows=6,831; path=C:\Users\patri\vrp_project\da..."



Direct comparison — one trade per bucket/date:


,candidate_id,candidate_subfamily,front_9d_include,front_9d_forecast_vol_pct_min,trades,total_pnl,profit_factor,profit_factor_pnl_per_day,avg_pnl_per_day,pnl_per_day_drawdown,worst_20_trade_pnl_per_day_sum,trades_le_neg_100k,avg_pnl_per_day_2025,avg_pnl_per_day_2026,delta_vs_ex9d_trades,delta_vs_ex9d_avg_pnl_per_day,delta_vs_ex9d_total_pnl,delta_vs_ex9d_profit_factor,delta_vs_ex9d_pnl_per_day_drawdown,delta_vs_ex9d_avg_pnl_per_day_2026
0,core_front_9d_rule_0004,front_conditional_9d,True,20.0,420,5.745171e+06,4.266415,3.583912,604.785125,-29677.448469,-23715.163620,2,651.802316,66.164231,0,5.434511,-20799.372769,-0.011825,0.0,0.000000
1,core_front_9d_rule_0001,front_ex_9d,False,NaN,420,5.765970e+06,4.278241,3.560693,599.350614,-29677.448469,-23715.163620,2,639.354946,66.164231,0,0.000000,0.000000,0.000000,0.0,0.000000
2,core_front_9d_rule_0003,front_conditional_9d,True,18.0,421,5.609588e+06,4.055074,3.215378,574.513452,-29677.448469,-24889.323006,2,651.802316,66.164231,1,-24.837163,-156382.139908,-0.223167,0.0,0.000000
3,core_front_9d_rule_0002,front_conditional_9d,True,16.0,425,5.522829e+06,3.951584,3.050389,558.471372,-29677.448469,-24889.323006,2,652.290375,74.509040,5,-40.879242,-243141.315638,-0.326656,0.0,8.344809



Candidate summary — all selection universes:


...


By-year detail:


,candidate_id,selection_universe,year,trades,unique_trade_dates,date_min,date_max,tenor_min,tenor_max,avg_tenor,win_rate,total_pnl,avg_pnl,median_pnl,worst_trade,best_trade,profit_factor,selected_trade_drawdown,worst_5_trade_sum,worst_10_trade_sum,worst_20_trade_sum,trades_le_neg_50k,trades_le_neg_100k,trades_le_neg_150k,total_pnl_per_day_sum,avg_pnl_per_day,median_pnl_per_day,worst_trade_pnl_per_day,best_trade_pnl_per_day,profit_factor_pnl_per_day,pnl_per_day_drawdown,worst_5_trade_pnl_per_day_sum,worst_10_trade_pnl_per_day_sum,worst_20_trade_pnl_per_day_sum,trades_2025,pnl_2025,avg_pnl_per_day_2025,worst_trade_2025,trades_2026,pnl_2026,avg_pnl_per_day_2026,worst_trade_2026,candidate_family,candidate_subfamily,candidate_description,front_9d_include,front_9d_rule_type,front_9d_model_vrp_log,front_9d_model_vrp_z,front_9d_RSI14_max,front_9d_forecast_vol_pct_min,front_12_15_model_vrp_log,front_12_15_model_vrp_z,front_12_15_RSI14_max,front_12_15_forecast_vol_pct_min,middle_model_vrp_log,middle_model_vrp_z,middle_RSI14_max,middle_forecast_vol_pct_min,back_model_vrp_log,back_model_vrp_z,back_RSI14_max,back_forecast_vol_pct_min
0,core_front_9d_rule_0001,all_qualified_trades,2021,29,5,2021-11-24,2021-12-06,12,33,21.931034,0.862069,5.285488e+05,18225.821619,21593.370395,-3607.390045,28384.414940,74.259174,-7.214780e+03,23541.623240,96227.046726,3.340393e+05,0,0,0,25003.349579,862.184468,998.702931,-171.780478,1527.694232,78.628843,-322.088397,760.081350,5548.209180,16561.573463,0,0.000000e+00,NaN,NaN,0,0.000000,NaN,NaN,core_front_9d_rule_test,front_ex_9d,Exclude 9D from Core Front; use Front rule for...,False,excluded,0.7,0.65,68.0,NaN,0.7,0.65,68.0,14.5,0.7,0.7,68.0,10.0,0.7,0.7,70.0,8.5
1,core_front_9d_rule_0001,all_qualified_trades,2022,521,80,2022-01-20,2022-12-12,12,33,22.554702,0.892514,9.178802e+06,17617.661731,24823.610979,-108475.796818,35371.772564,5.668845,-1.213894e+06,-510586.535163,-851762.028286,-1.119570e+06,10,5,0,409657.091558,786.290003,1066.055302,-7782.261885,1978.114770,4.931846,-55545.022099,-25710.022310,-40701.663776,-51948.701006,0,0.000000e+00,NaN,NaN,0,0.000000,NaN,NaN,core_front_9d_rule_test,front_ex_9d,Exclude 9D from Core Front; use Front rule for...,False,excluded,0.7,0.65,68.0,NaN,0.7,0.65,68.0,14.5,0.7,0.7,68.0,10.0,0.7,0.7,70.0,8.5
2,core_front_9d_rule_0001,all_qualified_trades,2023,21,8,2023-03-13,2023-10-19,12,30,17.285714,1.000000,3.925858e+05,18694.560080,18782.454977,12815.804660,23438.457453,inf,0.000000e+00,82494.078617,180160.932713,3.727453e+05,0,0,0,23890.298490,1137.633261,1123.411904,738.125423,1653.370542,inf,0.000000,4680.895783,9772.979516,22236.927949,0,0.000000e+00,NaN,NaN,0,0.000000,NaN,NaN,core_front_9d_rule_test,front_ex_9d,Exclude 9D from Core Front; use Front rule for...,False,excluded,0.7,0.65,68.0,NaN,0.7,0.65,68.0,14.5,0.7,0.7,68.0,10.0,0.7,0.7,70.0,8.5
3,core_front_9d_rule_0001,all_qualified_trades,2024,101,24,2024-08-02,2024-12-18,12,33,24.504950,0.940594,1.369539e+06,13559.789639,14743.901854,-12048.089968,28845.059994,26.535988,-2.409618e+04,19590.945161,71816.644756,1.691292e+05,0,0,0,57460.266310,568.913528,567.989641,-669.338332,1617.071545,21.277023,-1243.056901,382.442930,2033.985609,5488.362638,0,0.000000e+00,NaN,NaN,0,0.000000,NaN,NaN,core_front_9d_rule_test,front_ex_9d,Exclude 9D from Core Front; use Front rule for...,False,excluded,0.7,0.65,68.0,NaN,0.7,0.65,68.0,14.5,0.7,0.7,68.0,10.0,0.7,0.7,70.0,8.5
4,core_front_9d_rule_0001,all_qualified_trades,2025,192,43,2025-03-04,2025-11-20,12,33,25.265625,0.963542,2.855671e+06,14873.285925,14766.743116,-92588.332880,45095.398744,10.445328,-2.926175e+05,-285758.580619,-284321.767855,9.454786e+04,3,0,0,121832.033880,634.541843,661.260645,-4430.458987,2839.785902,9.960017,-12787.352083,-12460.737257,-12362.932080,7556.420941,192,2.855671e+06,634.541843,-92588.332880,0,0.000000,NaN,NaN,core_front_9d_rule_test,front_ex_9d,Exclude 9D from Core Front; use Front rule for...,False,excluded,0.7,0.65,68.0,NaN,0.7,0.65,68.0,14.5,0.7,


Pass diagnostics:


,candidate_id,pass_rows_all_dates,pass_rows_with_outcomes,Front_pass_rows_with_outcomes,Middle_pass_rows_with_outcomes,Back_pass_rows_with_outcomes,tenor_9_included,tenor_9_pass_rows_with_outcomes,tenor_12_pass_rows_with_outcomes,tenor_15_pass_rows_with_outcomes,front_9d_forecast_vol_pct_min
0,core_front_9d_rule_0001,1064,1064,209,406,449,False,0,107,102,NaN
1,core_front_9d_rule_0002,1139,1139,284,406,449,True,75,107,102,16.0
2,core_front_9d_rule_0003,1101,1101,246,406,449,True,37,107,102,18.0
3,core_front_9d_rule_0004,1075,1075,220,406,449,True,11,107,102,20.0



Saved outputs:
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\10_core_front_9d_rule_sweep_candidate_summary_20260704_222153.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\10_core_front_9d_rule_sweep_by_bucket_20260704_222153.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\10_core_front_9d_rule_sweep_by_tenor_20260704_222153.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\10_core_front_9d_rule_sweep_by_year_20260704_222153.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\10_core_front_9d_rule_sweep_candidate_pass_diagnostics_20260704_222153.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\10_core_front_9d_rule_sweep_ex9d_vs_conditional_comparison_20260704_222153.csv
  C:\Users\patri\vrp_project\data\processed\vrp_unifi

In [15]:
# ======================================================================================
# Cell 13 — Middle weakness diagnostic and candidate design
# ======================================================================================
#
# Objective:
#   With Front provisionally set from Cell 12, diagnose the remaining Middle weakness
#   and create a small Middle-focused candidate design for a later sweep.
#
# Provisional Front rule from Cell 12:
#   9D:
#       model_vrp_log > 0.70
#       model_vrp_z   > 0.65
#       RSI14         < 68
#       forecast vol  > 20.0
#
#   12D / 15D:
#       model_vrp_log > 0.70
#       model_vrp_z   > 0.65
#       RSI14         < 68
#       forecast vol  > 14.5
#
# Current Middle rule to diagnose:
#   18D / 21D / 24D:
#       model_vrp_log > 0.70
#       model_vrp_z   > 0.70
#       RSI14         < 68
#       forecast vol  > 10.0
#
# Fixed Back:
#   Back remains locked unchanged.
#
# Diagnostic questions:
#   1. Is Middle weakness concentrated in 18D, 21D, or 24D?
#   2. Is the Middle 2026 problem caused by low forecast vol, high RSI, weak VRP,
#      or all Middle tenors broadly?
#   3. What small Middle candidate set should be swept next?
#
# Candidate design only:
#   middle_rule_0001: current Middle rule
#   middle_rule_0002: Middle FV floor 11.5
#   middle_rule_0003: Middle FV floor 13.0
#   middle_rule_0004: Middle FV floor 14.5
#   middle_rule_0005: Middle RSI < 66
#   middle_rule_0006: Middle FV floor 13.0 and RSI < 66
#
# Important:
#   This cell does NOT sweep the new Middle candidates.
#   This cell does NOT select final thresholds.
#   This cell does NOT change sizing.
#   This cell does NOT lock production.
#   No Secondary.
# ======================================================================================

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 280)
pd.set_option("display.width", 460)

# ======================================================================================
# 0. Fallback config
# ======================================================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

if "NEW_BRANCH" not in globals():
    NEW_BRANCH = "vrp_unified_fds_core_signal_threshold_research_v1"

if "OUT_PROCESSED_DIR" not in globals():
    OUT_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / NEW_BRANCH
    OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if "OUT_AUDIT_DIR" not in globals():
    OUT_AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / NEW_BRANCH
    OUT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CELL13_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if "LOCKED_MODEL_SPEC" not in globals():
    LOCKED_MODEL_SPEC = "unified_fds_no_min_return"

if "ALL_TENORS" not in globals():
    ALL_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

if "TENOR_BUCKET_MAP" not in globals():
    TENOR_BUCKET_MAP = {
        9: "Front",
        12: "Front",
        15: "Front",
        18: "Middle",
        21: "Middle",
        24: "Middle",
        27: "Back",
        30: "Back",
        33: "Back",
    }

if "BUCKET_ORDER_MAP" not in globals():
    BUCKET_ORDER_MAP = {
        "Front": 1,
        "Middle": 2,
        "Back": 3,
    }

if "LOCKED_BACK_CORE_THRESHOLDS" not in globals():
    LOCKED_BACK_CORE_THRESHOLDS = {
        "model_vrp_log": 0.70,
        "model_vrp_z_3m": 0.70,
        "model_vrp_z_1y": 0.70,
        "RSI14_max": 70.0,
        "forecast_vol_pct_min": 8.5,
    }

PROVISIONAL_FRONT_CANDIDATE_ID = "core_front_9d_rule_0004"
PRIMARY_SELECTION_UNIVERSE = "one_trade_per_bucket_per_date_best_rank"
ALL_QUALIFIED_UNIVERSE = "all_qualified_trades"

MIDDLE_CANDIDATE_FAMILY = "core_middle_rule_test"

# ======================================================================================
# 1. Helpers
# ======================================================================================

def latest_file(directory: Path, pattern: str, required: bool = True) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"No file found in {directory} matching {pattern}")
    return None


def profit_factor_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = float(pnl[pnl > 0].sum())
    gross_loss = float(pnl[pnl < 0].sum())

    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0:
        return np.inf
    return np.nan


def max_drawdown_from_pnl(pnl: pd.Series) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)

    if len(pnl) == 0:
        return np.nan

    cumulative = pnl.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max

    return float(drawdown.min())


def worst_rolling_sum(pnl: pd.Series, window: int) -> float:
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0.0)

    if len(pnl) == 0:
        return np.nan

    if len(pnl) < window:
        return float(pnl.sum())

    return float(pnl.rolling(window=window, min_periods=window).sum().min())


def threshold_dict(model_vrp_log: float, model_vrp_z: float, rsi_max: float, forecast_vol_pct_min: float) -> dict:
    return {
        "model_vrp_log": float(model_vrp_log),
        "model_vrp_z": float(model_vrp_z),
        "model_vrp_z_3m": float(model_vrp_z),
        "model_vrp_z_1y": float(model_vrp_z),
        "RSI14_max": float(rsi_max),
        "forecast_vol_pct_min": float(forecast_vol_pct_min),
    }


def summarize_trades(df: pd.DataFrame, group_cols: list[str], diagnostic_label: str) -> pd.DataFrame:
    d = df.copy()

    if d.empty:
        return pd.DataFrame()

    d["trade_date"] = pd.to_datetime(d["trade_date"], errors="coerce").dt.normalize()

    numeric_cols = [
        "tenor",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
        "forecast_vol_pct",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "middle_min_z",
    ]

    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d[
        d["normalized_pnl_dollars"].notna()
        & d["normalized_pnl_per_day"].notna()
    ].copy()

    if d.empty:
        return pd.DataFrame()

    d = d.sort_values(["trade_date", "tenor"]).copy()

    rows = []

    for keys, g in d.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            "diagnostic_label": diagnostic_label,
        }

        for col, key in zip(group_cols, keys):
            row[col] = key

        pnl = pd.to_numeric(g["normalized_pnl_dollars"], errors="coerce")
        pnl_day = pd.to_numeric(g["normalized_pnl_per_day"], errors="coerce")
        wins = pd.to_numeric(g["win"], errors="coerce") if "win" in g.columns else pd.Series(np.nan, index=g.index)

        y2025 = g[g["trade_date"].dt.year == 2025].copy()
        y2026 = g[g["trade_date"].dt.year == 2026].copy()

        row.update({
            "trades": int(len(g)),
            "unique_trade_dates": int(g["trade_date"].nunique()),
            "date_min": g["trade_date"].min(),
            "date_max": g["trade_date"].max(),
            "tenor_min": int(g["tenor"].min()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "tenor_max": int(g["tenor"].max()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "avg_tenor": float(g["tenor"].mean()) if "tenor" in g.columns and g["tenor"].notna().any() else np.nan,
            "win_rate": float(wins.mean()) if wins.notna().sum() else np.nan,

            "total_pnl": float(pnl.sum()),
            "avg_pnl": float(pnl.mean()),
            "median_pnl": float(pnl.median()),
            "worst_trade": float(pnl.min()),
            "best_trade": float(pnl.max()),
            "profit_factor": profit_factor_from_pnl(pnl),
            "selected_trade_drawdown": max_drawdown_from_pnl(pnl),
            "worst_5_trade_sum": worst_rolling_sum(pnl, 5),
            "worst_10_trade_sum": worst_rolling_sum(pnl, 10),
            "worst_20_trade_sum": worst_rolling_sum(pnl, 20),
            "trades_le_neg_50k": int((pnl <= -50_000).sum()),
            "trades_le_neg_100k": int((pnl <= -100_000).sum()),
            "trades_le_neg_150k": int((pnl <= -150_000).sum()),

            "total_pnl_per_day_sum": float(pnl_day.sum()),
            "avg_pnl_per_day": float(pnl_day.mean()),
            "median_pnl_per_day": float(pnl_day.median()),
            "worst_trade_pnl_per_day": float(pnl_day.min()),
            "best_trade_pnl_per_day": float(pnl_day.max()),
            "profit_factor_pnl_per_day": profit_factor_from_pnl(pnl_day),
            "pnl_per_day_drawdown": max_drawdown_from_pnl(pnl_day),
            "worst_5_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 5),
            "worst_10_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 10),
            "worst_20_trade_pnl_per_day_sum": worst_rolling_sum(pnl_day, 20),

            "trades_2025": int(len(y2025)),
            "pnl_2025": float(y2025["normalized_pnl_dollars"].sum()) if len(y2025) else 0.0,
            "avg_pnl_per_day_2025": float(y2025["normalized_pnl_per_day"].mean()) if len(y2025) else np.nan,
            "worst_trade_2025": float(y2025["normalized_pnl_dollars"].min()) if len(y2025) else np.nan,

            "trades_2026": int(len(y2026)),
            "pnl_2026": float(y2026["normalized_pnl_dollars"].sum()) if len(y2026) else 0.0,
            "avg_pnl_per_day_2026": float(y2026["normalized_pnl_per_day"].mean()) if len(y2026) else np.nan,
            "worst_trade_2026": float(y2026["normalized_pnl_dollars"].min()) if len(y2026) else np.nan,
        })

        for signal_col in ["forecast_vol_pct", "RSI14", "model_vrp_log", "model_vrp_z_3m", "model_vrp_z_1y", "middle_min_z"]:
            if signal_col in g.columns:
                row[f"avg_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").mean())
                row[f"min_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").min())
                row[f"max_{signal_col}"] = float(pd.to_numeric(g[signal_col], errors="coerce").max())

        rows.append(row)

    return pd.DataFrame(rows)


def attach_rule_columns(summary_df: pd.DataFrame, grid_wide: pd.DataFrame) -> pd.DataFrame:
    if summary_df.empty:
        return summary_df

    cols = [
        "candidate_id",
        "candidate_family",
        "candidate_subfamily",
        "candidate_description",

        "front_9d_model_vrp_log",
        "front_9d_model_vrp_z",
        "front_9d_RSI14_max",
        "front_9d_forecast_vol_pct_min",
        "front_12_15_model_vrp_log",
        "front_12_15_model_vrp_z",
        "front_12_15_RSI14_max",
        "front_12_15_forecast_vol_pct_min",

        "middle_model_vrp_log",
        "middle_model_vrp_z",
        "middle_RSI14_max",
        "middle_forecast_vol_pct_min",

        "back_model_vrp_log",
        "back_model_vrp_z",
        "back_RSI14_max",
        "back_forecast_vol_pct_min",
    ]

    available = [c for c in cols if c in grid_wide.columns]

    return summary_df.merge(
        grid_wide[available].drop_duplicates("candidate_id"),
        on="candidate_id",
        how="left",
        validate="many_to_one",
    )


# ======================================================================================
# 2. Load inputs
# ======================================================================================

signal_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "01_unified_fds_signal_base_panel_*.parquet",
    required=True,
)

cell12_selected_trade_panel_path = latest_file(
    OUT_PROCESSED_DIR,
    "10_core_front_9d_rule_sweep_selected_trade_panel_*.parquet",
    required=True,
)

cell12_candidate_summary_path = latest_file(
    OUT_AUDIT_DIR,
    "10_core_front_9d_rule_sweep_candidate_summary_*.csv",
    required=True,
)

cell12_by_bucket_path = latest_file(
    OUT_AUDIT_DIR,
    "10_core_front_9d_rule_sweep_by_bucket_*.csv",
    required=True,
)

cell12_by_tenor_path = latest_file(
    OUT_AUDIT_DIR,
    "10_core_front_9d_rule_sweep_by_tenor_*.csv",
    required=True,
)

cell12_by_year_path = latest_file(
    OUT_AUDIT_DIR,
    "10_core_front_9d_rule_sweep_by_year_*.csv",
    required=True,
)

signal = pd.read_parquet(signal_panel_path)
selected_panel = pd.read_parquet(cell12_selected_trade_panel_path)
cell12_summary = pd.read_csv(cell12_candidate_summary_path)
cell12_by_bucket = pd.read_csv(cell12_by_bucket_path)
cell12_by_tenor = pd.read_csv(cell12_by_tenor_path)
cell12_by_year = pd.read_csv(cell12_by_year_path)

signal["trade_date"] = pd.to_datetime(signal["trade_date"], errors="coerce").dt.normalize()
selected_panel["trade_date"] = pd.to_datetime(selected_panel["trade_date"], errors="coerce").dt.normalize()

for df in [signal, selected_panel]:
    for c in [
        "tenor",
        "tenor_bucket_order",
        "model_vrp_log",
        "model_vrp_z_3m",
        "model_vrp_z_1y",
        "RSI14",
        "forecast_vol_pct",
        "normalized_pnl_dollars",
        "normalized_pnl_per_day",
        "win",
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

# Focus on the provisional Cell 12 winner: 9D allowed only if FV > 20.0.
provisional_panel = selected_panel[
    selected_panel["candidate_id"].eq(PROVISIONAL_FRONT_CANDIDATE_ID)
].copy()

provisional_all_qualified = provisional_panel[
    provisional_panel["selection_universe"].eq(ALL_QUALIFIED_UNIVERSE)
].copy()

provisional_primary = provisional_panel[
    provisional_panel["selection_universe"].eq(PRIMARY_SELECTION_UNIVERSE)
].copy()

middle_all_qualified = provisional_all_qualified[
    provisional_all_qualified["tenor_bucket"].eq("Middle")
].copy()

middle_primary = provisional_primary[
    provisional_primary["tenor_bucket"].eq("Middle")
].copy()

for df in [middle_all_qualified, middle_primary]:
    df["middle_min_z"] = df[["model_vrp_z_3m", "model_vrp_z_1y"]].min(axis=1)

# ======================================================================================
# 3. Middle diagnostics under provisional Front rule
# ======================================================================================

# Basic Middle diagnostics.
middle_by_tenor_all = summarize_trades(
    middle_all_qualified,
    group_cols=["selection_universe", "tenor", "tenor_bucket", "tenor_bucket_order"],
    diagnostic_label="middle_all_qualified_by_tenor",
)

middle_by_tenor_primary = summarize_trades(
    middle_primary,
    group_cols=["selection_universe", "tenor", "tenor_bucket", "tenor_bucket_order"],
    diagnostic_label="middle_primary_by_tenor",
)

middle_by_tenor = pd.concat(
    [middle_by_tenor_all, middle_by_tenor_primary],
    ignore_index=True,
)

middle_all_year = middle_all_qualified.copy()
middle_all_year["year"] = middle_all_year["trade_date"].dt.year.astype(int)

middle_primary_year = middle_primary.copy()
middle_primary_year["year"] = middle_primary_year["trade_date"].dt.year.astype(int)

middle_by_year = pd.concat(
    [
        summarize_trades(
            middle_all_year,
            group_cols=["selection_universe", "year"],
            diagnostic_label="middle_all_qualified_by_year",
        ),
        summarize_trades(
            middle_primary_year,
            group_cols=["selection_universe", "year"],
            diagnostic_label="middle_primary_by_year",
        ),
    ],
    ignore_index=True,
)

middle_by_tenor_year = pd.concat(
    [
        summarize_trades(
            middle_all_year,
            group_cols=["selection_universe", "tenor", "year"],
            diagnostic_label="middle_all_qualified_by_tenor_year",
        ),
        summarize_trades(
            middle_primary_year,
            group_cols=["selection_universe", "tenor", "year"],
            diagnostic_label="middle_primary_by_tenor_year",
        ),
    ],
    ignore_index=True,
)

# State diagnostics.
middle_state = pd.concat(
    [
        middle_all_qualified.assign(state_source="all_qualified"),
        middle_primary.assign(state_source="primary_selected"),
    ],
    ignore_index=True,
)

middle_state["middle_min_z"] = middle_state[["model_vrp_z_3m", "model_vrp_z_1y"]].min(axis=1)

middle_state["forecast_vol_state"] = pd.cut(
    middle_state["forecast_vol_pct"],
    bins=[-np.inf, 10.0, 11.5, 13.0, 14.5, 16.0, 18.0, np.inf],
    labels=["<=10.0", "10.0-11.5", "11.5-13.0", "13.0-14.5", "14.5-16.0", "16.0-18.0", ">18.0"],
    include_lowest=True,
)

middle_state["RSI14_state"] = pd.cut(
    middle_state["RSI14"],
    bins=[-np.inf, 60.0, 64.0, 66.0, 68.0, 70.0, np.inf],
    labels=["<=60", "60-64", "64-66", "66-68", "68-70", ">70"],
    include_lowest=True,
)

middle_state["model_vrp_log_state"] = pd.cut(
    middle_state["model_vrp_log"],
    bins=[-np.inf, 0.70, 0.75, 0.80, 0.90, 1.00, np.inf],
    labels=["<=0.70", "0.70-0.75", "0.75-0.80", "0.80-0.90", "0.90-1.00", ">1.00"],
    include_lowest=True,
)

middle_state["middle_min_z_state"] = pd.cut(
    middle_state["middle_min_z"],
    bins=[-np.inf, 0.70, 0.80, 1.00, 1.25, 1.50, np.inf],
    labels=["<=0.70", "0.70-0.80", "0.80-1.00", "1.00-1.25", "1.25-1.50", ">1.50"],
    include_lowest=True,
)

middle_state_forecast_vol = summarize_trades(
    middle_state,
    group_cols=["state_source", "forecast_vol_state"],
    diagnostic_label="middle_by_forecast_vol_state",
)

middle_state_rsi = summarize_trades(
    middle_state,
    group_cols=["state_source", "RSI14_state"],
    diagnostic_label="middle_by_RSI14_state",
)

middle_state_vrp_log = summarize_trades(
    middle_state,
    group_cols=["state_source", "model_vrp_log_state"],
    diagnostic_label="middle_by_model_vrp_log_state",
)

middle_state_min_z = summarize_trades(
    middle_state,
    group_cols=["state_source", "middle_min_z_state"],
    diagnostic_label="middle_by_min_z_state",
)

middle_state_diagnostics = pd.concat(
    [
        middle_state_forecast_vol,
        middle_state_rsi,
        middle_state_vrp_log,
        middle_state_min_z,
    ],
    ignore_index=True,
)

# 2026 Middle detail for inspection.
middle_2026_detail = middle_primary[
    middle_primary["trade_date"].dt.year.eq(2026)
].copy()

middle_2026_detail = middle_2026_detail[
    [
        c for c in [
            "candidate_id",
            "selection_universe",
            "trade_date",
            "tenor",
            "tenor_bucket",
            "normalized_pnl_dollars",
            "normalized_pnl_per_day",
            "win",
            "forecast_vol_pct",
            "RSI14",
            "model_vrp_log",
            "model_vrp_z_3m",
            "model_vrp_z_1y",
            "middle_min_z",
            "implied_vol_pct",
            "forecast_vol_pct",
            "model_vrp_ratio",
        ]
        if c in middle_2026_detail.columns
    ]
].sort_values(["trade_date", "tenor"]).reset_index(drop=True)

# ======================================================================================
# 4. Build Middle candidate design
# ======================================================================================

FRONT_9D_RULE = threshold_dict(
    model_vrp_log=0.70,
    model_vrp_z=0.65,
    rsi_max=68.0,
    forecast_vol_pct_min=20.0,
)

FRONT_12_15_RULE = threshold_dict(
    model_vrp_log=0.70,
    model_vrp_z=0.65,
    rsi_max=68.0,
    forecast_vol_pct_min=14.5,
)

BACK_RULE = threshold_dict(
    model_vrp_log=LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"],
    model_vrp_z=LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"],
    rsi_max=LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"],
    forecast_vol_pct_min=LOCKED_BACK_CORE_THRESHOLDS["forecast_vol_pct_min"],
)

middle_candidate_rules = [
    {
        "candidate_id": "core_middle_rule_0001",
        "candidate_subfamily": "current_middle",
        "candidate_description": "Current Middle rule: log > 0.70, z > 0.70, RSI < 68, forecast vol > 10.0.",
        "middle_rule": threshold_dict(0.70, 0.70, 68.0, 10.0),
    },
    {
        "candidate_id": "core_middle_rule_0002",
        "candidate_subfamily": "middle_higher_forecast_vol",
        "candidate_description": "Middle forecast-vol floor > 11.5.",
        "middle_rule": threshold_dict(0.70, 0.70, 68.0, 11.5),
    },
    {
        "candidate_id": "core_middle_rule_0003",
        "candidate_subfamily": "middle_higher_forecast_vol",
        "candidate_description": "Middle forecast-vol floor > 13.0.",
        "middle_rule": threshold_dict(0.70, 0.70, 68.0, 13.0),
    },
    {
        "candidate_id": "core_middle_rule_0004",
        "candidate_subfamily": "middle_higher_forecast_vol",
        "candidate_description": "Middle forecast-vol floor > 14.5.",
        "middle_rule": threshold_dict(0.70, 0.70, 68.0, 14.5),
    },
    {
        "candidate_id": "core_middle_rule_0005",
        "candidate_subfamily": "middle_tighter_RSI",
        "candidate_description": "Middle RSI < 66.",
        "middle_rule": threshold_dict(0.70, 0.70, 66.0, 10.0),
    },
    {
        "candidate_id": "core_middle_rule_0006",
        "candidate_subfamily": "middle_higher_forecast_vol_and_tighter_RSI",
        "candidate_description": "Middle forecast-vol floor > 13.0 and RSI < 66.",
        "middle_rule": threshold_dict(0.70, 0.70, 66.0, 13.0),
    },
]

wide_rows = []

for spec in middle_candidate_rules:
    m = spec["middle_rule"]

    wide_rows.append({
        "candidate_id": spec["candidate_id"],
        "candidate_family": MIDDLE_CANDIDATE_FAMILY,
        "candidate_subfamily": spec["candidate_subfamily"],
        "candidate_description": spec["candidate_description"],
        "locked_model_spec": LOCKED_MODEL_SPEC,

        "front_9d_include": True,
        "front_9d_model_vrp_log": FRONT_9D_RULE["model_vrp_log"],
        "front_9d_model_vrp_z": FRONT_9D_RULE["model_vrp_z"],
        "front_9d_RSI14_max": FRONT_9D_RULE["RSI14_max"],
        "front_9d_forecast_vol_pct_min": FRONT_9D_RULE["forecast_vol_pct_min"],

        "front_12_15_include": True,
        "front_12_15_model_vrp_log": FRONT_12_15_RULE["model_vrp_log"],
        "front_12_15_model_vrp_z": FRONT_12_15_RULE["model_vrp_z"],
        "front_12_15_RSI14_max": FRONT_12_15_RULE["RSI14_max"],
        "front_12_15_forecast_vol_pct_min": FRONT_12_15_RULE["forecast_vol_pct_min"],

        "middle_include": True,
        "middle_model_vrp_log": m["model_vrp_log"],
        "middle_model_vrp_z": m["model_vrp_z"],
        "middle_RSI14_max": m["RSI14_max"],
        "middle_forecast_vol_pct_min": m["forecast_vol_pct_min"],

        "back_include": True,
        "back_model_vrp_log": BACK_RULE["model_vrp_log"],
        "back_model_vrp_z": BACK_RULE["model_vrp_z"],
        "back_RSI14_max": BACK_RULE["RSI14_max"],
        "back_forecast_vol_pct_min": BACK_RULE["forecast_vol_pct_min"],
    })

candidate_grid_wide = pd.DataFrame(wide_rows)

candidate_grid_wide["front_fixed_to_cell12_9d_fv20_rule"] = (
    (candidate_grid_wide["front_9d_forecast_vol_pct_min"] == 20.0)
    & (candidate_grid_wide["front_12_15_forecast_vol_pct_min"] == 14.5)
    & (candidate_grid_wide["front_9d_model_vrp_z"] == 0.65)
    & (candidate_grid_wide["front_12_15_model_vrp_z"] == 0.65)
)

candidate_grid_wide["back_locked"] = (
    (candidate_grid_wide["back_model_vrp_log"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_log"])
    & (candidate_grid_wide["back_model_vrp_z"] == LOCKED_BACK_CORE_THRESHOLDS["model_vrp_z_3m"])
    & (candidate_grid_wide["back_RSI14_max"] == LOCKED_BACK_CORE_THRESHOLDS["RSI14_max"])
    & (candidate_grid_wide["back_forecast_vol_pct_min"] == LOCKED_BACK_CORE_THRESHOLDS["forecast_vol_pct_min"])
)

long_rows = []

for _, row in candidate_grid_wide.iterrows():
    for tenor in ALL_TENORS:
        bucket = TENOR_BUCKET_MAP[int(tenor)]

        if bucket == "Front" and tenor == 9:
            rule_scope = "front_9d"
            rule_type = "fixed_cell12_9d_fv20_rule"
            model_vrp_log = row["front_9d_model_vrp_log"]
            model_vrp_z = row["front_9d_model_vrp_z"]
            rsi_max = row["front_9d_RSI14_max"]
            fv_floor = row["front_9d_forecast_vol_pct_min"]

        elif bucket == "Front" and tenor in [12, 15]:
            rule_scope = "front_12_15"
            rule_type = "fixed_cell12_front_12_15_rule"
            model_vrp_log = row["front_12_15_model_vrp_log"]
            model_vrp_z = row["front_12_15_model_vrp_z"]
            rsi_max = row["front_12_15_RSI14_max"]
            fv_floor = row["front_12_15_forecast_vol_pct_min"]

        elif bucket == "Middle":
            rule_scope = "middle"
            rule_type = "candidate_middle_rule"
            model_vrp_log = row["middle_model_vrp_log"]
            model_vrp_z = row["middle_model_vrp_z"]
            rsi_max = row["middle_RSI14_max"]
            fv_floor = row["middle_forecast_vol_pct_min"]

        elif bucket == "Back":
            rule_scope = "back"
            rule_type = "locked_back_rule"
            model_vrp_log = row["back_model_vrp_log"]
            model_vrp_z = row["back_model_vrp_z"]
            rsi_max = row["back_RSI14_max"]
            fv_floor = row["back_forecast_vol_pct_min"]

        else:
            raise ValueError(f"Unexpected tenor/bucket combination: tenor={tenor}, bucket={bucket}")

        long_rows.append({
            "candidate_id": row["candidate_id"],
            "candidate_family": row["candidate_family"],
            "candidate_subfamily": row["candidate_subfamily"],
            "locked_model_spec": row["locked_model_spec"],
            "tenor": int(tenor),
            "tenor_bucket": bucket,
            "tenor_bucket_order": BUCKET_ORDER_MAP[bucket],
            "rule_scope": rule_scope,
            "rule_type": rule_type,
            "include_tenor": True,

            "model_vrp_log": float(model_vrp_log),
            "model_vrp_z": float(model_vrp_z),
            "model_vrp_z_3m": float(model_vrp_z),
            "model_vrp_z_1y": float(model_vrp_z),
            "RSI14_max": float(rsi_max),
            "forecast_vol_pct_min": float(fv_floor),

            "is_front_9d": bucket == "Front" and tenor == 9,
            "is_front_12_15": bucket == "Front" and tenor in [12, 15],
            "is_middle_candidate_rule": bucket == "Middle",
            "is_back_locked": bucket == "Back",
        })

candidate_grid_long = (
    pd.DataFrame(long_rows)
    .sort_values(["candidate_id", "tenor"])
    .reset_index(drop=True)
)

# Attach design rule columns to diagnostics where useful.
middle_by_tenor["candidate_id"] = PROVISIONAL_FRONT_CANDIDATE_ID
middle_by_year["candidate_id"] = PROVISIONAL_FRONT_CANDIDATE_ID
middle_by_tenor_year["candidate_id"] = PROVISIONAL_FRONT_CANDIDATE_ID
middle_state_diagnostics["candidate_id"] = PROVISIONAL_FRONT_CANDIDATE_ID
middle_2026_detail["candidate_id"] = PROVISIONAL_FRONT_CANDIDATE_ID

# ======================================================================================
# 5. Diagnostic conclusion
# ======================================================================================

middle_primary_2026 = middle_by_year[
    middle_by_year["diagnostic_label"].eq("middle_primary_by_year")
    & middle_by_year["year"].eq(2026)
].copy()

middle_2026_avg = (
    float(middle_primary_2026["avg_pnl_per_day"].iloc[0])
    if not middle_primary_2026.empty
    else np.nan
)

middle_primary_by_tenor_2026 = middle_by_tenor_year[
    middle_by_tenor_year["diagnostic_label"].eq("middle_primary_by_tenor_year")
    & middle_by_tenor_year["year"].eq(2026)
].copy()

worst_middle_2026_tenor = np.nan
worst_middle_2026_avg = np.nan

if not middle_primary_by_tenor_2026.empty:
    worst_row = middle_primary_by_tenor_2026.sort_values("avg_pnl_per_day").iloc[0]
    worst_middle_2026_tenor = int(worst_row["tenor"])
    worst_middle_2026_avg = float(worst_row["avg_pnl_per_day"])

diagnostic_conclusion = pd.DataFrame([
    {
        "diagnostic": "provisional_front_candidate_loaded",
        "status": "PASS" if not provisional_panel.empty else "FAIL",
        "detail": f"candidate_id={PROVISIONAL_FRONT_CANDIDATE_ID}; rows={len(provisional_panel):,}",
    },
    {
        "diagnostic": "middle_primary_2026_avg_pnl_per_day",
        "status": "WARN" if pd.notna(middle_2026_avg) and middle_2026_avg < 0 else "PASS",
        "detail": f"middle_primary_2026_avg_pnl_per_day={middle_2026_avg}",
    },
    {
        "diagnostic": "worst_middle_2026_tenor",
        "status": "INFO",
        "detail": f"worst_middle_2026_tenor={worst_middle_2026_tenor}; avg_pnl_per_day={worst_middle_2026_avg}",
    },
    {
        "diagnostic": "middle_candidate_design_count",
        "status": "PASS" if len(candidate_grid_wide) == 6 else "FAIL",
        "detail": f"candidate_count={len(candidate_grid_wide):,}",
    },
    {
        "diagnostic": "no_middle_sweep_performed",
        "status": "PASS",
        "detail": "Cell 13 creates Middle candidate design only. Cell 14 should sweep.",
    },
])

# ======================================================================================
# 6. Save outputs
# ======================================================================================

middle_by_tenor_path = OUT_AUDIT_DIR / f"11_middle_weakness_current_middle_by_tenor_{CELL13_TIMESTAMP}.csv"
middle_by_year_path = OUT_AUDIT_DIR / f"11_middle_weakness_current_middle_by_year_{CELL13_TIMESTAMP}.csv"
middle_by_tenor_year_path = OUT_AUDIT_DIR / f"11_middle_weakness_current_middle_by_tenor_year_{CELL13_TIMESTAMP}.csv"
middle_state_diagnostics_path = OUT_AUDIT_DIR / f"11_middle_weakness_current_middle_state_diagnostics_{CELL13_TIMESTAMP}.csv"
middle_2026_detail_path = OUT_AUDIT_DIR / f"11_middle_weakness_current_middle_2026_trade_detail_{CELL13_TIMESTAMP}.csv"
diagnostic_conclusion_path = OUT_AUDIT_DIR / f"11_middle_weakness_diagnostic_conclusion_{CELL13_TIMESTAMP}.csv"

wide_csv_path = OUT_AUDIT_DIR / f"11_core_middle_rule_candidate_grid_wide_{CELL13_TIMESTAMP}.csv"
wide_parquet_path = OUT_PROCESSED_DIR / f"11_core_middle_rule_candidate_grid_wide_{CELL13_TIMESTAMP}.parquet"
long_csv_path = OUT_AUDIT_DIR / f"11_core_middle_rule_candidate_grid_long_{CELL13_TIMESTAMP}.csv"
long_parquet_path = OUT_PROCESSED_DIR / f"11_core_middle_rule_candidate_grid_long_{CELL13_TIMESTAMP}.parquet"
ranges_path = OUT_AUDIT_DIR / f"11_core_middle_rule_candidate_grid_ranges_{CELL13_TIMESTAMP}.json"
validation_path = OUT_AUDIT_DIR / f"11_core_middle_rule_candidate_grid_validation_{CELL13_TIMESTAMP}.csv"
manifest_path = OUT_AUDIT_DIR / f"11_core_middle_rule_candidate_grid_manifest_{CELL13_TIMESTAMP}.json"

middle_by_tenor.to_csv(middle_by_tenor_path, index=False)
middle_by_year.to_csv(middle_by_year_path, index=False)
middle_by_tenor_year.to_csv(middle_by_tenor_year_path, index=False)
middle_state_diagnostics.to_csv(middle_state_diagnostics_path, index=False)
middle_2026_detail.to_csv(middle_2026_detail_path, index=False)
diagnostic_conclusion.to_csv(diagnostic_conclusion_path, index=False)

candidate_grid_wide.to_csv(wide_csv_path, index=False)
candidate_grid_wide.to_parquet(wide_parquet_path, index=False)
candidate_grid_long.to_csv(long_csv_path, index=False)
candidate_grid_long.to_parquet(long_parquet_path, index=False)

grid_ranges = {
    "candidate_family": MIDDLE_CANDIDATE_FAMILY,
    "provisional_front_candidate_id": PROVISIONAL_FRONT_CANDIDATE_ID,
    "front_9d_rule": FRONT_9D_RULE,
    "front_12_15_rule": FRONT_12_15_RULE,
    "middle_candidate_rules": [
        {
            "candidate_id": spec["candidate_id"],
            "candidate_subfamily": spec["candidate_subfamily"],
            "candidate_description": spec["candidate_description"],
            "middle_rule": spec["middle_rule"],
        }
        for spec in middle_candidate_rules
    ],
    "back_rule": BACK_RULE,
}

with open(ranges_path, "w", encoding="utf-8") as f:
    json.dump(grid_ranges, f, indent=2, default=str)

# ======================================================================================
# 7. Validation
# ======================================================================================

validation_rows = []

def add_validation(check, status, detail):
    validation_rows.append({
        "check": check,
        "status": status,
        "detail": detail,
    })

required_selected_cols = [
    "candidate_id",
    "selection_universe",
    "trade_date",
    "tenor",
    "tenor_bucket",
    "tenor_bucket_order",
    "normalized_pnl_dollars",
    "normalized_pnl_per_day",
    "win",
    "forecast_vol_pct",
    "RSI14",
    "model_vrp_log",
    "model_vrp_z_3m",
    "model_vrp_z_1y",
]

missing_selected_cols = [c for c in required_selected_cols if c not in selected_panel.columns]

candidate_count = len(candidate_grid_wide)
expected_candidate_count = 6

long_row_count = len(candidate_grid_long)
expected_long_row_count = candidate_count * len(ALL_TENORS)

bad_z_equality = candidate_grid_long[
    candidate_grid_long["model_vrp_z_3m"].ne(candidate_grid_long["model_vrp_z_1y"])
]

bad_front_9d = candidate_grid_long[
    candidate_grid_long["is_front_9d"]
    & (
        candidate_grid_long["model_vrp_log"].ne(FRONT_9D_RULE["model_vrp_log"])
        | candidate_grid_long["model_vrp_z"].ne(FRONT_9D_RULE["model_vrp_z"])
        | candidate_grid_long["RSI14_max"].ne(FRONT_9D_RULE["RSI14_max"])
        | candidate_grid_long["forecast_vol_pct_min"].ne(FRONT_9D_RULE["forecast_vol_pct_min"])
    )
]

bad_front_12_15 = candidate_grid_long[
    candidate_grid_long["is_front_12_15"]
    & (
        candidate_grid_long["model_vrp_log"].ne(FRONT_12_15_RULE["model_vrp_log"])
        | candidate_grid_long["model_vrp_z"].ne(FRONT_12_15_RULE["model_vrp_z"])
        | candidate_grid_long["RSI14_max"].ne(FRONT_12_15_RULE["RSI14_max"])
        | candidate_grid_long["forecast_vol_pct_min"].ne(FRONT_12_15_RULE["forecast_vol_pct_min"])
    )
]

bad_back = candidate_grid_long[
    candidate_grid_long["is_back_locked"]
    & (
        candidate_grid_long["model_vrp_log"].ne(BACK_RULE["model_vrp_log"])
        | candidate_grid_long["model_vrp_z"].ne(BACK_RULE["model_vrp_z"])
        | candidate_grid_long["RSI14_max"].ne(BACK_RULE["RSI14_max"])
        | candidate_grid_long["forecast_vol_pct_min"].ne(BACK_RULE["forecast_vol_pct_min"])
    )
]

middle_fv_values = sorted(
    candidate_grid_wide["middle_forecast_vol_pct_min"].dropna().unique().tolist()
)
middle_rsi_values = sorted(
    candidate_grid_wide["middle_RSI14_max"].dropna().unique().tolist()
)

trade_metric_cols = [
    "trades",
    "win_rate",
    "total_pnl",
    "avg_pnl",
    "avg_pnl_per_day",
    "profit_factor",
    "worst_trade",
    "pnl_per_day_drawdown",
]
unexpected_trade_cols = [
    c for c in trade_metric_cols
    if c in candidate_grid_wide.columns or c in candidate_grid_long.columns
]

add_validation(
    "signal_panel_loaded",
    "PASS" if len(signal) > 0 else "FAIL",
    f"rows={len(signal):,}; path={signal_panel_path}",
)

add_validation(
    "cell12_selected_trade_panel_loaded",
    "PASS" if len(selected_panel) > 0 else "FAIL",
    f"rows={len(selected_panel):,}; path={cell12_selected_trade_panel_path}",
)

add_validation(
    "required_selected_columns_available",
    "PASS" if not missing_selected_cols else "FAIL",
    f"missing={missing_selected_cols}",
)

add_validation(
    "provisional_front_candidate_available",
    "PASS" if not provisional_panel.empty else "FAIL",
    f"candidate_id={PROVISIONAL_FRONT_CANDIDATE_ID}; rows={len(provisional_panel):,}",
)

add_validation(
    "middle_primary_diagnostics_created",
    "PASS" if not middle_by_tenor.empty and not middle_by_year.empty else "FAIL",
    f"by_tenor_rows={len(middle_by_tenor):,}; by_year_rows={len(middle_by_year):,}",
)

add_validation(
    "middle_2026_detail_created",
    "PASS" if len(middle_2026_detail) > 0 else "WARN",
    f"rows={len(middle_2026_detail):,}",
)

add_validation(
    "middle_state_diagnostics_created",
    "PASS" if not middle_state_diagnostics.empty else "WARN",
    f"rows={len(middle_state_diagnostics):,}",
)

add_validation(
    "middle_candidate_grid_count",
    "PASS" if candidate_count == expected_candidate_count else "FAIL",
    f"candidate_count={candidate_count:,}; expected={expected_candidate_count:,}",
)

add_validation(
    "long_grid_has_all_tenors_per_candidate",
    "PASS" if long_row_count == expected_long_row_count else "FAIL",
    f"long_rows={long_row_count:,}; expected={expected_long_row_count:,}",
)

add_validation(
    "front_9d_fixed_to_fv20",
    "PASS" if bad_front_9d.empty else "FAIL",
    f"bad_rows={len(bad_front_9d):,}",
)

add_validation(
    "front_12_15_fixed",
    "PASS" if bad_front_12_15.empty else "FAIL",
    f"bad_rows={len(bad_front_12_15):,}",
)

add_validation(
    "back_locked",
    "PASS" if bad_back.empty else "FAIL",
    f"bad_rows={len(bad_back):,}",
)

add_validation(
    "middle_candidate_fv_values",
    "PASS" if middle_fv_values == [10.0, 11.5, 13.0, 14.5] else "FAIL",
    f"found={middle_fv_values}",
)

add_validation(
    "middle_candidate_rsi_values",
    "PASS" if middle_rsi_values == [66.0, 68.0] else "FAIL",
    f"found={middle_rsi_values}",
)

add_validation(
    "z_3m_equals_z_1y_within_rules",
    "PASS" if bad_z_equality.empty else "FAIL",
    f"bad_rows={len(bad_z_equality):,}",
)

add_validation(
    "no_trade_metrics_in_candidate_design",
    "PASS" if not unexpected_trade_cols else "FAIL",
    f"unexpected_trade_metric_cols={unexpected_trade_cols}",
)

add_validation(
    "no_new_middle_sweep",
    "PASS",
    "Cell 13 creates Middle candidate design only.",
)

add_validation(
    "no_final_candidate_selection",
    "PASS",
    "No final thresholds selected.",
)

add_validation(
    "no_secondary",
    "PASS",
    "Core only.",
)

add_validation(
    "no_sizing_changes",
    "PASS",
    "No sizing fields included.",
)

add_validation(
    "no_production_lock",
    "PASS",
    "No production threshold lock performed.",
)

validation = pd.DataFrame(validation_rows)
validation.to_csv(validation_path, index=False)

failures = validation[validation["status"] == "FAIL"]

manifest = {
    "notebook": "07_unified_fds_core_signal_threshold_research_v1.ipynb",
    "cell": "Cell 13 — Middle weakness diagnostic and candidate design",
    "timestamp": CELL13_TIMESTAMP,
    "locked_model_spec": LOCKED_MODEL_SPEC,
    "provisional_front_candidate_id": PROVISIONAL_FRONT_CANDIDATE_ID,
    "middle_candidate_family": MIDDLE_CANDIDATE_FAMILY,
    "candidate_count": int(candidate_count),
    "long_grid_rows": int(long_row_count),
    "signal_panel_path": str(signal_panel_path),
    "cell12_selected_trade_panel_path": str(cell12_selected_trade_panel_path),
    "cell12_candidate_summary_path": str(cell12_candidate_summary_path),
    "cell12_by_bucket_path": str(cell12_by_bucket_path),
    "cell12_by_tenor_path": str(cell12_by_tenor_path),
    "cell12_by_year_path": str(cell12_by_year_path),
    "middle_by_tenor_path": str(middle_by_tenor_path),
    "middle_by_year_path": str(middle_by_year_path),
    "middle_by_tenor_year_path": str(middle_by_tenor_year_path),
    "middle_state_diagnostics_path": str(middle_state_diagnostics_path),
    "middle_2026_detail_path": str(middle_2026_detail_path),
    "diagnostic_conclusion_path": str(diagnostic_conclusion_path),
    "wide_csv_path": str(wide_csv_path),
    "wide_parquet_path": str(wide_parquet_path),
    "long_csv_path": str(long_csv_path),
    "long_parquet_path": str(long_parquet_path),
    "ranges_path": str(ranges_path),
    "validation_path": str(validation_path),
    "front_9d_rule": FRONT_9D_RULE,
    "front_12_15_rule": FRONT_12_15_RULE,
    "back_rule": BACK_RULE,
    "middle_candidate_rules": [
        {
            "candidate_id": spec["candidate_id"],
            "candidate_subfamily": spec["candidate_subfamily"],
            "candidate_description": spec["candidate_description"],
            "middle_rule": spec["middle_rule"],
        }
        for spec in middle_candidate_rules
    ],
    "hard_constraints": [
        "Front fixed to Cell 12 provisional decision: 9D FV > 20, 12D/15D FV > 14.5.",
        "Back locked unchanged.",
        "Middle candidate design only.",
        "No Middle sweep in Cell 13.",
        "No final threshold selection.",
        "No Secondary.",
        "No sizing changes.",
        "No production lock.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

if not failures.empty:
    display(validation)
    raise RuntimeError("Cell 13 validation failed. Review validation output above.")

# ======================================================================================
# 8. Display review
# ======================================================================================

print("=" * 100)
print("Cell 13 — Middle weakness diagnostic and candidate design")
print("=" * 100)
print(f"Signal panel:                         {signal_panel_path}")
print(f"Cell 12 selected trade panel:          {cell12_selected_trade_panel_path}")
print(f"Provisional Front candidate:           {PROVISIONAL_FRONT_CANDIDATE_ID}")
print(f"Middle all-qualified rows reviewed:    {len(middle_all_qualified):,}")
print(f"Middle primary rows reviewed:          {len(middle_primary):,}")
print(f"Middle 2026 primary rows reviewed:     {len(middle_2026_detail):,}")
print(f"Middle candidate count:                {candidate_count:,}")
print(f"Long candidate grid rows:              {long_row_count:,}")
print("Front fixed:                           True")
print("Back locked:                           True")
print("No Middle sweep:                       True")
print("No final candidate selection:          True")
print("No Secondary:                          True")
print("No sizing changes:                     True")
print("No production lock:                    True")

print()
print("Validation:")
display(validation)

print()
print("Diagnostic conclusions:")
display(diagnostic_conclusion)

print()
print("Middle diagnostics by tenor:")
display(middle_by_tenor)

print()
print("Middle diagnostics by year:")
display(middle_by_year)

print()
print("Middle diagnostics by tenor/year:")
display(middle_by_tenor_year)

print()
print("Middle state diagnostics:")
display(middle_state_diagnostics)

print()
print("Middle 2026 primary selected trade detail:")
display(middle_2026_detail)

print()
print("Middle candidate grid — wide:")
display(candidate_grid_wide)

print()
print("Middle candidate grid — long:")
display(candidate_grid_long)

print()
print("Saved outputs:")
for p in [
    middle_by_tenor_path,
    middle_by_year_path,
    middle_by_tenor_year_path,
    middle_state_diagnostics_path,
    middle_2026_detail_path,
    diagnostic_conclusion_path,
    wide_csv_path,
    wide_parquet_path,
    long_csv_path,
    long_parquet_path,
    ranges_path,
    validation_path,
    manifest_path,
]:
    print(f"  {p}")

print("\nCELL 13 PASSED")

C:\Users\patri\AppData\Local\Temp\ipykernel_21560\808355830.py:222: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for keys, g in d.groupby(group_cols, dropna=False):
C:\Users\patri\AppData\Local\Temp\ipykernel_21560\808355830.py:222: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for keys, g in d.groupby(group_cols, dropna=False):
C:\Users\patri\AppData\Local\Temp\ipykernel_21560\808355830.py:222: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and si

Cell 13 — Middle weakness diagnostic and candidate design
Signal panel:                         C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\01_unified_fds_signal_base_panel_20200102_20260701_20260704_211636.parquet
Cell 12 selected trade panel:          C:\Users\patri\vrp_project\data\processed\vrp_unified_fds_core_signal_threshold_research_v1\10_core_front_9d_rule_sweep_selected_trade_panel_20260704_222153.parquet
Provisional Front candidate:           core_front_9d_rule_0004
Middle all-qualified rows reviewed:    406
Middle primary rows reviewed:          149
Middle 2026 primary rows reviewed:     29
Middle candidate count:                6
Long candidate grid rows:              54
Front fixed:                           True
Back locked:                           True
No Middle sweep:                       True
No final candidate selection:          True
No Secondary:                          True
No sizing changes:                     

C:\Users\patri\AppData\Local\Temp\ipykernel_21560\808355830.py:222: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for keys, g in d.groupby(group_cols, dropna=False):


,check,status,detail
0,signal_panel_loaded,PASS,"rows=14,688; path=C:\Users\patri\vrp_project\d..."
1,cell12_selected_trade_panel_loaded,PASS,"rows=6,831; path=C:\Users\patri\vrp_project\da..."
2,required_selected_columns_available,PASS,missing=[]
3,provisional_front_candidate_available,PASS,"candidate_id=core_front_9d_rule_0004; rows=1,685"
4,middle_primary_diagnostics_created,PASS,by_tenor_rows=6; by_year_rows=12
5,middle_2026_detail_created,PASS,rows=29
6,middle_state_diagnostics_created,PASS,rows=40
7,middle_candidate_grid_count,PASS,candidate_count=6; expected=6
8,long_grid_has_all_tenors_per_candidate,PASS,long_rows=54; expected=54
9,front_9d_fixed_to_fv20,PASS,bad_rows=0



Diagnostic conclusions:


...


Middle candidate grid — wide:


,candidate_id,candidate_family,candidate_subfamily,candidate_description,locked_model_spec,front_9d_include,front_9d_model_vrp_log,front_9d_model_vrp_z,front_9d_RSI14_max,front_9d_forecast_vol_pct_min,front_12_15_include,front_12_15_model_vrp_log,front_12_15_model_vrp_z,front_12_15_RSI14_max,front_12_15_forecast_vol_pct_min,middle_include,middle_model_vrp_log,middle_model_vrp_z,middle_RSI14_max,middle_forecast_vol_pct_min,back_include,back_model_vrp_log,back_model_vrp_z,back_RSI14_max,back_forecast_vol_pct_min,front_fixed_to_cell12_9d_fv20_rule,back_locked
0,core_middle_rule_0001,core_middle_rule_test,current_middle,"Current Middle rule: log > 0.70, z > 0.70, RSI...",unified_fds_no_min_return,True,0.7,0.65,68.0,20.0,True,0.7,0.65,68.0,14.5,True,0.7,0.7,68.0,10.0,True,0.7,0.7,70.0,8.5,True,True
1,core_middle_rule_0002,core_middle_rule_test,middle_higher_forecast_vol,Middle forecast-vol floor > 11.5.,unified_fds_no_min_return,True,0.7,0.65,68.0,20.0,True,0.7,0.65,68.0,14.5,True,0.7,0.7,68.0,11.5,True,0.7,0.7,70.0,8.5,True,True
2,core_middle_rule_0003,core_middle_rule_test,middle_higher_forecast_vol,Middle forecast-vol floor > 13.0.,unified_fds_no_min_return,True,0.7,0.65,68.0,20.0,True,0.7,0.65,68.0,14.5,True,0.7,0.7,68.0,13.0,True,0.7,0.7,70.0,8.5,True,True
3,core_middle_rule_0004,core_middle_rule_test,middle_higher_forecast_vol,Middle forecast-vol floor > 14.5.,unified_fds_no_min_return,True,0.7,0.65,68.0,20.0,True,0.7,0.65,68.0,14.5,True,0.7,0.7,68.0,14.5,True,0.7,0.7,70.0,8.5,True,True
4,core_middle_rule_0005,core_middle_rule_test,middle_tighter_RSI,Middle RSI < 66.,unified_fds_no_min_return,True,0.7,0.65,68.0,20.0,True,0.7,0.65,68.0,14.5,True,0.7,0.7,66.0,10.0,True,0.7,0.7,70.0,8.5,True,True
5,core_middle_rule_0006,core_middle_rule_test,middle_higher_forecast_vol_and_tighter_RSI,Middle forecast-vol floor > 13.0 and RSI < 66.,unified_fds_no_min_return,True,0.7,0.65,68.0,20.0,True,0.7,0.65,68.0,14.5,True,0.7,0.7,66.0,13.0,True,0.7,0.7,70.0,8.5,True,True



Middle candidate grid — long:


,candidate_id,candidate_family,candidate_subfamily,locked_model_spec,tenor,tenor_bucket,tenor_bucket_order,rule_scope,rule_type,include_tenor,model_vrp_log,model_vrp_z,model_vrp_z_3m,model_vrp_z_1y,RSI14_max,forecast_vol_pct_min,is_front_9d,is_front_12_15,is_middle_candidate_rule,is_back_locked
0,core_middle_rule_0001,core_middle_rule_test,current_middle,unified_fds_no_min_return,9,Front,1,front_9d,fixed_cell12_9d_fv20_rule,True,0.7,0.65,0.65,0.65,68.0,20.0,True,False,False,False
1,core_middle_rule_0001,core_middle_rule_test,current_middle,unified_fds_no_min_return,12,Front,1,front_12_15,fixed_cell12_front_12_15_rule,True,0.7,0.65,0.65,0.65,68.0,14.5,False,True,False,False
2,core_middle_rule_0001,core_middle_rule_test,current_middle,unified_fds_no_min_return,15,Front,1,front_12_15,fixed_cell12_front_12_15_rule,True,0.7,0.65,0.65,0.65,68.0,14.5,False,True,False,False
3,core_middle_rule_0001,core_middle_rule_test,current_middle,unified_fds_no_min_return,18,Middle,2,middle,candidate_middle_rule,True,0.7,0.70,0.70,0.70,68.0,10.0,False,False,True,False
4,core_middle_rule_0001,core_middle_rule_test,current_middle,unified_fds_no_min_return,21,Middle,2,middle,candidate_middle_rule,True,0.7,0.70,0.70,0.70,68.0,10.0,False,False,True,False
5,core_middle_rule_0001,core_middle_rule_test,current_middle,unified_fds_no_min_return,24,Middle,2,middle,candidate_middle_rule,True,0.7,0.70,0.70,0.70,68.0,10.0,False,False,True,False
6,core_middle_rule_0001,core_middle_rule_test,current_middle,unified_fds_no_min_return,27,Back,3,back,locked_back_rule,True,0.7,0.70,0.70,0.70,70.0,8.5,False,False,False,True
7,core_middle_rule_0001,core_middle_rule_test,current_middle,unified_fds_no_min_return,30,Back,3,back,locked_back_rule,True,0.7,0.70,0.70,0.70,70.0,8.5,False,False,False,True
8,core_middle_rule_0001,core_middle_rule_test,current_middle,unified_fds_no_min_return,33,Back,3,back,locked_back_rule,True,0.7,0.70,0.70,0.70,70.0,8.5,False,False,False,True
9,core_middle_rule_0002,core_middle_rule_test,middle_higher_forecast_vol,unified_fds_no_min_return,9,Front,1,front_9d,fixed_cell12_9d_fv20_rule,True,0.7,0.65,0.65,0.65,68.0,20.0,True,False,False,False



Saved outputs:
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\11_middle_weakness_current_middle_by_tenor_20260704_222837.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\11_middle_weakness_current_middle_by_year_20260704_222837.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\11_middle_weakness_current_middle_by_tenor_year_20260704_222837.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\11_middle_weakness_current_middle_state_diagnostics_20260704_222837.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\11_middle_weakness_current_middle_2026_trade_detail_20260704_222837.csv
  C:\Users\patri\vrp_project\data\audit\vrp_unified_fds_core_signal_threshold_research_v1\11_middle_weakness_diagnostic_conclusion_20260704_222837.csv
  C:\Users\patri\vrp_project\data\audit\vrp_un